# GOKO record linkage MVP (B1) on Spark and Delta Lake: extracted entity records against a watchlist

The same record linkage as `goko_record_linkage_mvp_b1.ipynb` -- the same matching core, the
same data preparation, the same estimation, scoring, clusters, roll-ups and self-checks, cell for
cell -- run on a Databricks cluster: the inputs are Delta tables, every heavy step is spread
across all the cluster's cores as Spark tasks, and every output table is a Delta table.

**What Spark changes, and what it does not.** The pipeline hands its heavy work to an executor
pool as a list of tasks (chunks of extracted entities to score, batches of pairs to compare,
bootstrap replicates to fit, addresses to parse). Here that pool is the cluster: each task is a
Spark task on an executor core, running the same function; what the tasks read (party frames,
model, candidate keys) is broadcast once and reused, and each chunk's pairs and evidence are
written by the executor that scored it to a Unity Catalog volume, then loaded into Delta by Spark
in parallel. Nothing is sampled away or cut short: every number is what a single process
computes (a self-test runs the same small input both ways and compares every table).

**To run on your data**: open the "Run" section below, set the two input tables, the output
schema and a volume folder in the Settings cell, then run every cell top to bottom. With
`RUN_NAME = "synthetic"` the notebook builds its own small synthetic set (written to Delta first)
as a sanity check that needs no data of yours.

**Cluster.** Databricks Runtime 15.4 LTS or later (Python 3.11+), with a **Dedicated (single
user) access mode** cluster: executors write to the volume folder, and tasks are spread with
Spark's RDD API (`sparkContext.parallelize`, broadcast variables), which Standard (shared) access
mode and serverless compute do not offer. Memory-optimized
workers suit it best (8 GB per core, e.g. Azure E-series, AWS r-series): each worker machine holds
one copy of the data (a few GB for a 500K-row watchlist) and, per core, the pages of it that
core's process touches plus the chunk it is scoring (`PAIRS_PER_TASK`); if a machine runs short,
it continues with fewer processes and the results do not change. Use a fixed number of workers
(autoscaling off), so every step sees all of them; the driver holds the inputs and the model
(also memory-optimized, 32 GB or more). The repository must
be a Git folder in the workspace (Workspace > Git folder), so `mappings/` and `reference/` sit
next to this notebook.

**Rules this notebook keeps**: every link carries a score and a basis, name-only links are
visible and never upgraded, conflicting identifiers veto; every gate has an unresolved
outcome -- an extracted party with no candidate is shown as such, empty rows and dropped
oversized blocking keys are listed; truth files (when present, e.g. in a self-test) are read
only by diagnostics and self-tests, never by the pipeline.


In [ ]:
# Libraries the notebook needs beyond the Databricks runtime (notebook-scoped: the executors get them too).
%pip install -q jellyfish rapidfuzz usaddress xlsxwriter openpyxl psutil
%pip install -q --no-deps recordlinkage==0.16
import os
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    dbutils.library.restartPython()   # noqa: F821 -- Databricks only: load the new libraries


## Matching core v1.1

**This section is self-contained.** It only does the matching math: comparisons, per-value
statistics, parameter estimation, scoring and clustering. Nothing here reads a file, writes a
spreadsheet or knows about note text -- so it can be dropped unchanged into a different
caller later. The self-test section hashes every code cell between this heading and the
"End of the matching core" marker and compares the hash with a recorded value, as a check
that nobody edited it without meaning to. Change it only deliberately: bump `CORE_VERSION`
and re-record the hash (see the self-test section).

**Contract.** Plain tables in, plain tables out. Nothing here reads a file, writes a
spreadsheet or knows about note text. The caller builds a *party frame* per side, one row per
party, with these columns (empty string when unknown):

| Column | Meaning |
|---|---|
| `party_id`, `part` | stable id; `person` or `business` |
| `first`, `middle`, `last` | cleaned name parts (`clean_person`) |
| `first_roots`, `last_nysiis`, `name_key` | nickname roots (`|`-joined), NYSIIS of the surname, `FIRST LAST` |
| `holder_key` | who holds a value, for single-holder counts: `LAST|F` (surname and first initial) for a person, the first alias for a business |
| `org_aliases`, `name_key` (business) | `|`-joined aliases from `org_aliases`; first alias is the name key |
| `dob`, `dob_int` | ISO date; `YYYYMMDD` as int (0 = none) |
| `addr_full`, `addr_street`, `zip`, `city_state`, `state` | address keys (`address_keys`) |
| `id_ssn` `id_npi` `id_dl` `id_tin` `id_license` `id_email` `id_vin` `id_plate` `id_cnpi` | valid, non-junk identifier values |
| `phone_own`, `phone_row` | a phone the party owns; a phone its row holds |
| `specialty`, `category`, `cat_strength` | canonical specialty; category; `strong` or `weak` |
| `tie` | `party_id` of the other part of the same row, or empty |

Pairs are positional: `il[i]` indexes the left frame, `ir[i]` the right one. The right frame
is the reference population whose value frequencies give per-value `u` (the watchlist in [B]).

**Where Splink would do better (whole core).** Splink compiles comparisons to SQL and runs
them in DuckDB or Spark, so 1M x 300k is routine and multi-core by default; here numpy and
per-unique-value Python loops do the work on one core. Splink's comparison library has
tested levels for names, dates and addresses; the levels below are hand-written. Splink
draws a waterfall chart per pair; the evidence table below is its tabular equivalent.

In [ ]:
# ---- Matching core v1.1: parameters and field definitions ---------------------------------
import math, re, unicodedata, hashlib, json
from dataclasses import dataclass, field, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
import jellyfish
import recordlinkage as rl
from rapidfuzz import fuzz as _rf_fuzz
from rapidfuzz.distance import JaroWinkler as _rf_jw, Levenshtein as _rf_lev

CORE_VERSION = "1.1"


@dataclass
class CoreParams:
    """Every number the core uses. The caller's run configuration embeds one of these and the
    manifest dumps it: there are no thresholds anywhere else in the core."""
    # name similarity: several measures, combined into graded levels (v1.1)
    jw_close: float = 0.92          # Jaro-Winkler at or above: a spelling variant
    lev_close: float = 0.85         # normalized Levenshtein similarity at or above: a spelling variant
    tsr_close: float = 90.0         # rapidfuzz token_sort_ratio at or above: reordered parts
    jw_sound: float = 0.85          # Jaro-Winkler at or above (below close): sounds alike
    lev_sound: float = 0.75         # normalized Levenshtein at or above (below close): sounds alike
    org_rare_idf: float = 10.0      # bits: a word fewer than ~1 in 1,000 organizations use
    org_distinct_min_len: int = 4   # a distinctive org word has at least this many letters
    org_close_sim: float = 0.75     # rarity-weighted share of the name's words that agree (v1.1)
    org_close_tsr: float = 85.0     # rapidfuzz token_set_ratio at or above, with org_close_sim
    surname_floor: int = 50         # count for a surname below the Census cutoff
    firstname_floor: int = 20       # count for a first name below the SSA cutoff
    org_df_floor: int = 1           # organizations using a word absent from NPPES
    own_org_min_count: int = 5      # the reference population's own word share counts from this many users
    flat_freq: float = 1e-3         # used only when an outside table is missing; stamped FLAT
    alpha: float = 5.0              # shrinkage of an m estimate toward the next source (prior chain)
    n_min: int = 50                 # informative pairs below which simulation joins the chain
    em_max_iter: int = 300
    em_tol: float = 1e-8
    em_prior_strength: float = 20.0  # Dirichlet (multi-level Beta) pseudo-pairs on m in EM (v1.1)
    m_floor: float = 0.005          # no level's m below this: bounds every disagreement weight (v1.1)
    bits_min: float = -10.0         # no single field counts below this many bits (v1.1)
    bits_max: float = 40.0          # ... or above this many
    unstable_ci_bits: float = 3.0   # bootstrap 95% interval wider than this (bits): flagged unstable
    unstable_pass_bits: float = 3.0  # training passes disagreeing by more than this (bits): flagged
    u_pseudo: float = 0.5           # pseudo-count for a level never seen among random pairs
    prior_pseudo: float = 0.5       # pseudo-count for a prior group with no strict pair
    junk_holders: int = 25          # a value held under more names than this is junk
    u_floor: float = 1e-12
    dob_min_year: int = 1900
    dob_max_year: int = 2026        # dates after this year are invalid (fixed for determinism)
    coparty_min_p: float = 0.9      # a business link must reach this to anchor a co-party
    ci_z: float = 1.96              # 95% intervals


# Fields per part, and their levels best-first. Every field also has a NULL level (-1): one
# side has nothing to compare. It is Splink's null level: excluded from m and u, worth 0 bits.
EMPTY = -1
FIELD_LEVELS = {
    "name": ["exact", "first_nick", "first_close", "last_close_first_agrees",
             "last_sound_first_agrees", "initial_agrees", "swapped", "first_empty",
             "last_fuzzy_first_weak", "first_differs", "else"],
    "middle": ["exact", "initial", "differs"],
    "org": ["exact", "dba", "short_form", "close", "rare_shared", "common_shared", "sibling", "none"],
    "dob": ["exact", "swap_or_typo", "year_month", "year", "differs"],
    "address": ["exact", "street", "zip", "city_state", "state", "differs"],
    "ssn": ["exact", "near", "differs"],
    "npi": ["exact", "near", "differs"],
    "dl": ["exact", "near", "differs"],
    "tin": ["exact", "near", "differs"],
    "license": ["exact", "differs"],
    "email": ["exact", "differs"],
    "vin": ["exact", "differs"],
    "plate": ["exact", "differs"],
    "cnpi": ["exact", "differs"],
    "phone": ["exact_owned_single", "exact_shared", "differs"],
    "spec_cat": ["specialty", "category_id", "category_weak", "differs"],
    "co_party": ["anchored", "not_anchored"],
}
PART_FIELDS = {
    "person": ["name", "middle", "dob", "address", "ssn", "npi", "dl", "license", "email",
               "vin", "plate", "phone", "spec_cat", "co_party"],
    "business": ["org", "address", "tin", "cnpi", "email", "phone", "spec_cat"],
}
COMPARED_FIELDS = {p: [f for f in fs if f != "co_party"] for p, fs in PART_FIELDS.items()}
# Correlated fields are compared as one graded group each, never as independent fields, so
# nothing is counted twice: first + last name (one `name` level; `middle` is compared only when
# the surnames agree, with its u taken among such pairs), organization aliases, the address
# (number/street/unit/ZIP/city/state), and specialty + category.
FIELD_GROUPS = {"name": ["first", "last", "middle (conditional on the surname)"],
                "org": ["every alias, d/b/a"],
                "address": ["number", "street", "unit", "zip", "city", "state"],
                "spec_cat": ["specialty", "category"]}
# Levels that mean "agrees": their bits never go below 0, so an extra agreeing field can
# never lower p.
AGREEMENT_LEVELS = {
    "name": {"exact", "first_nick", "first_close", "last_close_first_agrees",
             "last_sound_first_agrees", "initial_agrees", "swapped"},
    "middle": {"exact", "initial"}, "org": {"exact", "dba", "short_form", "close", "rare_shared"},
    "dob": {"exact", "swap_or_typo"}, "address": {"exact", "street"},
    "ssn": {"exact", "near"}, "npi": {"exact", "near"}, "dl": {"exact", "near"},
    "tin": {"exact", "near"}, "license": {"exact"}, "email": {"exact"}, "vin": {"exact"},
    "plate": {"exact"}, "cnpi": {"exact"}, "phone": {"exact_owned_single", "exact_shared"},
    "spec_cat": {"specialty", "category_id", "category_weak"}, "co_party": {"anchored"},
}
# Levels that mean "disagrees": their bits never go above 0 (a sparse level can otherwise come
# out with m > u by chance, and a disagreement must never count for a match).
DISAGREEMENT_LEVELS = {
    "name": {"else"}, "middle": {"differs"}, "org": {"sibling", "none"}, "dob": {"differs"},
    "address": {"differs"}, "ssn": {"differs"}, "npi": {"differs"}, "dl": {"differs"},
    "tin": {"differs"}, "license": {"differs"}, "email": {"differs"}, "vin": {"differs"},
    "plate": {"differs"}, "cnpi": {"differs"}, "phone": {"differs"}, "spec_cat": {"differs"},
}
# Levels whose bits are forced to 0: no evidence either way, by declaration.
ZERO_LEVELS = {"co_party": {"not_anchored"}}
IDENTIFIER_FIELDS = ["ssn", "npi", "dl", "tin", "license", "email", "vin", "plate", "cnpi", "phone"]
VETO_FIELDS = ["ssn", "npi", "dl", "tin"]          # one per party: two single-holder values veto
NEAR_FIELDS = {"ssn", "npi", "dl", "tin"}
NAME_FIELDS = {"name", "org"}
BASIS_ORDER = ["identifier", "address", "dob", "co_party", "contextual", "name_only", "none"]


def level_code(field_, level):
    return FIELD_LEVELS[field_].index(level)

In [ ]:
# ---- Matching core v1.1: normalizers ------------------------------------------------------
# Turns raw name / address / identifier text into the clean, comparable values the rest of
# the core works with: uppercase-ASCII folding, title/credential/suffix stripping, name
# splitting ("SMITH, JOHN" -> first/last), organization alias extraction (d/b/a, legal-suffix
# stripping, abbreviation expansion), and per-identifier-type validity checks.

TITLES = {"DR", "DOCTOR", "MR", "MRS", "MS", "MISS", "HON", "JUDGE", "PROF"}
CREDENTIALS = {"MD", "DO", "DC", "DPM", "DDS", "DMD", "PHD", "PT", "DPT", "NP", "PA", "RN",
               "LPN", "LAC", "ESQ", "JD", "CPA", "OT", "OTR", "FACS", "FACP", "MPH", "LCSW",
               "PSYD"}
BARE_CREDENTIALS = CREDENTIALS - {"DO", "PA", "PT", "OT"}
NAME_SUFFIXES = {"JR", "SR", "II", "III", "IV"}
ORG_LEGAL = {"PC", "PLLC", "LLC", "INC", "CORP", "CORPORATION", "CO", "LTD", "LLP", "LP",
             "PA", "COMPANY", "INCORPORATED", "THE", "AND", "OF"}
ORG_ABBREV = {"INS": "INSURANCE", "MED": "MEDICAL", "CTR": "CENTER", "CNTR": "CENTER",
              "SVCS": "SERVICES", "SVC": "SERVICES", "ASSOC": "ASSOCIATES", "ASSN": "ASSOCIATION",
              "HOSP": "HOSPITAL", "MGMT": "MANAGEMENT", "INTL": "INTERNATIONAL", "NATL": "NATIONAL",
              "DIAG": "DIAGNOSTIC", "REHAB": "REHABILITATION", "PHYS": "PHYSICAL", "THER": "THERAPY",
              "CHIRO": "CHIROPRACTIC", "GRP": "GROUP", "SVS": "SERVICES", "LAB": "LABORATORY",
              "LABS": "LABORATORIES", "PHARM": "PHARMACY", "TRANS": "TRANSPORTATION"}
ROLE_WORDS_UP = {"DEFENDANT", "DEFENDANTS", "PLAINTIFF", "PLAINTIFFS", "ANSWERING", "CLAIMANT",
                 "CLAIMANTS", "INSURED", "COUNSEL", "ATTORNEY", "RESPONDENT", "PETITIONER", "THE",
                 "HIS", "HER", "THEIR"}
_DBA_RE = re.compile(r"\b(?:d\s*/\s*b\s*/\s*a|dba|a\s*/\s*k\s*/\s*a|aka|f\s*/\s*k\s*/\s*a|fka|"
                     r"doing business as|trading as|t\s*/\s*a)\b", re.I)


def fold(s):
    """Uppercase ASCII: accents dropped ('JOSÉ' -> 'JOSE'), whitespace collapsed."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().upper()


def _undot(s):
    """'M.D.' -> 'MD', 'P.C.' -> 'PC', then any remaining dot is a space ('A.' -> 'A')."""
    s = re.sub(r"\b((?:[A-Za-z]{1,2}\.){2,})", lambda m: m.group(1).replace(".", ""), s)
    return s.replace(".", " ")


def _words(s):
    return [w.upper() for w in re.findall(r"[A-Za-z][A-Za-z'\-]*", s)]


def parse_person(name):
    """Split a full name into first / middle initial / last, with credentials.
    Handles 'Dr. A. Monroe', 'WILLIAM A. WEINER, D.O.', 'Moy, Marvin' and 'Mr. Pierre'."""
    s = _undot(fold(name))
    head, _, tail = s.partition(",")
    hw, tw = _words(head), _words(tail)
    creds = {w for w in tw if w in CREDENTIALS} | {w for w in hw[1:] if w in BARE_CREDENTIALS}
    title = next((w for w in hw if w in TITLES), None)
    drop = TITLES | CREDENTIALS | NAME_SUFFIXES | ROLE_WORDS_UP
    core_h = [w.strip("'-") for w in hw if w not in drop]
    core_t = [w.strip("'-") for w in tw if w not in drop]
    core = core_t + core_h if (len(core_h) == 1 and core_t) else core_h
    core = [w for w in core if w]
    first = middle = last = ""
    if len(core) == 1:
        last = core[0]
    elif len(core) >= 2:
        first, last = core[0], core[-1]
        middle = core[1][0] if len(core) > 2 else ""
    return {"first": first, "middle": middle, "last": last, "creds": sorted(creds), "title": title}


def _clean_part(s, drop_creds=True):
    words = [w.replace("'", "").strip("-") for w in _words(_undot(fold(s)))]
    drop = TITLES | NAME_SUFFIXES | (BARE_CREDENTIALS if drop_creds else set())
    return " ".join(w for w in words if w and w not in drop)


def clean_person(first, middle, last):
    """Cleaned (first, middle, last) from name parts as a form holds them.

    'Dr. John' -> 'JOHN'; 'SMITH JR' -> 'SMITH'; "O'Brien" -> 'OBRIEN'. A full name typed into
    one box is split with parse_person: 'SMITH, JOHN' in last, or 'JOHN SMITH' in first with
    no last."""
    f, m, l = fold(first), fold(middle), fold(last)
    if not l and f and len(_words(f)) >= 2 and not m:
        p = parse_person(f)
        f, m, l = p["first"], p["middle"], p["last"]
    elif l and not f and "," in l:
        p = parse_person(l)
        f, m, l = p["first"], p["middle"], p["last"]
    return _clean_part(f), _clean_part(m), _clean_part(l)


def org_aliases(name):
    """Name variants of one organization: d/b/a parts split, legal suffixes
    stripped, abbreviations expanded. Returns a list of word lists."""
    out = []
    for p in _DBA_RE.split(fold(name)):
        toks = re.findall(r"[A-Z0-9]+", _undot(p).upper().replace("'", "").replace("&", " AND "))
        joined, i = [], 0
        while i < len(toks):                     # OCR splits a suffix ('COMP ANY'): rejoin it
            if i + 1 < len(toks) and toks[i] + toks[i + 1] in ORG_LEGAL:
                joined.append(toks[i] + toks[i + 1]); i += 2
            else:
                joined.append(toks[i]); i += 1
        toks = [ORG_ABBREV.get(t, t) for t in joined]
        toks = [t for t in toks if len(t) > 1 and t not in ORG_LEGAL and t not in ROLE_WORDS_UP]
        if toks and toks not in out:
            out.append(toks)
    # a short form that is a subset of a longer alias adds nothing
    keep = [a for a in out if not any(set(a) < set(b) for b in out)] or out
    return keep


def org_alias_string(name):
    """'|'-joined aliases, each alias its words joined by a space: the party-frame form."""
    return "|".join(" ".join(a) for a in org_aliases(name))


# ---- identifiers ---------------------------------------------------------------------------
def _digits(s):
    return re.sub(r"\D", "", fold(s))


def _all_same(d):
    return len(set(d)) <= 1


_SEQ = {"123456789", "987654321", "1234567890", "0123456789", "012345678"}


def norm_ssn(s):
    """(value, valid, reason). Invalid: not 9 digits, area 000/666/9xx, group 00, serial 0000,
    one repeated digit, or a sequence."""
    d = _digits(s)
    if not d:
        return "", False, ""
    if len(d) != 9:
        return d, False, "length"
    if d[:3] in ("000", "666") or d[0] == "9" or d[3:5] == "00" or d[5:] == "0000":
        return d, False, "range"
    if _all_same(d) or d in _SEQ:
        return d, False, "placeholder"
    return d, True, ""


_BAD_EIN_PREFIX = {"00", "07", "08", "09", "17", "18", "19", "28", "29", "49", "69", "70", "78",
                   "79", "89", "96", "97"}


def norm_tin(s):
    d = _digits(s)
    if not d:
        return "", False, ""
    if len(d) != 9:
        return d, False, "length"
    if _all_same(d) or d in _SEQ:
        return d, False, "placeholder"
    if d[:2] in _BAD_EIN_PREFIX:
        return d, False, "prefix"
    return d, True, ""


def npi_luhn_ok(d):
    """NPI check digit: Luhn over '80840' + the first nine digits."""
    if len(d) != 10 or not d.isdigit():
        return False
    total = 0
    for i, ch in enumerate(reversed("80840" + d[:9])):
        v = int(ch)
        if i % 2 == 0:
            v *= 2
            if v > 9:
                v -= 9
        total += v
    return (10 - total % 10) % 10 == int(d[9])


def norm_npi(s):
    d = _digits(s)
    if not d or not d.strip("0"):
        return "", False, ""                       # LEIE writes 0000000000 for "none"
    if len(d) != 10:
        return d, False, "length"
    if not npi_luhn_ok(d):
        return d, False, "check_digit"
    return d, True, ""


def norm_phone(s):
    d = _digits(s)
    if len(d) == 11 and d[0] == "1":
        d = d[1:]
    if not d:
        return "", False, ""
    if len(d) != 10:
        return d, False, "length"
    if d[0] in "01" or d[3] in "01":
        return d, False, "range"
    if _all_same(d) or d in _SEQ or (d[3:6] == "555" and d[6:8] == "01"):
        return d, False, "placeholder"
    return d, True, ""


_JUNK_EMAIL_LOCAL = {"none", "na", "n/a", "noemail", "no", "unknown", "test", "noreply",
                     "nomail", "declined", "null", "email"}


def norm_email(s):
    e = str(s or "").strip().lower()
    if not e or e in ("nan",):
        return "", False, ""
    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", e):
        return e, False, "format"
    local, dom = e.split("@", 1)
    if local in _JUNK_EMAIL_LOCAL or dom.split(".")[0] in ("none", "na", "unknown", "noemail"):
        return e, False, "placeholder"
    return e, True, ""


def _alnum(s):
    return re.sub(r"[^A-Z0-9]", "", fold(s))


US_STATES = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "DISTRICT OF COLUMBIA": "DC",
    "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL",
    "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA",
    "MAINE": "ME", "MARYLAND": "MD", "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN",
    "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK", "OREGON": "OR",
    "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC", "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA",
    "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY",
    "PUERTO RICO": "PR", "GUAM": "GU", "VIRGIN ISLANDS": "VI", "AMERICAN SAMOA": "AS",
    "NORTHERN MARIANA ISLANDS": "MP"}
STATE_CODES = set(US_STATES.values()) | {"AA", "AE", "AP"}


def norm_state(s):
    t = re.sub(r"[^A-Z ]", "", fold(s)).strip()
    if t in STATE_CODES:
        return t
    return US_STATES.get(t, "")


def _qualified(number, state, min_len=4):
    """'STATE:NUMBER' (or ':NUMBER' when no state). Invalid when too short or one repeated
    character."""
    n = _alnum(number)
    if not n:
        return "", False, ""
    st = norm_state(state)
    v = f"{st}:{n}"
    if len(n) < min_len:
        return v, False, "length"
    if _all_same(n) or not n.strip("0"):
        return v, False, "placeholder"
    return v, True, ""


def norm_dl(number, state):
    return _qualified(number, state, 4)


def norm_license(number, state):
    return _qualified(number, state, 3)


def norm_plate(number, state):
    return _qualified(number, state, 2)


_VIN_MAP = {**{str(i): i for i in range(10)},
            "A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "H": 8, "J": 1, "K": 2,
            "L": 3, "M": 4, "N": 5, "P": 7, "R": 9, "S": 2, "T": 3, "U": 4, "V": 5, "W": 6,
            "X": 7, "Y": 8, "Z": 9}
_VIN_W = [8, 7, 6, 5, 4, 3, 2, 10, 0, 9, 8, 7, 6, 5, 4, 3, 2]


def vin_check_ok(v):
    if len(v) != 17 or any(c not in _VIN_MAP for c in v):
        return False
    r = sum(_VIN_MAP[c] * w for c, w in zip(v, _VIN_W)) % 11
    return v[8] == ("X" if r == 10 else str(r))


def norm_vin(s):
    v = _alnum(s)
    if not v:
        return "", False, ""
    if len(v) != 17 or set(v) & set("IOQ"):
        return v, False, "format"
    if not vin_check_ok(v):
        return v, False, "check_digit"
    return v, True, ""


_PLACEHOLDER_DOB = {"1900-01-01", "1901-01-01", "1899-12-31", "1800-01-01", "9999-12-31",
                    "1111-11-11", "1910-01-01"}


def norm_dob(s, params=None):
    """(ISO date, valid, reason). Accepts ISO, US m/d/y (2-digit years pivot at the max year),
    YYYYMMDD. Placeholders (1900-01-01 ...), impossible and future dates are invalid."""
    p = params or CoreParams()
    t = str(s or "").strip()
    if not t or t.lower() == "nan":
        return "", False, ""
    t = t.split("T")[0].split(" ")[0]
    y = m = d = None
    if re.fullmatch(r"\d{8}", t):
        y, m, d = int(t[:4]), int(t[4:6]), int(t[6:])
    elif re.fullmatch(r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}", t):
        y, m, d = (int(x) for x in re.split(r"[-/.]", t))
    elif re.fullmatch(r"\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}", t):
        m, d, y = (int(x) for x in re.split(r"[-/.]", t))
        if y < 100:
            y += 1900 if y > p.dob_max_year % 100 else 2000
    else:
        return t, False, "format"
    try:
        iso = pd.Timestamp(year=y, month=m, day=d).strftime("%Y-%m-%d")
    except (ValueError, OverflowError):
        return t, False, "impossible"
    if iso in _PLACEHOLDER_DOB:
        return iso, False, "placeholder"
    if y < p.dob_min_year or y > p.dob_max_year:
        return iso, False, "range"
    return iso, True, ""


# ---- addresses -----------------------------------------------------------------------------
STREET_TYPES = {"STREET": "ST", "STR": "ST", "ST": "ST", "AVENUE": "AVE", "AV": "AVE", "AVE": "AVE",
                "BOULEVARD": "BLVD", "BLVD": "BLVD", "ROAD": "RD", "RD": "RD", "DRIVE": "DR",
                "DR": "DR", "LANE": "LN", "LN": "LN", "COURT": "CT", "CT": "CT", "PLACE": "PL",
                "PL": "PL", "TERRACE": "TER", "TER": "TER", "PARKWAY": "PKWY", "PKWY": "PKWY",
                "HIGHWAY": "HWY", "HWY": "HWY", "CIRCLE": "CIR", "CIR": "CIR", "WAY": "WAY",
                "TRAIL": "TRL", "TRL": "TRL", "SQUARE": "SQ", "SQ": "SQ", "PLAZA": "PLZ",
                "PLZ": "PLZ", "COVE": "CV", "CV": "CV", "EXPRESSWAY": "EXPY", "EXPY": "EXPY",
                "TURNPIKE": "TPKE", "TPKE": "TPKE", "ALLEY": "ALY", "ALY": "ALY", "LOOP": "LOOP",
                "PIKE": "PIKE", "ROW": "ROW", "WALK": "WALK", "PATH": "PATH", "RUN": "RUN",
                "CROSSING": "XING", "XING": "XING", "POINT": "PT", "PT": "PT", "RIDGE": "RDG",
                "RDG": "RDG"}
DIRECTIONS = {"NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W", "NORTHEAST": "NE",
              "NORTHWEST": "NW", "SOUTHEAST": "SE", "SOUTHWEST": "SW", "N": "N", "S": "S",
              "E": "E", "W": "W", "NE": "NE", "NW": "NW", "SE": "SE", "SW": "SW"}
UNIT_WORDS = {"APT", "APARTMENT", "UNIT", "STE", "SUITE", "RM", "ROOM", "FL", "FLOOR", "BLDG",
              "LOT", "SPC", "SPACE", "TRLR", "DEPT", "#"}
_CITY_PREFIX = {"ST": "SAINT", "FT": "FORT", "MT": "MOUNT"}


def _addr_words(s):
    return re.sub(r"[^A-Z0-9# ]", " ", _undot(fold(s)).replace("-", " - ")).split()


def norm_street_parts(number, direction, name, stype, unit):
    """Normalized (number, direction, name, type, unit). A type or direction left inside the
    name ('MAIN STREET', 'W TAYLOR') is moved to its own part."""
    num = re.sub(r"[^A-Z0-9\-]", "", fold(number))
    words = [w for w in _addr_words(name) if w != "-"]
    d = DIRECTIONS.get(fold(direction).replace(".", ""), "")
    t = STREET_TYPES.get(fold(stype).replace(".", ""), fold(stype).replace(".", ""))
    if words and not d and words[0] in DIRECTIONS and len(words) > 1:
        d = DIRECTIONS[words[0]]; words = words[1:]
    if words and not t and words[-1] in STREET_TYPES and len(words) > 1:
        t = STREET_TYPES[words[-1]]; words = words[:-1]
    if words and words[-1] in DIRECTIONS and len(words) > 1:      # post-direction: 'AVE W'
        words = words[:-1]
    u = [w for w in _addr_words(unit) if w not in UNIT_WORDS and w != "-"]
    return num, d, " ".join(words), t, "".join(u)


def parse_street_line(line):
    """'2161 UNIVERSITY AVENUE W, STE 5' -> ('2161', '', 'UNIVERSITY', 'AVE', '5').
    'P O BOX 2161' -> ('2161', '', 'PO BOX', '', '')."""
    s = fold(line)
    if not s:
        return "", "", "", "", ""
    box = re.match(r"^(?:P\s*\.?\s*O\.?\s*BOX|POST OFFICE BOX|BOX)\s*#?\s*([A-Z0-9\-]+)", s)
    if box:
        return box.group(1), "", "PO BOX", "", ""
    main, _, rest = s.partition(",")
    unit = rest.strip()
    m = re.search(r"\s(?:APT|APARTMENT|UNIT|STE|SUITE|RM|ROOM|FL|FLOOR|BLDG|LOT|SPC|#)\s*\.?\s*#?\s*([A-Z0-9\-]+)\s*$", main)
    if m:
        unit = unit or m.group(1)
        main = main[:m.start()]
    elif "#" in main:
        main, _, u2 = main.partition("#")
        unit = unit or u2.strip()
    mnum = re.match(r"^\s*(\d+[A-Z]?(?:-\d+[A-Z]?)?)\s+(.*)$", main.strip())
    num, rest_name = (mnum.group(1), mnum.group(2)) if mnum else ("", main.strip())
    ws = rest_name.split()
    d = ""
    if len(ws) > 2 and ws[0] in ("N", "S") and ws[1] in ("E", "W"):      # 'N W 45TH' -> NW
        ws = [ws[0] + ws[1]] + ws[2:]
    if len(ws) > 1 and ws[0].replace(".", "") in DIRECTIONS:
        d = DIRECTIONS[ws[0].replace(".", "")]; ws = ws[1:]
    t = ""
    if len(ws) > 1 and ws[-1].replace(".", "") in DIRECTIONS:
        ws = ws[:-1]
    if len(ws) > 1 and ws[-1].replace(".", "") in STREET_TYPES:
        t = STREET_TYPES[ws[-1].replace(".", "")]; ws = ws[:-1]
    unit = " ".join(w for w in re.sub(r"[^A-Z0-9 ]", " ", unit).split() if w not in UNIT_WORDS)
    return num, d, " ".join(ws), t, unit.replace(" ", "")


def norm_city(s):
    ws = re.sub(r"[^A-Z ]", " ", _undot(fold(s))).split()
    if ws and ws[0] in _CITY_PREFIX:
        ws[0] = _CITY_PREFIX[ws[0]]
    return " ".join(ws)


def norm_zip(s):
    d = _digits(s)
    if len(d) in (3, 4):
        d = d.zfill(5)                             # a leading zero lost in a spreadsheet
    d = d[:5]
    return d if len(d) == 5 and d != "00000" else ""


def address_keys(number, name, unit, zip5, city, state):
    """The five keys the address levels compare, best first. Each is empty when a part it
    needs is missing."""
    street = f"{number}|{name}" if number and name else ""
    full = f"{street}|{unit}|{zip5}" if street and zip5 else ""
    city_state = f"{city}|{state}" if city and state else ""
    return full, street, zip5, city_state, state


# ---- nicknames and phonetics ---------------------------------------------------------------
class Nicknames:
    """Nickname roots from a (name1, relationship, name2) table (carltonnorthern/nicknames).
    The table lists relations both ways ('bill has_nickname robert'), so it is read as
    undirected, and a name's roots are itself plus every related name longer than it: BILL
    {BILL, ROBERT, WILLIAM} meets WILLIAM {WILLIAM}, and BILLY meets WILL through WILLIAM, but
    ROBERT {ROBERT} does not meet WILLIAM through BILL."""

    def __init__(self, table):
        nb = defaultdict(set)
        if table is not None and len(table):
            for a, b in zip(table["name1"].map(fold), table["name2"].map(fold)):
                nb[a].add(b)
                nb[b].add(a)
        self._roots = {n: {x for x in rel if len(x) > len(n)} for n, rel in nb.items()}
        self.size = len(table) if table is not None else 0

    def roots(self, first):
        cache = self.__dict__.setdefault("_cache", {})
        r = cache.get(first)
        if r is None:
            f = fold(first).split(" ")[0] if first else ""
            r = cache[first] = frozenset(self._roots.get(f, set()) | {f}) if f else frozenset()
        return r

    def roots_string(self, first):
        return "|".join(sorted(self.roots(first)))


def nysiis(s):
    s = re.sub(r"[^A-Z]", "", fold(s))
    return jellyfish.nysiis(s) if s else ""


def map_unique(values, func):
    """Apply func once per distinct value; values is a Series. Returns a Series."""
    s = pd.Series(values)
    uniq = pd.unique(s)
    table = {v: func(v) for v in uniq}
    return s.map(table)

In [ ]:
# ---- Matching core v1.1: outside rarity -----------------------------------------------------
# How surprising it is that two parties share a name, surname or organization word, from
# outside frequency tables (Census surnames, SSA first names, NPPES organization words) rather
# than from the two input files alone. org_idf takes the larger of NPPES's share and the
# reference population's own share of a word, so a word rare among NPPES health-care
# organizations but common in this particular watchlist (e.g. "AUTO", "COLLISION" in a list
# full of body shops) is not treated as rare just because NPPES rarely sees it.

class Rarity:
    """Value-specific chance agreement from outside tables. Tables arrive as dicts
    {value: count} plus the metadata totals; a missing table gives a flat rate that is stamped
    'FLAT' in the source so the run cannot be mistaken for a real one."""

    def __init__(self, surnames=None, first_names=None, org_df=None, meta=None, params=None,
                 own_org_counts=None, own_org_total=0):
        self.p = params or CoreParams()
        meta = meta or {}
        self.surnames, self.firsts, self.org_df = surnames or {}, first_names or {}, org_df or {}
        self.surname_total = meta.get("census", {}).get("people") or max(1, sum(self.surnames.values()))
        self.first_total = meta.get("ssa", {}).get("births") or max(1, sum(self.firsts.values()))
        self.org_n = meta.get("nppes", {}).get("organizations") or 1
        ini = defaultdict(int)
        for n, c in self.firsts.items():
            ini[n[0]] += c
        self.initial = {k: v / self.first_total for k, v in ini.items()}
        self.own_org = own_org_counts or {}
        self.own_org_total = own_org_total
        self.source = {"surnames": "census2010" if self.surnames else "FLAT",
                       "first_names": "ssa1930-2005" if self.firsts else "FLAT",
                       "org_tokens": ("nppes" if (self.org_df and self.org_n > 1) else "FLAT")
                                     + ("+reference_population" if self.own_org_total else "")}
        self._cache_org = {}

    def surname_freq(self, last):
        if not self.surnames:
            return self.p.flat_freq
        parts = [last] + [x for x in re.split(r"[ \-]", last) if x and x != last]
        return min(self.surnames.get(x, self.p.surname_floor) for x in parts) / self.surname_total

    def first_freq(self, first):
        if not self.firsts:
            return self.p.flat_freq
        f = first.split(" ")[0]
        return self.firsts.get(f, self.p.firstname_floor) / self.first_total

    def initial_share(self, letter):
        return self.initial.get(letter[:1], 1 / 26) if self.firsts else 1 / 26

    def org_idf(self, tok):
        """Bits of surprise in two organizations sharing this name word."""
        if tok in self._cache_org:
            return self._cache_org[tok]
        if not (self.org_df and self.org_n > 1):
            idf = 6.0
        else:
            idf = math.log2(self.org_n / self.org_df.get(tok, self.p.org_df_floor))
        if self.own_org_total:
            own = math.log2(self.own_org_total / max(1, self.own_org.get(tok, 0)))
            if self.own_org.get(tok, 0) >= self.p.own_org_min_count:
                idf = min(idf, own)
        self._cache_org[tok] = idf
        return idf


def own_org_word_counts(org_alias_strings):
    """How many parties of the reference population use each org word (for Rarity)."""
    counts = defaultdict(int)
    n = 0
    for s in org_alias_strings:
        if not s:
            continue
        n += 1
        for w in set(" ".join(s.split("|")).split()):
            counts[w] += 1
    return dict(counts), n

In [ ]:
# ---- Matching core v1.1: value statistics for per-value u and vetoes -----------------------

VALUE_FIELDS = {  # field -> party-frame columns holding its values
    "ssn": ["id_ssn"], "npi": ["id_npi"], "dl": ["id_dl"], "tin": ["id_tin"],
    "license": ["id_license"], "email": ["id_email"], "vin": ["id_vin"], "plate": ["id_plate"],
    "cnpi": ["id_cnpi"], "phone": ["phone_own", "phone_row"],
}
KEY_FIELDS = ["dob", "addr_full", "addr_street", "zip", "city_state", "state", "specialty",
              "category"]


@dataclass
class ValueStats:
    """Per-value counts the comparisons read. `ref_counts[key][value]`: parties of the
    reference (right) population holding the value; `ref_n[key]`: those holding any value.
    `names[field][value]`: distinct names holding an identifier value across both sides.
    `single[field]`: values held under exactly one name."""
    ref_counts: dict
    ref_n: dict
    names: dict
    single: dict


def build_value_stats(left, right):
    """Counts from the two party frames. Identifier u uses how many distinct names hold the
    value across both sides (so a shared or placeholder SSN weighs little); dates, addresses,
    specialty and category use the reference side's own frequencies."""
    ref_counts, ref_n, names, single = {}, {}, {}, {}
    for k in KEY_FIELDS:
        v = right[k][right[k] != ""] if k in right else pd.Series([], dtype=object)
        if k == "category":
            v = v[v != "other"]
        ref_counts[k] = v.value_counts()
        ref_n[k] = int(len(v))
    # v1.1: the reference population's own share of a full person name ("FIRST LAST"), so an
    # exact name's u is never below how often the list itself repeats the name
    if "part" in right and "first" in right:
        pm = (right["part"] == "person") & (right["first"] != "") & (right["last"] != "")
        v = (right.loc[pm, "first"] + " " + right.loc[pm, "last"])
        ref_counts["person_name"] = v.value_counts()
        ref_n["person_name"] = int(len(v))
    both = pd.concat([left, right], ignore_index=True)
    for f, cols in VALUE_FIELDS.items():
        parts = [pd.DataFrame({"v": both[c], "n": both["holder_key"]}) for c in cols if c in both]
        long = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame({"v": [], "n": []})
        long = long[long["v"] != ""]
        nn = long.drop_duplicates().groupby("v")["n"].size()
        names[f] = nn
        single[f] = set(nn.index[nn == 1])
        rv = pd.concat([right[c] for c in cols if c in right], ignore_index=True) if cols else pd.Series([], dtype=object)
        rv = rv[rv != ""]
        ref_n[f] = int((pd.concat([right[c] != "" for c in cols], axis=1).any(axis=1)).sum()) if len(right) else 0
        ref_counts[f] = rv.value_counts()
    return ValueStats(ref_counts, ref_n, names, single)


def junk_values(stats, params):
    """Identifier values held under more names than params.junk_holders: {field: set}."""
    return {f: set(nn.index[nn > params.junk_holders]) for f, nn in stats.names.items()}

In [ ]:
# ---- Matching core v1.1: comparisons (recordlinkage.Compare + custom features) -------------
# One level per field, the same code for candidates, random pairs, anchors and simulated
# pairs. Each feature returns its level codes and, where the level is value-specific, the
# value's own chance-agreement factor `uv` (NaN where the field-level u applies).


@dataclass
class CompareContext:
    params: CoreParams
    rarity: Rarity
    nick: Nicknames
    stats: ValueStats
    dba_pairs: list = field(default_factory=list)      # [(set, set)] declared as one org
    components: dict = field(default_factory=dict)    # random-pair rates for name sub-parts


def _unique_pairs(a, b):
    """Joint factorization of two aligned arrays: (inverse, unique a, unique b)."""
    ca, ua = pd.factorize(np.asarray(a, dtype=object))
    cb, ub = pd.factorize(np.asarray(b, dtype=object))
    nb = len(ub) + 1
    key = ca.astype(np.int64) * nb + cb.astype(np.int64)
    uk, inv = np.unique(key, return_inverse=True)
    return inv, np.asarray(ua, dtype=object)[uk // nb], np.asarray(ub, dtype=object)[uk % nb]


def _arr(s):
    return np.asarray(s.to_numpy() if hasattr(s, "to_numpy") else s, dtype=object)


def _lookup(counts, values):
    """counts: Series value->count. Returns float array (0 where absent)."""
    if counts is None or not len(counts):
        return np.zeros(len(values))
    return pd.Series(values).map(counts).fillna(0).to_numpy(dtype=float)


_JW_CACHE = {}


def _jw(a, b):
    """Jaro-Winkler, memoized: name and org words repeat across millions of pairs."""
    k = (a, b)
    v = _JW_CACHE.get(k)
    if v is None:
        if len(_JW_CACHE) > 5_000_000:
            _JW_CACHE.clear()
        v = _JW_CACHE[k] = jellyfish.jaro_winkler_similarity(a, b)
    return v


_NYSIIS_CACHE = {}
_METAPHONE_CACHE = {}


def _nysiis_c(s):
    v = _NYSIIS_CACHE.get(s)
    if v is None:
        v = _NYSIIS_CACHE[s] = nysiis(s)
    return v


def _metaphone_c(s):
    v = _METAPHONE_CACHE.get(s)
    if v is None:
        t = re.sub(r"[^A-Z]", "", s)
        v = _METAPHONE_CACHE[s] = jellyfish.metaphone(t) if t else ""
    return v


def name_similarity(a, b):
    """The measures one name comparison combines (v1.1): Jaro-Winkler, normalized Levenshtein,
    rapidfuzz token-sort and token-set ratios, and two phonetic codes (NYSIIS, Metaphone)."""
    return {"jaro_winkler": _jw(a, b), "levenshtein": _rf_lev.normalized_similarity(a, b),
            "token_sort": _rf_fuzz.token_sort_ratio(a, b), "token_set": _rf_fuzz.token_set_ratio(a, b),
            "nysiis": bool(a and b and _nysiis_c(a) == _nysiis_c(b)),
            "metaphone": bool(a and b and _metaphone_c(a) and _metaphone_c(a) == _metaphone_c(b))}


LAST_CMP = ["exact", "close", "sound", "differs"]


def _last_cmp(a, b, p):
    """0 exact, 1 close (a spelling variant, a reordered or dropped compound part), 2 sounds
    alike or is moderately similar, 3 differs, -1 empty. Several measures, one graded answer."""
    if not a or not b:
        return -1
    if a == b:
        return 0
    ta, tb = set(re.split(r"[ \-]", a)) - {""}, set(re.split(r"[ \-]", b)) - {""}
    if ta <= tb or tb <= ta:
        return 1
    jw = _jw(a, b)
    lev = _rf_lev.normalized_similarity(a, b)
    if jw >= p.jw_close or lev >= p.lev_close or _rf_fuzz.token_sort_ratio(a, b) >= p.tsr_close:
        return 1
    if (_nysiis_c(a) == _nysiis_c(b) or (_metaphone_c(a) and _metaphone_c(a) == _metaphone_c(b))
            or jw >= p.jw_sound or lev >= p.lev_sound):
        return 2
    return 3


FIRST_CMP = ["exact", "nick", "close", "initial", "empty", "differs"]


def _first_cmp(a, b, p, nick):
    """0 exact, 1 nickname, 2 close (spelling, sound, or one name inside a double name),
    3 initial agrees, 4 empty, 5 differs."""
    if not a or not b:
        return 4
    if len(a) == 1 or len(b) == 1:
        return 3 if a[0] == b[0] else 5
    if a == b:
        return 0
    if nick.roots(a) & nick.roots(b):
        return 1
    if (_jw(a, b) >= p.jw_close or _rf_lev.normalized_similarity(a, b) >= p.lev_close
            or (" " in a or " " in b) and _rf_fuzz.token_set_ratio(a, b) >= 100
            or (_metaphone_c(a) and _metaphone_c(a) == _metaphone_c(b))):
        return 2
    return 5


def _surname_cmp_array(last_l, last_r, p):
    ll, lr = _arr(last_l), _arr(last_r)
    inv, ua, ub = _unique_pairs(ll, lr)
    return np.array([_last_cmp(a, b, p) for a, b in zip(ua, ub)], dtype=np.int8)[inv]


def f_name(first_l, last_l, first_r, last_r, ctx):
    """Person name, one graded group (see FIELD_LEVELS['name'])."""
    p, R = ctx.params, ctx.rarity
    fl, ll, fr, lr = _arr(first_l), _arr(last_l), _arr(first_r), _arr(last_r)
    lc = _surname_cmp_array(ll, lr, p)
    inv, ua, ub = _unique_pairs(fl, fr)
    fc = np.array([_first_cmp(a, b, p, ctx.nick) for a, b in zip(ua, ub)], dtype=np.int8)[inv]
    swapped = (fl != "") & (ll != "") & (fl == lr) & (ll == fr) & (ll != lr)
    last_exact, last_close, last_sound = lc == 0, lc == 1, lc == 2
    first_agree = (fc == 0) | (fc == 1) | (fc == 2)
    conds = [lc < 0, last_exact & (fc == 0), last_exact & (fc == 1), last_exact & (fc == 2), swapped,
             last_close & first_agree, last_sound & first_agree, last_exact & (fc == 3),
             last_exact & (fc == 4), (last_close | last_sound) & ((fc == 3) | (fc == 4)),
             last_exact & (fc == 5)]
    codes = [EMPTY, 0, 1, 2, 6, 3, 4, 5, 7, 8, 9]
    level = np.select(conds, codes, default=10).astype(np.int8)
    # value-specific u: outside tables (and the list's own share) for the agreeing values,
    # random-pair rates for the parts that are not value-specific
    C = ctx.components or {}
    fL = map_unique(pd.Series(lr), R.surname_freq).to_numpy(dtype=float)
    fF = map_unique(pd.Series(fr), lambda v: R.first_freq(v) if v else 1.0).to_numpy(dtype=float)
    ini = map_unique(pd.Series(fr), lambda v: R.initial_share(v) if v else 1.0).to_numpy(dtype=float)
    fLsw = map_unique(pd.Series(fl), R.surname_freq).to_numpy(dtype=float)     # swapped: x.first is w.last
    st = ctx.stats
    own = np.zeros(len(level))
    m0 = level == 0
    if m0.any() and st.ref_n.get("person_name", 0):
        own[m0] = _lookup(st.ref_counts.get("person_name"),
                          (pd.Series(fr[m0]) + " " + pd.Series(lr[m0])).to_numpy()) / st.ref_n["person_name"]
    fuzzy_first = np.where(fc == 0, fF, C.get("first_agrees_fuzzy", np.nan))
    uv = np.full(len(level), np.nan)
    uv = np.where(level == 0, np.maximum(fL * fF, own), uv)
    uv = np.where(level == 1, fL * C.get("first_nick", np.nan), uv)
    uv = np.where(level == 2, fL * C.get("first_close", np.nan), uv)
    uv = np.where(level == 3, C.get("last_close", np.nan) * fuzzy_first, uv)
    uv = np.where(level == 4, C.get("last_sound", np.nan) * fuzzy_first, uv)
    uv = np.where(level == 5, fL * ini, uv)
    uv = np.where(level == 6, fLsw * fF, uv)
    uv = np.where(level == 7, fL * C.get("first_empty", np.nan), uv)
    uv = np.where(level == 9, fL * C.get("first_differs", np.nan), uv)
    return level, uv, fc, lc


def f_middle(ml, ll, mr, lr, ctx):
    """Middle name, compared only when the surnames agree (exact, close or alike): a middle
    name agreeing between two different surnames says nothing, and among pairs whose surnames
    agree its u is taken separately, so the name group is not counted twice."""
    a, b = _arr(ml), _arr(mr)
    lc = _surname_cmp_array(ll, lr, ctx.params)
    empty = (a == "") | (b == "") | (lc < 0) | (lc == 3)
    exact = (a == b) & ~empty & (np.char.str_len(a.astype(str)) > 1)
    ia = np.array([x[:1] for x in a], dtype=object)
    ib = np.array([x[:1] for x in b], dtype=object)
    initial = (ia == ib) & ~empty & ~exact
    level = np.select([empty, exact, initial], [EMPTY, 0, 1], default=2).astype(np.int8)
    return level, np.full(len(level), np.nan)


def _digit_arrays(v, width):
    return np.stack([(v // 10 ** k) % 10 for k in range(width)], axis=1)


def f_dob(dl, dr, ctx):
    a = np.asarray(dl, dtype=np.int64)
    b = np.asarray(dr, dtype=np.int64)
    empty = (a == 0) | (b == 0)
    ya, ma, da = a // 10000, (a // 100) % 100, a % 100
    yb, mb, db = b // 10000, (b // 100) % 100, b % 100
    exact = (a == b) & ~empty
    swap = (ya == yb) & (ma == db) & (da == mb) & ~exact
    hamming = (_digit_arrays(a, 8) != _digit_arrays(b, 8)).sum(axis=1)
    typo = (hamming == 1) & ~exact
    level = np.select([empty, exact, swap | typo, (ya == yb) & (ma == mb), ya == yb],
                      [EMPTY, 0, 1, 2, 3], default=4).astype(np.int8)
    st = ctx.stats
    uv = np.where(level == 0, _lookup(st.ref_counts.get("dob"),
                                      pd.Series(b).map(_dob_str).to_numpy()) / max(1, st.ref_n.get("dob", 0)),
                  np.nan)
    return level, uv


def _dob_str(v):
    v = int(v)
    return f"{v // 10000:04d}-{(v // 100) % 100:02d}-{v % 100:02d}" if v else ""


def f_address(full_l, street_l, zip_l, cs_l, st_l, full_r, street_r, zip_r, cs_r, st_r, ctx):
    fl, sl, zl, cl, tl = (_arr(x) for x in (full_l, street_l, zip_l, cs_l, st_l))
    fr, sr, zr, cr, tr = (_arr(x) for x in (full_r, street_r, zip_r, cs_r, st_r))
    eq = lambda x, y: (x == y) & (x != "")
    comparable = ((tl != "") & (tr != "")) | ((zl != "") & (zr != "")) | ((sl != "") & (sr != ""))
    conds = [eq(fl, fr), eq(sl, sr), eq(zl, zr), eq(cl, cr), eq(tl, tr), comparable]
    level = np.select(conds, [0, 1, 2, 3, 4, 5], default=EMPTY).astype(np.int8)
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    for code, key, vals in ((0, "addr_full", fr), (1, "addr_street", sr), (2, "zip", zr),
                            (3, "city_state", cr), (4, "state", tr)):
        m = level == code
        if m.any():
            uv[m] = _lookup(st.ref_counts.get(key), vals[m]) / max(1, st.ref_n.get(key, 0))
    return level, uv


def _near(a, b):
    """One substituted character or two adjacent ones exchanged (same length)."""
    if len(a) != len(b) or a == b:
        return False
    diff = [i for i, (x, y) in enumerate(zip(a, b)) if x != y]
    if len(diff) == 1:
        return True
    return len(diff) == 2 and diff[1] == diff[0] + 1 and a[diff[0]] == b[diff[1]] and a[diff[1]] == b[diff[0]]


def _split_q(v):
    st, _, n = v.partition(":")
    return st, n


def f_identifier(vl, vr, ctx, field_):
    """Exact (value-specific u) / near (typo, for SSN, NPI, DL, TIN) / differs. Qualified
    values (DL, licence, plate: 'STATE:NUMBER') agree when the numbers match and the states
    match or one is unstated. Returns level, uv, veto (two different single-holder values of a
    one-per-party identifier; DL only within one state)."""
    a, b = _arr(vl), _arr(vr)
    n = len(a)
    empty = (a == "") | (b == "")
    qualified = field_ in ("dl", "license", "plate")
    level = np.full(n, EMPTY, dtype=np.int8)
    veto = np.zeros(n, dtype=bool)
    idx = np.flatnonzero(~empty)
    if len(idx):
        inv, ua, ub = _unique_pairs(a[idx], b[idx])
        near_ok = field_ in NEAR_FIELDS
        single = ctx.stats.single.get(field_, set())
        res = []
        for x, y in zip(ua, ub):
            if qualified:
                sx, nx = _split_q(x)
                sy, ny = _split_q(y)
                states_ok = (sx == sy) or not sx or not sy
                if nx == ny and states_ok:
                    res.append((0, False))
                elif near_ok and states_ok and _near(nx, ny):
                    res.append((1, False))
                else:
                    v = (field_ == "dl" and sx and sx == sy and x in single and y in single)
                    res.append((len(FIELD_LEVELS[field_]) - 1, bool(v)))
            else:
                if x == y:
                    res.append((0, False))
                elif near_ok and _near(x, y):
                    res.append((1, False))
                else:
                    v = field_ in VETO_FIELDS and x in single and y in single
                    res.append((len(FIELD_LEVELS[field_]) - 1, bool(v)))
        r = np.array(res, dtype=np.int64).reshape(-1, 2)[inv]
        level[idx] = r[:, 0]
        veto[idx] = r[:, 1].astype(bool)
    st = ctx.stats
    uv = np.full(n, np.nan)
    m = level == 0
    if m.any():
        names = _lookup(st.names.get(field_), b[m])
        refc = _lookup(st.ref_counts.get(field_), b[m])
        uv[m] = np.maximum(np.maximum(names, refc), 1.0) / max(1, st.ref_n.get(field_, 0))
    return level, uv, veto


def f_phone(own_l, row_l, own_r, row_r, ctx):
    """exact_owned_single: the agreeing number is owned on both sides and held under one name;
    exact_shared: agrees otherwise (row-level, or several holders); differs."""
    ol, rwl, orr, rwr = _arr(own_l), _arr(row_l), _arr(own_r), _arr(row_r)
    single = ctx.stats.single.get("phone", set())
    has_l = (ol != "") | (rwl != "")
    has_r = (orr != "") | (rwr != "")
    eq = lambda x, y: (x == y) & (x != "")
    own_own = eq(ol, orr)
    any_eq = own_own | eq(ol, rwr) | eq(rwl, orr) | eq(rwl, rwr)
    own_single = own_own & pd.Series(ol).isin(single).to_numpy()
    level = np.select([~(has_l & has_r), own_single, any_eq], [EMPTY, 0, 1], default=2).astype(np.int8)
    matched = np.where(own_own, ol, np.where(eq(ol, rwr), ol, np.where(eq(rwl, orr), rwl, rwl)))
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    m = (level == 0) | (level == 1)
    if m.any():
        names = _lookup(st.names.get("phone"), matched[m])
        refc = _lookup(st.ref_counts.get("phone"), matched[m])
        uv[m] = np.maximum(np.maximum(names, refc), 1.0) / max(1, st.ref_n.get("phone", 0))
    return level, uv


def f_spec_cat(spec_l, cat_l, str_l, spec_r, cat_r, str_r, ctx):
    """Specialty and category, one correlated group: same specialty / same category, both
    identifier- or mapping-backed / same category, one side given or keyword / different."""
    sl, cl, gl, sr, cr, gr = (_arr(x) for x in (spec_l, cat_l, str_l, spec_r, cat_r, str_r))
    cl = np.where(cl == "other", "", cl)
    cr = np.where(cr == "other", "", cr)
    same_spec = (sl == sr) & (sl != "")
    same_cat = (cl == cr) & (cl != "")
    strong = (gl == "strong") & (gr == "strong")
    comparable = ((cl != "") & (cr != "")) | ((sl != "") & (sr != ""))
    level = np.select([same_spec, same_cat & strong, same_cat, comparable], [0, 1, 2, 3],
                      default=EMPTY).astype(np.int8)
    st = ctx.stats
    uv = np.full(len(level), np.nan)
    m = level == 0
    if m.any():
        uv[m] = _lookup(st.ref_counts.get("specialty"), sr[m]) / max(1, st.ref_n.get("specialty", 0))
    m = (level == 1) | (level == 2)
    if m.any():
        uv[m] = _lookup(st.ref_counts.get("category"), cr[m]) / max(1, st.ref_n.get("category", 0))
    return level, uv


# ---- organization names: levels from word pairing and word rarity -----------------------------
def _rejoin(x, y):
    """Undo OCR splits: 'CASUAL','TY' -> 'CASUALTY' when the other has it."""
    ys, out, i = set(y), [], 0
    while i < len(x):
        if i + 1 < len(x) and x[i] + x[i + 1] in ys and x[i] not in ys:
            out.append(x[i] + x[i + 1]); i += 2
        else:
            out.append(x[i]); i += 1
    return out


def _match_words(x, y, p):
    """Word pairing between two aliases: exact first, then Jaro-Winkler >= jw_close.
    Returns (matched words of x, words only in x, words only in y)."""
    x, y = _rejoin(x, y), _rejoin(y, x)
    left, matched = list(y), []
    for t in x:
        hit = t if t in left else max(left, key=lambda u: _jw(t, u), default=None)
        if hit is not None and (hit == t or _jw(t, hit) >= p.jw_close):
            matched.append(t); left.remove(hit)
    only_x = [t for t in x if t not in matched]
    return matched, only_x, left


def _fuzzy_family(a, b, p):
    """One name's words all appear (spelling-tolerant) in the other."""
    small, big = (a, b) if len(a) <= len(b) else (b, a)
    return bool(small) and all(any(t == u or _jw(t, u) >= p.jw_close for u in big) for t in small)


def declared_dba_pairs(org_alias_strings):
    """Alias pairs that one party declares to be one organization
    ('X d/b/a Y'). A trade name shares no words with the legal name; without this the sibling
    level would split a party from its own d/b/a."""
    pairs, seen = [], set()
    for s in org_alias_strings:
        al = [a for a in s.split("|") if a] if s else []
        for i in range(len(al)):
            for j in range(i + 1, len(al)):
                k = (al[i], al[j])
                if k not in seen:
                    seen.add(k)
                    pairs.append((frozenset(al[i].split()), frozenset(al[j].split())))
    return pairs


def _dba_index(ctx):
    """word -> indexes of declared d/b/a pairs using it (built once per context)."""
    idx = getattr(ctx, "_dba_idx", None)
    if idx is None or idx[0] is not ctx.dba_pairs:
        d = defaultdict(set)
        R = ctx.rarity
        for i, (q1, q2) in enumerate(ctx.dba_pairs):
            for q in (q1, q2):
                # index on the side's distinctive words (common ones like MEDICAL would put
                # every declaration in front of every pair); all words when none is distinctive
                rare = [w for w in q if R.org_idf(w) >= 6.0] or list(q)
                for w in rare:
                    d[w].add(i)
        idx = (ctx.dba_pairs, d)
        ctx._dba_idx = idx
    return idx[1]


def _declared_same(xs, ys, ctx):
    """Some record declares the two names to be one organization.
    Only declarations sharing a word with each side are checked."""
    if not ctx.dba_pairs:
        return False
    d = _dba_index(ctx)
    cx = set().union(*(d.get(w, set()) for a in xs for w in a))
    cy = set().union(*(d.get(w, set()) for b in ys for w in b))
    p = ctx.params
    for i in sorted(cx & cy):
        q1, q2 = ctx.dba_pairs[i]
        for a in xs:
            for b in ys:
                if (_fuzzy_family(set(a), q1, p) and _fuzzy_family(set(b), q2, p)) or                         (_fuzzy_family(set(a), q2, p) and _fuzzy_family(set(b), q1, p)):
                    return True
    return False


def _org_words_u(words, R):
    """Chance two organizations share these words: the rarest counts in full, the others at
    half, because the words of one name travel together."""
    idfs = sorted((R.org_idf(t) for t in words), reverse=True)
    if not idfs:
        return 1.0
    return 2.0 ** -(idfs[0] + 0.5 * sum(idfs[1:]))


def org_level(al, ar, ctx):
    """(level code, uv, detail) for two '|'-joined alias strings."""
    p, R = ctx.params, ctx.rarity
    if not al or not ar:
        return EMPTY, np.nan, ""
    xs = [a.split() for a in al.split("|") if a]
    ys = [a.split() for a in ar.split("|") if a]
    for x in xs:
        for y in ys:
            if x == y:
                return 0, max(p.u_floor, _org_words_u(y, R)), " ".join(y)
    best = None
    for x in xs:
        for y in ys:
            matched, ox, oy = _match_words(x, y, p)
            score = sum(R.org_idf(t) for t in matched)
            if best is None or score > best[0]:
                best = (score, matched, ox, oy, x, y)
    _, matched, ox, oy, x, y = best
    declared = _declared_same(xs, ys, ctx)
    if declared and not matched:
        return 1, max(p.u_floor, _org_words_u(y, R)), "declared d/b/a"
    if matched and (not ox or not oy):
        return 2, max(p.u_floor, _org_words_u(matched, R)), " ".join(matched)
    dist = lambda ws: [t for t in ws if len(t) >= p.org_distinct_min_len and R.org_idf(t) >= p.org_rare_idf]
    if matched and dist(ox) and dist(oy) and not declared:
        return 6, np.nan, f"{'+'.join(dist(ox))} vs {'+'.join(dist(oy))}"
    if declared:
        return 1, max(p.u_floor, _org_words_u(y, R)), "declared d/b/a"
    if matched:
        # v1.1: word rarity plus token-set similarity. 'close' when most of the rarity weight
        # of both names agrees and the token-set ratio agrees too.
        wm = sum(R.org_idf(t) for t in matched)
        sim = wm / max(wm + sum(R.org_idf(t) for t in ox), wm + sum(R.org_idf(t) for t in oy), 1e-9)
        if sim >= p.org_close_sim and _rf_fuzz.token_set_ratio(" ".join(x), " ".join(y)) >= p.org_close_tsr:
            return 3, max(p.u_floor, _org_words_u(matched, R)), " ".join(matched)
        top = max(matched, key=R.org_idf)
        if R.org_idf(top) >= p.org_rare_idf:
            return 4, max(p.u_floor, 2.0 ** -R.org_idf(top)), top
        return 5, np.nan, " ".join(matched)
    return 7, np.nan, ""


def f_org(al, ar, ctx):
    a, b = _arr(al), _arr(ar)
    inv, ua, ub = _unique_pairs(a, b)
    res = [org_level(x, y, ctx) for x, y in zip(ua, ub)]
    level = np.array([r[0] for r in res], dtype=np.int8)[inv]
    uv = np.array([r[1] for r in res], dtype=float)[inv]
    return level, uv


# Which party-frame columns each feature reads.
FEATURE_COLUMNS = {
    "name": (["first", "last"], ["first", "last"]),
    "middle": (["middle", "last"], ["middle", "last"]),
    "dob": (["dob_int"], ["dob_int"]),
    "address": (["addr_full", "addr_street", "zip", "city_state", "state"],
                ["addr_full", "addr_street", "zip", "city_state", "state"]),
    "phone": (["phone_own", "phone_row"], ["phone_own", "phone_row"]),
    "spec_cat": (["specialty", "category", "cat_strength"], ["specialty", "category", "cat_strength"]),
    "org": (["org_aliases"], ["org_aliases"]),
    **{f: ([f"id_{f}"], [f"id_{f}"]) for f in ["ssn", "npi", "dl", "tin", "license", "email",
                                                "vin", "plate", "cnpi"]},
}


def _feature(field_, ctx):
    if field_ == "name":
        return lambda a, b, c, d: f_name(a, b, c, d, ctx), [field_, f"uv_{field_}", "name_first_cmp", "name_last_cmp"]
    if field_ == "middle":
        return lambda a, b, c, d: f_middle(a, b, c, d, ctx), [field_, f"uv_{field_}"]
    if field_ == "dob":
        return lambda a, b: f_dob(a, b, ctx), [field_, f"uv_{field_}"]
    if field_ == "address":
        return lambda *s: f_address(*s, ctx), [field_, f"uv_{field_}"]
    if field_ == "phone":
        return lambda a, b, c, d: f_phone(a, b, c, d, ctx), [field_, f"uv_{field_}"]
    if field_ == "spec_cat":
        return lambda *s: f_spec_cat(*s, ctx), [field_, f"uv_{field_}"]
    if field_ == "org":
        return lambda a, b: f_org(a, b, ctx), [field_, f"uv_{field_}"]
    return (lambda a, b, f=field_: f_identifier(a, b, ctx, f),
            [field_, f"uv_{field_}", f"veto_{field_}"])


def compare_pairs(left, right, il, ir, part, ctx):
    """Level per field for positional pairs (il into left, ir into right), computed through
    recordlinkage.Compare with one custom vectorized feature per field. Returns a DataFrame
    with one row per pair: `<field>` level codes, `uv_<field>` value-specific u factors,
    `veto_<field>` flags and the name sub-comparisons."""
    il = np.asarray(il, dtype=np.int64)
    ir = np.asarray(ir, dtype=np.int64)
    cols = ["l", "r"]
    if len(il) == 0:
        out = pd.DataFrame({"l": il, "r": ir})
        for f in COMPARED_FIELDS[part]:
            _, labels = _feature(f, ctx)
            for lab in labels:
                out[lab] = pd.Series([], dtype=float if lab.startswith("uv_") else
                                     (bool if lab.startswith("veto_") else np.int8))
        return out
    pairs = pd.MultiIndex.from_arrays([il, ir], names=cols)
    cmp = rl.Compare(indexing_type="position")
    for f in COMPARED_FIELDS[part]:
        func, labels = _feature(f, ctx)
        lcols, rcols = FEATURE_COLUMNS[f]
        cmp.compare_vectorized(func, lcols, rcols, label=labels)
    feats = cmp.compute(pairs, left, right)
    feats = feats.reset_index(drop=True)
    feats.insert(0, "r", ir)
    feats.insert(0, "l", il)
    for c in feats.columns:
        if c in FIELD_LEVELS or c in ("name_first_cmp", "name_last_cmp"):
            feats[c] = feats[c].astype(np.int8)
        elif c.startswith("veto_"):
            feats[c] = feats[c].astype(bool)
    return feats


def coparty_proxy_levels(left, right, il, ir):
    """Co-party level for pairs used in estimation, where business links are not scored yet:
    'anchored' when the two tied businesses share a TIN or clinic NPI. Scoring uses the real
    anchors (business links on an identifier at p >= coparty_min_p, no veto)."""
    tl = left["tie_pos"].to_numpy()[il]
    tr = right["tie_pos"].to_numpy()[ir]
    has = (tl >= 0) & (tr >= 0)
    level = np.full(len(il), EMPTY, dtype=np.int8)
    if has.any():
        agree = np.zeros(int(has.sum()), dtype=bool)
        # v1.1 fix: the tied business's identifiers (tie_tin, tie_cnpi on the person frame);
        # v1.0 read the person frame's own (always empty) TIN columns, so this was never anchored
        for c in ("tie_tin", "tie_cnpi"):
            if c not in left or c not in right:
                continue
            a = left[c].to_numpy()[il][has]
            b = right[c].to_numpy()[ir][has]
            agree |= (a == b) & (a != "")
        level[has] = np.where(agree, 0, 1)
    return level

In [ ]:
# ---- Matching core v1.1: parameter estimation ------------------------------------------------
# u: exact-agreement levels in closed form from value frequencies over all pairs (no sampling);
# the fuzzy levels from a very large random sample, rescaled to the probability the closed form
# leaves. m: a prior centre from level counts per source (anchors, watchlist duplicates,
# simulation, published values, shrunk in a chain), then Splink-style EM with u fixed in several
# training passes, each blind to the fields its pairs were selected on, with a Dirichlet prior
# (a Beta per level) toward that centre, bootstrapped intervals, a floor on m and bounded bits.


def level_counts(levels, fields, weights=None, exclude=None):
    """Counts of each level among informative pairs (level != EMPTY). `exclude[field]` is a
    boolean mask of pairs whose value of that field must not count (the anchor's own field).
    Returns {field: (counts array per level, informative total)}."""
    out = {}
    w = np.ones(len(levels)) if weights is None else np.asarray(weights, dtype=float)
    for f in fields:
        if f not in levels:
            continue
        lv = levels[f].to_numpy()
        ok = lv >= 0
        if exclude is not None and f in exclude:
            ok &= ~exclude[f]
        k = len(FIELD_LEVELS[f])
        counts = np.bincount(lv[ok].astype(np.int64), weights=w[ok], minlength=k)[:k]
        out[f] = (counts, float(w[ok].sum()))
    return out


def estimate_u(levels_random, fields, params):
    """Field-level u per level from random pairs, smoothed by a pseudo-count."""
    rows = []
    cnt = level_counts(levels_random, fields)
    for f in fields:
        counts, n = cnt.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))
        k = len(counts)
        u = (counts + params.u_pseudo) / (n + params.u_pseudo * k)
        for i, lev in enumerate(FIELD_LEVELS[f]):
            rows.append({"field": f, "level": lev, "level_code": i, "u": float(u[i]),
                         "u_count": float(counts[i]), "u_pairs": float(n),
                         "u_source": f"random pairs ({int(n):,} informative)", "u_method": "sample"})
    return pd.DataFrame(rows)


# ---- u in closed form ------------------------------------------------------------------------
def _keys(frame, cols):
    """One key per party joining `cols`; '' when any part is empty (a joint key agrees only when
    every part agrees and none is empty)."""
    if isinstance(cols, str):
        cols = [cols]
    arrs = [frame[c].to_numpy(dtype=object) if c in frame else np.full(len(frame), "", dtype=object)
            for c in cols]
    out = arrs[0].astype(object)
    ok = out != ""
    for a in arrs[1:]:
        ok &= a != ""
        out = out + "\x1f" + a
    return np.where(ok, out, "")


def agreement_count(kx, kw=None):
    """Pairs whose keys agree, from value frequencies alone: sum over values of cx * cw between
    two frames, or cx (cx - 1) / 2 among distinct parties of one frame (kw None). kx, kw: key
    arrays ('' = none) or lists of key arrays for multi-valued fields (a party is counted once
    per value it holds)."""
    def vc(k):
        if isinstance(k, list):
            parts = [pd.DataFrame({"i": np.arange(len(a)), "v": np.asarray(a, dtype=object)}) for a in k]
            d = pd.concat(parts, ignore_index=True)
            d = d[d["v"] != ""].drop_duplicates()
            return d["v"].value_counts()
        s = pd.Series(np.asarray(k, dtype=object))
        return s[s != ""].value_counts()
    cx = vc(kx).astype(float)
    if kw is None:
        return float((cx * (cx - 1) / 2).sum())
    cw = vc(kw).astype(float)
    common = cx.index.intersection(cw.index)
    return float((cx[common] * cw[common]).sum())


def _pair_count(mask_x, mask_w=None):
    nx = float(np.asarray(mask_x, bool).sum())
    if mask_w is None:
        return nx * (nx - 1) / 2
    return nx * float(np.asarray(mask_w, bool).sum())


def _pattern_pairs(px, pw=None):
    """Pairs whose patterns (small ints, bit flags of which parts are filled) share a filled
    part: the informative pairs of a group field."""
    cx = pd.Series(px).value_counts()
    if pw is None:
        tot = 0.0
        items = list(cx.items())
        for i, (a, na) in enumerate(items):
            for b, nb in items[i:]:
                if a & b:
                    tot += na * (na - 1) / 2 if a == b else na * nb
        return tot
    cw = pd.Series(pw).value_counts()
    return float(sum(na * nb for a, na in cx.items() for b, nb in cw.items() if a & b))


def _hierarchy(keys_x, keys_w, same):
    """Probability mass (as pair counts) of each level of a field whose levels are 'key i
    agrees and no earlier key agrees' (address, specialty + category), by inclusion-exclusion
    over joint keys. keys_x / keys_w: list of key arrays best-first."""
    from itertools import combinations
    out = []
    for i in range(len(keys_x)):
        tot = 0.0
        earlier = list(range(i))
        for r in range(len(earlier) + 1):
            for S in combinations(earlier, r):
                idx = (i,) + S

                def joint(keys):
                    k = keys[idx[0]]
                    ok = k != ""
                    out_ = k.astype(object)
                    for j in idx[1:]:
                        ok &= keys[j] != ""
                        out_ = out_ + "\x1e" + keys[j]
                    return np.where(ok, out_, "")
                c = agreement_count(joint(keys_x), None if same else joint(keys_w))
                tot += (-1) ** r * c
        out.append(max(0.0, tot))
    return out


def closed_form_u(left, right, part, single_phone=None, same_frame=False):
    """Exact-agreement u per field in closed form: agreeing pairs over all informative pairs,
    counted from value frequencies (no sampling). `same_frame`: left and right are one frame
    (deduplication); pairs are distinct parties. Returns {field: {level: (u, informative
    pairs)}} for the levels it can count exactly; the fuzzy levels come from the sample."""
    L, R_ = left, (left if same_frame else right)
    same = same_frame
    kw = (lambda cols: None) if same else (lambda cols: _keys(R_, cols))
    out = {}

    def simple(field_, col):
        kx_, kw_ = _keys(L, col), (None if same else _keys(R_, col))
        n = _pair_count(kx_ != "", None if same else kw_ != "")
        if n > 0:
            out[field_] = {"exact": (agreement_count(kx_, kw_) / n, n)}
    for f in ("ssn", "npi", "tin", "email", "vin", "cnpi"):
        if f in PART_FIELDS[part]:
            simple(f, f"id_{f}")
    for f in ("dl", "license", "plate"):
        if f in PART_FIELDS[part] and f"id_{f}" in L:
            num = lambda F: np.array([v.partition(":")[2] if v else "" for v in F[f"id_{f}"].to_numpy(dtype=object)],
                                     dtype=object)
            kx_, kw_ = num(L), (None if same else num(R_))
            n = _pair_count(kx_ != "", None if same else kw_ != "")
            if n > 0:
                out[f] = {"exact": (agreement_count(kx_, kw_) / n, n)}
    if part == "person":
        simple("dob", "dob")
        lx = _keys(L, "last")
        n = _pair_count(lx != "", None if same else _keys(R_, "last") != "")
        if n > 0:
            out["name"] = {"exact": (agreement_count(_keys(L, ["first", "last"]),
                                                     None if same else _keys(R_, ["first", "last"])) / n, n)}
    else:
        ax = [np.array([(s.split("|") + [""] * 4)[i] if s else "" for s in L["org_aliases"]], dtype=object)
              for i in range(4)]
        aw = None if same else [np.array([(s.split("|") + [""] * 4)[i] if s else "" for s in R_["org_aliases"]],
                                         dtype=object) for i in range(4)]
        n = _pair_count(L["org_aliases"] != "", None if same else R_["org_aliases"] != "")
        if n > 0:
            out["org"] = {"exact": (min(1.0, agreement_count(ax, aw) / n), n)}
    # address: exact / street / zip / city_state / state, one hierarchy; informative when a
    # state, a ZIP or a street is on both sides
    cols = ["addr_full", "addr_street", "zip", "city_state", "state"]
    pat = lambda F: ((F["state"] != "").astype(int) + 2 * (F["zip"] != "").astype(int)
                     + 4 * (F["addr_street"] != "").astype(int)).to_numpy()
    n = _pattern_pairs(pat(L), None if same else pat(R_))
    if n > 0:
        h = _hierarchy([_keys(L, c) for c in cols], None if same else [_keys(R_, c) for c in cols], same)
        out["address"] = {lev: (c / n, n) for lev, c in zip(FIELD_LEVELS["address"][:5], h)}
    # specialty + category: specialty / category, both strong / category, one weak
    def sc_keys(F):
        cat = np.where(F["category"].to_numpy(dtype=object) == "other", "", F["category"].to_numpy(dtype=object))
        strong = np.where(F["cat_strength"].to_numpy(dtype=object) == "strong", cat, "")
        return [F["specialty"].to_numpy(dtype=object), strong.astype(object), cat.astype(object)]
    kx_ = sc_keys(L)
    kw_ = None if same else sc_keys(R_)
    patf = lambda k: (k[2] != "").astype(int) + 2 * (k[0] != "").astype(int)
    n = _pattern_pairs(patf(kx_), None if same else patf(kw_))
    if n > 0:
        h = _hierarchy(kx_, kw_, same)
        out["spec_cat"] = {"specialty": (h[0] / n, n), "category_id": (h[1] / n, n), "category_weak": (h[2] / n, n)}
    # phones: any number agreeing (owned or row-level); the owned, single-holder share apart
    has = lambda F: (F["phone_own"] != "") | (F["phone_row"] != "")
    n = _pair_count(has(L), None if same else has(R_))
    if n > 0:
        anyx = [L["phone_own"].to_numpy(dtype=object), L["phone_row"].to_numpy(dtype=object)]
        anyw = None if same else [R_["phone_own"].to_numpy(dtype=object), R_["phone_row"].to_numpy(dtype=object)]
        single = single_phone or set()
        own1 = lambda F: np.array([v if v in single else "" for v in F["phone_own"].to_numpy(dtype=object)], dtype=object)
        a_own = agreement_count(own1(L), None if same else own1(R_))
        a_any = agreement_count(anyx, anyw)
        out["phone"] = {"exact_owned_single": (a_own / n, n), "exact_shared": (max(0.0, a_any - a_own) / n, n)}
    return out


def combine_u(u_sample, closed, field_, params=None):
    """u per level of one field: closed-form levels as counted (with the same pseudo-count as
    the sample, so a level never seen among all pairs is rare, not impossible); the others
    from the sample, rescaled so every level sums to 1. `u_sample`: estimate_u rows."""
    params = params or CoreParams()
    s = u_sample[u_sample["field"] == field_].sort_values("level_code").copy()
    cf = closed.get(field_, {})
    if not cf:
        return s
    fixed = np.array([lev in cf for lev in s["level"]])
    k = len(s)
    cf_u = np.array([(cf[lev][0] * cf[lev][1] + params.u_pseudo) / (cf[lev][1] + params.u_pseudo * k)
                     if lev in cf else 0.0 for lev in s["level"]])
    rest = s["u"].to_numpy()
    room = max(0.0, 1.0 - cf_u.sum())
    tot_rest = rest[~fixed].sum()
    u = np.where(fixed, np.maximum(cf_u, 1e-15), rest * (room / tot_rest if tot_rest > 0 else 0.0))
    n = next(iter(cf.values()))[1]
    s["u"] = np.maximum(u, 1e-15)
    s["u_method"] = np.where(fixed, "closed form", "sample, rescaled")
    s["u_source"] = np.where(fixed, f"closed form over all {n:,.0f} informative pairs (value frequencies)",
                             s["u_source"] + " rescaled to the mass the closed form leaves")
    s["u_pairs"] = np.where(fixed, n, s["u_pairs"])
    return s


def name_components(levels_random):
    """Random-pair rates of the name sub-parts (first-name part given informative names, and
    'last name close' / 'sounds alike'), for the value-specific u of the partial name levels."""
    ok = levels_random["name"].to_numpy() >= 0 if "name" in levels_random else np.zeros(0, bool)
    n = max(1, int(ok.sum()))
    fc = levels_random["name_first_cmp"].to_numpy()[ok] if ok.any() else np.array([], dtype=int)
    lc = levels_random["name_last_cmp"].to_numpy()[ok] if ok.any() else np.array([], dtype=int)
    rate = lambda mask: (float(mask.sum()) + 0.5) / (n + 1.0)
    return {"first_nick": rate(fc == 1), "first_close": rate(fc == 2),
            "first_agrees_fuzzy": rate((fc == 1) | (fc == 2)), "first_initial": rate(fc == 3),
            "first_empty": rate(fc == 4), "first_differs": rate(fc == 5),
            "last_close": rate(lc == 1), "last_sound": rate(lc == 2), "pairs": n}


# ---- EM --------------------------------------------------------------------------------------
def em_fixed_u(levels, free_fields, fixed_fields, m_fixed, u_table, params, exclude=None):
    """Expectation-maximization over the pairs in `levels`, with u fixed for every field and m
    fixed for `fixed_fields`; learns m for `free_fields` and the match share lambda.
    `m_fixed[f]` and `u_table[f]` are arrays per level. Returns ({field: (m array, n_eff,
    expected counts)}, lambda, iterations). Used for the loose-anchor name estimate that
    centres the prior; the training passes use em_patterns."""
    n = len(levels)
    if n == 0:
        return {}, float("nan"), 0
    fields = [f for f in list(free_fields) + list(fixed_fields) if f in levels]
    lv = {f: levels[f].to_numpy().astype(np.int64) for f in fields}
    ok = {f: (lv[f] >= 0) & (~exclude[f] if exclude is not None and f in exclude else True)
          for f in fields}
    m = {f: (np.asarray(m_fixed[f], dtype=float) if f in fixed_fields
             else np.linspace(0.9, 0.1, len(FIELD_LEVELS[f])) / np.linspace(0.9, 0.1, len(FIELD_LEVELS[f])).sum())
         for f in fields}
    u = {f: np.asarray(u_table[f], dtype=float) for f in fields}
    lam, it = 0.5, 0
    for it in range(1, params.em_max_iter + 1):
        log_m = np.zeros(n)
        log_u = np.zeros(n)
        for f in fields:
            idx = np.where(ok[f], lv[f], 0)
            log_m += np.where(ok[f], np.log(np.clip(m[f][idx], 1e-12, 1)), 0.0)
            log_u += np.where(ok[f], np.log(np.clip(u[f][idx], 1e-12, 1)), 0.0)
        a = np.log(lam) + log_m
        b = np.log(1 - lam) + log_u
        w = 1.0 / (1.0 + np.exp(np.clip(b - a, -700, 700)))
        new_lam = float(np.clip(w.mean(), 1e-6, 1 - 1e-6))
        delta = abs(new_lam - lam)
        for f in free_fields:
            if f not in lv:
                continue
            k = len(FIELD_LEVELS[f])
            c = np.bincount(lv[f][ok[f]], weights=w[ok[f]], minlength=k)[:k]
            tot = c.sum()
            if tot > 0:
                newm = (c + 1e-3) / (tot + 1e-3 * k)
                delta = max(delta, float(np.abs(newm - m[f]).max()))
                m[f] = newm
        lam = new_lam
        if delta < params.em_tol:
            break
    out = {}
    for f in free_fields:
        if f in lv:
            out[f] = (m[f], float(w[ok[f]].sum()), np.bincount(lv[f][ok[f]], weights=w[ok[f]],
                                                             minlength=len(FIELD_LEVELS[f]))[:len(FIELD_LEVELS[f])])
    return out, lam, it


def pattern_table(levels, fields):
    """Distinct level patterns over `fields` (rows; -1 = null level) and how many pairs show
    each. EM and its bootstrap run on this table, so their cost does not grow with the pairs."""
    fields = [f for f in fields if f in levels]
    if not len(levels) or not fields:
        return np.zeros((0, len(fields)), np.int64), np.zeros(0), fields
    arr = np.stack([levels[f].to_numpy().astype(np.int64) for f in fields], axis=1)
    uniq, counts = np.unique(arr, axis=0, return_counts=True)
    return uniq, counts.astype(float), fields


def em_patterns(pat, counts, fields, u_tab, m_prior, alpha, params, m_init=None, lam_init=0.05,
                max_iter=None, tol=None):
    """EM with u fixed, over level patterns weighted by their counts. The null level (-1)
    carries no information for either class. m gets a Dirichlet prior centred on `m_prior`
    with `alpha` pseudo-pairs (a Beta prior per level): m_k = (E[n_k] + alpha m0_k) /
    (E[n] + alpha), so a field seen in few matches shrinks toward the centre.
    Returns {"m": {f: array}, "lambda", "iterations", "converged", "n_match": {f: expected
    informative matches}}."""
    max_iter = max_iter or params.em_max_iter
    tol = tol or params.em_tol
    if not len(pat) or counts.sum() <= 0:
        return {"m": {f: np.asarray(m_prior[f], float) for f in fields}, "lambda": float("nan"),
                "iterations": 0, "converged": False, "n_match": {f: 0.0 for f in fields}}
    m = {f: np.asarray((m_init or m_prior)[f], dtype=float).copy() for f in fields}
    m0 = {f: np.asarray(m_prior[f], dtype=float) for f in fields}
    u = {f: np.clip(np.asarray(u_tab[f], dtype=float), 1e-15, 1.0) for f in fields}
    cols = {f: pat[:, j] for j, f in enumerate(fields)}
    okc = {f: cols[f] >= 0 for f in fields}
    idx = {f: np.where(okc[f], cols[f], 0) for f in fields}
    log_u = np.zeros(len(pat))
    for f in fields:
        log_u += np.where(okc[f], np.log(u[f][idx[f]]), 0.0)
    lam, it, converged = float(lam_init), 0, False
    tot = counts.sum()
    nm = {}
    for it in range(1, max_iter + 1):
        log_m = np.zeros(len(pat))
        for f in fields:
            log_m += np.where(okc[f], np.log(np.clip(m[f][idx[f]], 1e-15, 1.0)), 0.0)
        a = math.log(lam) + log_m
        b = math.log(1 - lam) + log_u
        w = 1.0 / (1.0 + np.exp(np.clip(b - a, -700, 700)))
        wc = w * counts
        new_lam = float(np.clip(wc.sum() / tot, 1e-9, 1 - 1e-9))
        delta = abs(new_lam - lam) / max(lam, 1e-9)
        for f in fields:
            k = len(FIELD_LEVELS[f])
            c = np.bincount(cols[f][okc[f]], weights=wc[okc[f]], minlength=k)[:k]
            newm = (c + alpha * m0[f]) / (c.sum() + alpha)
            delta = max(delta, float(np.abs(newm - m[f]).max()))
            m[f] = newm
            nm[f] = float(c.sum())
        lam = new_lam
        if delta < tol:
            converged = True
            break
    return {"m": m, "lambda": lam, "iterations": it, "converged": converged, "n_match": nm}


def training_pass_em(levels, fields, u_tab, m_prior, params, name, rng=None, reps=0):
    """One Splink-style training pass: EM with u fixed on pairs chosen by one rule, over the
    fields the rule did not use. With reps > 0, also a bootstrap: pattern counts resampled
    (multinomial) and EM rerun from the point estimate. Returns the point estimate and the
    replicate m's."""
    pat, counts, fields = pattern_table(levels, fields)
    alpha = params.em_prior_strength
    est = em_patterns(pat, counts, fields, u_tab, m_prior, alpha, params)
    est.update({"pass": name, "pairs": int(counts.sum()), "patterns": len(pat), "fields": list(fields)})
    boots = []
    if reps and len(pat) and counts.sum() > 0:
        n = int(counts.sum())
        prob = counts / counts.sum()
        for _ in range(reps):
            cb = rng.multinomial(n, prob).astype(float)
            b = em_patterns(pat, cb, fields, u_tab, m_prior, alpha, params, m_init=est["m"],
                            lam_init=est["lambda"], max_iter=200, tol=1e-6)
            boots.append({"m": b["m"], "lambda": b["lambda"]})
    est["boot"] = boots
    return est


def combine_passes(passes, field_, m_prior, u, params):
    """One field's m from the training passes that saw it: the mean of the passes' estimates
    (Splink's rule), each already carrying the Dirichlet prior; the prior centre when no pass
    saw the field. 95% interval from the bootstrap (replicate b averages replicate b of every
    pass). Returns (m, lo, hi, per-pass m dict, bits lo, bits hi, flags)."""
    k = len(FIELD_LEVELS[field_])
    used = [p for p in passes if field_ in p["m"] and p["pairs"] > 0 and p.get("n_match", {}).get(field_, 0) > 0]
    m0 = np.asarray(m_prior, dtype=float)
    if not used:
        nan = np.full(k, np.nan)
        return m0, nan, nan, {}, nan, nan, ["no training pass saw this field: prior centre only"], np.zeros(k)
    m = np.mean([p["m"][field_] for p in used], axis=0)
    per = {p["pass"]: p["m"][field_] for p in used}
    reps = min((len(p["boot"]) for p in used), default=0)
    flags = []
    if reps:
        R_ = np.array([floor_m(np.mean([p["boot"][b]["m"][field_] for p in used], axis=0), params)
                       for b in range(reps)])
        lo, hi = np.percentile(R_, 2.5, axis=0), np.percentile(R_, 97.5, axis=0)
        bits_r = np.clip(np.log2(np.clip(R_, 1e-15, 1) / np.clip(u, 1e-15, 1)), params.bits_min, params.bits_max)
        blo, bhi = np.percentile(bits_r, 2.5, axis=0), np.percentile(bits_r, 97.5, axis=0)
    else:
        lo = hi = blo = bhi = np.full(k, np.nan)
    if len(used) > 1:
        bits_p = np.array([np.clip(np.log2(np.clip(floor_m(p["m"][field_], params), 1e-15, 1) / np.clip(u, 1e-15, 1)),
                                   params.bits_min, params.bits_max) for p in used])
        spread = bits_p.max(axis=0) - bits_p.min(axis=0)
    else:
        spread = np.zeros(k)
    for p in used:
        if not p["converged"]:
            flags.append(f"pass {p['pass']} did not converge")
    return m, lo, hi, per, blo, bhi, flags, spread


def combine_m(field_, sources, published, params):
    """One field's prior-centre m per level from sources in preference order.

    `sources`: list of (name, counts array, informative n) best first. Each estimate is shrunk
    toward the next one (alpha pseudo-pairs), bottom-up from the published value. Simulation
    (a source named 'simulation') joins the chain only when the sources before it hold fewer
    than n_min informative pairs. Returns rows with m, the chain, pairs and a 95% interval."""
    k = len(FIELD_LEVELS[field_])
    pub = np.asarray(published, dtype=float)
    pub = pub / pub.sum() if pub.sum() > 0 else np.full(k, 1.0 / k)
    before_sim = 0.0
    used = []
    for name, counts, n in sources:
        if name == "simulation" and before_sim >= params.n_min:
            continue
        if n > 0:
            used.append((name, np.asarray(counts, dtype=float), float(n)))
        if name != "simulation":
            before_sim += n
    m = pub.copy()
    for name, counts, n in reversed(used):
        m = (counts + params.alpha * m) / (n + params.alpha)
    n_total = sum(n for _, _, n in used)
    n_eff = n_total + params.alpha
    se = np.sqrt(np.clip(m * (1 - m), 0, None) / n_eff)
    chain = " > ".join(f"{name}({n:,.0f})" for name, _, n in used) or "published only"
    primary = used[0][0] if used else "published"
    rows = []
    for i, lev in enumerate(FIELD_LEVELS[field_]):
        rows.append({"field": field_, "level": lev, "level_code": i, "m": float(m[i]),
                     "m_lo": float(max(0.0, m[i] - params.ci_z * se[i])),
                     "m_hi": float(min(1.0, m[i] + params.ci_z * se[i])),
                     "m_source": primary, "m_chain": chain, "m_pairs": float(n_total),
                     "m_published": float(pub[i])})
    return rows


def floor_m(m, params):
    """No level below params.m_floor (then renormalized): bounds every disagreement weight."""
    m = np.maximum(np.asarray(m, dtype=float), params.m_floor)
    return m / m.sum()


def weights_table(m_rows, u_df, components, params=None):
    """Join m and u per field and level; field-level bits; flag agreement levels where m < u
    (those count 0 bits), disagreement levels where m > u, and unstable estimates. Value-specific
    levels say what their u rests on."""
    params = params or CoreParams()
    w = pd.DataFrame(m_rows).merge(u_df, on=["field", "level", "level_code"], how="left")
    w["u"] = w["u"].fillna(1.0)
    w["agreement"] = [lev in AGREEMENT_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    w["zero_by_rule"] = [lev in ZERO_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    w["bits_field_level"] = np.clip(np.log2(np.clip(w["m"], 1e-12, 1) / np.clip(w["u"], 1e-12, 1)),
                                    params.bits_min, params.bits_max)
    w["disagreement"] = [lev in DISAGREEMENT_LEVELS.get(f, set()) for f, lev in zip(w["field"], w["level"])]
    flag = np.where(w["agreement"] & (w["m"] < w["u"]), "m<u: agreement counts 0 bits",
                    np.where(w["disagreement"] & (w["m"] > w["u"]), "m>u: disagreement counts 0 bits", ""))
    if "unstable" in w:
        flag = np.where(w["unstable"].fillna("") != "",
                        np.where(flag != "", flag + "; ", "") + "UNSTABLE: " + w["unstable"].fillna(""), flag)
    w["flag"] = flag
    w["u_value_specific"] = [value_specific(f, lev) for f, lev in zip(w["field"], w["level"])]
    return w


VALUE_U_SOURCE = {
    ("name", "exact"): "census surname x SSA first name, or the list's own share of the full name if larger (per value)",
    ("name", "first_nick"): "census surname (per value) x random-pair rate of a nickname first name",
    ("name", "first_close"): "census surname (per value) x random-pair rate of a close first name",
    ("name", "last_close_first_agrees"): "random-pair rate of a close surname x SSA first name (per value)",
    ("name", "last_sound_first_agrees"): "random-pair rate of a like-sounding surname x SSA first name (per value)",
    ("name", "initial_agrees"): "census surname x SSA share of the initial (per value)",
    ("name", "swapped"): "census x SSA for the swapped values",
    ("name", "first_empty"): "census surname (per value) x random-pair rate of an empty first name",
    ("name", "first_differs"): "census surname (per value) x random-pair rate of a differing first name",
    ("org", "exact"): "NPPES / reference-population word shares (per value)",
    ("org", "dba"): "NPPES / reference-population word shares (per value)",
    ("org", "short_form"): "NPPES / reference-population word shares of the shared words",
    ("org", "close"): "NPPES / reference-population word shares of the shared words",
    ("org", "rare_shared"): "NPPES / reference-population share of the rarest shared word",
    ("dob", "exact"): "reference population: holders of the date",
    ("address", "exact"): "reference population: holders of the address",
    ("address", "street"): "reference population: holders of number + street",
    ("address", "zip"): "reference population: holders of the ZIP",
    ("address", "city_state"): "reference population: holders of city + state",
    ("address", "state"): "reference population: holders of the state",
    ("spec_cat", "specialty"): "reference population: holders of the specialty",
    ("spec_cat", "category_id"): "reference population: holders of the category",
    ("spec_cat", "category_weak"): "reference population: holders of the category",
}
for _f in IDENTIFIER_FIELDS:
    VALUE_U_SOURCE[(_f, "exact" if _f != "phone" else "exact_owned_single")] = "distinct names holding the value / reference holders of the field"
VALUE_U_SOURCE[("phone", "exact_shared")] = "distinct names holding the value / reference holders of the field"


def value_specific(f, lev):
    return VALUE_U_SOURCE.get((f, lev), "")


def strict_recall(fill, m_lookup):
    """How often a true match fires at least one strict rule, from field fill rates and the
    estimated m (rules treated as independent). `fill[key]`: share of pairs where both sides
    hold the field; `m_lookup(field, level)`: estimated m."""
    q = []
    ids = [f for f in ("ssn", "npi", "dl", "tin", "license", "email", "vin", "plate", "cnpi")]
    for f in ids:
        if fill.get(f, 0) > 0:
            q.append(fill[f] * m_lookup(f, "exact"))
    if fill.get("name", 0) > 0:
        q.append(fill["name"] * fill.get("dob", 0) * m_lookup("name", "exact") * m_lookup("dob", "exact"))
        q.append(fill["name"] * fill.get("street", 0) * m_lookup("name", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "street")))
    if fill.get("org", 0) > 0:
        q.append(fill["org"] * fill.get("zip", 0) * m_lookup("org", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "zip")))
        q.append(fill["org"] * fill.get("street", 0) * m_lookup("org", "exact")
                 * (m_lookup("address", "exact") + m_lookup("address", "street")))
    return float(1 - np.prod([1 - min(1.0, x) for x in q])) if q else 0.0


def prior_estimate(n_strict, n_pairs, recall, params):
    """P(a random pair of the group matches) = strict pairs / recall / all pairs. A zero count
    uses a pseudo-count and is flagged."""
    flagged = n_strict == 0 or recall <= 0
    num = (n_strict if n_strict > 0 else params.prior_pseudo) / max(recall, 1e-3)
    prior = float(min(0.5, num / max(1.0, float(n_pairs))))
    return prior, flagged

In [ ]:
# ---- Matching core v1.1: scoring, vetoes, basis, co-party, evidence -------------------------


def _lookup_table(w, f, col):
    sub = w[w["field"] == f].sort_values("level_code")
    return sub[col].to_numpy(dtype=float)


def used_u(levels, f, weights):
    """The u each pair's level of field f is scored with: value-specific where the level has
    one, field-level otherwise; for a pooled entity that offered k distinct values of the field
    (column k_<f>), an agreement level's u is multiplied by k (the chance that any of k values
    agrees by chance is at most k u)."""
    n = len(levels)
    lv = levels[f].to_numpy().astype(np.int64)
    ut = _lookup_table(weights, f, "u")
    vspec = np.array([bool(value_specific(f, lev)) for lev in FIELD_LEVELS[f]])
    agree = np.array([lev in AGREEMENT_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
    idx = np.where(lv >= 0, lv, 0)
    uv = levels[f"uv_{f}"].to_numpy(dtype=float) if f"uv_{f}" in levels else np.full(n, np.nan)
    use_v = vspec[idx] & ~np.isnan(uv)
    u = np.where(use_v, uv, ut[idx])
    if f"k_{f}" in levels:
        k = np.maximum(levels[f"k_{f}"].to_numpy(dtype=float), 1.0)
        u = np.where(agree[idx], u * k, u)
    return np.clip(u, 1e-15, 1.0), use_v


def score_pairs(levels, part, weights, prior_logit, params=None):
    """Fellegi-Sunter in bits. For each field: log2(m / u), where u is the value-specific u on
    levels that have one and the field-level u otherwise (times k for a pooled entity); a null
    (empty) field is 0 bits; an agreement level never goes below 0 and a disagreement level
    never above 0; no field outside [bits_min, bits_max]; levels declared 'no evidence' are 0.
    Vetoes set p = 0 and stay visible. Returns a DataFrame: bits_<field>, bits, logit, p, veto,
    basis."""
    params = params or CoreParams()
    n = len(levels)
    out = pd.DataFrame(index=levels.index)
    total = np.zeros(n)
    for f in PART_FIELDS[part]:
        if f not in levels:
            out[f"bits_{f}"] = 0.0
            continue
        lv = levels[f].to_numpy().astype(np.int64)
        mt = _lookup_table(weights, f, "m")
        agree = np.array([lev in AGREEMENT_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        zero = np.array([lev in ZERO_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        idx = np.where(lv >= 0, lv, 0)
        u, _ = used_u(levels, f, weights)
        bits = np.clip(np.log2(np.clip(mt[idx], 1e-15, 1.0) / u), params.bits_min, params.bits_max)
        disagree = np.array([lev in DISAGREEMENT_LEVELS.get(f, set()) for lev in FIELD_LEVELS[f]])
        bits = np.where(agree[idx], np.maximum(bits, 0.0), bits)
        bits = np.where(disagree[idx], np.minimum(bits, 0.0), bits)
        bits = np.where(zero[idx] | (lv < 0), 0.0, bits)
        out[f"bits_{f}"] = bits
        total += bits
    veto = np.full(n, "", dtype=object)
    for f in VETO_FIELDS:
        c = f"veto_{f}"
        if c in levels:
            v = levels[c].to_numpy(dtype=bool)
            veto = np.where(v, np.where(veto == "", f, veto + "+" + f), veto)
    out["bits"] = total
    out["logit"] = np.asarray(prior_logit, dtype=float) + total
    p = 1.0 / (1.0 + np.exp2(-np.clip(out["logit"].to_numpy(), -1000, 1000)))
    out["p"] = np.where(veto != "", 0.0, p)
    out["veto"] = veto
    out["basis"] = basis_of(levels, out, part)
    return out


def basis_of(levels, bits, part):
    """What the probability rests on, never upgraded by the score:
    identifier > address > dob > co_party > contextual (name + location/specialty/category)
    > name_only > none."""
    n = len(bits)
    pos = lambda f: (bits[f"bits_{f}"].to_numpy() > 0) if f"bits_{f}" in bits else np.zeros(n, bool)
    lv = lambda f: levels[f].to_numpy() if f in levels else np.full(n, EMPTY)
    ident = np.zeros(n, bool)
    for f in IDENTIFIER_FIELDS:
        if f in PART_FIELDS[part] and f in levels:
            exact = lv(f) == 0        # phone: only an owned, single-holder number
            ident |= exact & pos(f)
    addr = pos("address") & np.isin(lv("address"), [0, 1])
    location = pos("address") & np.isin(lv("address"), [2, 3, 4])
    dob = pos("dob") & np.isin(lv("dob"), [0, 1]) if part == "person" else np.zeros(n, bool)
    cop = pos("co_party")
    name = pos("name") | pos("org")
    context = name & (location | pos("spec_cat"))
    return np.select([ident, addr, dob & name, cop & name, context, name],
                     ["identifier", "address", "dob", "co_party", "contextual", "name_only"],
                     default="none").astype(object)


def apply_coparty(levels, scored, weights, prior_logit, anchored, params=None):
    """One-way co-party evidence for person pairs. `anchored`: boolean array, True where the
    two persons' tied businesses are linked on an identifier (p >= coparty_min_p, no veto),
    decided on business scores alone. Bits are added only where the names already agree, so
    a co-party can strengthen a name but never make one, and name-only links never vouch for
    each other. Returns re-scored pairs."""
    lev = levels.copy()
    has_tie = lev["co_party"].to_numpy() >= 0 if "co_party" in lev else np.zeros(len(lev), bool)
    name_agrees = scored["bits_name"].to_numpy() > 0
    code = np.where(has_tie, 1, EMPTY)
    code = np.where(has_tie & anchored & name_agrees, 0, code)
    lev["co_party"] = code.astype(np.int8)
    lev["uv_co_party"] = np.nan
    return lev, score_pairs(lev, "person", weights, prior_logit, params)


EVIDENCE_COLUMNS = ["pair_id", "field", "value_extracted", "value_watchlist", "level", "m", "m_source",
                    "m_pairs", "u", "u_source", "u_pairs", "values_offered", "bits", "veto"]


def evidence_rows(pair_ids, levels, scored, part, weights, values_l, values_r):
    """Long evidence: one row per pair and non-empty field (plus vetoing fields), with both
    values, the level, m and u with their sources and pair counts, how many distinct values
    a pooled entity offered for the field, and the bits. The rows of a pair sum to its total
    bits (the self-tests check it)."""
    rows = []
    w = weights.set_index(["field", "level_code"])
    for f in PART_FIELDS[part]:
        if f not in levels:
            continue
        lv = levels[f].to_numpy()
        keep = lv >= 0
        if not keep.any():
            continue
        idx = np.flatnonzero(keep)
        codes = lv[idx]
        sub = w.loc[[(f, int(c)) for c in codes]]
        u_all, use_v = used_u(levels, f, weights)
        k = levels[f"k_{f}"].to_numpy()[idx] if f"k_{f}" in levels else np.ones(len(idx))
        rows.append(pd.DataFrame({
            "pair_id": np.asarray(pair_ids)[idx], "field": f,
            "value_extracted": values_l.get(f, np.full(len(lv), ""))[idx] if f in values_l else "",
            "value_watchlist": values_r.get(f, np.full(len(lv), ""))[idx] if f in values_r else "",
            "level": [FIELD_LEVELS[f][c] for c in codes],
            "m": sub["m"].to_numpy(), "m_source": sub["m_source"].to_numpy(),
            "m_pairs": sub["m_pairs"].to_numpy(),
            "u": u_all[idx],
            "u_source": np.where(use_v[idx], sub["u_value_specific"].to_numpy(), sub["u_source"].to_numpy()),
            "u_pairs": sub["u_pairs"].to_numpy(),
            "values_offered": k,
            "bits": scored[f"bits_{f}"].to_numpy()[idx],
            "veto": levels[f"veto_{f}"].to_numpy()[idx] if f"veto_{f}" in levels else False,
        }))
    if not rows:
        return pd.DataFrame(columns=EVIDENCE_COLUMNS)
    ev = pd.concat(rows, ignore_index=True)
    return ev.sort_values(["pair_id", "field"], kind="stable").reset_index(drop=True)

In [ ]:
# ---- Matching core v1.1: pooled entities and consistent clusters -----------------------------
# A pooled entity (several records about one party) is compared through its members: every
# member against the other record, then per field the best level any member reached. Clusters
# are built by greedy union-find with cannot-link constraints, so they are transitive and never
# hold two different values of a one-per-party identifier.


def aggregate_member_levels(lv, group, fields, k=None):
    """Member-level comparisons -> one row per group (an entity x a record): per field the best
    level any member reached (the null level only when no member had a value), the u factor of
    the member that reached it (the smallest on ties), the row of `lv` that supplied it
    (src_<field>), and a veto only when some member vetoes and none agrees exactly. `group`:
    dense group index 0..G-1 per row of lv. `k`: optional {field: array per group} of the
    distinct values the entity offered, written as k_<field> for the scorer."""
    group = np.asarray(group, dtype=np.int64)
    G = int(group.max()) + 1 if len(group) else 0
    rows = np.arange(len(lv))
    out = {}
    for f in fields:
        if f not in lv:
            continue
        code = lv[f].to_numpy().astype(np.int64)
        key = np.where(code >= 0, code, 1000)
        uv = lv[f"uv_{f}"].to_numpy(dtype=float) if f"uv_{f}" in lv else np.full(len(lv), np.nan)
        uvk = np.where(np.isnan(uv), np.inf, uv)
        order = np.lexsort((rows, uvk, key, group))
        gs = group[order]
        first = np.r_[True, gs[1:] != gs[:-1]] if len(gs) else np.zeros(0, bool)
        pick = order[first]
        gid = group[pick]
        best = np.full(G, EMPTY, dtype=np.int8)
        best[gid] = code[pick]
        out[f] = best
        ub = np.full(G, np.nan)
        ub[gid] = uv[pick]
        out[f"uv_{f}"] = ub
        src = np.full(G, -1, dtype=np.int64)
        src[gid] = pick
        out[f"src_{f}"] = src
        if f"veto_{f}" in lv:
            v = lv[f"veto_{f}"].to_numpy(dtype=bool)
            anyv = np.bincount(group, weights=v.astype(float), minlength=G) > 0
            anyx = np.bincount(group, weights=(code == 0).astype(float), minlength=G) > 0
            out[f"veto_{f}"] = anyv & ~anyx
        if k is not None and f in k:
            out[f"k_{f}"] = np.asarray(k[f], dtype=float)
    return pd.DataFrame(out)


def _values_conflict(field_, a, b):
    """Two value sets of a one-per-party identifier conflict when together they hold two
    different values; driver licences only within one state."""
    u = set(a) | set(b)
    if field_ == "dl":
        by = defaultdict(set)
        for v in u:
            st, _, num = v.partition(":")
            if st:
                by[st].add(num)
        return any(len(s) > 1 for s in by.values())
    return len(u) > 1


def constrained_clusters(n, a, b, p, node_values, fields=VETO_FIELDS):
    """Greedy agglomeration with cannot-link constraints. Edges (a[i], b[i]) are taken best
    first (p descending, then a, b); two clusters merge only when no field in `fields` would
    then hold conflicting values (`node_values[field][node]`: a set of single-holder values).
    Transitive by construction; a refused edge keeps the field that refused it.
    Returns (cluster id per node: its smallest node index, accepted mask, refusal reason)."""
    parent = np.arange(n)
    vals = {}

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    def values(r):
        if r not in vals:
            vals[r] = {f: set(node_values[f][r]) if f in node_values else set() for f in fields}
        return vals[r]
    a = np.asarray(a, dtype=np.int64)
    b = np.asarray(b, dtype=np.int64)
    p = np.asarray(p, dtype=float)
    order = np.lexsort((b, a, -p))
    accepted = np.zeros(len(a), dtype=bool)
    reason = np.full(len(a), "", dtype=object)
    size = np.ones(n, dtype=np.int64)
    for i in order:
        ra, rb = find(a[i]), find(b[i])
        if ra == rb:
            accepted[i] = True
            continue
        va, vb = values(ra), values(rb)
        bad = [f for f in fields if va[f] and vb[f] and _values_conflict(f, va[f], vb[f])]
        if bad:
            reason[i] = "conflicting " + "+".join(bad)
            continue
        if size[ra] < size[rb]:
            ra, rb, va, vb = rb, ra, vb, va
        parent[rb] = ra
        size[ra] += size[rb]
        for f in fields:
            va[f] |= vb[f]
        vals.pop(rb, None)
        accepted[i] = True
    roots = np.array([find(i) for i in range(n)], dtype=np.int64)
    first = pd.Series(np.arange(n)).groupby(roots).transform("min").to_numpy()
    return first, accepted, reason

*End of the matching core.*

## 0 · Setup

One `RunConfig` holds every path, seed, sample size, chunk size, cap, floor and K; the
matching core's own numbers sit inside it as `core` (`CoreParams`). The manifest dumps both.
No threshold appears anywhere else.

For a real run, two settings matter: `EXTRACTED_CSV_PATH` and `WATCHLIST_CSV_PATH` (edited
directly in the "Run" section's Settings cell). A `synthetic` dataset also exists, built in
this notebook from a random-but-realistic universe with a known truth file -- it exists only
so the self-tests and `run_notebook_check.py` have something to run against without real data.

**Where Splink would do better.** Splink's settings dictionary is a declarative, versioned
model: blocking rules, comparisons and trained parameters serialize to one JSON and reload.
Here the configuration is a dataclass and the trained parameters live in the manifest; a
saved model cannot yet be reloaded to score new data without re-estimating.


In [ ]:
import os, sys, time, json, hashlib, platform, gc
from dataclasses import dataclass, field, asdict
from pathlib import Path


def _find_root():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "mappings" / "columns.csv").exists() and (p / "reference").exists():
            return p
    return here


@dataclass
class RunConfig:
    dataset: str = "synthetic"
    root: str = ""
    seed: int = 20260925
    # ---- switches (the Settings cell sets them; see README) ---------------------------------
    exhaustive: bool = False             # compare every same-type pair (streamed in chunks) instead of blocking
    address_standardizer: str = "usaddress"   # "usaddress" (offline) | "given" (already standardized) | "smarty"
    baseline: bool = False               # also run the user's current method (token-sort name + category gate)
    self_check: bool = True              # held-out identifier check and noise-injection test
    # ---- parallel execution (changes where the work runs, never what it computes) --------------
    workers: int = 0                     # worker processes; 0 = every core the free memory allows
    pairs_per_task: int = 250_000        # comparisons one worker holds at a time (chunk size)
    out_root: str = ""                   # where out/ goes ('': under the repository; Spark: a volume)
    # ---- u: exact levels in closed form; fuzzy levels from a very large random sample -------
    random_pairs: int = 5_000_000        # random extracted x watchlist pairs (per part, split by share)
    dedup_random_pairs: int = 5_000_000  # random extracted x extracted pairs for the deduplication model
    # ---- m -----------------------------------------------------------------------------------
    sim_records: int = 50_000            # watchlist records copied with noise (prior centre, last resort)
    anchor_pairs_per_value: int = 50     # cap per anchoring value (strict anchors)
    loose_holders: tuple = (2, 20)       # holders of a value for the loose anchor set
    em_pairs_per_pass: int = 2_000_000   # pairs one training pass may use (sampled beyond this)
    em_key_cap: int = 2_000              # a training key proposing more pairs is sampled down to this
    bootstrap_reps: int = 200            # bootstrap replicates for every m and the prior
    # ---- candidates ----------------------------------------------------------------------------
    chunk_size: int = 20_000             # extracted parties per indexing chunk
    exhaustive_pairs_per_chunk: int = 4_000_000   # pairs per chunk when exhaustive
    block_pair_cap: int = 5_000_000      # a blocking key proposing more pairs is refined by state
    sn_window: int = 7                   # sorted-neighbourhood window
    # ---- extracted-side deduplication and clusters ---------------------------------------------
    dedup_p_min: float = 0.95            # pool two extracted records at or above this p ...
    dedup_basis: tuple = ("identifier", "address", "dob", "co_party")   # ... resting on one of these
    link_cluster_p: float = 0.5          # an entity-watchlist link joins a cluster at or above this p
    # ---- keeping and output --------------------------------------------------------------------
    keep_p_floor: float = 0.0            # keep every scored candidate: the user's
                                          # priority is quality, not table size or
                                          # time/memory, so nothing scored is ever
                                          # silently dropped from the output ...
    top_k: int = 3                       # ... or among its entity's top K (ties kept) ...
    #                                      ... or when it rests on an identifier or is vetoed
    flag_p: float = 0.5                  # the workbook's flagged entities: best p at or above this
    workbook_flagged_max: int = 200_000  # flagged entities in the workbook (all in the files)
    workbook_sample_per_stratum: int = 25   # candidate pairs sampled per basis x p band
    excel_row_limit: int = 1_048_575     # data rows per sheet before a continuation sheet
    sensitivity_prior_mult: tuple = (0.1, 1.0, 10.0)
    sensitivity_recall: tuple = (0.5, 1.0)   # multiplied into the estimated recall, capped at 1
    p_bands: tuple = (0.5, 0.8, 0.9, 0.95, 0.99)
    # ---- held-out identifier check and noise-injection test ------------------------------------
    holdout_ids: tuple = ("ssn", "npi", "tin", "dl")
    holdout_mode: str = "each"           # "each": one run per identifier hidden | "all": one run, all hidden
    holdout_max_holders: int = 2         # truth only from values held under at most this many holders
    holdout_min_pairs: int = 5           # identifiers with fewer true pairs are not run
    noise_sources: int = 2_000           # watchlist records copied with noise
    noise_max_copies: int = 3            # 1..this many noisy copies of each (tests the deduplication too)
    noise_decoys: int = 2_000            # fictional parties not on the list
    # ---- baseline: the user's current method (defaults ASSUMED until the user supplies theirs) -
    baseline_threshold: float = 90.0     # rapidfuzz token_sort_ratio at or above: a match
    baseline_strip_legal: bool = True    # drop legal suffixes (INC, LLC, PC ...)
    baseline_strip_titles: bool = True   # drop titles and credentials (DR, MD, JR ...)
    baseline_include_middle: bool = False
    baseline_gate_missing: str = "block"  # a pair with a missing category: "block" | "pass"
    # ---- address standardization (SmartyStreets: never called without credentials) -------------
    smarty_url: str = "https://us-street.api.smarty.com/street-address"
    smarty_batch: int = 100
    # ---- the built-in synthetic test set (self-tests and run_notebook_check.py only) -----------
    synth_persons: int = 2_000
    synth_businesses: int = 500
    core: CoreParams = field(default_factory=CoreParams)

    @property
    def paths(self):
        r = Path(self.root)
        o = Path(self.out_root) if self.out_root else r / "out"
        return {"root": r, "mappings": r / "mappings", "reference": r / "reference",
                "data": r / "data", "out": o / self.dataset, "review": r / "review",
                "notebook": r / "goko_record_linkage_mvp_b1.ipynb", "tables": o / self.dataset / "tables",
                "smarty_cache": r / "data" / "smarty_cache.json"}


def make_config(dataset=None, **overrides):
    """The run configuration: a `RunConfig` with its defaults, one preset ("synthetic", used
    only by the self-tests and run_notebook_check.py) applied if asked for, then any explicit
    keyword overrides applied on top (e.g. make_config("production", baseline=True))."""
    ds = dataset or "production"
    cfg = RunConfig(dataset=ds, root=str(_find_root()))
    if ds == "synthetic":
        cfg.random_pairs = cfg.dedup_random_pairs = 1_000_000
        cfg.noise_sources, cfg.noise_decoys = 800, 800
    for k, v in overrides.items():
        if not hasattr(cfg, k):
            raise AttributeError(f"RunConfig has no setting {k!r}")
        setattr(cfg, k, v)
    if cfg.address_standardizer not in ("usaddress", "given", "smarty"):
        raise ValueError(f"address_standardizer must be usaddress, given or smarty, not {cfg.address_standardizer!r}")
    for key in ("data", "out", "review"):
        cfg.paths[key].mkdir(parents=True, exist_ok=True)
    return cfg


def config_records(cfg):
    """The run configuration as manifest rows."""
    d = asdict(cfg)
    core = d.pop("core")
    rows = [{"section": "config", "key": k, "value": json.dumps(v) if not isinstance(v, str) else v,
             "source": "RunConfig"} for k, v in d.items()]
    rows += [{"section": "config.core", "key": k, "value": json.dumps(v), "source": "CoreParams"}
             for k, v in core.items()]
    return rows


class Stopwatch:
    """Wall time and peak resident memory per step (peak sampled after each step)."""

    def __init__(self):
        self.t0 = time.time()
        self.rows = []
        self._last = self.t0
        try:
            import psutil
            self._proc = psutil.Process()
        except ImportError:
            self._proc = None
        self.peak = 0

    def rss(self):
        if self._proc is None:
            return 0
        try:
            info = self._proc.memory_info()
            peak = getattr(info, "peak_wset", None) or info.rss   # Windows: peak working set
            return max(info.rss, peak)
        except Exception:
            return 0

    def mark(self, step):
        now = time.time()
        r = self.rss()
        self.peak = max(self.peak, r)
        self.rows.append({"step": step, "seconds": round(now - self._last, 2),
                          "elapsed": round(now - self.t0, 2), "peak_rss_gb": round(self.peak / 1e9, 3)})
        self._last = now
        print(f"[{now - self.t0:7.1f}s  peak {self.peak / 1e9:5.2f} GB] {step}")

## 0a · Parallel execution and the chunk store

**Workers.** `Parallel` runs a function over a list of tasks on up to `N_WORKERS` processes.
On Linux (a Databricks driver, for example) the workers are *forked*: each starts with the
driver's memory already in place (party frames, model, candidate keys), so nothing large is
copied or pickled, and a worker only sends back its task's result. The driver hands out one task
at a time, so one slow chunk never holds up the others, and results come back in task order. On
Windows and macOS, which cannot fork, the same tasks run one after another in this process:
slower, same results.

**Memory.** Every task gives the same result wherever and however often it runs, so the pool
can change how many workers run without changing any output:

- *How many.* `N_WORKERS = 0` is decided when each step starts, from the memory free then: one
  worker needs about 0.45 x the driver's own memory (the pages of its data a worker touches get
  copied) plus about 0.5 GB per 100,000 comparisons of `PAIRS_PER_TASK`.
- *Recycled.* A worker is replaced by a fresh fork after `WORKER_RECYCLE_TASKS` (16) tasks, which
  hands back the pages it had copied (they grow with every task it runs).
- *Floor.* When free memory falls below `memory_floor()` (1 GB or 8% of the machine, whichever
  is more), a worker that finishes a task gets no new one: one worker fewer from then on.
- *Stopped.* If the system still runs out of memory it stops a worker (workers mark themselves
  as its first choice, so the notebook's own process survives); that worker's task goes back in
  the queue and runs again. Only a task that cannot finish even on a single worker ends the run,
  with a message to lower `PAIRS_PER_TASK`.

Each large step prints how many workers it used and the largest worker's memory, which is the
number to size a machine by.

**The chunk store.** A run's three large tables (candidates, evidence, dedup pairs) are folders
of Parquet parts, one part per chunk, all with the same columns, named so that reading them in
name order gives the rows in the order a single process produces. `TableHandle` reads a table
whole (`read()`), part by part (`iter()`), or one part (`read_part()`).

**Where Splink would do better.** Splink runs on DuckDB or Spark, which parallelize and spill
to disk inside the engine, across a whole cluster with Spark; here the parallelism is processes
on one machine (the Spark notebook spreads the same work across a cluster). Splink's outputs
are database tables; these are Parquet folders.


In [ ]:
# ---- Parallel execution and the chunk store ------------------------------------------------------
# Nothing here changes what is computed, only where and when: work is split into tasks that
# workers run with the same functions, results come back in task order, and large tables go to
# disk chunk by chunk instead of accumulating in memory. A run on 64 cores and a run on one core
# produce identical tables.
import multiprocessing as _mp
import shutil
import warnings
from types import SimpleNamespace

_SHARED = {}          # read-only state for the current parallel phase (forked workers inherit it)
_IN_WORKER = False    # True inside a worker: a nested parallel call runs in that worker, in order
WORKER_RECYCLE_TASKS = 16    # a worker is replaced by a fresh fork after this many tasks (see _fork_map)


def _worker_init():
    global _IN_WORKER
    _IN_WORKER = True
    warnings.filterwarnings("ignore")
    try:                          # if the machine runs out of memory, the kernel stops a worker, not the driver
        with open("/proc/self/oom_score_adj", "w") as fh:
            fh.write("1000")
    except OSError:
        pass


def _fork_available():
    return "fork" in _mp.get_all_start_methods()


def _private_bytes():
    """This process's private memory (Linux; 0 elsewhere): what it would free by exiting."""
    try:
        with open("/proc/self/smaps_rollup") as fh:
            return sum(int(l.split()[1]) for l in fh if l.startswith("Private")) * 1024
    except OSError:
        return 0


def _available_bytes():
    import psutil
    return psutil.virtual_memory().available


def memory_floor():
    """Workers stop taking new tasks when the machine's free memory falls below this."""
    import psutil
    return max(1.0e9, 0.08 * psutil.virtual_memory().total)


def _trim_caches():
    """The Jaro-Winkler memo is a pure speed-up; a worker keeps it bounded so its memory stays flat."""
    if len(_JW_CACHE) > 1_000_000:
        _JW_CACHE.clear()


def worker_memory_bytes(pairs_per_task, holder_bytes=None):
    """About how much private memory one worker reaches: the pages of the process that forked
    it (party frames, lookups, model) that it touches and so copies -- measured at about 0.3 x
    that process's own private memory within the WORKER_RECYCLE_TASKS a worker lives, 0.45 x
    allowed -- plus one task's working set (levels, scores and evidence rows: about 0.5 GB per
    100,000 comparisons)."""
    held = _private_bytes() if holder_bytes is None else holder_bytes
    return 0.45 * held + 5_000 * pairs_per_task


def auto_worker_count(cap, pairs_per_task):
    """Every core (up to `cap`), unless memory is the limit: as many workers as fit in the memory
    free now, above the floor workers keep clear of. Measured when each step starts, so it
    follows the size of the data and what the machine has left."""
    per = worker_memory_bytes(pairs_per_task)
    room = _available_bytes() - memory_floor()
    return int(max(1, min(cap, os.cpu_count() or 1, room // max(per, 1))))


class Parallel:
    """Runs a function over a list of tasks on several cores. On Linux (a Databricks driver, for
    example) the workers are forked: they start with the driver's memory (party frames, model,
    candidate keys) already in place, so nothing large is copied or pickled. Where fork is not
    available (Windows, macOS) the tasks run one after another in this process. Either way the
    results come back in task order, identical to a one-process run.

    `workers`: 0 (auto) -- every core the free memory allows, decided when each step starts; a
    number -- at most that many. `pairs_per_task` sizes the memory a worker needs (auto)."""

    def __init__(self, workers=0, pairs_per_task=250_000):
        self.auto = not workers
        self.workers = max(1, int(workers or os.cpu_count() or 1))
        self.pairs_per_task = pairs_per_task

    @property
    def parallel(self):
        return self.workers > 1 and _fork_available() and not _IN_WORKER

    def map(self, fn, tasks, shared=None, workers=None):
        global _SHARED
        tasks = list(tasks)
        prev, _SHARED = _SHARED, (shared if shared is not None else {})
        try:
            n = min(workers or self.workers, len(tasks))
            if n <= 1 or not self.parallel:
                return [fn(t) for t in tasks]
            if self.auto:
                n = min(n, auto_worker_count(n, self.pairs_per_task))
            return _fork_map(fn, tasks, n)
        finally:
            _SHARED = prev


def _worker_loop(conn, fn, tasks):
    """A forked worker: runs the task index it is sent, sends back (index, ok, result, its
    private memory), and stops when it is sent None."""
    import traceback
    _worker_init()
    while True:
        i = conn.recv()
        if i is None:
            break
        try:
            conn.send((i, True, fn(tasks[i]), _private_bytes()))
        except BaseException:
            conn.send((i, False, traceback.format_exc(), 0))
            break
    conn.close()


def _fork_map(fn, tasks, n_workers, say=print):
    """Runs fn over tasks on up to n_workers forked workers and returns the results in task
    order. The driver hands out one task index at a time over a private pipe per worker (no lock
    shared with a worker, so a worker that dies cannot hold the others up). The function and its
    data are inherited, never pickled; only results travel.

    Memory: a task that is sent out, run and returned gives the same result wherever and however
    often it runs, so the pool may change how many workers run without changing any result:
    - a worker is replaced by a fresh fork after WORKER_RECYCLE_TASKS tasks, which returns the
      pages it copied from the driver (they grow with every task a worker runs);
    - when the machine's free memory falls below memory_floor(), a worker that finishes a task
      is not given another (at least one always continues);
    - if the operating system still stops a worker (out of memory; workers are its first choice,
      never the driver), its task goes back in the queue and the pool continues with one worker
      fewer. Only a task that fails on a single worker ends the step, with a message."""
    from multiprocessing.connection import wait as _wait
    ctx = _mp.get_context("fork")
    floor = memory_floor()
    queue = list(range(len(tasks)))[::-1]        # pop() hands out the lowest index first
    out, got = [None] * len(tasks), 0
    workers = {}                                 # pipe -> [process, task in hand, tasks done]
    lost_alone = set()                           # tasks lost while a single worker was running
    notes = set()
    peak = 0                                     # the largest worker (private memory), for the log

    def give(conn, i):
        """Hands task i to the worker behind conn. A worker the system stopped a moment ago
        cannot take it: its pipe then reports the end on the next wait, and lost() re-queues i."""
        workers[conn][1] = i
        try:
            conn.send(i)
        except OSError:
            pass

    def start():
        conn, child = ctx.Pipe()
        gc.freeze()               # inherited objects stay out of the collector: fewer pages copied
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", DeprecationWarning)
                p = ctx.Process(target=_worker_loop, args=(child, fn, tasks), daemon=True)
                p.start()
        finally:
            gc.unfreeze()
        child.close()
        workers[conn] = [p, None, 0]
        if queue:
            give(conn, queue.pop())

    def stop(conn):
        p = workers.pop(conn)[0]
        try:
            conn.send(None)
        except OSError:
            pass
        p.join()
        conn.close()

    def lost(conn):
        """The worker behind conn was stopped by the system: its task goes back in the queue."""
        p, i, _ = workers.pop(conn)
        p.join()
        conn.close()
        if i is not None:
            if not workers and i in lost_alone:
                raise RuntimeError(f"task {i} needs more memory than this machine has free, even on one worker: "
                                   f"lower PAIRS_PER_TASK in the Settings cell and run again")
            if not workers:
                lost_alone.add(i)
            queue.append(i)                      # it runs again, next
        if "killed" not in notes:
            say(f"  (memory: the system stopped a worker (exit code {p.exitcode}); its task runs again, "
                f"with {max(1, len(workers))} worker(s) from here on -- results are unchanged)")
            notes.add("killed")
        if not workers:
            start()

    try:
        for _ in range(min(n_workers, len(tasks))):
            start()
        while got < len(tasks):
            ready = _wait(list(workers), timeout=5)
            for conn in ready:
                try:
                    i, ok, val, mem = conn.recv()
                except (EOFError, OSError):
                    lost(conn)
                    continue
                if not ok:
                    raise RuntimeError(f"task {i} failed in a worker process:\n{val}")
                w = workers[conn]
                out[i], w[1] = val, None
                got += 1
                peak = max(peak, mem)
                w[2] += 1
                if not queue:
                    stop(conn)
                elif len(workers) > 1 and _available_bytes() < floor:
                    stop(conn)                       # memory is short: one worker fewer from here on
                    if "floor" not in notes:
                        say(f"  (memory: less than {floor / 1e9:.1f} GB free, continuing with "
                            f"{len(workers)} worker(s) -- results are unchanged)")
                        notes.add("floor")
                elif w[2] >= WORKER_RECYCLE_TASKS:
                    stop(conn)                       # a fresh fork in its place
                    start()
                else:
                    give(conn, queue.pop())
            if not ready:                            # a worker gone without closing its pipe
                for conn in [c for c, w in workers.items() if w[0].exitcode is not None]:
                    lost(conn)
    finally:
        for conn, w in list(workers.items()):
            if w[0].is_alive():
                w[0].terminate()
            w[0].join()
            conn.close()
    if len(tasks) >= 64:
        say(f"  ({len(tasks)} tasks on up to {n_workers} worker processes; the largest held "
            f"{peak / 1e9:.2f} GB of its own memory)")
    return out


PAR = Parallel(1)       # the Settings cell replaces it (N_WORKERS)


def _w_map_unique(task):
    S = _SHARED
    lo, hi = task
    f = S["func"]
    return [f(v) for v in S["values"][lo:hi]]


def map_unique_par(values, func, min_values=20_000):
    """map_unique (the core's apply-once-per-distinct-value) with the distinct values spread over
    the workers when there are enough of them to be worth it. Same values, same results."""
    s = pd.Series(values)
    uniq = pd.unique(s)
    if len(uniq) < min_values or not PAR.parallel:
        return map_unique(s, func)
    k = PAR.workers * 4
    step = max(1, -(-len(uniq) // k))
    tasks = [(lo, min(len(uniq), lo + step)) for lo in range(0, len(uniq), step)]
    res = PAR.map(_w_map_unique, tasks, {"func": func, "values": uniq})
    table = {}
    for (lo, hi), r in zip(tasks, res):
        table.update(zip(uniq[lo:hi], r))
    return s.map(table)


class RunStore:
    """Where one run keeps its large tables: a folder per table holding one Parquet part per
    chunk, every part with the same columns. Part names sort in the table's row order (part
    order, then chunk), so reading the parts in name order gives the rows in the order a
    single-process run produces them. `tables_root`: the published tables (candidates, evidence,
    dedup_pairs); `work_root`: intermediate tables, deleted when the run is done."""

    PUBLISHED = ("candidates", "evidence", "dedup_pairs")

    def __init__(self, tables_root, work_root):
        self.tables_root, self.work_root = Path(tables_root), Path(work_root)

    def dir(self, table, create=True):
        d = (self.tables_root if table in self.PUBLISHED else self.work_root) / table
        if create:
            d.mkdir(parents=True, exist_ok=True)
        return d

    @staticmethod
    def name(order, part, ci):
        return f"{order}-{part}-{ci:06d}"

    def write(self, table, name, df):
        df.to_parquet(self.dir(table) / f"{name}.parquet", index=False)

    def parts(self, table, part=None):
        d = self.dir(table, create=False)
        if not d.exists():
            return []
        return [p for p in sorted(d.glob("*.parquet")) if part is None or p.stem.split("-")[1] == part]

    def read_part(self, table, name, columns=None):
        p = self.dir(table, create=False) / f"{name}.parquet"
        return pd.read_parquet(p, columns=columns) if p.exists() else None

    def iter(self, table, part=None, columns=None):
        for p in self.parts(table, part):
            yield p.stem, pd.read_parquet(p, columns=columns)

    def read(self, table, part=None, columns=None):
        got = [df for _, df in self.iter(table, part, columns)]
        got = [g for g in got if len(g)] or got[:1]
        return pd.concat(got, ignore_index=True) if got else pd.DataFrame(columns=columns or [])

    def rows(self, table):
        import pyarrow.parquet as pq
        return sum(pq.ParquetFile(p).metadata.num_rows for p in self.parts(table))

    def drop(self, table):
        shutil.rmtree(self.dir(table, create=False), ignore_errors=True)

    def drop_work(self):
        shutil.rmtree(self.work_root, ignore_errors=True)


class TableHandle:
    """One large table of a run, held on disk as Parquet parts in row order. `read()` loads it
    whole (small runs, tests); `iter()` streams it part by part; `read_part(name)` loads one."""

    def __init__(self, store, table, rows=None):
        self.store, self.table, self._rows = store, table, rows

    def iter(self, part=None, columns=None):
        return self.store.iter(self.table, part, columns)

    def read(self, part=None, columns=None):
        return self.store.read(self.table, part, columns)

    def part_names(self, part=None):
        return [p.stem for p in self.store.parts(self.table, part)]

    def read_part(self, name, columns=None):
        return self.store.read_part(self.table, name, columns)

    @property
    def folder(self):
        return self.store.dir(self.table, create=False)

    def __len__(self):
        if self._rows is None:
            self._rows = self.store.rows(self.table)
        return self._rows


## 0b · Spark: the executor pool is the cluster

`SparkParallel` takes the local worker pool's place (`PAR`): the pipeline's parallel steps call
`PAR.map(function, tasks, shared)`. Here the tasks are dealt out over the cluster's executors
(worker machines), one Spark task per executor. `shared` -- what every task of the step reads --
is broadcast once per object and reused by every later step (a comparison context is re-sent
after it gains its name sub-part rates, so no step reads an earlier state); each executor loads
it once and runs its share of the tasks on every core of its machine with the fork pool of
section 0a, so the cores of a machine share one copy of the data (one copy per core would not
fit: the data of a 500K-row watchlist is a few GB), and the pool keeps that machine within its
memory the way it does the driver in the pandas notebook. Results come back in task order. A
step with a single task runs on the driver. Each task of a scoring step writes its chunk's rows
to the run's folder on the volume; only small summaries come back to the driver.

**Delta in and out.** `delta_input` reads a Delta table (and its version, for the manifest);
`write_delta_folder` loads a folder of Parquet parts into a Delta table with Spark (no row
passes through the driver), optionally Z-ordered for fast lookups; `write_delta_pandas` writes a
small table.

**Where Splink would do better.** Splink on Spark expresses blocking and comparisons as Spark
SQL, so the candidate pairs are produced by distributed joins and never need the watchlist's
blocking keys in every executor; here each executor core holds the broadcast party frames and
keys (fine for a watchlist of a few million rows, costly far beyond). Splink's comparisons run
in the JVM; these run in Python workers (the same Python functions as the pandas notebook, which
is what keeps the two identical).


In [ ]:
# ---- Spark: the same tasks across the whole cluster ------------------------------------------------
# SparkParallel takes the place of the local worker pool (PAR). Every parallel step of the pipeline
# (comparison batches, bootstrap replicates, address parsing, the scoring chunks and their second
# pass) calls PAR.map(function, tasks, shared). Here the tasks are dealt out over the cluster's
# executors (worker machines), one Spark task per executor; that Spark task loads what the tasks
# read (party frames, model, candidate keys -- broadcast once per object) and runs its share of
# the tasks on every core of its machine with the same memory-governed fork pool the pandas
# notebook uses on one machine, so the cores of a machine share one copy of the data instead of
# holding one each. Results come back in task order. The chunk files go to a folder every executor
# can write and the driver can read (a Unity Catalog volume), so large results never pass through
# the driver.
from pyspark.broadcast import Broadcast
from pyspark import cloudpickle as _cloudpickle


def _broadcast_key(v):
    """A key naming one broadcast on an executor for as long as it exists: the file Spark wrote
    it to (Python object ids are reused, so they cannot name it). None when Spark handed the
    value over a socket (encrypted I/O): then it is loaded per task."""
    return getattr(v, "_path", None) or None


class SparkParallel:
    """PAR for a Spark cluster (see the section text)."""
    parallel = True
    auto = True

    def __init__(self, spark, pairs_per_task=250_000):
        self.spark = spark
        self.sc = spark.sparkContext
        self.workers = int(self.sc.defaultParallelism)       # every executor core in the cluster
        self.pairs_per_task = pairs_per_task
        self._bc = {}

    def executors(self):
        """Executors (worker machines) running now; 1 in local mode (the driver is the executor)."""
        try:
            n = int(self.sc._jsc.sc().getExecutorMemoryStatus().size()) - 1    # the driver is listed too
        except Exception:
            n = 0
        return max(1, n)

    def _ship(self, v):
        if v is None or isinstance(v, (bool, int, float, str, bytes, np.integer, np.floating)):
            return v
        # One broadcast per object (kept alive, so its id stays its own). A comparison context
        # gains its name sub-part rates after the random-pair step: the key includes them, so
        # a later step never reads the earlier state. The value travels as cloudpickle bytes:
        # classes defined in this notebook go by value (plain pickle, which Spark's broadcast
        # uses, would name them as attributes of the executor's own main module).
        k = (id(v), id(getattr(v, "components", None)))
        if k not in self._bc:
            self._bc[k] = (v, self.sc.broadcast(_cloudpickle.dumps(v, protocol=5)))
        return self._bc[k][1]

    def map(self, fn, tasks, shared=None, workers=None):
        global _SHARED
        tasks = list(tasks)
        if len(tasks) <= 1:                    # nothing to spread: run it here, as the local pool does
            prev, _SHARED = _SHARED, (shared if shared is not None else {})
            try:
                return [fn(t) for t in tasks]
            finally:
                _SHARED = prev
        for cache in (_JW_CACHE, _NYSIIS_CACHE, _METAPHONE_CACHE):
            cache.clear()                      # memo caches travel with the functions: keep them empty
        shipped = {k: self._ship(v) for k, v in (shared or {}).items()}
        n_groups = min(self.executors(), len(tasks))
        groups = [list(range(g, len(tasks), n_groups)) for g in range(n_groups)]   # dealt out in turn
        ppt, cap = self.pairs_per_task, workers

        def run(it):
            # the executor's Python process loads each broadcast once and keeps it for the tasks
            # of this step (and forgets values no later step ships)
            loaded = _cloudpickle.__dict__.setdefault("_rl_loaded", {})
            keys = {k: _broadcast_key(v) for k, v in shipped.items() if isinstance(v, Broadcast)}
            for b in [b for b in loaded if b not in set(keys.values())]:
                del loaded[b]
            gc.collect()
            S = {}
            for k, v in shipped.items():
                if isinstance(v, Broadcast):
                    b = keys[k]
                    if b is None:                          # no stable key: load it for this task only
                        S[k] = _cloudpickle.loads(v.value)
                        continue
                    if b not in loaded:
                        loaded[b] = _cloudpickle.loads(v.value)
                        v.__dict__.pop("_value", None)     # the bytes can be read again from the file
                    S[k] = loaded[b]
                else:
                    S[k] = v
            g = fn.__globals__
            g["_SHARED"] = S
            # the process that runs a task packs its result as cloudpickle bytes: classes defined
            # in this notebook go by value (the fork pool's pipes use plain pickle, which would
            # look them up in the executor's own main module), and the bytes travel as they are
            pack = lambda t: _cloudpickle.dumps(fn(t), protocol=5)
            for idxs in it:
                sub = [tasks[i] for i in idxs]
                # every core of this machine, as many as its free memory allows (the fork pool
                # then keeps them within it)
                n = auto_worker_count(min(cap or len(sub), len(sub)), ppt) if _fork_available() else 1
                g["_IN_WORKER"] = True                     # a nested parallel call runs in order
                res = _fork_map(pack, sub, n, say=lambda *a: None) if n > 1 else [pack(t) for t in sub]
                yield from zip(idxs, res)
        got = dict(self.sc.parallelize(groups, n_groups).mapPartitions(run).collect())
        return [_cloudpickle.loads(got[i]) for i in range(len(tasks))]

    def release(self):
        """Drop every broadcast of the finished run (executors free their copies)."""
        for _, b in self._bc.values():
            try:
                b.destroy()
            except Exception:
                pass
        self._bc = {}


# ---- Delta Lake in and out ------------------------------------------------------------------------
def delta_input(spark, name):
    """A Delta table (catalog.schema.table, or a path) as the input frame, every column text, with
    the table version read, so the manifest can say exactly which snapshot was matched.

    Spark returns a table's rows in no fixed order, and a few steps depend on row positions (the
    seeded random-pair sample behind the u rates, the self-checks' samples), so the rows are put in
    one fixed order: a `_row` column's order when the table has one (the synthetic set is written
    with its generation order there), otherwise `record_id` order. The same table version therefore
    always gives the same numbers."""
    is_path = str(name).startswith(("/", "dbfs:", "s3:", "abfss:", "gs:"))
    df = spark.read.format("delta").load(name) if is_path else spark.read.table(name)
    try:
        ref = f"delta.`{name}`" if is_path else name
        version = int(spark.sql(f"DESCRIBE HISTORY {ref} LIMIT 1").collect()[0]["version"])
    except Exception:
        version = None
    pdf = df.toPandas()
    if "_row" in pdf.columns:
        pdf = pdf.sort_values("_row", kind="stable").drop(columns="_row")
    elif "record_id" in pdf.columns:
        pdf = pdf.sort_values("record_id", kind="stable", key=lambda s: s.astype(str))
    pdf = pdf.reset_index(drop=True)
    pdf = pdf.astype(str).replace({"None": "", "nan": "", "NaT": "", "<NA>": ""})
    return pdf, version


def delta_column(name):
    """A column name Delta accepts (no spaces or ,;{}()=): the claims table's per-basis-and-band
    counts, 'address p>=0.9', become 'address_p_ge_0.9'. Other names are unchanged."""
    n = str(name)
    for a, b in ((">=", "_ge_"), ("<=", "_le_"), ("<", "_lt_"), (">", "_gt_"), ("=", "_eq_")):
        n = n.replace(a, b)
    n = re.sub(r"[ ,;{}()\n\t]+", "_", n)
    return re.sub(r"_+", "_", n).strip("_") or "column"


def delta_table_name(schema, prefix, name):
    return f"{schema}.{prefix}{name}" if schema else f"{prefix}{name}"


def write_delta_pandas(spark, df, table, mode="overwrite"):
    """A small output table (a pandas frame) as a Delta table."""
    d = plain_table(df.drop(columns=[c for c in df.columns if str(c).startswith("_")], errors="ignore"))
    if not len(d.columns):
        d = pd.DataFrame({"empty": pd.Series([], dtype=str)})
    d.columns = [delta_column(c) for c in d.columns]
    assert d.columns.is_unique, f"two columns share a Delta name in {table}"
    sdf = spark.createDataFrame(d) if len(d) else spark.createDataFrame(d.astype(str))
    sdf.write.format("delta").mode(mode).option("overwriteSchema", "true").saveAsTable(table)
    return len(d)


def write_delta_folder(spark, folder, table, mode="overwrite", zorder=None):
    """A large table (a folder of Parquet parts written by the workers) as a Delta table: Spark
    reads the parts in parallel and writes them, so no row passes through the driver."""
    sdf = spark.read.parquet(str(folder))
    sdf = sdf.drop(*[c for c in sdf.columns if c.startswith("_")])     # internal positions stay in the parts
    sdf = sdf.toDF(*[delta_column(c) for c in sdf.columns])
    sdf.write.format("delta").mode(mode).option("overwriteSchema", "true").saveAsTable(table)
    if zorder:
        try:
            spark.sql(f"OPTIMIZE {table} ZORDER BY ({zorder})")
        except Exception as ex:                 # OPTIMIZE is a Databricks (or Delta 2+) command
            print(f"  (OPTIMIZE {table} skipped: {str(ex)[:120]})")
    return spark.read.table(table).count()


def local_spark():
    """A local Spark session with Delta Lake, for running this notebook outside Databricks
    (a test machine): every core of this machine is one executor core."""
    import tempfile
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    home = os.environ.get("RL_SPARK_HOME") or tempfile.mkdtemp(prefix="rl_spark_")   # warehouse, metastore
    b = (SparkSession.builder.appName("record-linkage").master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(home, "warehouse"))
         .config("spark.driver.extraJavaOptions", f"-Dderby.system.home={os.path.join(home, 'derby')}")
         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
         .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.driver.memory", os.environ.get("RL_SPARK_DRIVER_MEMORY", "4g"))
         .config("spark.driver.maxResultSize", "2g")
         .config("spark.ui.showConsoleProgress", "false"))
    return configure_spark_with_delta_pip(b).getOrCreate()


## 1 · Mappings, reference tables and inputs

**Mappings** (`mappings/*.csv`) are the reviewable tables: column routing, category maps,
keyword rules, specialty synonyms, the simulation noise table and the published starting
values of m. **Reference tables** (`reference/`) are copied outside tables, checked against
the SHA-256 recorded in `reference/SOURCES.md`. **Inputs** are two CSVs with the schema in
`SCHEMA` below (the next code cell): `record_id` is required and unique per file, every
other column may be empty,
missing schema columns are added empty, and unknown columns are kept as `x_<name>`
(displayed, never compared). All-empty columns are reported.

**Where Splink would do better.** Splink reads Parquet or a database table straight into
DuckDB without a pandas copy, so a 1M-row input costs little memory. Here pandas holds the
rows as text.

In [ ]:
import csv, gzip, re

SCHEMA = ["record_id", "claim_id", "note_id", "category", "first_name", "middle_name",
          "last_name", "dob", "ssn", "driver_license_number", "driver_license_state",
          "provider_npi", "professional_license_number", "professional_license_state",
          "professional_license_type", "provider_specialty", "business_name", "tin", "clinic_npi",
          "street_number", "street_direction", "street_name", "street_type", "unit", "city",
          "state", "zip", "home_phone", "work_phone", "email", "vin", "plate_number", "plate_state"]


class InputError(ValueError):
    pass


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_mappings(mdir):
    """The reviewable tables, validated: known categories only, rates in [0, 1], m per field
    summing to 1."""
    mdir = Path(mdir)
    read = lambda n: pd.read_csv(mdir / n, dtype=str, keep_default_na=False)
    maps = {"columns": read("columns.csv"), "category_map": read("category_map.csv"),
            "category_keywords": read("category_keywords.csv"),
            "specialty_canon": read("specialty_canon.csv"),
            "simulation_noise": read("simulation_noise.csv"), "published_m": read("published_m.csv")}
    cats = {"medical", "legal", "repair shop", "witness", "claimant", "other"}
    for n in ("category_map", "category_keywords"):
        bad = set(maps[n]["category"]) - cats
        if bad:
            raise InputError(f"{n}.csv: unknown categories {sorted(bad)}")
    missing = set(SCHEMA) - set(maps["columns"]["column"])
    if missing:
        raise InputError(f"columns.csv does not route {sorted(missing)}")
    rates = maps["simulation_noise"]["rate"].astype(float)
    if ((rates < 0) | (rates > 1)).any():
        raise InputError("simulation_noise.csv: a rate outside [0, 1]")
    pm = maps["published_m"].assign(m=lambda d: d["m"].astype(float))
    for f, levels in FIELD_LEVELS.items():
        sub = pm[pm["field"] == f]
        if set(sub["level"]) != set(levels):
            raise InputError(f"published_m.csv: field {f} needs levels {levels}")
        if abs(sub["m"].sum() - 1) > 1e-6:
            raise InputError(f"published_m.csv: m for {f} sums to {sub['m'].sum():.4f}, not 1")
    maps["published_m"] = pm
    return maps


def load_reference(rdir, params):
    """Outside tables as plain dicts, plus a hash check against reference/SOURCES.md."""
    rdir = Path(rdir)

    def counts(fname):
        p = rdir / fname
        if not p.exists():
            return None
        with gzip.open(p, "rt", encoding="utf-8") as fh:
            rd = csv.reader(fh); next(rd)
            return {k: int(v) for k, v in rd}

    meta_p = rdir / "reference_meta.json"
    meta = json.loads(meta_p.read_text(encoding="utf-8")) if meta_p.exists() else {}
    nick_p = rdir / "nicknames.csv"
    nick = pd.read_csv(nick_p, dtype=str, keep_default_na=False) if nick_p.exists() else None
    recorded = {}
    src = rdir / "SOURCES.md"
    if src.exists():
        for line in src.read_text(encoding="utf-8").splitlines():
            m = re.match(r"\|\s*`([^`]+)`\s*\|.*`([0-9a-f]{64})`\s*\|\s*$", line)
            if m:
                recorded[m.group(1)] = m.group(2)
    checks = []
    for name, want in sorted(recorded.items()):
        p = rdir / name
        got = sha256_file(p) if p.exists() else ""
        checks.append({"file": name, "recorded": want, "actual": got,
                       "status": "ok" if got == want else ("missing" if not got else "CHANGED")})
    return {"surnames": counts("surnames.csv.gz"), "first_names": counts("first_names.csv.gz"),
            "org_tokens": counts("org_tokens.csv.gz"), "meta": meta, "nicknames": nick,
            "hash_checks": pd.DataFrame(checks)}


def load_input(path, source):
    """One input CSV as text. Returns (frame, report)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_filter=False, encoding="utf-8")
    df.columns = [c.strip() for c in df.columns]
    return validate_input(df, source)


def validate_input(df, source):
    if "record_id" not in df.columns:
        raise InputError(f"{source}: record_id column is required")
    df = df.copy()
    df["record_id"] = df["record_id"].astype(str).str.strip()
    empty_ids = int((df["record_id"] == "").sum())
    if empty_ids:
        raise InputError(f"{source}: {empty_ids} rows have no record_id")
    dup = df["record_id"][df["record_id"].duplicated(keep=False)]
    if len(dup):
        raise InputError(f"{source}: record_id not unique, e.g. {sorted(set(dup))[:5]}")
    unknown = [c for c in df.columns if c not in SCHEMA]
    df = df.rename(columns={c: (c if c.startswith("x_") else f"x_{c}") for c in unknown})
    added = [c for c in SCHEMA if c not in df.columns]
    for c in added:
        df[c] = ""
    for c in df.columns:
        df[c] = df[c].fillna("").astype(str)
    all_null = [c for c in SCHEMA if c != "record_id" and (df[c].str.strip() == "").all()]
    report = {"source": source, "rows": len(df), "added_columns": added,
              "passthrough_columns": [c for c in df.columns if c.startswith("x_")],
              "all_empty_columns": all_null}
    return df[SCHEMA + [c for c in df.columns if c.startswith("x_")]], report

## 2 · Normalize

Every column is normalized per `mappings/columns.csv` with the core's normalizers, once per
distinct value (a name that appears 3,000 times is parsed once). Each identifier gets a
validity flag and a reason: NPI Luhn check, VIN check digit, SSN ranges, EIN prefixes,
placeholder values such as 000000000, 123456789 or 1900-01-01. Invalid values stay visible in
the details table and are never compared.

**Where Splink would do better.** Splink leaves cleaning to the user but its comparison
library normalizes inside SQL, in parallel. Here name and address parsing are Python loops
over distinct values; at 1.3M rows they are the second-largest cost after comparisons.

In [ ]:
ID_TYPES = ["ssn", "npi", "dl", "license", "tin", "cnpi", "email", "vin", "plate"]


def _norm3(series, func):
    """Apply a (value, valid, reason) normalizer once per distinct value."""
    res = map_unique_par(series, func)
    return (res.map(lambda t: t[0]), res.map(lambda t: t[1]), res.map(lambda t: t[2]))


def normalize_rows(df, maps, nick, params, source, std=None):
    """Row-level normalized frame (same index as df). `std`: the address standardizer
    (section 2a); None uses the components as given."""
    std = std or AddressStandardizer("given")
    ad = std.standardize(df)
    out = pd.DataFrame({"record_id": df["record_id"], "source": source,
                        "claim_id": df["claim_id"].str.strip(), "note_id": df["note_id"].str.strip()})
    key = df["first_name"] + "\x1f" + df["middle_name"] + "\x1f" + df["last_name"]
    parts = map_unique_par(key, lambda k: clean_person(*k.split("\x1f")))
    out["first"] = parts.map(lambda t: t[0])
    out["middle"] = parts.map(lambda t: t[1])
    out["last"] = parts.map(lambda t: t[2])
    out["business_raw"] = df["business_name"].map(fold)
    out["org_aliases"] = map_unique_par(df["business_name"], org_alias_string)
    out["dob"], out["dob_valid"], out["dob_reason"] = _norm3(df["dob"], lambda v: norm_dob(v, params))
    out["ssn"], out["ssn_valid"], out["ssn_reason"] = _norm3(df["ssn"], norm_ssn)
    out["npi"], out["npi_valid"], out["npi_reason"] = _norm3(df["provider_npi"], norm_npi)
    out["tin"], out["tin_valid"], out["tin_reason"] = _norm3(df["tin"], norm_tin)
    out["cnpi"], out["cnpi_valid"], out["cnpi_reason"] = _norm3(df["clinic_npi"], norm_npi)
    out["email"], out["email_valid"], out["email_reason"] = _norm3(df["email"], norm_email)
    out["vin"], out["vin_valid"], out["vin_reason"] = _norm3(df["vin"], norm_vin)
    q = lambda a, b: map_unique_par(df[a] + "\x1f" + df[b], lambda k: k.split("\x1f"))
    out["dl"], out["dl_valid"], out["dl_reason"] = _norm3(
        df["driver_license_number"] + "\x1f" + df["driver_license_state"], lambda k: norm_dl(*k.split("\x1f")))
    out["license"], out["license_valid"], out["license_reason"] = _norm3(
        df["professional_license_number"] + "\x1f" + df["professional_license_state"],
        lambda k: norm_license(*k.split("\x1f")))
    out["plate"], out["plate_valid"], out["plate_reason"] = _norm3(
        df["plate_number"] + "\x1f" + df["plate_state"], lambda k: norm_plate(*k.split("\x1f")))
    out["home_phone"], out["home_phone_valid"], out["home_phone_reason"] = _norm3(df["home_phone"], norm_phone)
    out["work_phone"], out["work_phone_valid"], out["work_phone_reason"] = _norm3(df["work_phone"], norm_phone)
    out["license_type"] = df["professional_license_type"].map(fold)
    canon = dict(zip(maps["specialty_canon"]["raw"].map(fold), maps["specialty_canon"]["canonical"].map(fold)))
    out["specialty_raw"] = df["provider_specialty"].map(fold)
    out["specialty"] = out["specialty_raw"].map(lambda s: canon.get(s, s))
    out["category_given"] = df["category"].str.strip().str.lower()
    akey = (ad["number"] + "\x1f" + ad["direction"] + "\x1f" + ad["name"] + "\x1f" + ad["type"] + "\x1f"
            + ad["unit"])
    a = map_unique_par(akey, lambda k: norm_street_parts(*k.split("\x1f")))
    out["addr_number"] = a.map(lambda t: t[0]); out["addr_dir"] = a.map(lambda t: t[1])
    out["addr_name"] = a.map(lambda t: t[2]); out["addr_type"] = a.map(lambda t: t[3])
    out["addr_unit"] = a.map(lambda t: t[4])
    out["city"] = map_unique_par(ad["city"], norm_city)
    out["state"] = map_unique_par(ad["state"], norm_state)
    out["zip"] = map_unique_par(ad["zip"], norm_zip)
    keys = map_unique_par(out["addr_number"] + "\x1f" + out["addr_name"] + "\x1f" + out["addr_unit"] + "\x1f"
                      + out["zip"] + "\x1f" + out["city"] + "\x1f" + out["state"],
                      lambda k: address_keys(*k.split("\x1f")))
    out["addr_full"] = keys.map(lambda t: t[0]); out["addr_street"] = keys.map(lambda t: t[1])
    out["city_state"] = keys.map(lambda t: t[3])
    xcols = [c for c in df.columns if c.startswith("x_")]
    for c in xcols:
        out[c] = df[c]
    # raw display values
    out["raw_name"] = (df["first_name"] + " " + df["middle_name"] + " " + df["last_name"]).str.split().str.join(" ")
    out["raw_business"] = df["business_name"].str.strip()
    out["raw_address"] = (df["street_number"] + " " + df["street_direction"] + " " + df["street_name"] + " "
                          + df["street_type"] + " " + df["unit"] + ", " + df["city"] + " " + df["state"]
                          + " " + df["zip"]).str.split().str.join(" ").str.strip(", ")
    return out.reset_index(drop=True)

## 2a · Address standardization (US)

One setting, `address_standardizer`, chooses how the street address becomes components. All
three produce the same fields (number, pre-direction, street name, street type, unit, city,
state, ZIP), which the core's normalizers then abbreviate the USPS way:

| Setting | What it does |
|---|---|
| `given` | the input is already standardized (by SmartyStreets upstream, say): the components are used as given |
| `usaddress` (default, offline) | the street columns are rejoined into one line and parsed with the `usaddress` tagger, so a whole line typed into `street_name`, a unit inside the name or a missing type are repaired; a line the tagger cannot parse keeps its given components (counted) |
| `smarty` | the SmartyStreets US Street API: credentials from `SMARTY_AUTH_ID` / `SMARTY_AUTH_TOKEN`, batches of 100, every answer cached in `data/smarty_cache.json`. Without credentials, or for an address Smarty cannot match, the row falls back to `usaddress`, and the manifest says how many. The self-tests never call the service (a fake stands in) |

City, state and ZIP come from their own columns in every mode except `smarty`, which returns
its own.

**Where Splink would do better.** Nowhere: Splink leaves address cleaning to the user and
has no address parser; its comparison library compares address strings as given. The
parsing here is ours.

In [ ]:
ADDRESS_COMPONENTS = ["number", "direction", "name", "type", "unit", "city", "state", "zip"]


def _street_line(number, direction, name, stype, unit):
    u = fold(unit)
    if u and not re.match(r"^(APT|APARTMENT|UNIT|STE|SUITE|RM|ROOM|FL|FLOOR|BLDG|LOT|SPC|SPACE|TRLR|DEPT|#)\b", u):
        u = "# " + u
    return " ".join(x for x in (fold(number), fold(direction), fold(name), fold(stype), u) if x)


def parse_usaddress(line):
    """One street line -> (number, direction, name, type, unit) with the usaddress tagger, or
    None when it cannot be parsed. PO boxes become ('<box>', '', 'PO BOX', '', '')."""
    import usaddress
    if not line:
        return ("", "", "", "", "")
    try:
        tags, _ = usaddress.tag(line)
    except Exception:
        return None
    if "USPSBoxID" in tags:
        return (tags["USPSBoxID"], "", "PO BOX", "", "")
    number = " ".join(tags.get(k, "") for k in ("AddressNumberPrefix", "AddressNumber", "AddressNumberSuffix")).strip()
    name = " ".join(tags.get(k, "") for k in ("StreetNamePreModifier", "StreetNamePreType", "StreetName")).strip()
    predir = tags.get("StreetNamePreDirectional", "").replace(".", "")
    ws = name.split()
    if predir in ("N", "S") and len(ws) > 1 and ws[0] in ("E", "W"):      # 'N W 45TH' -> NW 45TH
        predir, name = predir + ws[0], " ".join(ws[1:])
    unit = " ".join(tags.get(k, "") for k in ("OccupancyIdentifier", "SubaddressIdentifier")).replace("#", "").strip()
    if not (number or name):
        return None
    return (number, predir, name, tags.get("StreetNamePostType", ""), unit)


class AddressStandardizer:
    """Street address -> components, by the chosen mode (see the section text). `post` is the
    HTTP call used by the smarty mode (injectable, so tests never reach the network)."""

    def __init__(self, mode="usaddress", cache_path=None, url="", batch=100, post=None, env=None):
        self.mode, self.cache_path, self.url, self.batch = mode, cache_path, url, batch
        self.post = post or self._http_post
        env = os.environ if env is None else env
        self.auth = (env.get("SMARTY_AUTH_ID", ""), env.get("SMARTY_AUTH_TOKEN", ""))
        self.report = defaultdict(int)
        self.report["mode"] = mode
        self._cache = None

    # ---- smarty ------------------------------------------------------------------------------
    def _load_cache(self):
        if self._cache is None:
            self._cache = {}
            if self.cache_path and Path(self.cache_path).exists():
                self._cache = json.loads(Path(self.cache_path).read_text(encoding="utf-8"))
        return self._cache

    def _save_cache(self):
        if self.cache_path and self._cache is not None:
            Path(self.cache_path).write_text(json.dumps(self._cache, indent=0, sort_keys=True), encoding="utf-8")

    def _http_post(self, url, params, payload):
        import urllib.request, urllib.parse
        q = urllib.parse.urlencode(params)
        req = urllib.request.Request(f"{url}?{q}", data=json.dumps(payload).encode("utf-8"),
                                     headers={"Content-Type": "application/json; charset=utf-8"})
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read().decode("utf-8"))

    @staticmethod
    def _smarty_key(street, city, state, zipc):
        return hashlib.sha1("\x1f".join((street, city, state, zipc)).encode("utf-8")).hexdigest()

    def _smarty(self, queries):
        """queries: {key: (street, city, state, zip)} -> {key: component tuple or None}."""
        cache = self._load_cache()
        out = {k: cache[k] for k in queries if k in cache}
        self.report["smarty_cache_hits"] += len(out)
        todo = [k for k in queries if k not in cache]
        if todo and not all(self.auth):
            self.report["smarty_no_credentials"] += len(todo)
            return out
        keys = sorted(todo)
        for i in range(0, len(keys), self.batch):
            chunk = keys[i:i + self.batch]
            payload = [{"input_id": k, "street": queries[k][0], "city": queries[k][1], "state": queries[k][2],
                        "zipcode": queries[k][3], "candidates": 1} for k in chunk]
            try:
                resp = self.post(self.url, {"auth-id": self.auth[0], "auth-token": self.auth[1]}, payload)
                self.report["smarty_calls"] += 1
            except Exception as ex:                               # a failed call is counted, not fatal
                self.report["smarty_errors"] += 1
                self.report["smarty_last_error"] = str(ex)[:200]
                continue
            got = {r.get("input_id"): r for r in resp or []}
            for k in chunk:
                r = got.get(k)
                if r is None:
                    cache[k] = None
                    continue
                c = r.get("components", {})
                cache[k] = [c.get("primary_number", ""), c.get("street_predirection", ""), c.get("street_name", ""),
                            c.get("street_suffix", ""), c.get("secondary_number", ""), c.get("city_name", ""),
                            c.get("state_abbreviation", ""), c.get("zipcode", "")]
            for k in chunk:
                if k in cache:
                    out[k] = cache[k]
        self._save_cache()
        return out

    # ---- all modes ---------------------------------------------------------------------------
    def standardize(self, df):
        """Input-schema rows -> DataFrame of ADDRESS_COMPONENTS (raw strings, same index)."""
        given = pd.DataFrame({"number": df["street_number"], "direction": df["street_direction"],
                              "name": df["street_name"], "type": df["street_type"], "unit": df["unit"],
                              "city": df["city"], "state": df["state"], "zip": df["zip"]}, index=df.index)
        self.report["rows"] += len(df)
        if self.mode == "given":
            return given
        lines = [_street_line(*t) for t in zip(df["street_number"], df["street_direction"], df["street_name"],
                                               df["street_type"], df["unit"])]
        out = given.copy()
        if self.mode == "smarty":
            q = {}
            keys = []
            for i, (line, c, s, z) in enumerate(zip(lines, df["city"], df["state"], df["zip"])):
                if not line:
                    keys.append("")
                    continue
                k = self._smarty_key(line, fold(c), fold(s), str(z).strip())
                q[k] = (line, fold(c), fold(s), str(z).strip())
                keys.append(k)
            res = self._smarty(q)
            done = np.zeros(len(df), dtype=bool)
            vals = {c: out[c].to_numpy(dtype=object).copy() for c in ADDRESS_COMPONENTS}
            for i, k in enumerate(keys):
                r = res.get(k) if k else None
                if r:
                    for c, v in zip(ADDRESS_COMPONENTS, r):
                        vals[c][i] = v or ""
                    done[i] = True
                elif k and k in res:
                    self.report["smarty_no_match"] += 1
            for c in ADDRESS_COMPONENTS:
                out[c] = vals[c]
            self.report["smarty_standardized"] += int(done.sum())
            rest = ~done
        else:
            rest = np.ones(len(df), dtype=bool)
        idx = np.flatnonzero(rest & (np.array(lines, dtype=object) != ""))
        if len(idx):
            if self.mode == "smarty":
                self.report["smarty_fallback_usaddress"] += len(idx)
            parsed = map_unique_par(pd.Series([lines[i] for i in idx]), parse_usaddress)
            vals = {c: out[c].to_numpy(dtype=object).copy() for c in ADDRESS_COMPONENTS[:5]}
            for j, t in zip(idx, parsed):
                if t is None:
                    self.report["usaddress_unparsed_kept_given"] += 1
                    continue
                for c, v in zip(ADDRESS_COMPONENTS[:5], t):
                    vals[c][j] = v
                self.report["usaddress_parsed"] += 1
            for c in ADDRESS_COMPONENTS[:5]:
                out[c] = vals[c]
        return out


def make_standardizer(cfg, post=None, env=None):
    return AddressStandardizer(cfg.address_standardizer, cfg.paths["smarty_cache"], cfg.smarty_url,
                               cfg.smarty_batch, post=post, env=env)

## 3 · Breakdown

Each row yields up to two parties, tied by the row: a **person part** when there is a first
or last name, and a **business part** when there is a business name, or a TIN or clinic NPI
with no business name. Details go to their owner per `mappings/columns.csv`: address and work
phone are held by both parts with ownership `row`; the email goes to the person, or to the
business when the row has no person. Rows that yield no party are reported, never dropped
silently.

**Where Splink would do better.** Splink links records, not parts of records; the split is
ours. What Splink would add is its array comparisons for multi-valued fields (a party with
several phones), which this schema does not need because a row holds one value per column.

In [ ]:
def breakdown(rows, source, nick):
    """Rows -> (parties, details, ties, empty_rows)."""
    has_person = (rows["first"] != "") | (rows["last"] != "")
    raw_tin_or_cnpi = (rows["tin"] != "") | (rows["cnpi"] != "")
    has_business = (rows["org_aliases"] != "") | raw_tin_or_cnpi
    pi = np.flatnonzero(has_person.to_numpy())
    bi = np.flatnonzero(has_business.to_numpy())
    empty = rows.loc[~(has_person | has_business), ["record_id"]]

    def base(idx, part):
        r = rows.iloc[idx]
        tag = "P" if part == "person" else "B"
        p = pd.DataFrame({"party_id": source + ":" + r["record_id"] + ":" + tag,
                          "source": source, "record_id": r["record_id"].to_numpy(), "part": part,
                          "row": idx, "claim_id": r["claim_id"].to_numpy(),
                          "note_id": r["note_id"].to_numpy()})
        return p

    P = base(pi, "person")
    B = base(bi, "business")
    rp, rb = rows.iloc[pi], rows.iloc[bi]
    for c in ("first", "middle", "last"):
        P[c] = rp[c].to_numpy()
        B[c] = ""
    P["org_aliases"] = ""
    B["org_aliases"] = rb["org_aliases"].to_numpy()
    P["name_key"] = np.where(P["first"] != "", P["first"] + " " + P["last"], P["last"])
    B["name_key"] = [a.split("|")[0] if a else "" for a in B["org_aliases"]]
    # who holds a value: surname + first initial, so 'R. Smith' and 'Robert Smith' are one holder
    P["holder_key"] = P["last"] + "|" + P["first"].str[:1]
    B["holder_key"] = B["name_key"]
    P["display_name"] = rp["raw_name"].to_numpy()
    B["display_name"] = np.where(rb["raw_business"].to_numpy() != "", rb["raw_business"].to_numpy(),
                                 "(no name: " + np.where(rb["tin"].to_numpy() != "", "TIN " + rb["tin"].to_numpy(),
                                                         "clinic NPI " + rb["cnpi"].to_numpy()) + ")")
    P["first_roots"] = map_unique_par(P["first"], nick.roots_string).to_numpy()
    B["first_roots"] = ""
    P["last_nysiis"] = map_unique_par(P["last"], nysiis).to_numpy()
    B["last_nysiis"] = ""
    # details owned by the person only
    for c, col in (("dob", "dob"),):
        P[c] = np.where(rp["dob_valid"].to_numpy(dtype=bool), rp["dob"].to_numpy(), "")
        B[c] = ""
    P["dob_int"] = [int(d.replace("-", "")) if d else 0 for d in P["dob"]]
    B["dob_int"] = 0
    valid = lambda r, t: np.where(r[f"{t}_valid"].to_numpy(dtype=bool), r[t].to_numpy(), "")
    for t in ("ssn", "npi", "dl", "license", "vin", "plate"):
        P[f"id_{t}"] = valid(rp, t)
        B[f"id_{t}"] = ""
    for t in ("tin", "cnpi"):
        B[f"id_{t}"] = valid(rb, t)
        P[f"id_{t}"] = ""
    # email: the person's, or the business's when the row has no person
    P["id_email"] = valid(rp, "email")
    B["id_email"] = np.where(has_person.to_numpy()[bi], "", valid(rb, "email"))
    P["phone_own"] = valid(rp, "home_phone")
    B["phone_own"] = ""
    P["phone_row"] = valid(rp, "work_phone")
    B["phone_row"] = valid(rb, "work_phone")
    for c in ("addr_full", "addr_street", "zip", "city_state", "state", "city", "addr_number",
              "addr_name", "addr_unit"):
        P[c] = rp[c].to_numpy()
        B[c] = rb[c].to_numpy()
    P["specialty"] = rp["specialty"].to_numpy()
    B["specialty"] = ""                        # provider_specialty is the person's
    P["specialty_raw_row"] = rp["specialty_raw"].to_numpy()
    B["specialty_raw_row"] = rb["specialty_raw"].to_numpy()   # category inference is row-level
    for c in ("license_type", "category_given", "raw_address"):
        P[c] = rp[c].to_numpy()
        B[c] = rb[c].to_numpy()
    P["business_raw"] = rp["business_raw"].to_numpy()
    B["business_raw"] = rb["business_raw"].to_numpy()
    parties = pd.concat([P, B], ignore_index=True)
    # ties: the other part of the same row (a row holds at most one person and one business)
    both = np.intersect1d(pi, bi)
    pp = pd.Series(P["party_id"].to_numpy(), index=pi).reindex(both).to_numpy()
    bb = pd.Series(B["party_id"].to_numpy(), index=bi).reindex(both).to_numpy()
    other = dict(zip(pp, bb))
    other.update(zip(bb, pp))
    parties["tie"] = parties["party_id"].map(other).fillna("")
    parties = parties.sort_values(["row", "part"], kind="stable").reset_index(drop=True)
    ties = parties[(parties["part"] == "person") & (parties["tie"] != "")][["record_id", "party_id", "tie"]]
    ties = ties.rename(columns={"party_id": "person_party", "tie": "business_party"}).reset_index(drop=True)
    details = build_details(rows, parties, pi, bi, has_person.to_numpy())
    return parties, details, ties, empty


DETAIL_SPEC = [  # (type, row column, owner, ownership, source column)
    ("dob", "dob", "person", "own", "dob"),
    ("ssn", "ssn", "person", "own", "ssn"),
    ("npi", "npi", "person", "own", "provider_npi"),
    ("dl", "dl", "person", "own", "driver_license_number"),
    ("license", "license", "person", "own", "professional_license_number"),
    ("vin", "vin", "person", "own", "vin"),
    ("plate", "plate", "person", "own", "plate_number"),
    ("phone", "home_phone", "person", "own", "home_phone"),
    ("phone", "work_phone", "both", "row", "work_phone"),
    ("email", "email", "person_else_business", "own", "email"),
    ("tin", "tin", "business", "own", "tin"),
    ("cnpi", "cnpi", "business", "own", "clinic_npi"),
    ("address", "addr_full", "both", "row", "street_number..zip"),
]


def build_details(rows, parties, pi, bi, has_person):
    """Long details table: party, type, normalized value, validity, ownership, source column."""
    pid_p = parties.set_index(["row", "part"])["party_id"]
    person_of = pd.Series(pid_p.xs("person", level="part")) if len(pi) else pd.Series(dtype=object)
    business_of = pd.Series(pid_p.xs("business", level="part")) if len(bi) else pd.Series(dtype=object)
    out = []
    for typ, col, owner, own, src in DETAIL_SPEC:
        vals = rows[col]
        present = vals != ""
        vflag = rows[f"{col}_valid"] if f"{col}_valid" in rows else pd.Series(True, index=rows.index)
        reason = rows[f"{col}_reason"] if f"{col}_reason" in rows else pd.Series("", index=rows.index)
        targets = []
        if owner in ("person", "both", "person_else_business"):
            targets.append(person_of)
        if owner in ("business", "both"):
            targets.append(business_of)
        if owner == "person_else_business":
            targets.append(business_of[~pd.Series(has_person)[business_of.index].to_numpy()])
        for tgt in targets:
            idx = tgt.index[present.to_numpy()[tgt.index]] if len(tgt) else []
            if len(idx) == 0:
                continue
            out.append(pd.DataFrame({"party_id": tgt.loc[idx].to_numpy(), "type": typ,
                                     "value": vals.to_numpy()[idx], "valid": vflag.to_numpy()[idx].astype(bool),
                                     "reason": reason.to_numpy()[idx], "ownership": own,
                                     "source_column": src}))
    if not out:
        return pd.DataFrame(columns=["party_id", "type", "value", "valid", "reason", "ownership", "source_column"])
    return pd.concat(out, ignore_index=True)

## 4 · Value index

For every (type, value): how many parties hold it in each file, how many distinct names hold
it, whether one name holds it (single-holder: the only kind that can veto or anchor), and
whether it is junk (fails validation, or is held under more than `junk_holders` names). It
drives shared-identifier counts, anchors, per-value u and the row-level evidence. Junk values
are blanked from the party frames before any comparison and listed in the manifest.

**Where Splink would do better.** Splink computes term frequencies per column in SQL and
joins them onto pairs; the idea is the same, and Splink's version is lazier with memory.

In [ ]:
def value_index(parties, details):
    """(type, value) -> holders per file, distinct names, single-holder and junk flags."""
    d = details.merge(parties[["party_id", "source", "name_key"]], on="party_id", how="left")
    g = d.groupby(["type", "value"], sort=True)
    vi = pd.DataFrame({
        "holders_extracted": g["source"].agg(lambda s: int((s == "X").sum())),
        "holders_watchlist": g["source"].agg(lambda s: int((s == "W").sum())),
        "distinct_names": g["name_key"].nunique(),
        "valid": g["valid"].all(),
        "invalid_reason": g["reason"].agg(lambda s: ",".join(sorted({x for x in s if x}))),
    }).reset_index()
    vi["single_holder"] = vi["valid"] & (vi["distinct_names"] == 1)
    return vi


def value_index_fast(parties, details):
    """Same as value_index, vectorized for large inputs."""
    d = details[["party_id", "type", "value", "valid", "reason"]].merge(
        parties[["party_id", "source", "holder_key"]].rename(columns={"holder_key": "name_key"}), on="party_id", how="left")
    d["is_x"] = (d["source"] == "X").astype(np.int64)
    d["is_w"] = (d["source"] == "W").astype(np.int64)
    g = d.groupby(["type", "value"], sort=True)
    vi = g.agg(holders_extracted=("is_x", "sum"), holders_watchlist=("is_w", "sum"),
               distinct_names=("name_key", "nunique"), valid=("valid", "all")).reset_index()
    bad = d[d["reason"] != ""].groupby(["type", "value"])["reason"].first()
    vi["invalid_reason"] = pd.MultiIndex.from_frame(vi[["type", "value"]]).map(bad.to_dict()).fillna("")
    vi["single_holder"] = vi["valid"] & (vi["distinct_names"] == 1)
    return vi


def mark_junk(vi, params):
    vi = vi.copy()
    many = vi["valid"] & (vi["distinct_names"] > params.junk_holders) & (vi["type"] != "address") & \
        (vi["type"] != "dob")
    vi["junk"] = ~vi["valid"] | many
    vi["junk_reason"] = np.where(~vi["valid"], "invalid:" + vi["invalid_reason"],
                                 np.where(many, "held under " + vi["distinct_names"].astype(str) + " names", ""))
    return vi


def blank_junk(parties, vi):
    """Blank identifier values held under too many names (invalid ones are already blank)."""
    many = vi[vi["junk"] & vi["valid"]]
    col = {"ssn": ["id_ssn"], "npi": ["id_npi"], "dl": ["id_dl"], "license": ["id_license"],
           "tin": ["id_tin"], "cnpi": ["id_cnpi"], "email": ["id_email"], "vin": ["id_vin"],
           "plate": ["id_plate"], "phone": ["phone_own", "phone_row"]}
    n = 0
    for t, cols in col.items():
        bad = set(many.loc[many["type"] == t, "value"])
        if not bad:
            continue
        for c in cols:
            m = parties[c].isin(bad)
            n += int(m.sum())
            parties.loc[m, c] = ""
    return n

## 5 · Category

**Extracted rows.** The given category is kept. An inferred category is added beside it,
with the rule that set it: a valid provider or clinic NPI → medical, a licence type that
`category_map.csv` maps to medical or legal → that category (both *identifier-backed*); a
business-name phrase from `category_keywords.csv` → its category (*keyword*, weaker, applied
to both parts of the row). Witness and claimant are never inferred; "other" counts as no
information. The category used in matching is identifier-backed → given → keyword.
A given category that disagrees with an inferred one is a data-quality finding; both stay
visible.

**Watchlist rows.** No category is given: it is inferred from the specialty, then the licence
type, through `category_map.csv`; unmapped values → unknown, listed in the manifest.

**Prior group** (from the extracted party): professional (medical, legal), business (repair
shop, or a business part unless identifier-backed medical/legal), private (witness,
claimant), unknown.

**Where Splink would do better.** Nothing specific: Splink has no category concept. Its
closest tool is blocking or training separately per group, which the prior groups mimic.

In [ ]:
GROUPS = ["professional", "business", "private", "unknown"]


def _keyword_rules(kw):
    rules = []
    for phrase, cat in zip(kw["phrase"].map(fold), kw["category"]):
        rules.append((re.compile(r"(?<![A-Z0-9])" + re.escape(phrase) + r"(?![A-Z0-9])"), cat, phrase))
    return rules


def keyword_category(names, kw):
    """First matching phrase wins (file order). Returns (category, phrase) per name."""
    rules = _keyword_rules(kw)

    def one(s):
        s = _undot(fold(s))
        for rx, cat, phrase in rules:
            if rx.search(s):
                return cat, phrase
        return "", ""
    return map_unique_par(pd.Series(names), one)


def assign_category_extracted(parties, maps):
    cm = maps["category_map"]
    lic = dict(zip(cm.loc[cm["source_field"] == "license_type", "value"].map(fold),
                   cm.loc[cm["source_field"] == "license_type", "category"]))
    p = parties
    given = p["category_given"].where(p["category_given"].isin(
        ["medical", "legal", "repair shop", "witness", "claimant"]), "")
    npi_backed = (p["id_npi"] != "") | (p["id_cnpi"] != "")
    lic_cat = p["license_type"].map(lambda v: lic.get(v, "")).where(p["part"] == "person", "")
    lic_cat = lic_cat.where(lic_cat.isin(["medical", "legal"]), "")
    id_cat = np.where(npi_backed, "medical", lic_cat)
    id_rule = np.where(npi_backed, np.where(p["id_npi"] != "", "provider_npi", "clinic_npi"),
                       np.where(lic_cat != "", "license_type:" + p["license_type"], ""))
    kwc = keyword_category(p["business_raw"], maps["category_keywords"])
    kw_cat = kwc.map(lambda t: t[0]).to_numpy()
    kw_rule = np.where(kw_cat != "", "keyword:" + kwc.map(lambda t: t[1]).to_numpy(), "")
    inferred = np.where(id_cat != "", id_cat, kw_cat)
    inferred_rule = np.where(id_cat != "", id_rule, kw_rule)
    used = np.where(id_cat != "", id_cat, np.where(given != "", given, kw_cat))
    strength = np.where(id_cat != "", "strong", np.where(used != "", "weak", ""))
    used_rule = np.where(id_cat != "", id_rule, np.where(given != "", "given", kw_rule))
    p["category_inferred"] = inferred
    p["category_rule"] = inferred_rule
    p["category"] = used
    p["category_used_rule"] = used_rule
    p["cat_strength"] = strength
    p["category_mismatch"] = (given != "") & (inferred != "") & (given != inferred)
    person = p["part"] == "person"
    group = np.select(
        [person & np.isin(used, ["medical", "legal"]),
         person & np.isin(used, ["witness", "claimant"]),
         person & (used == "repair shop"),
         ~person & (id_cat != "")],
        ["professional", "private", "business", "professional"],
        default=np.where(person, "unknown", "business"))
    p["prior_group"] = group
    return p


def assign_category_watchlist(parties, maps):
    cm = maps["category_map"]
    spec = dict(zip(cm.loc[cm["source_field"] == "specialty", "value"].map(fold),
                    cm.loc[cm["source_field"] == "specialty", "category"]))
    lic = dict(zip(cm.loc[cm["source_field"] == "license_type", "value"].map(fold),
                   cm.loc[cm["source_field"] == "license_type", "category"]))
    p = parties
    raw_spec = p["specialty_raw_row"] if "specialty_raw_row" in p else p["specialty"]
    s_cat = raw_spec.map(lambda v: spec.get(v, ""))
    l_cat = p["license_type"].map(lambda v: lic.get(v, ""))
    cat = np.where(s_cat != "", s_cat, l_cat)
    rule = np.where(s_cat != "", "specialty_map:" + raw_spec, np.where(l_cat != "", "license_type_map:" + p["license_type"], ""))
    p["category_inferred"] = cat
    p["category_rule"] = rule
    p["category"] = cat
    p["category_used_rule"] = rule
    p["cat_strength"] = np.where(cat != "", "strong", "")
    p["category_mismatch"] = False
    p["prior_group"] = ""
    unmapped = pd.concat([
        pd.DataFrame({"source_field": "specialty", "value": raw_spec[(raw_spec != "") & (s_cat == "")]}),
        pd.DataFrame({"source_field": "license_type", "value": p["license_type"][(p["license_type"] != "") & (l_cat == "") & (s_cat == "")]}),
    ]).value_counts().rename("parties").reset_index()
    return p, unmapped

## 6 · Outside rarity and party frames

The outside tables become a `Rarity` (Census surnames, SSA first names and initials, NPPES
organization words, with the watchlist's own business-word shares as the larger of the two
where it is larger) and a `Nicknames` index. Parties are split by part into the positional
frames the matching core compares, junk identifier values are blanked, and the comparison
context (value statistics, declared d/b/a pairs) is built.

**Where Splink would do better.** Splink's term-frequency tables come from the data being
linked; ours come from outside, which is deliberate (a fraud file over-represents exactly the
names in question) but means rarity is only as good as the Census/SSA/NPPES coverage.

In [ ]:
@dataclass
class Prepared:
    X: dict            # part -> positional party frame (extracted)
    W: dict            # part -> positional party frame (watchlist)
    parties: pd.DataFrame
    details: pd.DataFrame
    ties: pd.DataFrame
    vi: pd.DataFrame
    ctx: object        # comparison context for extracted x watchlist (u per value from the watchlist)
    ctx_dedup: object  # ... for extracted x extracted (u per value from the extracted file itself)
    reports: dict


def split_parts(parties):
    """part -> positional frame, with tie_pos: the tied party's position in the other part's
    frame of the same side (-1 when none)."""
    fr = {part: parties[parties["part"] == part].reset_index(drop=True) for part in ("person", "business")}
    pos = {part: pd.Series(np.arange(len(f)), index=f["party_id"].to_numpy()) for part, f in fr.items()}
    for part, other in (("person", "business"), ("business", "person")):
        t = fr[part]["tie"]
        fr[part]["tie_pos"] = t.map(pos[other]).fillna(-1).astype(np.int64).to_numpy()
    # the tied business's identifiers, on the person (the co-party proxy in estimation reads them)
    tp = fr["person"]["tie_pos"].to_numpy()
    for c, src in (("tie_tin", "id_tin"), ("tie_cnpi", "id_cnpi")):
        vals = fr["business"][src].to_numpy(dtype=object) if len(fr["business"]) else np.array([], dtype=object)
        fr["person"][c] = np.where(tp >= 0, vals[np.maximum(tp, 0)] if len(vals) else "", "")
    return fr


def rows_to_parties(df, source, maps, nick, params, watchlist, std=None):
    rows = normalize_rows(df, maps, nick, params, source, std)
    parties, details, ties, empty = breakdown(rows, source, nick)
    if watchlist:
        parties, unmapped = assign_category_watchlist(parties, maps)
    else:
        parties = assign_category_extracted(parties, maps)
        unmapped = None
    return rows, parties, details, ties, empty, unmapped


def prepare(xdf, wdf, maps, ref, cfg, log=print, std=None):
    """Inputs (validated frames) -> Prepared."""
    p = cfg.core
    nick = Nicknames(ref["nicknames"])
    std = std or make_standardizer(cfg)
    xrows, xpar, xdet, xtie, xempty, _ = rows_to_parties(xdf, "X", maps, nick, p, False, std)
    wrows, wpar, wdet, wtie, wempty, unmapped = rows_to_parties(wdf, "W", maps, nick, p, True, std)
    parties = pd.concat([xpar, wpar], ignore_index=True)
    details = pd.concat([xdet, wdet], ignore_index=True)
    vi = mark_junk(value_index_fast(parties, details), p)
    blanked_x = blank_junk(xpar, vi)
    blanked_w = blank_junk(wpar, vi)
    counts, total = own_org_word_counts(wpar.loc[wpar["part"] == "business", "org_aliases"])
    rarity = Rarity(ref["surnames"], ref["first_names"], ref["org_tokens"], ref["meta"], p,
                    own_org_counts=counts, own_org_total=total)
    stats = build_value_stats(xpar, wpar)
    dba = declared_dba_pairs(pd.concat([xpar["org_aliases"], wpar["org_aliases"]]))
    ctx = CompareContext(params=p, rarity=rarity, nick=nick, stats=stats, dba_pairs=dba)
    # deduplication: the extracted file is its own reference population for per-value u;
    # who holds a value (single-holder, vetoes) is still counted over both files
    stats_d = build_value_stats(xpar, xpar)
    stats_d.names, stats_d.single = stats.names, stats.single
    ctx_dedup = CompareContext(params=p, rarity=rarity, nick=nick, stats=stats_d, dba_pairs=dba)
    X, W = split_parts(xpar), split_parts(wpar)
    dq = details[~details["valid"]].groupby(["type", "reason"]).size().rename("values").reset_index()
    reports = {"empty_rows_extracted": xempty, "empty_rows_watchlist": wempty, "unmapped": unmapped,
               "invalid_values": dq, "junk_blanked": blanked_x + blanked_w,
               "category_mismatch": int(xpar["category_mismatch"].sum()),
               "rarity_source": rarity.source, "nickname_rows": nick.size, "dba_pairs": len(dba),
               "rows": {"extracted": len(xdf), "watchlist": len(wdf)},
               "address_standardizer": dict(std.report),
               "parties": {f"{s}_{part}": len(fr) for s, d in (("extracted", X), ("watchlist", W))
                           for part, fr in d.items()}}
    log(f"parties: {reports['parties']}  empty rows: extracted {len(xempty)}, watchlist {len(wempty)}")
    log(f"address standardizer: {dict(std.report)}")
    log(f"value index: {len(vi):,} values, {int(vi['single_holder'].sum()):,} single-holder, "
        f"{int(vi['junk'].sum()):,} junk; junk values blanked in {reports['junk_blanked']} party fields")
    log(f"rarity: {rarity.source}; nicknames: {nick.size} rows; declared d/b/a pairs: {len(dba)}")
    return Prepared(X=X, W=W, parties=parties, details=details, ties=pd.concat([xtie, wtie]), vi=vi,
                    ctx=ctx, ctx_dedup=ctx_dedup, reports=reports)

## 7 · Candidates

Only extracted × watchlist pairs of the same part type are ever proposed. The union of these
rules, each pair remembering which rules proposed it:

| Part | Rules |
|---|---|
| Person | exact SSN, provider NPI, DL (state:number), licence (state:number), email, VIN, plate; any phone; NYSIIS(last) + first initial; NYSIIS(last) + canonical first initial (Bill meets William); exact DOB + first initial (a surname change); swapped first/last; sorted neighbourhood on "last first" (window 5) |
| Business | exact TIN, clinic NPI, email, work phone; rarest word of each alias (d/b/a included); leading word + state; sorted neighbourhood on the sorted-word name |

**Before indexing**, every rule's pair count is computed from key counts on both sides and
printed. A key that would propose more than `block_pair_cap` pairs is refined with the state;
a refined key still over the cap is dropped and listed in the manifest with its pair count
(an unresolved outcome, never a silent loss). Keys shared by more than the cap in the sorted
neighbourhood are left to the refined blocks. Indexing runs in chunks of `chunk_size`
extracted parties through `recordlinkage.Index` (`Block`, `SortedNeighbourhood` with the
global sorting-key values, so a chunked run proposes exactly the pairs of an unchunked one).

**Chunks.** The extracted side is cut into chunks by the pairs they are estimated to propose
(each extracted party's keys against the watchlist's key counts, summed over the rules), so no
chunk holds more than about `pairs_per_task` comparisons, whatever the skew of common names.
The estimate only sizes the chunks: which pairs are proposed never depends on it.

**Where Splink would do better.** Splink's blocking runs as SQL joins in DuckDB, parallel
and out of core, and its `cumulative_comparisons_to_be_scored_from_blocking_rules_chart`
shows each rule's marginal pairs. recordlinkage's `Block` is a pandas merge per chunk.

In [ ]:
PERSON_RULES = ["ssn", "npi", "dl", "license", "email", "vin", "plate", "phone", "near_ids",
                "nysiis_initial", "nysiis_canon_initial", "nysiis_first_missing", "nysiis_last",
                "last_part_initial", "first_sound_last_initial", "dob_initial", "dob", "swapped",
                "sn_last_first", "sn_first_last", "street"]
BUSINESS_RULES = ["tin", "cnpi", "email", "phone", "near_ids", "rare_word", "rare_word_sound",
                  "lead_word_state", "sn_sorted_name", "street"]
IDENTIFIER_RULES = {"ssn", "npi", "dl", "license", "email", "vin", "plate", "phone", "tin", "cnpi", "near_ids"}
SN_RULES = {"sn_last_first", "sn_first_last", "sn_sorted_name"}
ASYMMETRIC_RULES = {"swapped", "nysiis_first_missing"}     # x-side and w-side keys differ
NEAR_ID_TYPES = {"person": ["ssn", "npi", "dl", "phone"], "business": ["tin", "phone"]}


def _deletion_keys(v):
    """Every string with one character deleted: two values one substitution or one adjacent
    transposition apart share at least one of these keys."""
    return {v[:i] + v[i + 1:] for i in range(len(v))}


def rule_keys(parties, part, rule, rarity, side="x", dba_pairs=None):
    """Long frame (pos, key, state) of one rule's keys; a party can have several keys."""
    p = parties
    pos = np.arange(len(p))
    st = p["state"].to_numpy(dtype=object)
    if rule in ("ssn", "npi", "dl", "license", "email", "vin", "plate", "tin", "cnpi"):
        k = p[f"id_{rule}"].to_numpy(dtype=object)
        if rule in ("dl", "license", "plate"):                   # number only: states may be unstated
            k = np.array([v.partition(":")[2] if v else "" for v in k], dtype=object)
        return _frame(pos, k, st)
    if rule == "phone":
        if part == "person":
            return pd.concat([_frame(pos, p["phone_own"].to_numpy(dtype=object), st),
                              _frame(pos, p["phone_row"].to_numpy(dtype=object), st)], ignore_index=True)
        return _frame(pos, p["phone_row"].to_numpy(dtype=object), st)
    if rule == "near_ids":
        # B1: one digit mistyped or two transposed in SSN, NPI, DL, TIN or a phone
        rows_p, rows_k = [], []
        for t in NEAR_ID_TYPES[part]:
            cols = ["phone_own", "phone_row"] if t == "phone" else [f"id_{t}"]
            for c in cols:
                if c not in p:
                    continue
                for i, v in enumerate(p[c].to_numpy(dtype=object)):
                    if not v:
                        continue
                    v = v.partition(":")[2] if t == "dl" else v
                    for key in _deletion_keys(v):
                        rows_p.append(i); rows_k.append(f"{t}:{key}")
        rp = np.array(rows_p, dtype=np.int64)
        return _frame(rp, np.array(rows_k, dtype=object), st[rp] if len(rp) else np.array([], dtype=object))
    if rule == "street":
        return _frame(pos, p["addr_street"].to_numpy(dtype=object), st)
    first = p["first"].to_numpy(dtype=object)
    last = p["last"].to_numpy(dtype=object)
    ny = p["last_nysiis"].to_numpy(dtype=object)
    ini = np.array([f[:1] for f in first], dtype=object)
    if rule == "nysiis_initial":
        k = np.where((ny != "") & (ini != ""), ny + "|" + ini, "")
        return _frame(pos, k, st)
    if rule == "nysiis_canon_initial":
        rows_p, rows_k = [], []
        for i, (n, f, roots) in enumerate(zip(ny, first, p["first_roots"].to_numpy(dtype=object))):
            if not n or not f:
                continue
            for letter in sorted({r[:1] for r in roots.split("|") if r}):
                rows_p.append(i); rows_k.append(f"{n}|{letter}")
        return _frame(np.array(rows_p, dtype=np.int64), np.array(rows_k, dtype=object), st[rows_p] if rows_p else np.array([], dtype=object))
    if rule == "nysiis_first_missing":
        # added by the build: a surname with no first name meets every holder of the surname
        k = np.where((ny != "") & ((ini == "") if side == "x" else True), ny, "")
        return _frame(pos, k, st)
    if rule == "nysiis_last":
        # B1: the surname's sound alone (any first name: an unknown nickname, a changed first name)
        return _frame(pos, ny, st)
    if rule == "last_part_initial":
        # B1: each part of a compound surname (GARCIA-VILLANUEVA meets GARCIA) + first initial
        rows_p, rows_k = [], []
        for i, (l_, f) in enumerate(zip(last, first)):
            if not l_ or not f:
                continue
            for w_ in {x for x in re.split(r"[ \-]", l_) if len(x) > 1}:
                rows_p.append(i); rows_k.append(f"{_nysiis_c(w_)}|{f[0]}")
        rp = np.array(rows_p, dtype=np.int64)
        return _frame(rp, np.array(rows_k, dtype=object), st[rp] if len(rp) else np.array([], dtype=object))
    if rule == "first_sound_last_initial":
        # B1: a changed or badly mistyped surname: the first name's sound + the surname's initial
        k = np.array([f"{_nysiis_c(f)}|{l_[0]}" if (len(f) > 1 and l_) else "" for f, l_ in zip(first, last)],
                     dtype=object)
        return _frame(pos, k, st)
    if rule == "dob_initial":
        dob = p["dob"].to_numpy(dtype=object)
        k = np.where((dob != "") & (ini != ""), dob + "|" + ini, "")
        return _frame(pos, k, st)
    if rule == "dob":
        # B1: the date of birth alone (a changed surname and a changed first name)
        return _frame(pos, p["dob"].to_numpy(dtype=object), st)
    if rule == "swapped":
        k = np.where((first != "") & (last != ""), (first + "|" + last) if side == "x" else (last + "|" + first), "")
        return _frame(pos, k, st)
    if rule == "sn_last_first":
        k = np.where(last != "", last + " " + first, "")
        return _frame(pos, k, st)
    if rule == "sn_first_last":
        # B1: sorted on "first last", so a surname typo early in the name still meets its match
        k = np.where((last != "") & (first != ""), first + " " + last, "")
        return _frame(pos, k, st)
    al = p["org_aliases"].to_numpy(dtype=object)
    if rule in ("rare_word", "rare_word_sound"):
        # the two rarest words of each alias (B1: two, not one), and of every alias another
        # record declares as its d/b/a partner (so a trade name meets the legal name);
        # rare_word_sound: their NYSIIS codes (a typo in the rare word)
        partners = defaultdict(set)
        for q1, q2 in (dba_pairs or []):
            a1, a2 = " ".join(sorted(q1)), " ".join(sorted(q2))
            partners[a1].add(q2); partners[a2].add(q1)
        rows_p, rows_k = [], []
        for i, s in enumerate(al):
            for a in (s.split("|") if s else []):
                ws = a.split()
                sets = [ws] + [sorted(q) for q in partners.get(" ".join(sorted(ws)), ())]
                for w_ in sets:
                    if w_:
                        for t in sorted(set(w_), key=lambda t: (-rarity.org_idf(t), t))[:2]:
                            rows_p.append(i)
                            rows_k.append(t if rule == "rare_word" else (_nysiis_c(t) if len(t) >= 4 else ""))
        rp = np.array(rows_p, dtype=np.int64)
        return _frame(rp, np.array(rows_k, dtype=object), st[rp] if len(rp) else np.array([], dtype=object))
    if rule == "lead_word_state":
        k = np.array([(s.split("|")[0].split()[0] + "|" + t) if s and t else "" for s, t in zip(al, st)], dtype=object)
        return _frame(pos, k, st)
    if rule == "sn_sorted_name":
        k = np.array([" ".join(sorted(s.split("|")[0].split())) if s else "" for s in al], dtype=object)
        return _frame(pos, k, st)
    raise ValueError(rule)


def _frame(pos, key, state):
    f = pd.DataFrame({"pos": np.asarray(pos, dtype=np.int64), "key": np.asarray(key, dtype=object),
                      "state": np.asarray(state, dtype=object)})
    return f[f["key"] != ""].drop_duplicates(["pos", "key"]).reset_index(drop=True)


def plan_rule(xk, wk, rule, cap):
    """Effective keys after refining oversized keys by state. Returns (xk, wk, report rows)."""
    cx = xk["key"].value_counts()
    cw = wk["key"].value_counts()
    both = cx.index.intersection(cw.index)
    prod = cx[both] * cw[both]
    total = int(prod.sum())
    big = set(prod.index[prod > cap]) if rule not in SN_RULES else set()
    rep = {"rule": rule, "pairs_before_refine": total, "oversized_keys": len(big),
           "dropped_keys": 0, "dropped_pairs": 0, "pairs_after_refine": total, "dropped_examples": ""}
    if big:
        def refine(f):
            f = f.copy()
            m = f["key"].isin(big)
            f.loc[m, "key"] = np.where(f.loc[m, "state"] != "", f.loc[m, "key"] + "|" + f.loc[m, "state"], "")
            return f[f["key"] != ""]
        xk, wk = refine(xk), refine(wk)
        cx2, cw2 = xk["key"].value_counts(), wk["key"].value_counts()
        b2 = cx2.index.intersection(cw2.index)
        prod2 = cx2[b2] * cw2[b2]
        drop = prod2[prod2 > cap]
        if len(drop):
            xk = xk[~xk["key"].isin(drop.index)]
            wk = wk[~wk["key"].isin(drop.index)]
        rep.update(dropped_keys=int(len(drop)), dropped_pairs=int(drop.sum()),
                   pairs_after_refine=int(prod2.sum() - drop.sum()),
                   dropped_examples="; ".join(f"{k} ({v:,})" for k, v in drop.sort_values(ascending=False).head(5).items()))
    elif rule in SN_RULES:
        # exact-duplicate keys over the cap stay with the refined blocks
        over = set(prod.index[prod > cap])
        if over:
            xk = xk[~xk["key"].isin(over)]
            wk = wk[~wk["key"].isin(over)]
            rep.update(dropped_keys=len(over), dropped_pairs=int(prod[list(over)].sum()),
                       dropped_examples="left to the name blocks: " + "; ".join(sorted(over)[:5]))
    return xk.reset_index(drop=True), wk.reset_index(drop=True), rep


class CandidatePlan:
    """Keys of every rule for one part, prepared once for all chunks. `same_frame`: the
    extracted file against itself (deduplication): each unordered pair once (l < r).
    `exhaustive`: every same-type pair is a candidate; the rules still say which of them
    would also have proposed it."""

    def __init__(self, xp, wp, part, rarity, cfg, dba_pairs=None, same_frame=False, exhaustive=None):
        self.part, self.cfg, self.same = part, cfg, same_frame
        self.exhaustive = cfg.exhaustive if exhaustive is None else exhaustive
        self.rules = list(PERSON_RULES if part == "person" else BUSINESS_RULES)
        if self.exhaustive:
            self.rules.append("exhaustive")
        self.bit = {r: 1 << i for i, r in enumerate(self.rules)}
        self.keys, self.report = {}, []
        for r in self.rules:
            if r == "exhaustive":
                continue
            xk, wk, rep = plan_rule(rule_keys(xp, part, r, rarity, dba_pairs=dba_pairs),
                                    rule_keys(wp, part, r, rarity, side="w", dba_pairs=dba_pairs), r,
                                    cfg.block_pair_cap)
            rep["part"] = part
            self.report.append(rep)
            sk = None
            if r in SN_RULES:
                sk = np.sort(pd.unique(np.concatenate([xk["key"].to_numpy(dtype=object),
                                                       wk["key"].to_numpy(dtype=object)])).astype(str))
            self.keys[r] = (xk, wk, sk)
        self.n_x, self.n_w = len(xp), len(wp)

    def _join(self, r, xk, wk, sk, pos):
        """Pairs (x-side pos in `pos`, w-side pos) proposed by rule r."""
        xc = xk[np.isin(xk["pos"].to_numpy(), pos)]
        if not len(xc) or not len(wk):
            return None
        xf = pd.DataFrame({"key": xc["key"].to_numpy(dtype=object)})
        wf = pd.DataFrame({"key": wk["key"].to_numpy(dtype=object)})
        idx = rl.Index()
        if r in SN_RULES:
            idx.add(rl.index.SortedNeighbourhood("key", "key", window=self.cfg.sn_window, sorting_key_values=sk))
        else:
            idx.add(rl.index.Block("key", "key"))
        mi = idx.index(xf, wf)
        if len(mi) == 0:
            return None
        return (xc["pos"].to_numpy()[mi.get_level_values(0).to_numpy()],
                wk["pos"].to_numpy()[mi.get_level_values(1).to_numpy()])

    def chunk_positions(self, pos):
        """Candidate pairs whose extracted party is in `pos`: (l, r, rules bit mask)."""
        pos = np.unique(np.asarray(pos, dtype=np.int64))
        ls, rs, bs = [], [], []
        for r in self.rules:
            if r == "exhaustive":
                continue
            xk, wk, sk = self.keys[r]
            got = self._join(r, xk, wk, sk, pos)
            if got is not None:
                ls.append(got[0]); rs.append(got[1]); bs.append(np.full(len(got[0]), self.bit[r], dtype=np.int64))
            if self.same and r in ASYMMETRIC_RULES:
                # the other direction, so each unordered pair is found from its smaller member
                got = self._join(r, wk, xk, sk, pos)
                if got is not None:
                    ls.append(got[0]); rs.append(got[1]); bs.append(np.full(len(got[0]), self.bit[r], dtype=np.int64))
        if self.exhaustive and len(pos):
            n_r = self.n_w
            ls.append(np.repeat(pos, n_r)); rs.append(np.tile(np.arange(n_r, dtype=np.int64), len(pos)))
            bs.append(np.full(len(pos) * n_r, self.bit["exhaustive"], dtype=np.int64))
        if not ls:
            return pd.DataFrame({"l": np.array([], np.int64), "r": np.array([], np.int64),
                                 "rules": np.array([], np.int64)})
        l = np.concatenate(ls); r_ = np.concatenate(rs); b = np.concatenate(bs)
        if self.same:
            keep = l < r_
            l, r_, b = l[keep], r_[keep], b[keep]
        key = l * np.int64(1 << 32) + r_
        order = np.argsort(key, kind="stable")
        key, b = key[order], b[order]
        if not len(key):
            return pd.DataFrame({"l": np.array([], np.int64), "r": np.array([], np.int64),
                                 "rules": np.array([], np.int64)})
        starts = np.flatnonzero(np.r_[True, key[1:] != key[:-1]])
        bits = np.bitwise_or.reduceat(b, starts)
        uk = key[starts]
        return pd.DataFrame({"l": uk >> 32, "r": uk & np.int64((1 << 32) - 1), "rules": bits})

    def chunk(self, lo, hi):
        """Candidate pairs for extracted parties with positions in [lo, hi)."""
        return self.chunk_positions(np.arange(lo, hi, dtype=np.int64))

    def chunks(self, n_parties=None):
        """Position chunks covering the extracted side: chunk_size parties (blocking), or as
        many as keep a chunk near exhaustive_pairs_per_chunk pairs (exhaustive)."""
        n = self.n_x if n_parties is None else n_parties
        step = self.cfg.chunk_size
        if self.exhaustive:
            step = max(1, min(step, self.cfg.exhaustive_pairs_per_chunk // max(1, self.n_w)))
        return [(lo, min(n, lo + step)) for lo in range(0, n, step)]

    def rule_names(self, bits):
        return [",".join(r for r in self.rules if b & self.bit[r]) for b in bits]


# ---- chunk sizes from estimated pair counts ---------------------------------------------------
def estimated_pairs(plan):
    """Upper bound of the candidate pairs each extracted position proposes: per rule, the
    watchlist parties sharing one of its keys (sorted-neighbourhood rules: those within the
    window), summed over the rules before their union. It only sizes the chunks, so it never
    changes which pairs are proposed."""
    est = np.zeros(plan.n_x, dtype=np.float64)
    h = (plan.cfg.sn_window - 1) // 2
    for r in plan.rules:
        if r == "exhaustive":
            est += plan.n_w
            continue
        xk, wk, sk = plan.keys[r]
        dirs = [(xk, wk)] + ([(wk, xk)] if plan.same and r in ASYMMETRIC_RULES else [])
        for a, b in dirs:
            if not len(a) or not len(b):
                continue
            if r in SN_RULES:
                rank = pd.Series(np.arange(len(sk)), index=sk)
                cb = np.bincount(rank.reindex(b["key"].to_numpy()).to_numpy(dtype=np.int64), minlength=len(sk))
                cum = np.r_[0, np.cumsum(cb)]
                ra = rank.reindex(a["key"].to_numpy()).to_numpy(dtype=np.int64)
                n = cum[np.minimum(ra + h + 1, len(sk))] - cum[np.maximum(ra - h, 0)]
            else:
                n = a["key"].map(b["key"].value_counts()).fillna(0).to_numpy(dtype=np.float64)
            np.add.at(est, a["pos"].to_numpy(dtype=np.int64), n)
    return est


def budget_cuts(weights, budget):
    """Consecutive ranges [lo, hi) whose summed weight stays near `budget` (at least one item
    each): the chunks a parallel pass hands to its workers."""
    cuts, acc, lo = [], 0.0, 0
    for i, w in enumerate(np.asarray(weights, dtype=np.float64)):
        if acc > 0 and acc + w > budget:
            cuts.append((lo, i))
            lo, acc = i, 0.0
        acc += w
    if lo < len(weights):
        cuts.append((lo, len(weights)))
    return cuts


## Noise engine

Noisy copies of input rows, driven by `mappings/simulation_noise.csv` (a pessimistic table,
for review). The same engine makes the simulated true-match pairs for m (step 10) and the
built-in synthetic test set. Each copy records which noises hit it.

**Where Splink would do better.** Splink has no simulator; its answer to missing labels is
EM plus the user's own labels. Simulation here is a last-resort source of m and is flagged
as such wherever it is used.

In [ ]:
_TYPO_ALPHA = "ABCDEFGHIJKLMNOPRSTUVWY"
SUFFIX_WORDS = {"INC", "LLC", "PC", "PLLC", "CORP", "CO", "LTD", "LLP", "PA", "INCORPORATED",
                "CORPORATION", "COMPANY", "P.C.", "L.L.C.", "INC."}
ABBREV_OF = {v: k for k, v in ORG_ABBREV.items() if len(k) >= 3}

# A small fictional-use geography: real city / state / ZIP-prefix triples, fictional streets.
GEOGRAPHY = [
    ("BROOKLYN", "NY", "112"), ("QUEENS", "NY", "113"), ("BRONX", "NY", "104"), ("ALBANY", "NY", "122"),
    ("NEWARK", "NJ", "071"), ("PATERSON", "NJ", "075"), ("PHILADELPHIA", "PA", "191"), ("PITTSBURGH", "PA", "152"),
    ("BOSTON", "MA", "021"), ("WORCESTER", "MA", "016"), ("HARTFORD", "CT", "061"), ("PROVIDENCE", "RI", "029"),
    ("BALTIMORE", "MD", "212"), ("RICHMOND", "VA", "232"), ("CHARLOTTE", "NC", "282"), ("ATLANTA", "GA", "303"),
    ("MIAMI", "FL", "331"), ("ORLANDO", "FL", "328"), ("TAMPA", "FL", "336"), ("HIALEAH", "FL", "330"),
    ("DETROIT", "MI", "482"), ("CLEVELAND", "OH", "441"), ("COLUMBUS", "OH", "432"), ("CHICAGO", "IL", "606"),
    ("MILWAUKEE", "WI", "532"), ("MINNEAPOLIS", "MN", "554"), ("SAINT LOUIS", "MO", "631"), ("MEMPHIS", "TN", "381"),
    ("NEW ORLEANS", "LA", "701"), ("HOUSTON", "TX", "770"), ("DALLAS", "TX", "752"), ("SAN ANTONIO", "TX", "782"),
    ("PHOENIX", "AZ", "850"), ("DENVER", "CO", "802"), ("LAS VEGAS", "NV", "891"), ("LOS ANGELES", "CA", "900"),
    ("SAN DIEGO", "CA", "921"), ("FRESNO", "CA", "937"), ("SEATTLE", "WA", "981"), ("PORTLAND", "OR", "972"),
]
STREET_NAMES = ["MAPLE", "OAK", "CEDAR", "PINE", "ELM", "WILLOW", "BIRCH", "SPRUCE", "CHESTNUT", "WALNUT",
                "HICKORY", "MAGNOLIA", "DOGWOOD", "SYCAMORE", "LAUREL", "JUNIPER", "HAWTHORN", "ASPEN",
                "LINDEN", "POPLAR", "CYPRESS", "HOLLY", "IVY", "ORCHARD", "MEADOW", "PRAIRIE", "RIDGE",
                "SUMMIT", "VALLEY", "HILLSIDE", "LAKEVIEW", "RIVERSIDE", "BAYVIEW", "HARBOR", "SUNSET",
                "SUNRISE", "PARK", "GARDEN", "CHURCH", "SCHOOL", "MILL", "BRIDGE", "MARKET", "UNION",
                "LIBERTY", "FRANKLIN", "WASHINGTON", "JEFFERSON", "MADISON", "MONROE", "JACKSON", "LINCOLN",
                "GRANT", "HAMILTON", "ADAMS", "CLINTON", "KENNEDY", "HIGHLAND", "FOREST", "GREEN", "SPRING",
                "CENTER", "MAIN", "BROAD", "HIGH", "WATER", "FRONT", "CANAL", "STATION", "RAILROAD", "COLONIAL",
                "EUCLID", "KINGS", "QUEENS", "WINDSOR", "OXFORD", "CAMBRIDGE", "STRATFORD", "ASHFORD",
                "BRADFORD", "CLIFTON", "FAIRVIEW", "GLENWOOD", "KENWOOD", "LAKEWOOD", "MAPLEWOOD", "OAKWOOD",
                "ROSEWOOD", "WOODLAND", "WESTGATE", "EASTGATE", "NORTHGATE", "SOUTHGATE", "CROSSWIND",
                "WINDMILL", "BLUEBIRD", "CARDINAL", "ORIOLE", "HERON", "FALCON", "EAGLE", "HAWK", "SPARROW",
                "1ST", "2ND", "3RD", "4TH", "5TH", "6TH", "7TH", "8TH", "9TH", "10TH", "12TH", "14TH", "21ST",
                "34TH", "45TH", "79TH", "110TH"]
STREET_SUFFIXES = ["ST", "AVE", "RD", "DR", "LN", "CT", "BLVD", "PL", "WAY", "TER"]
SPECIALTIES = ["CHIROPRACTIC", "PHYSICAL THERAPY", "ACUPUNCTURE", "ORTHOPEDICS", "GENERAL PRACTICE",
               "PAIN MANAGEMENT", "RADIOLOGY", "NEUROLOGY", "PSYCHOLOGY", "PODIATRY", "INTERNAL MEDICINE",
               "OSTEOPATHY", "DENTISTRY", "PHARMACY", "NURSING"]


def luhn_npi(prefix9):
    """A valid NPI from nine digits."""
    total = 0
    for i, ch in enumerate(reversed("80840" + prefix9)):
        v = int(ch)
        if i % 2 == 0:
            v *= 2
            if v > 9:
                v -= 9
        total += v
    return prefix9 + str((10 - total % 10) % 10)


def vin_with_check(body16, rng):
    """A VIN with a correct check digit from 16 characters (position 9 is replaced)."""
    v = list(body16[:8] + "0" + body16[8:16])
    r = sum(_VIN_MAP[c] * w for c, w in zip(v, _VIN_W)) % 11
    v[8] = "X" if r == 10 else str(r)
    return "".join(v)


class Fake:
    """Fictional values with realistic frequencies (names drawn from the Census and SSA
    tables), deterministic under one numpy Generator."""

    def __init__(self, ref, rng, n_surnames=30000, n_firsts=3000):
        self.rng = rng
        sn = sorted((ref.get("surnames") or {"SMITH": 1}).items(), key=lambda kv: -kv[1])[:n_surnames]
        fn = sorted((ref.get("first_names") or {"JOHN": 1}).items(), key=lambda kv: -kv[1])[:n_firsts]
        self.surnames = np.array([k for k, _ in sn], dtype=object)
        self.sw = np.array([v for _, v in sn], dtype=float); self.sw /= self.sw.sum()
        self.firsts = np.array([k for k, _ in fn], dtype=object)
        self.fw = np.array([v for _, v in fn], dtype=float); self.fw /= self.fw.sum()
        nick = ref.get("nicknames")
        self.nick_of = defaultdict(list)
        if nick is not None:
            for a, b in zip(nick["name1"].map(fold), nick["name2"].map(fold)):
                self.nick_of[a].append(b)

    def _draw(self, values, cum, n):
        """Weighted draws by inverse CDF (rng.choice with p re-sums the weights on every call)."""
        k = 1 if n is None else n
        idx = np.searchsorted(cum, self.rng.random(k) * cum[-1], side="right")
        out = values[np.minimum(idx, len(values) - 1)]
        return out[0] if n is None else out

    def surname(self, n=None):
        if not hasattr(self, "_scum"):
            self._scum = np.cumsum(self.sw)
        return self._draw(self.surnames, self._scum, n)

    def first(self, n=None):
        if not hasattr(self, "_fcum"):
            self._fcum = np.cumsum(self.fw)
        return self._draw(self.firsts, self._fcum, n)

    def digits(self, k):
        return "".join(str(x) for x in self.rng.integers(0, 10, k))

    def ssn(self):
        while True:
            s = f"{self.rng.integers(1, 899):03d}{self.rng.integers(1, 100):02d}{self.rng.integers(1, 10000):04d}"
            if norm_ssn(s)[1]:
                return s

    def tin(self):
        while True:
            s = f"{self.rng.integers(10, 99):02d}{self.rng.integers(0, 10 ** 7):07d}"
            if norm_tin(s)[1]:
                return s

    def npi(self):
        return luhn_npi(str(self.rng.integers(1, 3)) + self.digits(8))

    def phone(self):
        while True:
            s = f"{self.rng.integers(201, 990)}{self.rng.integers(200, 999)}{self.rng.integers(0, 10000):04d}"
            if norm_phone(s)[1]:
                return s

    def dob(self, lo=1940, hi=2002):
        y = int(self.rng.integers(lo, hi + 1)); m = int(self.rng.integers(1, 13)); d = int(self.rng.integers(1, 29))
        return f"{y:04d}-{m:02d}-{d:02d}"

    def address(self, state=None):
        g = [x for x in GEOGRAPHY if x[1] == state] if state else GEOGRAPHY
        if not g:
            g = GEOGRAPHY
        city, st, z3 = g[int(self.rng.integers(0, len(g)))]
        return {"street_number": str(int(self.rng.integers(1, 9999))), "street_direction": "",
                "street_name": str(self.rng.choice(STREET_NAMES)), "street_type": str(self.rng.choice(STREET_SUFFIXES)),
                "unit": (str(int(self.rng.integers(1, 40))) if self.rng.random() < 0.3 else ""),
                "city": city, "state": st, "zip": z3 + f"{int(self.rng.integers(0, 100)):02d}"}

    def email(self, first, last):
        dom = str(self.rng.choice(["mailbox.test", "post.test", "inbox.test", "webmail.test"]))
        return f"{first.lower()}.{last.lower()}{int(self.rng.integers(1, 999))}@{dom}".replace(" ", "")

    def dl(self):
        return str(self.rng.choice(list("ABCDEFGHJKLMNPRSTWY"))) + self.digits(7)

    def vin(self):
        chars = "ABCDEFGHJKLMNPRSTUVWXYZ0123456789"
        body = "".join(self.rng.choice(list(chars), 16))
        return vin_with_check(body, self.rng)

    def plate(self):
        return "".join(self.rng.choice(list("ABCDEFGHJKLMNPRSTUVWXYZ"), 3)) + self.digits(4)


def _typo(s, rng):
    s = str(s)
    if len(s) < 3:
        return s
    i = int(rng.integers(1, len(s) - 1))
    op = int(rng.integers(0, 4))
    c = _TYPO_ALPHA[int(rng.integers(0, len(_TYPO_ALPHA)))]
    if op == 0:
        return s[:i] + c + s[i + 1:]
    if op == 1:
        return s[:i] + s[i + 1:]
    if op == 2:
        return s[:i] + c + s[i:]
    return s[:i - 1] + s[i] + s[i - 1] + s[i + 1:]


def _digit_typo(s, rng):
    d = [i for i, ch in enumerate(s) if ch.isdigit()]
    if not d:
        return s
    i = d[int(rng.integers(0, len(d)))]
    new = str((int(s[i]) + int(rng.integers(1, 10))) % 10)
    return s[:i] + new + s[i + 1:]


def _transpose(s, rng):
    d = [i for i in range(len(s) - 1) if s[i].isdigit() and s[i + 1].isdigit() and s[i] != s[i + 1]]
    if not d:
        return s
    i = d[int(rng.integers(0, len(d)))]
    return s[:i] + s[i + 1] + s[i] + s[i + 2:]


def apply_noise(rows, noise, fake, rng, scale=1.0, id_prefix="sim"):
    """Noisy copies of input-schema rows. `noise`: the simulation_noise table; `scale`
    multiplies every rate. Returns (copies, log) where log lists the noises per copy."""
    rate = {n: float(r) * scale for n, r in zip(noise["noise"], noise["rate"])}
    hit = lambda n: rng.random() < rate.get(n, 0.0)
    cats = ["medical", "legal", "repair shop", "witness", "claimant", "other"]
    out, log = [], []
    for i, r in enumerate(rows.to_dict("records")):
        r = dict(r)
        applied = []

        def did(n):
            applied.append(n)
        has_person = bool(r.get("first_name") or r.get("last_name"))
        if has_person:
            last, first = r.get("last_name", ""), r.get("first_name", "")
            if last and hit("surname_typo"):
                last = _typo(last, rng); did("surname_typo")
            if last and re.search(r"[ \-]", last) and hit("surname_compound_dropped"):
                last = re.split(r"[ \-]", last)[0]; did("surname_compound_dropped")
            if last and hit("surname_change"):
                last = str(fake.surname()).title(); did("surname_change")
            f = fold(first)
            if f and hit("first_nickname") and fake.nick_of.get(f):
                opts = fake.nick_of[f]
                first = opts[int(rng.integers(0, len(opts)))].title(); did("first_nickname")
            elif f and hit("first_initial_only"):
                first = f[0]; did("first_initial_only")
            elif f and hit("first_typo"):
                first = _typo(first, rng); did("first_typo")
            elif f and hit("first_missing"):
                first = ""; did("first_missing")
            if first and last and hit("names_swapped"):
                first, last = last, first; did("names_swapped")
            r["first_name"], r["last_name"] = first, last
            mid = r.get("middle_name", "")
            if mid:
                if hit("middle_dropped"):
                    mid = ""; did("middle_dropped")
                elif hit("middle_initial"):
                    mid = mid[0]; did("middle_initial")
            r["middle_name"] = mid
            dob = r.get("dob", "")
            if dob:
                y, m, d = dob[:4], dob[5:7], dob[8:10]
                if hit("dob_missing"):
                    dob = ""; did("dob_missing")
                elif hit("dob_placeholder"):
                    dob = "1900-01-01"; did("dob_placeholder")
                elif int(d) <= 12 and d != m and hit("dob_day_month_swap"):
                    dob = f"{y}-{d}-{m}"; did("dob_day_month_swap")
                elif hit("dob_digit_typo"):
                    for _ in range(10):
                        cand = _digit_typo(dob.replace("-", ""), rng)
                        iso, ok, _r = norm_dob(cand)
                        if ok:
                            dob = iso; break
                    did("dob_digit_typo")
                elif hit("dob_year_off"):
                    dob = f"{int(y) + (1 if rng.random() < 0.5 else -1):04d}-{m}-{d}"; did("dob_year_off")
                r["dob"] = dob
            if r.get("provider_specialty") and hit("specialty_coarser"):
                r["provider_specialty"] = "PHYSICIAN"; did("specialty_coarser")
        for col in ("ssn", "provider_npi", "driver_license_number", "tin"):
            v = r.get(col, "")
            if v:
                if hit("id_digit_typo"):
                    r[col] = _digit_typo(v, rng); did(f"id_digit_typo:{col}")
                elif hit("id_transposition"):
                    r[col] = _transpose(v, rng); did(f"id_transposition:{col}")
        if r.get("street_name") or r.get("zip") or r.get("state"):
            if hit("moved_within_state"):
                r.update(fake.address(r.get("state") or None)); did("moved_within_state")
            elif hit("moved_other_state"):
                others = [g[1] for g in GEOGRAPHY if g[1] != r.get("state")]
                r.update(fake.address(others[int(rng.integers(0, len(others)))])); did("moved_other_state")
            if r.get("unit") and hit("unit_dropped"):
                r["unit"] = ""; did("unit_dropped")
            if r.get("zip") and hit("zip_typo"):
                r["zip"] = _digit_typo(r["zip"], rng); did("zip_typo")
        for col in ("home_phone", "work_phone"):
            if r.get(col) and hit("phone_changed"):
                r[col] = fake.phone(); did(f"phone_changed:{col}")
        if r.get("email") and hit("email_changed"):
            r["email"] = fake.email(fold(r.get("first_name") or "info") or "info",
                                    fold(r.get("last_name") or "office") or "office"); did("email_changed")
        b = r.get("business_name", "")
        if b:
            if _DBA_RE.search(b) and hit("business_dba_used"):
                b = _DBA_RE.split(b)[-1].strip(" ,"); did("business_dba_used")
            words = b.split()
            if len(words) > 1 and words[-1].upper().strip(",.") in SUFFIX_WORDS and hit("business_suffix_dropped"):
                b = " ".join(words[:-1]).strip(" ,"); did("business_suffix_dropped")
            words = b.split()
            ab = [j for j, w in enumerate(words) if w.upper() in ABBREV_OF]
            if ab and hit("business_abbreviated"):
                j = ab[int(rng.integers(0, len(ab)))]
                words[j] = ABBREV_OF[words[j].upper()].title(); b = " ".join(words); did("business_abbreviated")
            if hit("business_typo"):
                words = b.split()
                j = int(rng.integers(0, len(words)))
                if len(words[j]) >= 5:
                    words[j] = _typo(words[j], rng); b = " ".join(words); did("business_typo")
            r["business_name"] = b
        if r.get("category") and hit("category_disagrees"):
            r["category"] = str(rng.choice([c for c in cats if c != r["category"]])); did("category_disagrees")
        r["record_id"] = f"{id_prefix}:{r['record_id']}"
        out.append(r)
        log.append(";".join(applied))
    return pd.DataFrame(out, columns=list(rows.columns)), log

## Synthetic data

A fictional test set with a truth file, deterministic under the run seed.

* **Universe**: persons and businesses with canonical details; names drawn with Census and
  SSA frequencies (so "Smith" is as common as it is), every identifier fictional and valid
  (NPIs pass Luhn, VINs their check digit).
* **Watchlist rows**: 60% of the universe, lightly noised, ~10% listed twice (watchlist
  duplicates), plus watchlist-only parties up to the requested size.
* **Extracted rows**: 70% of the universe (so 40% of them are not listed), null-heavy, noised
  with the simulation table at half its rates, many parties in several claims (unmerged
  duplicates), grouped into claims and notes.
* **Edge cases**: ~45 hand-written cases with the expected best watchlist row, basis, veto
  and levels, listed in the table `EDGE_CASES`.

Truth is written beside the inputs and read only by diagnostics and self-tests, never by the
pipeline.

**Where Splink would do better.** Splink ships labelled demo datasets (febrl, historical
persons) with known truth; they are real benchmarks. This set is ours and shaped by our own
noise table, so results on it show that the code does what it says, not how well it would do
on client data.

In [ ]:
def blank_row(rid, **kw):
    r = {c: "" for c in SCHEMA}
    r["record_id"] = rid
    r.update({k: ("" if v is None else str(v)) for k, v in kw.items()})
    return r


def _npi(n):
    return luhn_npi(f"19{n:07d}")


def _ssn(n):
    return f"{200 + n % 600:03d}{10 + n % 80:02d}{1000 + n:04d}"


def _tin(n):
    return f"45{n:07d}"


def _addr(num, street, stype, city, state, zp, unit=""):
    return {"street_number": num, "street_name": street, "street_type": stype, "unit": unit,
            "city": city, "state": state, "zip": zp}


A_NY = _addr("118", "ORCHARD", "ST", "BROOKLYN", "NY", "11211")
A_NY2 = _addr("42", "HERON", "AVE", "QUEENS", "NY", "11354")
A_CA = _addr("901", "SUNSET", "BLVD", "LOS ANGELES", "CA", "90026")
A_IL = _addr("77", "CANAL", "ST", "CHICAGO", "IL", "60606", "4")
A_FL = _addr("3100", "BAYVIEW", "DR", "MIAMI", "FL", "33133")


def edge_cases():
    """Hand-written cases: (case, description, extracted rows, watchlist rows, expectations).
    An expectation names the extracted party (record_id, part), the watchlist record expected
    to be its best candidate (rank 1) or merely visible, and the basis / veto / levels."""
    C = []

    def case(cid, desc, x, w, expect):
        C.append({"case": cid, "desc": desc, "x": x, "w": w, "expect": expect})

    E = lambda **k: k
    case("E01", "person + business on one row, both identifier-linked",
         [blank_row("E01X", claim_id="CE01", note_id="NE01", first_name="Harold", last_name="Vantongeren",
                    provider_npi=_npi(101), business_name="Vantongeren Chiropractic PC", tin=_tin(101), **A_NY)],
         [blank_row("E01W1", first_name="Harold", last_name="Vantongeren", provider_npi=_npi(101), dob="1961-03-14", **A_NY),
          blank_row("E01W2", business_name="Vantongeren Chiropractic P.C.", tin=_tin(101), **A_NY)],
         [E(x="E01X", part="person", w="E01W1", basis="identifier", rank=1),
          E(x="E01X", part="business", w="E01W2", basis="identifier", rank=1)])
    case("E02", "a TIN shared by two businesses in the extracted file",
         [blank_row("E02X1", claim_id="CE02", note_id="NE02", business_name="Silverline Medical Supply", tin=_tin(102)),
          blank_row("E02X2", claim_id="CE02", note_id="NE02b", business_name="Quillfeather Diagnostics", tin=_tin(102))],
         [blank_row("E02W", business_name="Silverline Medical Supply Inc", tin=_tin(102), **A_FL)],
         [E(x="E02X1", part="business", w="E02W", basis="identifier", rank=1),
          E(x="E02X2", part="business", w="E02W", basis="identifier")])
    case("E03", "sibling companies: a shared head word, distinctive words on both sides",
         [blank_row("E03X", claim_id="CE03", note_id="NE03", business_name="Allport Indemnity Company", **A_NY2)],
         [blank_row("E03W1", business_name="Allport Indemnity Co"),
          blank_row("E03W2", business_name="Allport Casualty Surety Company", **A_NY)],
         [E(x="E03X", part="business", w="E03W1", rank=1, levels={"org": "exact"}),
          E(x="E03X", part="business", w="E03W2", levels={"org": "sibling"}, if_visible=True)])
    s4 = _ssn(104)
    case("E04", "one SSN under three names in the extracted file",
         [blank_row("E04X1", claim_id="CE04", note_id="NE04", first_name="Rosalind", last_name="Achterberg", ssn=s4),
          blank_row("E04X2", claim_id="CE04", note_id="NE04", first_name="Tobias", last_name="Wenceslas", ssn=s4),
          blank_row("E04X3", claim_id="CE04b", note_id="NE04b", first_name="Imelda", last_name="Fairweather", ssn=s4)],
         [blank_row("E04W", first_name="Rosalind", last_name="Achterberg", ssn=s4, dob="1970-02-02")],
         [E(x="E04X1", part="person", w="E04W", basis="identifier", rank=1),
          E(x="E04X2", part="person", w="E04W", basis="identifier")])
    case("E05", "junk SSN on both sides is never an identifier",
         [blank_row("E05X", claim_id="CE05", note_id="NE05", first_name="Gertrude", last_name="Oyelaran", ssn="123-45-6789", dob="1958-07-21")],
         [blank_row("E05W", first_name="Gertrude", last_name="Oyelaran", ssn="123456789", dob="1958-07-21")],
         [E(x="E05X", part="person", w="E05W", basis="dob", rank=1, levels={"ssn": "empty"})])
    case("E06", "two different single-holder SSNs veto",
         [blank_row("E06X", claim_id="CE06", note_id="NE06", first_name="Mortimer", last_name="Quackenbush", ssn=_ssn(106), dob="1966-11-02")],
         [blank_row("E06W", first_name="Mortimer", last_name="Quackenbush", ssn=_ssn(206), dob="1966-11-02")],
         [E(x="E06X", part="person", w="E06W", veto="ssn")])
    case("E07", "Bill meets William",
         [blank_row("E07X", claim_id="CE07", note_id="NE07", first_name="Bill", last_name="Szczepanski", dob="1972-05-30")],
         [blank_row("E07W", first_name="William", last_name="Szczepanski", dob="1972-05-30")],
         [E(x="E07X", part="person", w="E07W", basis="dob", rank=1, levels={"name": "first_nick"})])
    case("E08", "first-name typo, same address",
         [blank_row("E08X", claim_id="CE08", note_id="NE08", first_name="Jonathon", last_name="Kowalczyk", **A_NY2)],
         [blank_row("E08W", first_name="Jonathan", last_name="Kowalczyk", **A_NY2)],
         [E(x="E08X", part="person", w="E08W", basis="address", rank=1, levels={"address": "exact"})])
    case("E09", "first and last swapped",
         [blank_row("E09X", claim_id="CE09", note_id="NE09", first_name="Nwachukwu", last_name="Chidiebere", dob="1980-01-19")],
         [blank_row("E09W", first_name="Chidiebere", last_name="Nwachukwu", dob="1980-01-19")],
         [E(x="E09X", part="person", w="E09W", basis="dob", rank=1, levels={"name": "swapped"})])
    case("E10", "initial only, owned phone agrees",
         [blank_row("E10X", claim_id="CE10", note_id="NE10", first_name="R.", last_name="Featherstonehaugh", home_phone="(718) 392-4410")],
         [blank_row("E10W", first_name="Rupert", last_name="Featherstonehaugh", home_phone="7183924410")],
         [E(x="E10X", part="person", w="E10W", basis="identifier", rank=1, levels={"name": "initial_agrees", "phone": "exact_owned_single"})])
    case("E11", "a claim row with only a TIN",
         [blank_row("E11X", claim_id="CE11", note_id="NE11", tin=_tin(111))],
         [blank_row("E11W", business_name="Marchetti Towing LLC", tin=_tin(111), **A_NY)],
         [E(x="E11X", part="business", w="E11W", basis="identifier", rank=1)])
    case("E12", "a rare exact name and nothing else: visible as name_only",
         [blank_row("E12X", claim_id="CE12", note_id="NE12", first_name="Evangelina", last_name="Throckmorton")],
         [blank_row("E12W", first_name="Evangelina", last_name="Throckmorton", dob="1949-09-09", **A_CA)],
         [E(x="E12X", part="person", w="E12W", basis="name_only", rank=1)])
    case("E13", "John Smith and nothing else: every John Smith visible, name_only",
         [blank_row("E13X", claim_id="CE13", note_id="NE13", first_name="John", last_name="Smith")],
         [blank_row(f"E13W{i}", first_name="John", last_name="Smith", dob=f"19{50 + i}-01-0{i}") for i in (1, 2, 3)],
         [E(x="E13X", part="person", w=f"E13W{i}", basis="name_only") for i in (1, 2, 3)])
    case("E14", "day and month swapped",
         [blank_row("E14X", claim_id="CE14", note_id="NE14", first_name="Philippa", last_name="Gundersen", dob="1975-04-09")],
         [blank_row("E14W", first_name="Philippa", last_name="Gundersen", dob="1975-09-04")],
         [E(x="E14X", part="person", w="E14W", basis="dob", rank=1, levels={"dob": "swap_or_typo"})])
    case("E15", "moved to another state, SSN agrees",
         [blank_row("E15X", claim_id="CE15", note_id="NE15", first_name="Cornelius", last_name="Abernethy", ssn=_ssn(115), **A_NY)],
         [blank_row("E15W", first_name="Cornelius", last_name="Abernethy", ssn=_ssn(115), **A_CA)],
         [E(x="E15X", part="person", w="E15W", basis="identifier", rank=1, levels={"address": "differs"})])
    case("E16", "given category conflicts with an NPI",
         [blank_row("E16X", claim_id="CE16", note_id="NE16", category="legal", first_name="Winifred", last_name="Castellano", provider_npi=_npi(116))],
         [blank_row("E16W", first_name="Winifred", last_name="Castellano", provider_npi=_npi(116))],
         [E(x="E16X", part="person", w="E16W", basis="identifier", rank=1, party={"category_mismatch": True, "category": "medical"})])
    case("E17", "keyword business: repair shop",
         [blank_row("E17X", claim_id="CE17", note_id="NE17", business_name="Ramirez Auto Body & Collision")],
         [blank_row("E17W", business_name="Ramirez Auto Body and Collision Inc")],
         [E(x="E17X", part="business", w="E17W", rank=1, levels={"org": "exact"},
            party={"category_inferred": "repair shop"})])
    v18 = vin_with_check("1FTFW1E5JFA12345", None)
    case("E18", "VIN agrees",
         [blank_row("E18X", claim_id="CE18", note_id="NE18", first_name="Desmond", last_name="Okonkwo-Lind", vin=v18)],
         [blank_row("E18W", first_name="Desmond", last_name="Okonkwo-Lind", vin=v18)],
         [E(x="E18X", part="person", w="E18W", basis="identifier", rank=1)])
    case("E19", "a clinic phone shared by two doctors is not an identifier",
         [blank_row("E19X1", claim_id="CE19", note_id="NE19", first_name="Anneliese", last_name="Borowczyk", work_phone="3124640199", business_name="Northgate Spine Center"),
          blank_row("E19X2", claim_id="CE19", note_id="NE19", first_name="Tiberius", last_name="Maclean", work_phone="312-464-0199", business_name="Northgate Spine Center")],
         [blank_row("E19W", first_name="Anneliese", last_name="Borowczyk", work_phone="3124640199")],
         [E(x="E19X2", part="person", w="E19W", basis_not="identifier", levels={"phone": "exact_shared"})])
    case("E20", "co-party: the tied businesses share a TIN",
         [blank_row("E20X", claim_id="CE20", note_id="NE20", first_name="Leopold", last_name="Anyanwu", business_name="Anyanwu Family Medicine PC", tin=_tin(120))],
         [blank_row("E20W", first_name="Leopold", last_name="Anyanwu", business_name="Anyanwu Family Medicine PC", tin=_tin(120))],
         [E(x="E20X", part="business", w="E20W", basis="identifier", rank=1),
          E(x="E20X", part="person", w="E20W", basis="co_party", rank=1, levels={"co_party": "anchored"})])
    case("E21", "co-party needs an anchor: name-only businesses vouch for nothing",
         [blank_row("E21X", claim_id="CE21", note_id="NE21", first_name="Octavia", last_name="Pellegrini", business_name="Pellegrini Wellness")],
         [blank_row("E21W", first_name="Octavia", last_name="Pellegrini", business_name="Pellegrini Wellness")],
         [E(x="E21X", part="person", w="E21W", basis="name_only", rank=1)])
    case("E22", "watchlist duplicates: the same person listed twice",
         [blank_row("E22X", claim_id="CE22", note_id="NE22", first_name="Ingrid", last_name="Solberg-Haas", provider_npi=_npi(122))],
         [blank_row("E22W1", first_name="Ingrid", last_name="Solberg-Haas", provider_npi=_npi(122), **A_IL),
          blank_row("E22W2", first_name="Ingrid", last_name="Solberg Haas", provider_npi=_npi(122), **A_FL)],
         [E(x="E22X", part="person", w="E22W1", basis="identifier", true=True),
          E(x="E22X", part="person", w="E22W2", basis="identifier", true=True)])
    case("E23", "an invalid NPI is never an identifier",
         [blank_row("E23X", claim_id="CE23", note_id="NE23", first_name="Fenwick", last_name="Oduya", provider_npi="1234567890")],
         [blank_row("E23W", first_name="Fenwick", last_name="Oduya", provider_npi="1234567890")],
         [E(x="E23X", part="person", w="E23W", basis="name_only", rank=1, levels={"npi": "empty"})])
    case("E24", "an empty row yields no party and is reported",
         [blank_row("E24X", claim_id="CE24", note_id="NE24", city="BROOKLYN")], [],
         [E(x="E24X", part="none", empty_row=True)])
    case("E25", "one part of a compound surname dropped",
         [blank_row("E25X", claim_id="CE25", note_id="NE25", first_name="Maria", last_name="Garcia-Villanueva", dob="1983-06-17")],
         [blank_row("E25W", first_name="Maria", last_name="Garcia", dob="1983-06-17")],
         [E(x="E25X", part="person", w="E25W", basis="dob", rank=1, levels={"name": "last_close_first_agrees"})])
    case("E26", "surname changed, SSN and DOB agree",
         [blank_row("E26X", claim_id="CE26", note_id="NE26", first_name="Beatrix", last_name="Holloway", ssn=_ssn(126), dob="1979-12-01")],
         [blank_row("E26W", first_name="Beatrix", last_name="Lindqvist", ssn=_ssn(126), dob="1979-12-01")],
         [E(x="E26X", part="person", w="E26W", basis="identifier", rank=1, levels={"name": "else"})])
    case("E27", "two driver licences of one state veto",
         [blank_row("E27X", claim_id="CE27", note_id="NE27", first_name="Casimir", last_name="Vandermeer", driver_license_number="D2700001", driver_license_state="NY")],
         [blank_row("E27W", first_name="Casimir", last_name="Vandermeer", driver_license_number="K5839204", driver_license_state="NY")],
         [E(x="E27X", part="person", w="E27W", veto="dl")])
    case("E28", "driver licences of two states do not veto",
         [blank_row("E28X", claim_id="CE28", note_id="NE28", first_name="Rosamund", last_name="Ekwueme", driver_license_number="D2800001", driver_license_state="NY")],
         [blank_row("E28W", first_name="Rosamund", last_name="Ekwueme", driver_license_number="E2800002", driver_license_state="NJ")],
         [E(x="E28X", part="person", w="E28W", veto="", rank=1)])
    case("E29", "email agrees",
         [blank_row("E29X", claim_id="CE29", note_id="NE29", first_name="Thaddeus", last_name="Blackwood", email="T.Blackwood@Mailbox.test")],
         [blank_row("E29W", first_name="Thaddeus", last_name="Blackwood", email="t.blackwood@mailbox.test")],
         [E(x="E29X", part="person", w="E29W", basis="identifier", rank=1)])
    case("E30", "a placeholder DOB is no evidence",
         [blank_row("E30X", claim_id="CE30", note_id="NE30", first_name="Ottoline", last_name="Ferrante", dob="1900-01-01")],
         [blank_row("E30W", first_name="Ottoline", last_name="Ferrante", dob="01/01/1900")],
         [E(x="E30X", part="person", w="E30W", basis="name_only", rank=1, levels={"dob": "empty"})])
    case("E31", "middle initials differ, DOB agrees",
         [blank_row("E31X", claim_id="CE31", note_id="NE31", first_name="Lucius", middle_name="J", last_name="Harrowgate", dob="1964-08-08")],
         [blank_row("E31W", first_name="Lucius", middle_name="K", last_name="Harrowgate", dob="1964-08-08")],
         [E(x="E31X", part="person", w="E31W", basis="dob", rank=1, levels={"middle": "differs"})])
    case("E32", "the trade name of a declared d/b/a",
         [blank_row("E32X", claim_id="CE32", note_id="NE32", business_name="Kestrel Imaging LLC d/b/a Harborview Radiology")],
         [blank_row("E32W", business_name="Harborview Radiology")],
         [E(x="E32X", part="business", w="E32W", rank=1, levels={"org": "exact"})])
    case("E33", "a d/b/a declared by another record",
         [blank_row("E33X", claim_id="CE33", note_id="NE33", business_name="Tamarind Wellness Group"),
          blank_row("E33D", claim_id="CE33b", note_id="NE33b", business_name="Juniper Holistic Center d/b/a Tamarind Wellness Group")],
         [blank_row("E33W", business_name="Juniper Holistic Center")],
         [E(x="E33X", part="business", w="E33W", levels={"org": "dba"}, true=True)])
    case("E34", "abbreviations expanded",
         [blank_row("E34X", claim_id="CE34", note_id="NE34", business_name="Lakeshore Med Ctr")],
         [blank_row("E34W", business_name="Lakeshore Medical Center Inc")],
         [E(x="E34X", part="business", w="E34W", rank=1, levels={"org": "exact"})])
    case("E35", "a short form of the name",
         [blank_row("E35X", claim_id="CE35", note_id="NE35", business_name="Pinecrest Orthopedic")],
         [blank_row("E35W", business_name="Pinecrest Orthopedic Rehabilitation")],
         [E(x="E35X", part="business", w="E35W", rank=1, levels={"org": "short_form"})])
    case("E36", "name plus city and specialty: contextual",
         [blank_row("E36X", claim_id="CE36", note_id="NE36", first_name="Ignatius", last_name="Wojciechowski", provider_specialty="Chiropractor", city="Chicago", state="IL")],
         [blank_row("E36W", first_name="Ignatius", last_name="Wojciechowski", provider_specialty="CHIROPRACTIC", city="CHICAGO", state="IL")],
         [E(x="E36X", part="person", w="E36W", basis="contextual", rank=1)])
    s37 = _ssn(137)
    s37n = s37[:4] + str((int(s37[4]) + 1) % 10) + s37[5:]
    case("E37", "one SSN digit mistyped: near, no veto",
         [blank_row("E37X", claim_id="CE37", note_id="NE37", first_name="Serafina", last_name="Lindgren", ssn=s37n, dob="1990-03-03")],
         [blank_row("E37W", first_name="Serafina", last_name="Lindgren", ssn=s37, dob="1990-03-03")],
         [E(x="E37X", part="person", w="E37W", veto="", rank=1, levels={"ssn": "near"})])
    s38 = _ssn(138)
    s38t = s38[:6] + s38[7] + s38[6] + s38[8:]
    case("E38", "two SSN digits transposed: near, no veto",
         [blank_row("E38X", claim_id="CE38", note_id="NE38", first_name="Augustin", last_name="Merriweather", ssn=s38t, dob="1955-10-10")],
         [blank_row("E38W", first_name="Augustin", last_name="Merriweather", ssn=s38, dob="1955-10-10")],
         [E(x="E38X", part="person", w="E38W", veto="", rank=1, levels={"ssn": "near"})])
    case("E39", "a claim row with only a clinic NPI",
         [blank_row("E39X", claim_id="CE39", note_id="NE39", clinic_npi=_npi(139))],
         [blank_row("E39W", business_name="Harbor Point Physical Therapy", clinic_npi=_npi(139))],
         [E(x="E39X", part="business", w="E39W", basis="identifier", rank=1)])
    case("E40", "surname only, DOB agrees",
         [blank_row("E40X", claim_id="CE40", note_id="NE40", last_name="Quintanilla-Obi", dob="1987-02-25")],
         [blank_row("E40W", first_name="Esperanza", last_name="Quintanilla-Obi", dob="1987-02-25")],
         [E(x="E40X", part="person", w="E40W", basis="dob", rank=1, levels={"name": "first_empty"})])
    case("E41", "LLP: keyword legal",
         [blank_row("E41X", claim_id="CE41", note_id="NE41", business_name="Pemberton & Vasquez LLP")], [],
         [E(x="E41X", part="business", party={"category_inferred": "legal"})])
    case("E42", "a witness is a private party",
         [blank_row("E42X", claim_id="CE42", note_id="NE42", category="witness", first_name="Jeremiah", last_name="Tolliver")], [],
         [E(x="E42X", part="person", party={"prior_group": "private"})])
    case("E43", "one NPI under two different names: identifier basis, the names count against",
         [blank_row("E43X", claim_id="CE43", note_id="NE43", first_name="Alice", last_name="Brennan-Yoo", provider_npi=_npi(143))],
         [blank_row("E43W", first_name="Marcus", last_name="Delacroix-Ibe", provider_npi=_npi(143))],
         [E(x="E43X", part="person", w="E43W", basis="identifier", veto="")])
    case("E44", "licence plate agrees",
         [blank_row("E44X", claim_id="CE44", note_id="NE44", first_name="Horatio", last_name="Villalobos", plate_number="HVX 4412", plate_state="NY")],
         [blank_row("E44W", first_name="Horatio", last_name="Villalobos", plate_number="HVX4412", plate_state="NY")],
         [E(x="E44X", part="person", w="E44W", basis="identifier", rank=1)])
    case("E45", "professional licence agrees; licence type makes it medical",
         [blank_row("E45X", claim_id="CE45", note_id="NE45", first_name="Clementine", last_name="Arbuthnot", professional_license_number="045678", professional_license_state="NY", professional_license_type="DC")],
         [blank_row("E45W", first_name="Clementine", last_name="Arbuthnot", professional_license_number="45678", professional_license_state="NY")],
         [E(x="E45X", part="person", party={"category": "medical", "prior_group": "professional"})])
    return C


def _person_entity(fake, rng, eid, professional):
    first, last = str(fake.first()), str(fake.surname())
    if rng.random() < 0.08:
        last = f"{last}-{fake.surname()}"
    e = {"eid": eid, "kind": "person", "first_name": first.title(), "last_name": last.title(),
         "middle_name": str(fake.first()).title() if rng.random() < 0.6 else "",
         "dob": fake.dob(), "ssn": fake.ssn(), "home_phone": fake.phone(),
         "driver_license_number": fake.dl(), "email": fake.email(first, last)}
    a = fake.address()
    e.update(a)
    e["driver_license_state"] = a["state"]
    if rng.random() < 0.2:
        e["vin"] = fake.vin(); e["plate_number"] = fake.plate(); e["plate_state"] = a["state"]
    if professional:
        spec = str(rng.choice(SPECIALTIES))
        e.update({"provider_npi": fake.npi(), "professional_license_number": fake.digits(6),
                  "professional_license_state": a["state"], "provider_specialty": spec,
                  "category": "medical", "work_phone": fake.phone()})
        if rng.random() < 0.08:
            e.update({"category": "legal", "provider_npi": "", "provider_specialty": "",
                      "professional_license_type": "ATTORNEY"})
    else:
        e["category"] = str(rng.choice(["claimant", "witness", "claimant"]))
    return e


BUSINESS_KINDS = [("Medical", "PC", "medical"), ("Chiropractic", "PC", "medical"),
                  ("Physical Therapy", "PLLC", "medical"), ("Imaging", "LLC", "medical"),
                  ("Acupuncture", "PC", "medical"), ("Rehabilitation Center", "Inc", "medical"),
                  ("Pharmacy", "Inc", "medical"), ("Auto Body", "Inc", "repair shop"),
                  ("Collision", "LLC", "repair shop"), ("Towing", "Corp", "repair shop"),
                  ("Law Offices", "PC", "legal"), ("Legal Group", "LLP", "legal"),
                  ("Diagnostics", "LLC", "medical"), ("Orthopedic Associates", "PC", "medical")]


def _business_entity(fake, rng, eid):
    kind, suffix, cat = BUSINESS_KINDS[int(rng.integers(0, len(BUSINESS_KINDS)))]
    head = str(fake.surname()).title()
    if rng.random() < 0.3:                       # 'Maple Ridge Kowalski ...': not only a surname
        head = f"{str(rng.choice(STREET_NAMES)).title()} {head}"
    name = f"{head} {kind} {suffix}"
    if kind == "Law Offices":
        name = f"Law Offices of {str(fake.first()).title()} {head}"
    if rng.random() < 0.08:
        name = f"{name} d/b/a {str(rng.choice(STREET_NAMES)).title()} {kind}"
    e = {"eid": eid, "kind": "business", "business_name": name, "tin": fake.tin(),
         "work_phone": fake.phone(), "email": f"office{int(rng.integers(1, 99999))}@{head.lower()}.test",
         "category": cat}
    if cat == "medical":
        e["clinic_npi"] = fake.npi()
    e.update(fake.address())
    return e


W_KEEP = {"dob": 0.9, "ssn": 0.3, "provider_npi": 0.9, "clinic_npi": 0.8, "tin": 0.5, "home_phone": 0.4,
          "work_phone": 0.5, "email": 0.2, "driver_license_number": 0.2, "professional_license_number": 0.5,
          "vin": 0.3, "street_name": 0.95, "provider_specialty": 0.9, "category": 0.0, "middle_name": 0.7}
X_KEEP = {"dob": 0.55, "ssn": 0.25, "provider_npi": 0.6, "clinic_npi": 0.5, "tin": 0.5, "home_phone": 0.4,
          "work_phone": 0.3, "email": 0.25, "driver_license_number": 0.2, "professional_license_number": 0.3,
          "vin": 0.3, "street_name": 0.6, "provider_specialty": 0.5, "category": 0.7, "middle_name": 0.5}
_GROUPS = {"street_name": ["street_number", "street_direction", "street_name", "street_type", "unit", "zip"],
           "driver_license_number": ["driver_license_number", "driver_license_state"],
           "professional_license_number": ["professional_license_number", "professional_license_state"],
           "vin": ["vin", "plate_number", "plate_state"]}


def _thin(r, keep, rng):
    for k, p in keep.items():
        if rng.random() >= p:
            for c in _GROUPS.get(k, [k]):
                r[c] = ""
    return r


def _row_from(entities, rid, keep, rng):
    r = blank_row(rid)
    for e in entities:
        for k, v in e.items():
            if k in r and v and (not r[k] or k not in ("category",)):
                r[k] = v
    return _thin(r, keep, rng)


def make_dataset(n_persons, n_businesses, n_watchlist, ref, noise, seed, x_share=0.75, w_share=0.6,
                 w_dup=0.1, x_repeat=0.5, x_noise_scale=0.5, w_noise_scale=0.15, include_edges=True,
                 prefix="S"):
    """(extracted rows, watchlist rows, truth pairs, edge cases). Truth pairs are
    (extracted record_id, part, watchlist record_id) for the same entity and part."""
    rng = np.random.default_rng(seed)
    fake = Fake(ref, rng)
    persons = [_person_entity(fake, rng, f"{prefix}P{i}", rng.random() < 0.35) for i in range(n_persons)]
    biz = [_business_entity(fake, rng, f"{prefix}B{i}") for i in range(n_businesses)]
    owner = {}                                  # business -> professional person on the same rows
    profs = [p for p in persons if p.get("provider_npi") or p.get("professional_license_type")]
    for b in biz:
        if profs and rng.random() < 0.4:
            owner[b["eid"]] = profs[int(rng.integers(0, len(profs)))]
    owners = {o["eid"] for o in owner.values()}
    units = [[p] for p in persons if p["eid"] not in owners]
    units += [[b, owner[b["eid"]]] if b["eid"] in owner else [b] for b in biz]
    w_rows, x_rows, w_ent, x_ent = [], [], [], []
    for u in units:
        if rng.random() < w_share:
            for k in range(2 if rng.random() < w_dup else 1):
                rid = f"{prefix}W{len(w_rows):06d}"
                r = _row_from(u, rid, W_KEEP, rng)
                r["category"] = ""
                if k == 1:
                    r.update(fake.address(r.get("state") or None))
                w_rows.append(r); w_ent.append([e["eid"] for e in u])
        if rng.random() < x_share:
            for k in range(1 + int(rng.random() < x_repeat) + int(rng.random() < x_repeat / 3)):
                rid = f"{prefix}X{len(x_rows):06d}"
                x_rows.append(_row_from(u, rid, X_KEEP, rng)); x_ent.append([e["eid"] for e in u])
    while len(w_rows) < n_watchlist:                   # listed parties never seen in claims
        i = len(w_rows)
        e = _person_entity(fake, rng, f"{prefix}WO{i}", rng.random() < 0.5) if rng.random() < 0.85 else \
            _business_entity(fake, rng, f"{prefix}WO{i}")
        r = _row_from([e], f"{prefix}W{i:06d}", W_KEEP, rng)
        r["category"] = ""
        w_rows.append(r); w_ent.append([e["eid"]])
    W = pd.DataFrame(w_rows, columns=SCHEMA)
    X = pd.DataFrame(x_rows, columns=SCHEMA)
    W, _ = apply_noise(W, noise, fake, rng, scale=w_noise_scale, id_prefix="w")
    X, xlog = apply_noise(X, noise, fake, rng, scale=x_noise_scale, id_prefix="x")
    W["record_id"] = W["record_id"].str.replace("w:", "", regex=False)
    X["record_id"] = X["record_id"].str.replace("x:", "", regex=False)
    # claims and notes: shuffle the rows into claims of 1-5 rows, 1-3 notes each
    order = rng.permutation(len(X))
    ci, pos = 0, 0
    claim = np.empty(len(X), dtype=object); note = np.empty(len(X), dtype=object)
    while pos < len(order):
        k = int(rng.integers(1, 6)); idx = order[pos:pos + k]
        nn = int(rng.integers(1, 4))
        for j, t in enumerate(idx):
            claim[t] = f"{prefix}C{ci:06d}"; note[t] = f"{prefix}C{ci:06d}-N{j % nn}"
        ci += 1; pos += k
    X["claim_id"], X["note_id"] = claim, note
    # truth
    kind = {e["eid"]: e["kind"] for u in units for e in u}
    wi = defaultdict(list)
    for rid, ents in zip(W["record_id"], w_ent):
        for e in ents:
            wi[e].append(rid)
    truth = []
    for rid, ents in zip(X["record_id"], x_ent):
        for e in ents:
            part = "person" if kind.get(e) == "person" else "business"
            for wr in wi.get(e, []):
                truth.append((rid, part, wr))
    edges = edge_cases() if include_edges else []
    if edges:
        X = pd.concat([X, pd.DataFrame([r for c in edges for r in c["x"]], columns=SCHEMA)], ignore_index=True)
        W = pd.concat([W, pd.DataFrame([r for c in edges for r in c["w"]], columns=SCHEMA)], ignore_index=True)
        for c in edges:
            for ex in c["expect"]:
                if ex.get("w") and ex.get("true", ex.get("rank") == 1):
                    truth.append((ex["x"], ex["part"], ex["w"]))
    truth = pd.DataFrame(truth, columns=["x_record_id", "part", "w_record_id"]).drop_duplicates()
    X = X.fillna("").astype(str)
    W = W.fillna("").astype(str)
    return X, W, truth, edges, xlog

## 8-11 · Comparisons, u, m and the prior

**Comparisons (8).** The core's `compare_pairs` gives one level per field for any set of
pairs; the same code runs on candidates, random pairs, anchors, watchlist duplicates and
simulated pairs. Each part (person, business) has its own weights.

**u (9).** `random_pairs` random extracted × watchlist pairs per run (split between parts by
their share of extracted parties) through the same comparisons give the field-level u of
every level (Splink's method), and the rates of the name sub-parts. Exact levels use
value-specific u instead: names from the Census/SSA tables, organization words from NPPES
(or the watchlist's own share), identifiers, dates and addresses from how many hold the value.

**m (10)**, per field and level, in order of preference:

| Source | Pairs | Used for |
|---|---|---|
| 1a strict anchors | extracted pairs sharing a valid one-per-party identifier (SSN, NPI, DL; TIN for businesses) held under exactly one name; at most `anchor_pairs_per_value` pairs per value; same-note pairs excluded | every field except the names (the anchor conditions on the name) and except the anchoring field itself |
| 1b loose anchors | the same identifier types held by 2-20 extracted parties, no name condition; EM with u fixed and the non-name m fixed from 1a | name, middle, org name |
| 2 watchlist duplicates | watchlist pairs sharing an SSN / NPI (TIN / clinic NPI) held by 2-20 parties; persons also by exact name + DOB (then not for name or DOB) | fields still short of data |
| 3 simulation | `sim_records` watchlist rows and their noisy copies (noise table) | any field with < `n_min` informative pairs from 1-2 |
| 4 published | `mappings/published_m.csv` | the base every estimate is shrunk toward |

Each estimate is shrunk toward the next source with alpha = 5 pseudo-pairs; the manifest
records the chain, the pairs behind it and a 95% interval. An agreement level whose m falls
below its u is flagged and counts 0 bits.

**Prior (11)**, per group of the extracted party, over *all* pairs: pairs firing a strict
rule (single-holder identifier agrees with no veto; exact name + DOB; exact name + street;
exact org name + ZIP or + street) / how often a true match fires one of them (from the
estimated m and the fill rates) / all extracted × watchlist pairs of the group. A group with
no strict pair uses a pseudo-count and is flagged.

**Across the workers.** Every comparison above runs in slices spread over the workers. The
random pairs behind u are never held: each slice returns its level counts, which are whole
numbers and sum exactly. Each training pass's bootstrap replicates are drawn first, in the
order a single process draws them, then fitted in parallel. The estimates are identical to a
single-process run.

**Where Splink would do better.** Splink estimates u from up to 10^8 random pairs in
DuckDB and trains m with EM in several passes, each blocked on a different rule and with u
fixed, then averages the passes; here EM runs once, on the loose-anchor set. Splink's prior
comes from a deterministic-rules count with a user-given recall; here recall is computed from
the m estimates. Splink does not use anchors or simulation.

In [ ]:
ANCHOR_TYPES = {"person": ["ssn", "npi", "dl"], "business": ["tin"]}
NAME_FREE = {"person": ["name", "middle"], "business": ["org"]}
WDUP_TYPES = {"person": ["ssn", "npi"], "business": ["tin", "cnpi"]}
# Splink-style training passes: each selects pairs by one rule and learns m for every field
# except the ones the rule used (their agreement is forced by the selection).
TRAIN_RULES = {
    "person": [("dob", ["dob"]), ("nysiis_initial", ["name", "middle"]), ("street", ["address"]),
               ("ssn", ["ssn"]), ("npi", ["npi"]), ("phone", ["phone"]), ("email", ["email"])],
    "business": [("rare_word", ["org"]), ("street", ["address"]), ("tin", ["tin"]), ("cnpi", ["cnpi"]),
                 ("phone", ["phone"])],
}


@dataclass
class Model:
    """The trained parameters of one comparison setting: 'link' (extracted x watchlist) or
    'dedup' (extracted x extracted)."""
    kind: str
    weights: dict        # part -> weights table (one row per field and level)
    prior: dict          # (part, group) -> prior probability that a random pair matches
    prior_df: pd.DataFrame
    passes: dict         # part -> list of training-pass summaries
    u_sample_pairs: dict  # part -> random pairs behind the fuzzy levels of u


def _pairs_within(groups, cap, rng, exclude_same=None):
    """All pairs within each group of positions (at most cap per group, deterministic)."""
    ls, rs = [], []
    for g in groups:
        g = np.asarray(g)
        i, j = np.triu_indices(len(g), k=1)
        a, b = g[i], g[j]
        if exclude_same is not None:
            keep = exclude_same[a] != exclude_same[b]
            a, b = a[keep], b[keep]
        if len(a) > cap:
            sel = np.sort(rng.choice(len(a), cap, replace=False))
            a, b = a[sel], b[sel]
        ls.append(a); rs.append(b)
    if not ls:
        return np.array([], np.int64), np.array([], np.int64)
    return np.concatenate(ls).astype(np.int64), np.concatenate(rs).astype(np.int64)


def anchor_pairs(fr, types, holders, name_condition, cap, rng, exclude_notes=True):
    """Pairs of parties within one frame sharing a value of `types`. name_condition=True keeps
    values held under exactly one name (strict); holders=(lo, hi) bounds the parties per value.
    Returns (l, r, anchor-field mask dict)."""
    notes = fr["note_id"].to_numpy(dtype=object) if exclude_notes else None
    if notes is not None:
        notes = np.where(notes == "", "row:" + fr["record_id"].to_numpy(dtype=object), notes)
    found = []
    for t in types:
        col = f"id_{t}"
        v = fr[col].to_numpy(dtype=object)
        d = pd.DataFrame({"pos": np.arange(len(fr)), "v": v, "n": fr["holder_key"].to_numpy(dtype=object)})
        d = d[d["v"] != ""]
        if not len(d):
            continue
        g = d.groupby("v")
        size = g["pos"].size()
        names = g["n"].nunique()
        ok = (size >= holders[0]) & (size <= holders[1])
        if name_condition:
            ok &= names == 1
        keep = set(size.index[ok])
        grp = [x["pos"].to_numpy() for v, x in d[d["v"].isin(keep)].groupby("v", sort=True)]
        l, r = _pairs_within(grp, cap, rng, notes)
        found.append(pd.DataFrame({"l": l, "r": r, "t": t}))
    if not found:
        return np.array([], np.int64), np.array([], np.int64), {}
    f = pd.concat(found, ignore_index=True)
    piv = f.assign(one=True).pivot_table(index=["l", "r"], columns="t", values="one", aggfunc="any",
                                         fill_value=False)
    l = piv.index.get_level_values(0).to_numpy()
    r = piv.index.get_level_values(1).to_numpy()
    return l, r, {t: piv[t].to_numpy(dtype=bool) for t in piv.columns}


def name_dob_pairs(fr, cap, rng):
    d = pd.DataFrame({"pos": np.arange(len(fr)), "k": fr["name_key"].to_numpy(dtype=object) + "|" +
                      fr["dob"].to_numpy(dtype=object)})
    d = d[(fr["dob"].to_numpy() != "") & (fr["first"].to_numpy() != "")]
    size = d.groupby("k")["pos"].size()
    keep = set(size.index[(size >= 2) & (size <= 20)])
    grp = [x["pos"].to_numpy() for _, x in d[d["k"].isin(keep)].groupby("k", sort=True)]
    return _pairs_within(grp, cap, rng)


def with_coparty(levels, left, right, part):
    if part == "person":
        levels["co_party"] = coparty_proxy_levels(left, right, levels["l"].to_numpy(), levels["r"].to_numpy())
    return levels


# ---- u ---------------------------------------------------------------------------------------
def random_pairs(nl, nr, n, seed, same_frame):
    """n random pairs (with replacement) of positions; distinct parties within one frame."""
    rng = np.random.default_rng(seed)
    if same_frame:
        l = rng.integers(0, nl, n)
        r = rng.integers(0, nl - 1, n)
        r = np.where(r >= l, r + 1, r)
        return l.astype(np.int64), r.astype(np.int64)
    idx = rl.Index()
    idx.add(rl.index.Random(n, replace=True, random_state=seed))
    mi = idx.index(pd.DataFrame(index=np.arange(nl)), pd.DataFrame(index=np.arange(nr)))
    return mi.get_level_values(0).to_numpy().astype(np.int64), mi.get_level_values(1).to_numpy().astype(np.int64)


# ---- the prior centre of m: the build's source chain ------------------------------------------
def prior_centre_m(pr, maps, ref, cfg, wdf, u_link, rng, log=print):
    """m per field from anchors, watchlist duplicates, simulation and published values, shrunk
    in a chain (combine_m). It is the centre of the Dirichlet prior the training passes use,
    so thin data shrinks toward it. Returns ({part: rows}, sources, loose-anchor EM summary)."""
    p = cfg.core
    ctx = pr.ctx
    pub = maps["published_m"]
    pub_arr = lambda f: np.array([float(pub[(pub["field"] == f) & (pub["level"] == lev)]["m"].iloc[0])
                                  for lev in FIELD_LEVELS[f]])
    sources, em_info, centre = [], {}, {}
    t0 = time.time()
    # simulation: noisy copies of watchlist rows
    k = min(cfg.sim_records, len(wdf))
    pick = np.sort(rng.choice(len(wdf), size=k, replace=False)) if k else np.array([], int)
    src = wdf.iloc[pick][SCHEMA].reset_index(drop=True)
    fake = Fake(ref, rng)
    copies, _ = apply_noise(src, maps["simulation_noise"], fake, rng, scale=1.0, id_prefix="sim")
    _, cpar, _, _, _, _ = rows_to_parties(copies, "S", maps, ctx.nick, p, False)
    C = split_parts(cpar)
    sim_counts = {}
    for part in ("person", "business"):
        Cf, W = C[part], pr.W[part]
        wpos = pd.Series(np.arange(len(W)), index=W["record_id"].to_numpy())
        orig = Cf["record_id"].str.replace("sim:", "", regex=False)
        ok = orig.isin(wpos.index).to_numpy()
        l = np.flatnonzero(ok)
        r = wpos[orig[ok]].to_numpy() if ok.any() else np.array([], np.int64)
        lv = compare_chunked(Cf, W, l, r, part, ctx)
        sim_counts[part] = level_counts(lv, PART_FIELDS[part])
        sources.append({"part": part, "source": "simulation", "pairs": len(lv)})
    log(f"prior centre: simulation, {k:,} noisy copies compared ({time.time() - t0:.0f}s)")
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        fields = PART_FIELDS[part]
        non_name = [f for f in fields if f not in NAME_FREE[part]]
        l, r, amask = anchor_pairs(X, ANCHOR_TYPES[part], (2, 10 ** 9), True, cfg.anchor_pairs_per_value, rng)
        lv1 = compare_chunked(X, X, l, r, part, ctx)
        c1a = level_counts(lv1, non_name, exclude=amask)
        sources.append({"part": part, "source": "anchors_strict", "pairs": len(lv1)})
        l, r, dmask = anchor_pairs(W, WDUP_TYPES[part], (2, 20), False, cfg.anchor_pairs_per_value, rng,
                                   exclude_notes=False)
        lv2 = compare_chunked(W, W, l, r, part, ctx)
        c2 = level_counts(lv2, fields, exclude=dmask)
        n_wdup = len(lv2)
        if part == "person":
            l, r = name_dob_pairs(W, cfg.anchor_pairs_per_value, rng)
            lv2b = compare_chunked(W, W, l, r, part, ctx)
            c2b = level_counts(lv2b, [f for f in fields if f not in ("name", "dob")])
            for f, (cnt, n) in c2b.items():
                c0, n0 = c2.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))
                c2[f] = (c0 + cnt, n0 + n)
            n_wdup += len(lv2b)
        sources.append({"part": part, "source": "watchlist_duplicates", "pairs": n_wdup})
        rows = []
        z = lambda f: np.zeros(len(FIELD_LEVELS[f]))
        for f in non_name:
            srcs = [("anchors_strict", *c1a.get(f, (z(f), 0.0))), ("watchlist_duplicates", *c2.get(f, (z(f), 0.0))),
                    ("simulation", *sim_counts[part].get(f, (z(f), 0.0)))]
            rows += combine_m(f, srcs, pub_arr(f), p)
        mfix = pd.DataFrame(rows)
        u_df = u_link[part]
        l, r, lmask = anchor_pairs(X, ANCHOR_TYPES[part], cfg.loose_holders, False, cfg.anchor_pairs_per_value, rng)
        lv1b = compare_chunked(X, X, l, r, part, ctx)
        fixed = [f for f in non_name if f != "co_party"]
        m_fixed = {f: mfix[mfix["field"] == f].sort_values("level_code")["m"].to_numpy() for f in fixed}
        u_tab = {f: u_df[u_df["field"] == f].sort_values("level_code")["u"].to_numpy() for f in fields if f != "co_party"}
        em, lam, iters = em_fixed_u(lv1b, NAME_FREE[part], fixed, m_fixed, u_tab, p, exclude=lmask)
        em_info[part] = {"pairs": len(lv1b), "lambda": lam, "iterations": iters}
        sources.append({"part": part, "source": "anchors_loose_em", "pairs": len(lv1b)})
        for f in NAME_FREE[part]:
            cnt, n = (em[f][2], em[f][1]) if f in em else (z(f), 0.0)
            srcs = [("anchors_loose_em", cnt, n), ("watchlist_duplicates", *c2.get(f, (z(f), 0.0))),
                    ("simulation", *sim_counts[part].get(f, (z(f), 0.0)))]
            rows += combine_m(f, srcs, pub_arr(f), p)
        centre[part] = pd.DataFrame(rows)
        log(f"prior centre [{part}]: strict anchors {len(lv1):,}, loose anchors {len(lv1b):,} "
            f"(EM lambda {lam:.3f}), watchlist duplicates {n_wdup:,} ({time.time() - t0:.0f}s)")
    return centre, sources, em_info


# ---- training passes ---------------------------------------------------------------------------
def training_pairs(L, R, part, rule, ctx, cfg, rng, same_frame):
    """Pairs sharing a key of `rule` (at most about em_key_cap per key: each side of an
    oversized key is sampled), at most em_pairs_per_pass in all."""
    xk = rule_keys(L, part, rule, ctx.rarity, "x", ctx.dba_pairs)
    wk = rule_keys(L if same_frame else R, part, rule, ctx.rarity, "w", ctx.dba_pairs)
    side = max(2, int(math.sqrt(cfg.em_key_cap)))

    def cap(f):
        if not len(f):
            return f
        f = f.assign(_o=rng.random(len(f))).sort_values(["key", "_o"], kind="stable")
        return f[f.groupby("key").cumcount() < side]
    m = cap(xk)[["pos", "key"]].merge(cap(wk)[["pos", "key"]], on="key", suffixes=("_x", "_w"))
    if same_frame:
        m = m[m["pos_x"] != m["pos_w"]]
        a, b = np.minimum(m["pos_x"], m["pos_w"]), np.maximum(m["pos_x"], m["pos_w"])
        m = pd.DataFrame({"pos_x": a, "pos_w": b})
    m = m.drop_duplicates(["pos_x", "pos_w"])
    if len(m) > cfg.em_pairs_per_pass:
        m = m.iloc[np.sort(rng.choice(len(m), cfg.em_pairs_per_pass, replace=False))]
    return m["pos_x"].to_numpy(np.int64), m["pos_w"].to_numpy(np.int64)


def final_weights(part, passes, m0_rows, u_df, components, params):
    """Weights table of one part: m from the training passes (mean of the passes that saw the
    field, each with the Dirichlet prior), floored; the prior centre where no pass saw the
    field; bootstrap intervals; flags for unstable weights."""
    rows = []
    for f in PART_FIELDS[part]:
        c = m0_rows[m0_rows["field"] == f].sort_values("level_code")
        m0 = c["m"].to_numpy()
        u = u_df[u_df["field"] == f].sort_values("level_code")["u"].to_numpy()
        k = len(FIELD_LEVELS[f])
        if f == "co_party":
            m, lo, hi, per, blo, bhi, flags, spread = m0, c["m_lo"].to_numpy(), c["m_hi"].to_numpy(), {}, \
                np.full(k, np.nan), np.full(k, np.nan), [], np.zeros(k)
            note = "not trained by EM (a proxy level in estimation): prior centre"
            source = "prior centre: " + c["m_chain"].iloc[0]
        else:
            m, lo, hi, per, blo, bhi, flags, spread = combine_passes(passes, f, m0, u, params)
            note = "; ".join(flags)
            source = (f"EM, u fixed: mean of {len(per)} training passes ({', '.join(x.split(':')[-1] for x in per)}), "
                      f"Dirichlet prior ({params.em_prior_strength:g} pseudo-pairs) toward the prior centre"
                      if per else "prior centre only (no training pass saw this field)")
        mf = floor_m(m, params)
        n_match = sum(p_.get("n_match", {}).get(f, 0.0) for p_ in passes if f in p_.get("m", {}))
        for i, lev in enumerate(FIELD_LEVELS[f]):
            relevant = max(mf[i], u[i] if i < len(u) else 0) >= 0.01
            why = []
            if relevant and np.isfinite(bhi[i]) and (bhi[i] - blo[i]) > params.unstable_ci_bits:
                why.append(f"bootstrap 95% interval {bhi[i] - blo[i]:.1f} bits wide")
            if relevant and spread[i] > params.unstable_pass_bits:
                why.append(f"training passes disagree by {spread[i]:.1f} bits")
            rows.append({"field": f, "level": lev, "level_code": i, "m": float(mf[i]),
                         "m_lo": float(lo[i]) if np.isfinite(lo[i]) else np.nan,
                         "m_hi": float(hi[i]) if np.isfinite(hi[i]) else np.nan,
                         "m_source": source, "m_chain": c["m_chain"].iloc[0], "m_pairs": float(n_match),
                         "m_prior_centre": float(m0[i]), "m_published": float(c["m_published"].iloc[i]),
                         "m_passes": json.dumps({kk.split(":")[-1]: round(float(v[i]), 6) for kk, v in per.items()}),
                         "bits_lo": float(blo[i]) if np.isfinite(blo[i]) else np.nan,
                         "bits_hi": float(bhi[i]) if np.isfinite(bhi[i]) else np.nan,
                         "unstable": "; ".join(why), "m_note": note})
    return weights_table(rows, u_df, components, params)


def fill_rates(Xg, W, part):
    sh = lambda F, c: float((F[c] != "").mean()) if len(F) else 0.0
    both = lambda c: sh(Xg, c) * sh(W, c)
    fill = {t: both(f"id_{t}") for t in ("ssn", "npi", "dl", "license", "email", "vin", "plate", "tin", "cnpi")
            if f"id_{t}" in Xg}
    if part == "person":
        full = lambda F: float(((F["first"] != "") & (F["last"] != "")).mean()) if len(F) else 0.0
        fill["name"] = full(Xg) * full(W)
        fill["dob"] = both("dob")
    else:
        fill["org"] = both("org_aliases")
        fill["zip"] = both("zip")
    fill["street"] = both("addr_street")
    return fill


def _m_lookup(w):
    t = {(f, lev): m for f, lev, m in zip(w["field"], w["level"], w["m"])}
    return lambda f, lev: float(t.get((f, lev), 0.0))


def _replicate_lookups(passes, w, params):
    """m lookups for each bootstrap replicate (mean of the passes' replicate b; the final m
    where no pass saw the field)."""
    reps = min((len(p_["boot"]) for p_ in passes if p_.get("boot")), default=0)
    base = {(f, lev): m for f, lev, m in zip(w["field"], w["level"], w["m"])}
    out = []
    for b in range(reps):
        t = dict(base)
        fields = {f for p_ in passes for f in p_.get("m", {})}
        for f in fields:
            used = [p_ for p_ in passes if f in p_.get("m", {}) and p_.get("boot")]
            if used:
                m = floor_m(np.mean([p_["boot"][b]["m"][f] for p_ in used], axis=0), params)
                for i, lev in enumerate(FIELD_LEVELS[f]):
                    t[(f, lev)] = float(m[i])
        out.append(lambda f, lev, t=t: float(t.get((f, lev), 0.0)))
    return out


def estimate_prior(pr, weights, passes, cfg, log=print, same_frame=False, seed=0):
    """Prior per group (link) or per part (dedup): strict pairs / how often a true match fires
    a strict rule / all pairs, with a bootstrap 95% interval (extracted parties resampled, m
    from the matching EM replicate)."""
    rows, prior = [], {}
    rng = np.random.default_rng(seed)
    ctx = pr.ctx_dedup if same_frame else pr.ctx
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        R = X if same_frame else W
        w = weights[part]
        mlook = _m_lookup(w)
        reps = _replicate_lookups(passes.get(part, []), w, cfg.core)
        l, r = strict_pairs(X, W, part, ctx, same_frame)
        per_party = np.bincount(l, minlength=len(X)).astype(float) if len(l) else np.zeros(len(X))
        grp = X["prior_group"].to_numpy() if not same_frame else np.full(len(X), "all", dtype=object)
        for g in (GROUPS if not same_frame else ["all"]):
            gm = grp == g
            if not gm.any():
                continue
            n_strict = int(per_party[gm].sum())
            n_pairs = int(gm.sum()) * (len(R) - 1 if same_frame else len(R))
            if same_frame:
                n_pairs //= 2
            fill = fill_rates(X[gm], R, part)
            rec = strict_recall(fill, mlook)
            pv, flagged = prior_estimate(n_strict, n_pairs, rec, cfg.core)
            boots = []
            if reps:
                cnt = per_party[gm]
                for mb in reps:
                    wts = rng.poisson(1.0, len(cnt)).astype(float)
                    nb = float((wts * cnt).sum())
                    npairs = float(wts.sum()) * (n_pairs / max(1, gm.sum()))
                    boots.append(prior_estimate(nb, npairs, strict_recall(fill, mb), cfg.core)[0])
            lo, hi = (float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))) if boots else (np.nan, np.nan)
            prior[(part, g)] = pv
            rows.append({"part": part, "group": g, "extracted_parties": int(gm.sum()),
                         "watchlist_parties": len(R), "all_pairs": n_pairs, "strict_pairs": n_strict,
                         "recall": rec, "prior": pv, "prior_lo": lo, "prior_hi": hi,
                         "prior_logit_bits": math.log2(pv / (1 - pv)),
                         "flag": "pseudo-count: no strict pair or no recall" if flagged else ""})
    df = pd.DataFrame(rows)
    log(("dedup " if same_frame else "") + "prior:\n" +
        df[["part", "group", "extracted_parties", "strict_pairs", "recall", "prior", "prior_lo", "prior_hi", "flag"]]
        .to_string(index=False))
    return prior, df


# ---- comparisons across the workers ------------------------------------------------------------
COMPARE_STEP = 250_000     # pairs per task: the slices compare_chunked has always compared


def _w_compare(task):
    S = _SHARED
    lo, hi = task
    out = compare_pairs(S["L"], S["R"], S["l"][lo:hi], S["r"][lo:hi], S["part"], S["ctx"])
    _trim_caches()
    return out


def compare_parallel(L, R, l, r, part, ctx, step=None):
    """compare_pairs over many pairs, in slices of `step` spread over the workers and
    concatenated in order: the result equals one compare_pairs call over all the pairs."""
    step = step or COMPARE_STEP
    l, r = np.asarray(l, dtype=np.int64), np.asarray(r, dtype=np.int64)
    if len(l) <= step:
        return compare_pairs(L, R, l, r, part, ctx)
    tasks = [(lo, min(len(l), lo + step)) for lo in range(0, len(l), step)]
    parts = PAR.map(_w_compare, tasks, {"L": L, "R": R, "l": l, "r": r, "part": part, "ctx": ctx})
    return pd.concat(parts, ignore_index=True)


def compare_chunked(L, R, l, r, part, ctx, step=250_000):
    if not len(l):
        return with_coparty(compare_pairs(L, R, l, r, part, ctx), L, R, part)
    return with_coparty(compare_parallel(L, R, l, r, part, ctx, step), L, R, part)


# ---- u from random pairs: counts, not levels -------------------------------------------------------
NAME_COMPONENT_COUNTS = ("n", "fc1", "fc2", "fc12", "fc3", "fc4", "fc5", "lc1", "lc2")


def _w_random_counts(task):
    S = _SHARED
    lo, hi = task
    lv = compare_pairs(S["L"], S["R"], S["l"][lo:hi], S["r"][lo:hi], S["part"], S["ctx"])
    lv = with_coparty(lv, S["L"], S["R"], S["part"])
    cnt = level_counts(lv, PART_FIELDS[S["part"]])
    comp = None
    if "name" in lv:
        ok = lv["name"].to_numpy() >= 0
        fc = lv["name_first_cmp"].to_numpy()[ok]
        lc = lv["name_last_cmp"].to_numpy()[ok]
        comp = {"n": int(ok.sum()), "fc1": int((fc == 1).sum()), "fc2": int((fc == 2).sum()),
                "fc12": int(((fc == 1) | (fc == 2)).sum()), "fc3": int((fc == 3).sum()),
                "fc4": int((fc == 4).sum()), "fc5": int((fc == 5).sum()), "lc1": int((lc == 1).sum()),
                "lc2": int((lc == 2).sum())}
    _trim_caches()
    return cnt, comp


def random_pair_counts(L, R, l, r, part, ctx, step=250_000):
    """What estimate_u and name_components read from random pairs (level counts per field, the
    name sub-part counts), without keeping the pairs' levels: slices in parallel, counts summed.
    The counts are whole numbers, so the sums equal one pass over all the pairs."""
    l, r = np.asarray(l, dtype=np.int64), np.asarray(r, dtype=np.int64)
    tasks = [(lo, min(len(l), lo + step)) for lo in range(0, len(l), step)]
    res = PAR.map(_w_random_counts, tasks, {"L": L, "R": R, "l": l, "r": r, "part": part, "ctx": ctx})
    cnt, comp = {}, None
    for c, k in res:
        for f, (counts, n) in c.items():
            c0, n0 = cnt.get(f, (np.zeros(len(counts)), 0.0))
            cnt[f] = (c0 + counts, n0 + n)
        if k is not None:
            comp = k if comp is None else {x: comp[x] + k[x] for x in NAME_COMPONENT_COUNTS}
    return cnt, comp


def estimate_u_counts(cnt, fields, params):
    """estimate_u (matching core) from level counts instead of the levels themselves."""
    rows = []
    for f in fields:
        counts, n = cnt.get(f, (np.zeros(len(FIELD_LEVELS[f])), 0.0))
        k = len(counts)
        u = (counts + params.u_pseudo) / (n + params.u_pseudo * k)
        for i, lev in enumerate(FIELD_LEVELS[f]):
            rows.append({"field": f, "level": lev, "level_code": i, "u": float(u[i]),
                         "u_count": float(counts[i]), "u_pairs": float(n),
                         "u_source": f"random pairs ({int(n):,} informative)", "u_method": "sample"})
    return pd.DataFrame(rows)


def name_components_counts(comp):
    """name_components (matching core) from the counts _w_random_counts returns."""
    n = max(1, int(comp["n"])) if comp else 1
    rate = lambda k: (float(comp[k] if comp else 0) + 0.5) / (n + 1.0)
    return {"first_nick": rate("fc1"), "first_close": rate("fc2"), "first_agrees_fuzzy": rate("fc12"),
            "first_initial": rate("fc3"), "first_empty": rate("fc4"), "first_differs": rate("fc5"),
            "last_close": rate("lc1"), "last_sound": rate("lc2"), "pairs": n}


def estimate_u_model(L, R, part, ctx, n, seed, same_frame, log=print, label=""):
    """u for one part: exact levels in closed form over all pairs, the rest from n random
    pairs. Returns (u table, the name sub-part rates (persons; None otherwise), sample size)."""
    t0 = time.time()
    nl, nr = len(L), len(L if same_frame else R)
    total = nl * (nl - 1) // 2 if same_frame else nl * nr
    n = int(min(n, total))
    if n <= 0:
        u_s = estimate_u(pd.DataFrame(), PART_FIELDS[part], ctx.params)
        return u_s, None, 0
    l, r = random_pairs(nl, nr, n, seed, same_frame)
    cnt, comp = random_pair_counts(L, L if same_frame else R, l, r, part, ctx)
    u_s = estimate_u_counts(cnt, PART_FIELDS[part], ctx.params)
    closed = closed_form_u(L, R, part, ctx.stats.single.get("phone", set()), same_frame)
    u_df = pd.concat([combine_u(u_s, closed, f, ctx.params) for f in PART_FIELDS[part]], ignore_index=True)
    log(f"u[{label}{part}]: {n:,} random pairs for the fuzzy levels, "
        f"{sum(len(v) for v in closed.values())} levels in closed form ({time.time() - t0:.0f}s)")
    return u_df, (name_components_counts(comp) if comp is not None else None), n


# ---- training passes: the bootstrap across the workers ----------------------------------------------
BOOT_PARALLEL_MIN = 200_000   # patterns x replicates below which the bootstrap stays in one process


def _w_em_boot(task):
    S = _SHARED
    out = []
    for b in task:
        e = em_patterns(S["pat"], S["draws"][b], S["fields"], S["u_tab"], S["m_prior"], S["alpha"], S["params"],
                        m_init=S["m_init"], lam_init=S["lam_init"], max_iter=200, tol=1e-6)
        out.append({"m": e["m"], "lambda": e["lambda"]})
    return out


def training_pass_em_parallel(levels, fields, u_tab, m_prior, params, name, rng=None, reps=0):
    """training_pass_em (matching core) with its bootstrap replicates spread over the workers.
    The replicate counts are drawn first, in the order the core draws them, and each replicate
    is the core's em_patterns, so the estimates and intervals are the core's."""
    pat, counts, fields = pattern_table(levels, fields)
    alpha = params.em_prior_strength
    est = em_patterns(pat, counts, fields, u_tab, m_prior, alpha, params)
    est.update({"pass": name, "pairs": int(counts.sum()), "patterns": len(pat), "fields": list(fields)})
    boots = []
    if reps and len(pat) and counts.sum() > 0:
        n = int(counts.sum())
        prob = counts / counts.sum()
        draws = [rng.multinomial(n, prob).astype(float) for _ in range(reps)]
        k = max(1, min(reps, PAR.workers * 2)) if len(pat) * reps >= BOOT_PARALLEL_MIN else 1
        batches = [list(range(i, reps, k)) for i in range(k)]
        shared = {"pat": pat, "draws": draws, "fields": fields, "u_tab": u_tab, "m_prior": m_prior,
                  "alpha": alpha, "params": params, "m_init": est["m"], "lam_init": est["lambda"]}
        got = {}
        for batch, res in zip(batches, PAR.map(_w_em_boot, batches, shared)):
            got.update(dict(zip(batch, res)))
        boots = [got[b] for b in range(reps)]
    est["boot"] = boots
    return est


def run_training(L, R, part, ctx, u_df, m0_rows, cfg, rng, same_frame, label, log=print):
    """Every training pass of one part: EM with u fixed over the fields its rule did not use,
    Dirichlet prior toward the prior centre, bootstrap replicates."""
    t0 = time.time()
    fields_all = [f for f in PART_FIELDS[part] if f != "co_party"]
    u_tab = {f: u_df[u_df["field"] == f].sort_values("level_code")["u"].to_numpy() for f in fields_all}
    m0 = {f: floor_m(m0_rows[m0_rows["field"] == f].sort_values("level_code")["m"].to_numpy(), cfg.core)
          for f in fields_all}
    passes = []
    for rule, excluded in TRAIN_RULES[part]:
        l, r = training_pairs(L, R, part, rule, ctx, cfg, rng, same_frame)
        if len(l) < 20:
            passes.append({"pass": f"{label}:{rule}", "pairs": int(len(l)), "skipped": "fewer than 20 pairs",
                           "m": {}, "boot": [], "converged": False, "lambda": float("nan"), "iterations": 0,
                           "fields": [], "n_match": {}, "excluded": excluded})
            continue
        lv = compare_chunked(L, L if same_frame else R, l, r, part, ctx)
        fields = [f for f in fields_all if f not in excluded]
        est = training_pass_em_parallel(lv, fields, u_tab, m0, cfg.core, f"{label}:{rule}", rng, cfg.bootstrap_reps)
        est["excluded"] = excluded
        passes.append(est)
        del lv
    done = [p_ for p_ in passes if p_["pairs"] and not p_.get("skipped")]
    log(f"EM [{label}{part}]: {len(done)} training passes "
        + ", ".join(f"{p_['pass'].split(':')[-1]} {p_['pairs']:,} pairs lambda {p_['lambda']:.3g}" for p_ in done)
        + f" ({time.time() - t0:.0f}s, {cfg.bootstrap_reps} bootstrap replicates each)")
    return passes


def strict_pairs(X, W, part, ctx, same_frame=False):
    """Pairs firing a strict rule (not vetoed), as (l, r); within one frame l < r."""
    found = []
    R = X if same_frame else W

    def join(kx, kw):
        a = pd.DataFrame({"k": kx, "l": np.arange(len(X))})
        b = pd.DataFrame({"k": kw, "r": np.arange(len(R))})
        a, b = a[a["k"] != ""], b[b["k"] != ""]
        m = a.merge(b, on="k")
        if same_frame:
            m = m[m["l"] < m["r"]]
        found.append(m[["l", "r"]])

    single = ctx.stats.single
    ids = ["ssn", "npi", "dl", "license", "email", "vin", "plate"] if part == "person" else ["tin", "cnpi", "email"]
    for t in ids:
        kx = X[f"id_{t}"].to_numpy(dtype=object)
        kx = np.where(pd.Series(kx).isin(single.get(t, set())).to_numpy(), kx, "")
        join(kx, R[f"id_{t}"].to_numpy(dtype=object))
    nk = lambda F: np.where((F["first"] != "") | (F["part"] == "business"), F["name_key"], "").astype(object)
    if part == "person":
        join(np.where(X["dob"] != "", nk(X) + "|" + X["dob"], ""), np.where(R["dob"] != "", nk(R) + "|" + R["dob"], ""))
    else:
        join(np.where(X["zip"] != "", nk(X) + "|" + X["zip"], ""), np.where(R["zip"] != "", nk(R) + "|" + R["zip"], ""))
    join(np.where(X["addr_street"] != "", nk(X) + "|" + X["addr_street"], ""),
         np.where(R["addr_street"] != "", nk(R) + "|" + R["addr_street"], ""))
    f = pd.concat(found, ignore_index=True).drop_duplicates()
    if not len(f):
        return f["l"].to_numpy(), f["r"].to_numpy()
    lv = compare_parallel(X, R, f["l"].to_numpy(), f["r"].to_numpy(), part, ctx)
    veto = np.zeros(len(lv), bool)
    for t in VETO_FIELDS:
        if f"veto_{t}" in lv:
            veto |= lv[f"veto_{t}"].to_numpy(dtype=bool)
    return f["l"].to_numpy()[~veto], f["r"].to_numpy()[~veto]


def estimate_models(pr, maps, ref, cfg, wdf, log=print):
    """u, m and the prior for the link model (extracted x watchlist) and the deduplication
    model (extracted x extracted)."""
    rng = np.random.default_rng(cfg.seed)
    nx = {part: len(pr.X[part]) for part in ("person", "business")}
    tot = max(1, sum(nx.values()))
    u = {"link": {}, "dedup": {}}
    comps, n_sample = {}, {"link": {}, "dedup": {}}
    for kind, ctx, same, n_all in (("link", pr.ctx, False, cfg.random_pairs), ("dedup", pr.ctx_dedup, True,
                                                                                   cfg.dedup_random_pairs)):
        for part in ("person", "business"):
            X, W = pr.X[part], pr.W[part]
            if not len(X) or (not same and not len(W)):
                u[kind][part] = estimate_u(pd.DataFrame(), PART_FIELDS[part], cfg.core)
                n_sample[kind][part] = 0
                continue
            n = int(n_all * nx[part] / tot)
            u_df, comp, n_used = estimate_u_model(X, W, part, ctx, n, cfg.seed + (7 if same else 0), same, log,
                                                  label=f"{kind} ")
            u[kind][part] = u_df
            n_sample[kind][part] = n_used
            if part == "person" and n_used and comp is not None:
                comps[kind] = comp
        ctx.components = comps.get(kind, {})
    centre, sources, em_loose = prior_centre_m(pr, maps, ref, cfg, wdf, u["link"], rng, log)
    models = {}
    for kind, ctx, same in (("dedup", pr.ctx_dedup, True), ("link", pr.ctx, False)):
        weights, passes = {}, {}
        for part in ("person", "business"):
            X, W = pr.X[part], pr.W[part]
            passes[part] = (run_training(X, W, part, ctx, u[kind][part], centre[part], cfg, rng, same, kind, log)
                            if len(X) and (same or len(W)) else [])
            w = final_weights(part, passes[part], centre[part], u[kind][part], comps.get(kind), cfg.core)
            w.insert(0, "part", part)
            w.insert(0, "model", kind)
            weights[part] = w
        prior, prior_df = estimate_prior(pr, weights, passes, cfg, log, same_frame=same, seed=cfg.seed + 11)
        models[kind] = Model(kind=kind, weights=weights, prior=prior, prior_df=prior_df,
                             passes={part: [{k: v for k, v in p_.items() if k != "boot"} for p_ in ps]
                                     for part, ps in passes.items()},
                             u_sample_pairs=n_sample[kind])
        un = sum(int((w["unstable"] != "").sum()) for w in weights.values())
        log(f"{kind} model: {un} weight levels flagged unstable")
    return {"link": models["link"], "dedup": models["dedup"], "components": comps, "sources": sources,
            "em_loose": em_loose, "centre": centre}


## 12 · Score

**Two passes, in chunks, on every core.** The first pass scores chunks of extracted entities in
parallel workers; each worker writes its chunk's scored pairs and their evidence to Parquet and
returns only what the driver needs next: the links that may join a cluster, the co-party
anchors, per-entity counts. Clustering then runs once, on those links. The second pass gives
every pair its cluster status, writes the candidates table and returns each entity's roll-up.
An entity's pairs are always in one chunk, so its rank and roll-up are what one process gives.

In chunks of extracted parties: candidates → comparisons → the core's Fellegi-Sunter scorer
(sum of log2(m/u) per field, value-specific u on the levels that have one, 0 bits when a field
is empty, agreement never below 0) → vetoes (p = 0, visible, with the reason) → basis.
Businesses are scored first; their identifier links at p >= 0.9 with no veto are the anchors
for the **co-party** pass on persons, which adds bits only where the names already agree
(one way: a co-party can strengthen a name, never make one, and name-only links never vouch
for each other).

A pair is **kept** when p >= `keep_p_floor` (default 0.0: every scored candidate, since the user's priority is entity-resolution quality, not output size, time or memory), or it is among its entity's top K, or it rests on an identifier, or it is vetoed. `rank`, `top_k` and the veto flag stay on every row as metadata for sorting and display; they no longer exclude anything once the floor is 0. Pairs are never filtered by basis: a name-only match is kept on the same terms as any other, and it never silently disappears once it has been proposed and scored (this was a real gap found by acceptance testing on 2026-09-27: one true pair scored below the old 1e-3 floor, was not its entity's top candidate, and had no identifier, so it was dropped from every output table with nothing recording that it had existed).

**Where Splink would do better.** Splink's `predict()` scores every blocked pair in SQL with term-frequency adjustments and writes them all, the same "keep everything scored" choice made here; it has no equivalent size concern because it writes to a table, not a workbook (the workbook sheet is a summary; the full candidates/evidence tables carry every row regardless of format). Splink has no vetoes or basis classes: both are ours.


In [ ]:
P_BINS = np.array([0, 1e-3, 0.01, 0.1, 0.5, 0.8, 0.9, 0.99, 1.0000001])
DEDUP_KEEP_P = 0.0           # every scored dedup pair is kept (same principle as
                             # keep_p_floor: no display-size cutoff on correctness)


def _rank_within(l, p):
    """Rank of each pair within its entity (l), best first; ties share the rank ('min')."""
    d = pd.DataFrame({"l": l, "p": p})
    return d.groupby("l")["p"].rank(method="min", ascending=False).to_numpy()


def _key(a, b):
    return np.asarray(a, dtype=np.int64) * np.int64(1 << 32) + np.asarray(b, dtype=np.int64)


def single_values(frame, stats, fields=VETO_FIELDS):
    """Per party, the single-holder values of the one-per-party identifiers (for vetoes and
    cluster constraints): {field: list of frozensets}."""
    out = {}
    for f in fields:
        col = f"id_{f}"
        s = stats.single.get(f, set())
        v = frame[col].to_numpy(dtype=object) if col in frame else np.full(len(frame), "", dtype=object)
        out[f] = [frozenset([x]) if (x and x in s) else frozenset() for x in v]
    return out


@dataclass
class Entities:
    """Extracted parties of one part pooled into entities (deduplication)."""
    part: str
    e_of: np.ndarray       # party position -> entity index
    order: np.ndarray      # party positions sorted by entity
    start: np.ndarray      # entity e's members are order[start[e]:start[e + 1]]
    ids: np.ndarray        # entity id per entity
    size: np.ndarray
    weakest: np.ndarray    # weakest accepted dedup link inside the entity (nan: a single record)
    refused: int           # dedup links refused because the result would hold conflicting identifiers
    k: dict                # field -> distinct values the entity offers, per entity

    def members(self, e):
        return self.order[self.start[e]:self.start[e + 1]]


K_KEYS = {"name": ["first", "last"], "middle": ["middle"], "dob": ["dob"], "address": ["addr_full", "addr_street", "city_state", "state"],
          "phone": ["phone_own", "phone_row"], "spec_cat": ["specialty", "category"], "org": ["org_aliases"],
          **{f: [f"id_{f}"] for f in ["ssn", "npi", "dl", "tin", "license", "email", "vin", "plate", "cnpi"]}}


def build_entities_edges(pr, part, a, b, p, cfg):
    """Pool extracted parties from the accepted dedup links (a, b, p): constrained union-find,
    never pooling two different single-holder identifiers of one kind."""
    X = pr.X[part]
    n = len(X)
    a, b, p = np.asarray(a, np.int64), np.asarray(b, np.int64), np.asarray(p, float)
    cid, acc, why = constrained_clusters(n, a, b, p, single_values(X, pr.ctx.stats))
    uniq, e_of = np.unique(cid, return_inverse=True)
    order = np.argsort(e_of, kind="stable")
    size = np.bincount(e_of, minlength=len(uniq))
    start = np.r_[0, np.cumsum(size)]
    weakest = np.full(len(uniq), np.nan)
    if acc.any():
        ea = e_of[a[acc]]
        w = pd.Series(p[acc]).groupby(ea).min()
        weakest[w.index.to_numpy()] = w.to_numpy()
    rid = X["record_id"].to_numpy(dtype=object)
    first_rid = pd.Series(rid[order]).groupby(e_of[order]).min().to_numpy()
    ids = np.array([f"XE:{r}:{'P' if part == 'person' else 'B'}" for r in first_rid], dtype=object)
    k = {}
    for f, cols in K_KEYS.items():
        cols = [c for c in cols if c in X]
        if not cols:
            continue
        key = X[cols[0]].to_numpy(dtype=object).copy()
        if f in ("address",):                       # the most specific address key a party has
            for c in cols[1:]:
                key = np.where(key != "", key, X[c].to_numpy(dtype=object))
        elif f == "phone":
            key = np.where(X["phone_own"] != "", X["phone_own"], X["phone_row"])
        elif len(cols) > 1:
            for c in cols[1:]:
                key = key + "|" + X[c].to_numpy(dtype=object)
            key = np.where(np.char.str_len(key.astype(str)) > len(cols) - 1, key, "")
        d = pd.DataFrame({"e": e_of, "v": key})
        d = d[d["v"] != ""]
        k[f] = np.maximum(1, d.groupby("e")["v"].nunique().reindex(np.arange(len(uniq))).fillna(0).to_numpy())
    return Entities(part=part, e_of=e_of, order=order, start=start, ids=ids, size=size, weakest=weakest,
                    refused=int((~acc & (why != "")).sum()), k=k)


def build_entities(pr, part, D, cfg):
    """Pool extracted parties: accepted dedup links (p >= dedup_p_min, no veto, a basis in
    dedup_basis) joined by constrained union-find, never pooling two different single-holder
    identifiers of one kind. `D`: kept dedup pairs (l, r, p, veto, basis)."""
    if len(D):
        ok = (D["p"].to_numpy() >= cfg.dedup_p_min) & (D["veto"].to_numpy() == "") & \
             np.isin(D["basis"].to_numpy(), list(cfg.dedup_basis))
        a, b, p = D["l"].to_numpy()[ok], D["r"].to_numpy()[ok], D["p"].to_numpy()[ok]
    else:
        a = b = np.array([], np.int64); p = np.array([], float)
    return build_entities_edges(pr, part, a, b, p, cfg)


def entity_prior_group(X, ent):
    """An entity's prior group: professional if any member is, else business, private, unknown."""
    rank = {"professional": 0, "business": 1, "private": 2, "unknown": 3}
    g = X["prior_group"].map(rank).fillna(3).to_numpy()
    best = pd.Series(g).groupby(ent.e_of).min().reindex(np.arange(len(ent.ids))).fillna(3).to_numpy()
    inv = {v: k for k, v in rank.items()}
    return np.array([inv[int(x)] for x in best], dtype=object)


def _frames(part, X, W):
    """The minimal Prepared-like object evidence_for and pair_table read: X and W of one part."""
    return SimpleNamespace(X={part: X}, W={part: W})


# ---- 12a deduplication: extracted x extracted, one chunk ------------------------------------------
DEDUP_RAW_COLUMNS = ["l", "r", "p", "bits", "basis", "veto", "rules"]


def dedup_chunk(S, l, r, rules):
    """One chunk of candidate pairs (l < r) of extracted parties: compare, score, co-party for
    persons (through their businesses' identifier links), keep. Returns (kept rows, scored)."""
    X, ctx, plan, model, cfg, part = S["X"], S["ctx"], S["plan"], S["model"], S["cfg"], S["part"]
    lv = compare_pairs(X, X, l, r, part, ctx)
    pl = np.full(len(lv), S["logit"])
    anchors = S["anchors"]
    if part == "person":
        tl, tr = X["tie_pos"].to_numpy()[l], X["tie_pos"].to_numpy()[r]
        has = (tl >= 0) & (tr >= 0)
        lv["co_party"] = np.where(has, 1, EMPTY).astype(np.int8)
        sc = score_pairs(lv, part, model.weights[part], pl, cfg.core)
        anch = np.zeros(len(lv), bool)
        if anchors is not None and len(anchors) and has.any():
            anch[has] = np.isin(_key(np.minimum(tl[has], tr[has]), np.maximum(tl[has], tr[has])), anchors)
        if anch.any():
            lv, sc = apply_coparty(lv, sc, model.weights[part], pl, anch, cfg.core)
    else:
        sc = score_pairs(lv, part, model.weights[part], pl, cfg.core)
    p = sc["p"].to_numpy()
    keep = (p >= DEDUP_KEEP_P) | (sc["veto"].to_numpy() != "") | (sc["basis"].to_numpy() == "identifier")
    s = pd.concat([lv[keep].reset_index(drop=True), sc[keep].reset_index(drop=True)[[c_ for c_ in sc.columns]]], axis=1)
    s["rules"] = plan.rule_names(rules[keep])
    return s, len(sc)


def _w_dedup(task):
    S = _SHARED
    ci, lo, hi = task
    c = S["plan"].chunk(lo, hi)
    z = np.array([], np.int64)
    res = {"ci": ci, "scored": 0, "kept": 0, "a": z, "b": z, "p": np.array([], float), "anchors": z}
    if len(c):
        cfg = S["cfg"]
        s, n = dedup_chunk(S, c["l"].to_numpy(), c["r"].to_numpy(), c["rules"].to_numpy())
        S["store"].write("_dedup_raw", RunStore.name(S["order"], S["part"], ci), s[DEDUP_RAW_COLUMNS])
        p, veto, basis = s["p"].to_numpy(), s["veto"].to_numpy(), s["basis"].to_numpy()
        ok = (p >= cfg.dedup_p_min) & (veto == "") & np.isin(basis, list(cfg.dedup_basis))
        res.update(scored=n, kept=len(s), a=s["l"].to_numpy()[ok].astype(np.int64),
                   b=s["r"].to_numpy()[ok].astype(np.int64), p=p[ok])
        if S["part"] == "business":            # the person pass's co-party anchors (dedup_anchors)
            m = (basis == "identifier") & (p >= cfg.core.coparty_min_p) & (veto == "")
            res["anchors"] = _key(s["l"].to_numpy()[m], s["r"].to_numpy()[m])
    _trim_caches()
    return res


def score_dedup_part(pr, part, model, cfg, store, anchors=None, log=print):
    """Every candidate pair of extracted parties of one part (each unordered pair once), scored
    in parallel chunks; the kept pairs go to the store. Returns (the links entities may pool on,
    diagnostics, and for businesses the identifier links that anchor the person pass)."""
    X = pr.X[part]
    ctx = pr.ctx_dedup
    plan = CandidatePlan(X, X, part, ctx.rarity, cfg, ctx.dba_pairs, same_frame=True)
    logit = math.log2(model.prior[(part, "all")] / (1 - model.prior[(part, "all")])) if (part, "all") in model.prior else 0.0
    cuts = budget_cuts(estimated_pairs(plan), cfg.pairs_per_task) if len(X) else []
    shared = {"X": X, "ctx": ctx, "plan": plan, "model": model, "cfg": cfg, "part": part, "logit": logit,
              "anchors": anchors, "store": store, "order": 0 if part == "person" else 1}
    res = PAR.map(_w_dedup, [(i, lo, hi) for i, (lo, hi) in enumerate(cuts)], shared)
    cat = lambda k, dt: np.concatenate([x[k] for x in res]) if res else np.array([], dt)
    edges = (cat("a", np.int64), cat("b", np.int64), cat("p", float))
    n_scored, n_kept = sum(x["scored"] for x in res), sum(x["kept"] for x in res)
    anch = np.unique(cat("anchors", np.int64))
    log(f"  dedup {part}: {n_scored:,} extracted pairs scored, {n_kept:,} kept ({len(cuts)} chunks)")
    return edges, {"scored": n_scored, "kept": n_kept, "report": plan.report, "chunks": len(cuts)}, anch


# ---- 12b linking: extracted entities x watchlist, one chunk ------------------------------------------
def link_candidates(S, lo, hi):
    """The candidate (entity, watchlist party) pairs of entities lo..hi-1: every pair any member
    proposed, with the union of the rules that proposed it, sorted by (entity, party)."""
    ent, plan = S["ent"], S["plan"]
    mpos = np.concatenate([ent.members(e) for e in range(lo, hi)]) if hi > lo else np.array([], np.int64)
    c = plan.chunk_positions(mpos)
    if not len(c):
        z = np.array([], np.int64)
        return z, z, z
    e = ent.e_of[c["l"].to_numpy()]
    ek = _key(e, c["r"].to_numpy())
    order = np.argsort(ek, kind="stable")
    ek_s, b_s = ek[order], c["rules"].to_numpy()[order]
    st = np.flatnonzero(np.r_[True, ek_s[1:] != ek_s[:-1]])
    uk = ek_s[st]
    rules = np.bitwise_or.reduceat(b_s, st)
    return (uk >> 32).astype(np.int64), (uk & np.int64((1 << 32) - 1)).astype(np.int64), rules


def link_chunk(S, ce_e, ce_r, rules):
    """One chunk's candidates scored: every member of the entity compared with every candidate,
    co-party, the best level per field over the members (with the multiplicity correction),
    Fellegi-Sunter, the co-party pass, rank within the entity, keep. Returns (kept scores, kept
    aggregated levels, summary), rows sorted by (entity, watchlist party)."""
    X, W, ent, model, cfg, part, ctx = S["X"], S["W"], S["ent"], S["model"], S["cfg"], S["part"], S["ctx"]
    anchors = S["anchors"]
    fields = COMPARED_FIELDS[part] + (["co_party"] if part == "person" else [])
    n_e = ent.size[ce_e]
    g = np.repeat(np.arange(len(ce_e)), n_e)
    offs = np.arange(len(g)) - np.repeat(np.cumsum(n_e) - n_e, n_e)
    m = ent.order[ent.start[ce_e][g] + offs]
    r_m = ce_r[g]
    lv = compare_pairs(X, W, m, r_m, part, ctx)
    n_member_pairs = len(lv)
    if part == "person":
        tl, tr = X["tie_pos"].to_numpy()[m], W["tie_pos"].to_numpy()[r_m]
        has = (tl >= 0) & (tr >= 0)
        code = np.where(has, 1, EMPTY)
        if anchors is not None and len(anchors["keys"]) and has.any():
            eb = np.full(len(tl), -1, dtype=np.int64)
            eb[has] = anchors["e_of_business"][tl[has]]
            code = np.where(has & np.isin(_key(np.maximum(eb, 0), np.maximum(tr, 0)), anchors["keys"]), 0, code)
        lv["co_party"] = code.astype(np.int8)
        lv["uv_co_party"] = np.nan
    kk = {f: ent.k[f][ce_e] for f in ent.k if f in fields}
    agg = aggregate_member_levels(lv, g, fields, kk)
    del lv
    pl = S["e_logit"][ce_e]
    if part == "person":
        anchored = agg["co_party"].to_numpy() == 0
        agg["co_party"] = np.where(agg["co_party"].to_numpy() >= 0, 1, EMPTY).astype(np.int8)
        sc = score_pairs(agg, part, model.weights[part], pl, cfg.core)
        if anchored.any():
            agg, sc = apply_coparty(agg, sc, model.weights[part], pl, anchored, cfg.core)
    else:
        sc = score_pairs(agg, part, model.weights[part], pl, cfg.core)
    for f in fields:                              # the member that supplied each field's best level
        if f"src_{f}" in agg:
            s_ = agg[f"src_{f}"].to_numpy()
            agg[f"src_{f}"] = np.where(s_ >= 0, m[np.maximum(s_, 0)], -1)
    p = sc["p"].to_numpy()
    basis = sc["basis"].to_numpy()
    bi = pd.Series(basis).map({b: i for i, b in enumerate(BASIS_ORDER)}).to_numpy()
    pbin = np.clip(np.searchsorted(P_BINS, p, side="right") - 1, 0, len(P_BINS) - 2)
    vetoed = sc["veto"].to_numpy() != ""
    hist = np.zeros((len(BASIS_ORDER), len(P_BINS) - 1), dtype=np.int64)
    np.add.at(hist, (bi[~vetoed], pbin[~vetoed]), 1)
    rank = _rank_within(ce_e, np.where(vetoed, -1.0, p))
    keep = (p >= cfg.keep_p_floor) | (rank <= cfg.top_k) | (basis == "identifier") | vetoed
    s = sc[keep].copy()
    s.insert(0, "rank", np.where(vetoed[keep], np.nan, rank[keep]))
    s.insert(0, "rules", S["plan"].rule_names(rules[keep]))
    s.insert(0, "r", ce_r[keep]); s.insert(0, "e", ce_e[keep])
    s["prior_logit"] = pl[keep]
    s = s.reset_index(drop=True)
    a = agg[keep].reset_index(drop=True)
    order = np.lexsort((s["r"].to_numpy(), s["e"].to_numpy()))
    s, a = s.iloc[order].reset_index(drop=True), a.iloc[order].reset_index(drop=True)
    es = pd.DataFrame({"e": ce_e, "veto": vetoed}).groupby("e").agg(candidates=("veto", "size"), vetoed=("veto", "sum"))
    rule_counts = {name: int(((rules & bit) > 0).sum()) for name, bit in S["plan"].bit.items()}
    return s, a, {"hist": hist, "scored": len(sc), "member_pairs": n_member_pairs, "Es": es,
                  "rule_counts": rule_counts, "kept": len(s)}


def link_chunk_evidence(S, s, a, keys=None):
    """Evidence rows of one chunk's kept pairs (all of them, or those whose key is in `keys`)."""
    X, W, ent, part = S["X"], S["W"], S["ent"], S["part"]
    if keys is not None:
        m = np.isin(_key(s["e"].to_numpy(), s["r"].to_numpy()), keys)
        s, a = s[m].reset_index(drop=True), a[m].reset_index(drop=True)
    if not len(s):
        return None
    e = s["e"].to_numpy(dtype=np.int64)
    r = s["r"].to_numpy(dtype=np.int64)
    pairs_part = pd.DataFrame({"pair_id": ent.ids[e] + "~" + W["party_id"].to_numpy()[r]})
    return evidence_for(_frames(part, X, W), part, pairs_part, a, s, S["model"].weights[part], disp=S.get("disp"))


def _w_link(task):
    S = _SHARED
    ci, lo, hi = task
    cfg, part = S["cfg"], S["part"]
    name = RunStore.name(S["order"], part, ci)
    ce_e, ce_r, rules = link_candidates(S, lo, hi)
    z = np.array([], np.int64)
    out = {"ci": ci, "hist": np.zeros((len(BASIS_ORDER), len(P_BINS) - 1), dtype=np.int64), "scored": 0,
           "member_pairs": 0, "Es": pd.DataFrame(columns=["candidates", "vetoed"]), "rule_counts": {}, "kept": 0,
           "edges": (z, z, np.array([], float)), "anchors": z, "proposed": z}
    if S.get("keep"):
        out["keys"] = _key(ce_e, ce_r)
    if not len(ce_e):
        return out
    s, a, info = link_chunk(S, ce_e, ce_r, rules)
    out.update(info)
    S["store"].write("_link_raw", name, s)
    if S["evidence"] is not None:
        ev = link_chunk_evidence(S, s, a, None if S["evidence"] == "all" else S["evidence_keys"])
        if ev is not None:
            S["store"].write("evidence", name, ev)
    p = s["p"].to_numpy(dtype=float)
    veto = s["veto"].to_numpy() != ""
    basis = s["basis"].to_numpy()
    on_name = np.isin(basis, ["contextual", "name_only", "none"])
    edge = (p >= cfg.link_cluster_p) & ~veto & ~on_name          # the links that may join a cluster
    e, r = s["e"].to_numpy(dtype=np.int64), s["r"].to_numpy(dtype=np.int64)
    out["edges"] = (e[edge], r[edge], p[edge])
    if part == "business":                                       # link_anchors: the person pass's co-party
        m = (basis == "identifier") & (p >= cfg.core.coparty_min_p) & ~veto
        out["anchors"] = _key(e[m], r[m])
    q = S.get("probe_keys")
    if q is not None and len(q):
        uk = _key(ce_e, ce_r)
        out["proposed"] = uk[np.isin(uk, q)]
    if S.get("keep"):
        out["s"], out["a"] = s, a
    _trim_caches()
    return out


def score_link_part(pr, part, ent, model, cfg, store, anchors=None, log=print, probes=None, keep=False):
    """Every candidate (entity, watchlist party) of one part, scored in parallel chunks of
    entities; the kept pairs and their evidence go to the store. `keep`: also return every
    scored pair and its levels in memory (small runs: the self-tests)."""
    X, W = pr.X[part], pr.W[part]
    ctx = pr.ctx
    plan = CandidatePlan(X, W, part, ctx.rarity, cfg, ctx.dba_pairs)
    for rep in plan.report:
        log(f"  {part:<8} {rep['rule']:<24} pairs {rep['pairs_before_refine']:>12,}"
            + (f"  oversized keys {rep['oversized_keys']} -> dropped {rep['dropped_keys']} ({rep['dropped_pairs']:,} pairs)"
               if rep['oversized_keys'] or rep['dropped_keys'] else ""))
    grp_logit = {g: math.log2(model.prior[(part, g)] / (1 - model.prior[(part, g)])) for g in GROUPS if (part, g) in model.prior}
    e_group = entity_prior_group(X, ent)
    e_logit = pd.Series(e_group).map(grp_logit).fillna(0.0).to_numpy(dtype=float)
    E = len(ent.ids)
    # chunks of entities by estimated member comparisons: each member of an entity meets every
    # candidate any member proposed, so an entity costs about size x its members' pairs
    est = estimated_pairs(plan)
    per_e = np.bincount(ent.e_of, weights=est, minlength=E) * ent.size if E else np.zeros(0)
    cuts = budget_cuts(per_e, cfg.pairs_per_task) if E else []
    probe_keys = probes.keys(pr, part, ent) if probes is not None else None
    ev_keys = probes.evidence_keys(pr, part, ent) if probes is not None else None
    ev_mode = None if (probes is not None and probes.no_evidence) else ("all" if ev_keys is None else "keys")
    disp = {"X": display_values(X), "W": display_values(W)} if ev_mode is not None else None
    shared = {"X": X, "W": W, "ctx": ctx, "part": part, "ent": ent, "model": model, "cfg": cfg, "anchors": anchors,
              "plan": plan, "e_logit": e_logit, "store": store, "order": 0 if part == "business" else 1,
              "probe_keys": probe_keys, "evidence": ev_mode, "evidence_keys": ev_keys, "keep": keep, "disp": disp}
    res = PAR.map(_w_link, [(i, lo, hi) for i, (lo, hi) in enumerate(cuts)], shared)
    hist = sum((x["hist"] for x in res), np.zeros((len(BASIS_ORDER), len(P_BINS) - 1), dtype=np.int64))
    rule_counts = defaultdict(int)
    for x in res:
        for k, v in x["rule_counts"].items():
            rule_counts[k] += v
    n_scored, n_member, n_kept = (sum(x[k] for x in res) for k in ("scored", "member_pairs", "kept"))
    es = [x["Es"] for x in res if len(x["Es"])]
    Es = pd.concat(es).sort_index() if es else pd.DataFrame(columns=["candidates", "vetoed"])
    cat = lambda k, i, dt: np.concatenate([x[k][i] for x in res]) if res else np.array([], dt)
    edges = (cat("edges", 0, np.int64), cat("edges", 1, np.int64), cat("edges", 2, float))
    anch = np.unique(np.concatenate([x["anchors"] for x in res])) if res else np.array([], np.int64)
    proposed = np.concatenate([x["proposed"] for x in res]) if res else np.array([], np.int64)
    diag = {"hist": hist, "scored": n_scored, "member_pairs": n_member, "rule_counts": dict(rule_counts),
            "report": plan.report, "exhaustive": plan.exhaustive, "kept": n_kept, "chunks": len(cuts),
            "cuts": cuts, "proposed": proposed}
    S_ = L_ = None
    if keep:
        diag["keys"] = np.concatenate([x["keys"] for x in res]) if res else np.array([], np.int64)
        got = [(x["s"], x["a"]) for x in res if "s" in x]
        S_ = pd.concat([g[0] for g in got], ignore_index=True) if got else pd.DataFrame(
            columns=["e", "r", "rules", "rank", "p", "veto", "basis", "bits"])
        L_ = pd.concat([g[1] for g in got], ignore_index=True) if got else pd.DataFrame()
    log(f"  link {part}: {n_scored:,} entity x watchlist pairs scored ({n_member:,} member comparisons), "
        f"{n_kept:,} kept ({len(cuts)} chunks)")
    return S_, L_, Es, diag, edges, anch


def link_anchors(S_business, ent_business, cfg):
    """Business entity x watchlist links on an identifier (p >= coparty_min_p, no veto): the
    co-party anchors of the person pass, keyed (business entity, watchlist business)."""
    keys = np.array([], np.int64)
    if len(S_business):
        m = (S_business["basis"] == "identifier") & (S_business["p"] >= cfg.core.coparty_min_p) & (S_business["veto"] == "")
        keys = np.unique(_key(S_business.loc[m, "e"], S_business.loc[m, "r"]))
    return {"keys": keys, "e_of_business": ent_business.e_of}


# ---- 12c consistent clusters -------------------------------------------------------------------
class _NodeValues(dict):
    """node -> frozenset of single-holder values; filled for the nodes that can be reached
    through links (constrained_clusters reads no other node), every other node holds none."""
    def __missing__(self, key):
        return frozenset()


def cluster_links(pr, part, ent, edges, cfg):
    """Entities and watchlist parties joined by links at p >= link_cluster_p that rest on more
    than a name (edges, from every chunk), by constrained union-find: transitive, and never
    holding two different single-holder values of SSN / NPI / DL (one state) / TIN. Name-only
    links stay visible and are labelled, but never join a cluster. Returns (cluster id per node:
    entities, then watchlist parties; {pair key: 'linked' or 'conflict: ...'} per link)."""
    X, W = pr.X[part], pr.W[part]
    E, NW = len(ent.ids), len(W)
    e, r, p = edges
    if not len(e):
        return np.arange(E + NW), {}
    stats = pr.ctx.stats
    node_vals = {}
    for f in VETO_FIELDS:
        s = stats.single.get(f, set())
        col = f"id_{f}"
        xv = X[col].to_numpy(dtype=object) if col in X else np.full(len(X), "", dtype=object)
        ev = defaultdict(set)
        for pos, v in enumerate(xv):
            if v and v in s:
                ev[int(ent.e_of[pos])].add(v)
        nv = _NodeValues({i: frozenset(st) for i, st in ev.items()})
        wv = W[col].to_numpy(dtype=object) if col in W else np.full(len(W), "", dtype=object)
        for j in np.unique(r):
            v = wv[j]
            if v and v in s:
                nv[E + int(j)] = frozenset([v])
        node_vals[f] = nv
    cid, acc, why = constrained_clusters(E + NW, e, E + r, p, node_vals)
    st_e = np.where(acc, "linked", "conflict: " + why.astype(str))
    return cid, dict(zip(_key(e, r).tolist(), st_e.tolist()))


def score_all(pr, models, cfg, store, log=print, probes=None, keep=False):
    """Deduplication (businesses first: their identifier links anchor the persons' co-party),
    entities, linking, clusters. Large results go to `store`; the returned dict holds the
    entities, the per-part diagnostics and the clusters."""
    res = {"dedup": {}, "entities": {}, "link": {}, "clusters": {}}
    edges_b, dd_b, anch_b = score_dedup_part(pr, "business", models["dedup"], cfg, store, log=log)
    edges_p, dd_p, _ = score_dedup_part(pr, "person", models["dedup"], cfg, store, anchors=anch_b, log=log)
    res["dedup"] = {"business": (None, dd_b), "person": (None, dd_p)}
    for part, (a, b, p) in (("business", edges_b), ("person", edges_p)):
        ent = build_entities_edges(pr, part, a, b, p, cfg)
        res["entities"][part] = ent
        log(f"  entities [{part}]: {len(pr.X[part]):,} extracted parties -> {len(ent.ids):,} entities "
            f"({int((ent.size > 1).sum()):,} pooled, largest {int(ent.size.max()) if len(ent.size) else 0}); "
            f"{ent.refused} dedup links refused (conflicting identifiers)")
    eb, ep = res["entities"]["business"], res["entities"]["person"]
    Sb, Lb, Es_b, D_b, edges_lb, anch_lb = score_link_part(pr, "business", eb, models["link"], cfg, store, log=log,
                                                             probes=probes, keep=keep)
    anchors = {"keys": anch_lb, "e_of_business": eb.e_of}
    log(f"  co-party anchors (business entities linked on an identifier, p >= {cfg.core.coparty_min_p}): {len(anchors['keys'])}")
    Sp, Lp, Es_p, D_p, edges_lp, _ = score_link_part(pr, "person", ep, models["link"], cfg, store, anchors=anchors,
                                                       log=log, probes=probes, keep=keep)
    res["link"] = {"business": (Sb, Lb, Es_b, D_b), "person": (Sp, Lp, Es_p, D_p)}
    for part, edges in (("business", edges_lb), ("person", edges_lp)):
        ent = res["entities"][part]
        cid, es = cluster_links(pr, part, ent, edges, cfg)
        E = len(ent.ids)
        res["clusters"][part] = {"cid": cid, "edge_status": es, "e_cid": cid[:E], "w_cid": cid[E:]}
    return res


## 13 · Roll-up

**Entities**: one row per extracted party (record_id + part). Its probability is its single
best (non-vetoed) match; every other kept candidate is listed. `rests_on_name` is true when
the best match's basis is contextual or name_only: a filter, never a threshold. An extracted
party with no candidate is shown as such ("no candidate proposed"), not dropped.

The entity roll-up is computed chunk by chunk in the second pass (every pair of an entity is
in one chunk) and joined on the driver.

**Claims**: per claim, rows, parties, the highest p and its entity, counts by basis x p band,
and name-only matches at p >= 0.5.

**Where Splink would do better.** Splink clusters pairwise predictions into entities
(connected components at a threshold); here the unit is one extracted party against the list,
so the roll-up is a best-match choice with the others listed.

In [ ]:
def pair_table(pr, part, S, ent, status=None, e_cid=None, w_cid=None):
    """Kept entity x watchlist pairs with identities, names, ranking and cluster status."""
    if not len(S):
        return pd.DataFrame()
    X, W = pr.X[part], pr.W[part]
    e, r = S["e"].to_numpy(dtype=np.int64), S["r"].to_numpy(dtype=np.int64)
    first = ent.order[ent.start[e]]                     # a member for display
    out = pd.DataFrame({
        "pair_id": ent.ids[e] + "~" + W["party_id"].to_numpy()[r],
        "entity_id": ent.ids[e], "part": part, "entity_records": ent.size[e],
        "extracted_record_id": X["record_id"].to_numpy()[first],
        "claim_id": X["claim_id"].to_numpy()[first], "extracted_name": X["display_name"].to_numpy()[first],
        "watchlist_record_id": W["record_id"].to_numpy()[r], "watchlist_name": W["display_name"].to_numpy()[r],
        "p": S["p"].to_numpy(), "bits": S["bits"].to_numpy(), "prior_bits": S["prior_logit"].to_numpy(),
        "basis": S["basis"].to_numpy(), "rests_on_name": np.isin(S["basis"].to_numpy(), ["contextual", "name_only"]),
        "veto": S["veto"].to_numpy(), "rank": S["rank"].to_numpy(), "proposed_by": S["rules"].to_numpy(),
        "cluster_status": status if status is not None and len(status) == len(S) else "",
        "cluster_id": (np.array([f"C{part[0].upper()}{x}" for x in e_cid[e]], dtype=object) if e_cid is not None else ""),
        "_e": e, "_r": r})
    for f in PART_FIELDS[part]:
        out[f"bits_{f}"] = S[f"bits_{f}"].to_numpy()
    return out


def _listing(df, what):
    """'W1 (0.97, identifier); W2 (...)' per entity (_e)."""
    if not len(df):
        return pd.Series(dtype=object)
    txt = [f"{w} ({p:.3g}, {b})" for w, p, b in zip(df["watchlist_record_id"], df["p"], df[what])]
    return pd.Series(txt, index=df.index).groupby(df["_e"].to_numpy()).agg("; ".join)


def rollup_rows(pr, entities, ents):
    """One row per extracted party (record_id + part), pointing back to its entity's result."""
    out = []
    for part in ("person", "business"):
        X = pr.X[part]
        ent = ents[part]
        E = entities[entities["part"] == part].set_index("_e")
        e = ent.e_of
        d = pd.DataFrame({"party_id": X["party_id"], "record_id": X["record_id"], "part": part,
                          "claim_id": X["claim_id"], "note_id": X["note_id"], "name": X["display_name"],
                          "entity_id": ent.ids[e], "entity_records": ent.size[e]})
        for c in ("best_watchlist_record_id", "best_watchlist_name", "p", "basis", "rests_on_name", "match_status",
                  "cluster_id", "cluster_watchlist_ids", "status", "category_used", "prior_group"):
            d[c] = E[c].reindex(e).to_numpy()
        out.append(d)
    return pd.concat(out, ignore_index=True)


def p_band(p):
    return np.select([p >= 0.9, p >= 0.8, p >= 0.5, p >= 0.1], ["p>=0.9", "0.8-0.9", "0.5-0.8", "0.1-0.5"], "p<0.1")


def rollup_claims(rows, xrows_claims):
    """Per claim: rows, parties, the highest p and its party, counts by basis x p band."""
    e = rows.copy()
    e["band"] = p_band(e["p"].to_numpy())
    g = e.groupby("claim_id", sort=True)
    top = e.sort_values(["claim_id", "p"], ascending=[True, False], kind="stable").groupby("claim_id").head(1).set_index("claim_id")
    out = pd.DataFrame({"rows": xrows_claims.reindex(g.size().index).fillna(0).astype(int),
                        "parties": g.size(), "entities": g["entity_id"].nunique(), "max_p": g["p"].max()})
    out["max_p_party"] = top["party_id"].reindex(out.index)
    out["max_p_name"] = top["name"].reindex(out.index)
    out["max_p_basis"] = top["basis"].reindex(out.index)
    ct = pd.crosstab(e["claim_id"], e["basis"] + " " + e["band"])
    out = out.join(ct, how="left").fillna(0)
    out["name_only_at_p_0.5"] = e[(e["basis"] == "name_only") & (e["p"] >= 0.5)].groupby("claim_id").size().reindex(out.index).fillna(0).astype(int)
    return out.reset_index().rename(columns={"index": "claim_id"})


# ---- the second pass: cluster status, the candidates table, the entity roll-up ------------------------
PAIR_TABLE_COLUMNS = ["pair_id", "entity_id", "part", "entity_records", "extracted_record_id", "claim_id",
                      "extracted_name", "watchlist_record_id", "watchlist_name", "p", "bits", "prior_bits", "basis",
                      "rests_on_name", "veto", "rank", "proposed_by", "cluster_status", "cluster_id", "_e", "_r"]


def candidate_columns(cfg, parts):
    """One schema for every part file of the candidates table (so the folder reads as one
    table): the pair_table columns, then the bits of every field of the parts that have pairs,
    in the order the business and person tables concatenate, then the baseline's columns."""
    bits = []
    for part in ("business", "person"):
        if part in parts:
            bits += [f"bits_{f}" for f in PART_FIELDS[part] if f"bits_{f}" not in bits]
    return PAIR_TABLE_COLUMNS + bits + (["p_baseline", "baseline_match"] if cfg.baseline else [])


def link_chunk_final(S2, s):
    """pair_table for one chunk's scored pairs, each with the status the clustering gave it:
    vetoed; name only (p >= link_cluster_p but resting on a name); linked or conflict (a link
    that could join a cluster); same cluster (weak direct link); below threshold."""
    part, ent, X, W, cfg = S2["part"], S2["ent"], S2["X"], S2["W"], S2["cfg"]
    e = s["e"].to_numpy(dtype=np.int64)
    r = s["r"].to_numpy(dtype=np.int64)
    p = s["p"].to_numpy(dtype=float)
    veto = s["veto"].to_numpy() != ""
    basis = s["basis"].to_numpy()
    on_name = np.isin(basis, ["contextual", "name_only", "none"])
    edge = (p >= cfg.link_cluster_p) & ~veto & ~on_name
    status = np.full(len(s), "below threshold", dtype=object)
    status[veto] = "vetoed"
    status[(p >= cfg.link_cluster_p) & ~veto & on_name] = "name only: visible, not clustered"
    if edge.any():
        es = S2["edge_status"]
        status[np.flatnonzero(edge)] = [es[k] for k in _key(e[edge], r[edge]).tolist()]
    cid, E = S2["cid"], len(ent.ids)
    same = cid[e] == cid[E + r]
    status[same & (status == "below threshold")] = "same cluster (weak direct link)"
    P = pair_table(_frames(part, X, W), part, s, ent, status, cid[:E], cid[E:])
    if S2.get("baseline") is not None:
        P = baseline_columns(P, part, ent, S2["baseline"], cfg)
    return P.reindex(columns=S2["columns"])


def entity_pair_columns(P, cfg):
    """The pair-derived columns of the entity roll-up for the entities in P (one chunk: every
    pair of an entity is in the same chunk). Its p is the best link inside its cluster, or
    without a cluster its best candidate; conflicting candidates and the others are listed.
    Returns (best rows, best non-name p, conflicts, others, weakest cluster link)."""
    ok = P[P["veto"] == ""].sort_values(["_e", "p", "bits", "watchlist_record_id"],
                                        ascending=[True, False, False, True], kind="stable")
    linked = ok[ok["cluster_status"] == "linked"]
    best_l = linked.groupby("_e").head(1).set_index("_e")
    best_a = ok.groupby("_e").head(1).set_index("_e")
    best = best_a.copy()
    best.loc[best_l.index] = best_l
    best["match_status"] = np.where(best.index.isin(best_l.index), "linked in a consistent cluster",
                                    best["cluster_status"].to_numpy())
    non_name = ok[~ok["rests_on_name"] & (ok["basis"] != "none")].groupby("_e")["p"].max()
    conf = ok[ok["cluster_status"].astype(str).str.startswith("conflict")]
    conf_txt = _listing(conf, "cluster_status")
    best_r = best["watchlist_record_id"]
    rest = ok[ok["watchlist_record_id"].to_numpy() != best_r.reindex(ok["_e"]).to_numpy()]
    pos = rest.groupby("_e").cumcount().to_numpy()
    others = _listing(rest[pos < cfg.top_k], "basis")
    weak = linked.groupby("_e")["p"].min()
    return (best[["watchlist_record_id", "watchlist_name", "p", "basis", "match_status"]], non_name, conf_txt,
            others, weak)


def sensitivity_best(P):
    """Each entity's best non-vetoed pair by p, then bits (the sensitivity table's input)."""
    b = P[P["veto"] == ""].sort_values(["part", "_e", "p", "bits"], ascending=[True, True, False, False]) \
        .groupby(["part", "_e"]).head(1)
    return b[["part", "_e", "entity_id", "bits"]]


def _w_link_final(task):
    S2 = _SHARED
    ci = task
    name = RunStore.name(S2["order"], S2["part"], ci)
    s = S2["store"].read_part("_link_raw", name)
    if s is None or not len(s):
        return {"ci": ci, "rows": 0}
    P = link_chunk_final(S2, s)
    top = P[(P["basis"] == "name_only") & (P["veto"] == "")]
    out = {"ci": ci, "rows": len(P), "entity": entity_pair_columns(P, S2["cfg"]), "sens": sensitivity_best(P),
           "veto_counts": P.loc[P["veto"] != "", "veto"].value_counts(),
           "status_counts": P["cluster_status"].value_counts(),
           "groups": P.assign(band=p_band(P["p"].to_numpy())).groupby(["basis", "band"]).size(),
           "top_name_only": top.sort_values("p", ascending=False, kind="stable").head(20)}
    for q in S2["consumers"]:
        out[q.name] = q(S2, P)
    if S2["write"]:
        S2["store"].write("candidates", name, P)
    _trim_caches()
    return out


def finish_link_part(pr, part, ent, cluster, cfg, store, n_chunks, columns, consumers=(), write=True):
    """The second pass over one part's chunks, in parallel. Returns each chunk's pieces, in order."""
    pre = None
    if cfg.baseline:                    # the baseline's cleaned names and categories, once
        xn, xc = baseline_frame(pr.X[part], part, cfg, False)
        wn, wc = baseline_frame(pr.W[part], part, cfg, True)
        pre = (xn, xc, wn, wc)
    shared = {"X": pr.X[part], "W": pr.W[part], "part": part, "ent": ent, "cfg": cfg, "cid": cluster["cid"],
              "edge_status": cluster["edge_status"], "store": store, "order": 0 if part == "business" else 1,
              "baseline": pre, "consumers": tuple(consumers), "write": write, "columns": columns}
    return PAR.map(_w_link_final, list(range(n_chunks)), shared)


def _w_dedup_final(task):
    S2 = _SHARED
    name = RunStore.name(S2["order"], S2["part"], task)
    D = S2["store"].read_part("_dedup_raw", name)
    if D is None:
        return 0
    X, ent, part = S2["X"], S2["ent"], S2["part"]
    l, r = D["l"].to_numpy(dtype=np.int64), D["r"].to_numpy(dtype=np.int64)
    T = pd.DataFrame({
        "part": part, "record_id_1": X["record_id"].to_numpy()[l], "record_id_2": X["record_id"].to_numpy()[r],
        "name_1": X["display_name"].to_numpy()[l], "name_2": X["display_name"].to_numpy()[r],
        "claim_1": X["claim_id"].to_numpy()[l], "claim_2": X["claim_id"].to_numpy()[r],
        "p": D["p"].to_numpy(), "bits": D["bits"].to_numpy(), "basis": D["basis"].to_numpy(),
        "veto": D["veto"].to_numpy(), "proposed_by": D["rules"].to_numpy(),
        "same_entity": ent.e_of[l] == ent.e_of[r],
        "entity_id": np.where(ent.e_of[l] == ent.e_of[r], ent.ids[ent.e_of[l]], "")})
    S2["store"].write("dedup_pairs", name, T)
    return len(T)


def finish_dedup_part(pr, part, ent, store, n_chunks):
    """The dedup table (kept extracted x extracted pairs, with whether they pooled), per chunk."""
    shared = {"X": pr.X[part], "ent": ent, "part": part, "store": store, "order": 0 if part == "person" else 1}
    return sum(PAR.map(_w_dedup_final, list(range(n_chunks)), shared))


def _concat(frames):
    frames = [f for f in frames if f is not None and len(f)]
    return pd.concat(frames) if frames else None


def rollup_entities(pr, final, E_by_part, ents, clusters, cfg):
    """One row per extracted entity. Its p is its best link inside its cluster (consistent
    watchlist records only) or, without a cluster, its best candidate; its status says which;
    conflicting candidates are listed apart; every other candidate is listed. `final`: the
    second pass's per-chunk pieces of each part."""
    out = []
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        ent = ents[part]
        E = len(ent.ids)
        Es = E_by_part[part]
        first = ent.order[ent.start[:-1]] if E else np.array([], np.int64)
        members_rid = pd.Series(X["record_id"].to_numpy()[ent.order]).groupby(ent.e_of[ent.order]).agg("; ".join)
        claims = pd.Series(X["claim_id"].to_numpy()[ent.order]).groupby(ent.e_of[ent.order]).agg(
            lambda s: "; ".join(sorted({x for x in s if x})))
        base = pd.DataFrame({
            "entity_id": ent.ids, "part": part, "records": ent.size,
            "record_ids": members_rid.reindex(np.arange(E)).fillna("").to_numpy(),
            "claim_ids": claims.reindex(np.arange(E)).fillna("").to_numpy(),
            "name": X["display_name"].to_numpy()[first] if E else [],
            "category_given": X["category_given"].to_numpy()[first] if E else [],
            "category_inferred": X["category_inferred"].to_numpy()[first] if E else [],
            "category_used": X["category"].to_numpy()[first] if E else [],
            "category_mismatch": (pd.Series(X["category_mismatch"].to_numpy()[ent.order]).groupby(ent.e_of[ent.order]).any()
                                  .reindex(np.arange(E)).fillna(False).to_numpy()),
            "prior_group": entity_prior_group(X, ent) if E else [],
            "dedup_weakest_p": ent.weakest})
        idx = np.arange(E)
        base["candidates"] = Es["candidates"].reindex(idx).fillna(0).astype(int).to_numpy() if len(Es) else 0
        base["vetoed_candidates"] = Es["vetoed"].reindex(idx).fillna(0).astype(int).to_numpy() if len(Es) else 0
        e_cid = clusters[part]["e_cid"]
        w_cid = clusters[part]["w_cid"]
        base["cluster_id"] = [f"C{part[0].upper()}{c}" for c in e_cid]
        if len(W):
            wm = np.isin(w_cid, np.unique(e_cid))       # only clusters an entity belongs to are read
            in_cl = pd.Series(W["record_id"].to_numpy()[wm]).groupby(w_cid[wm]).agg(lambda s: "; ".join(sorted(s)))
        else:
            in_cl = pd.Series(dtype=object)
        base["cluster_watchlist_ids"] = pd.Series(e_cid).map(in_cl).fillna("").to_numpy()
        cols = dict(best_watchlist_record_id="", best_watchlist_name="", p=0.0, basis="none", match_status="",
                    best_non_name_p=0.0, other_candidates="", conflicting_candidates="", cluster_weakest_p=np.nan)
        for c, v in cols.items():
            base[c] = v
        chunks = final.get(part, [])
        if sum(f.get("rows", 0) for f in chunks):
            pieces = [f["entity"] for f in chunks if f.get("rows")]
            best = _concat([pc[0] for pc in pieces])
            if best is None:
                best = pd.DataFrame(columns=["watchlist_record_id", "watchlist_name", "p", "basis", "match_status"])
            one = lambda i, dt: _concat([pc[i] for pc in pieces]) if _concat([pc[i] for pc in pieces]) is not None \
                else pd.Series(dtype=dt)
            non_name, conf_txt, others, weak = one(1, float), one(2, object), one(3, object), one(4, float)
            base["best_watchlist_record_id"] = best["watchlist_record_id"].reindex(idx).fillna("").to_numpy()
            base["best_watchlist_name"] = best["watchlist_name"].reindex(idx).fillna("").to_numpy()
            base["p"] = best["p"].reindex(idx).fillna(0.0).to_numpy(dtype=float)
            base["basis"] = best["basis"].reindex(idx).fillna("none").to_numpy()
            base["match_status"] = best["match_status"].reindex(idx).fillna("").to_numpy()
            base["best_non_name_p"] = non_name.reindex(idx).fillna(0.0).to_numpy(dtype=float)
            base["other_candidates"] = others.reindex(idx).fillna("").to_numpy()
            base["conflicting_candidates"] = conf_txt.reindex(idx).fillna("").to_numpy()
            base["cluster_weakest_p"] = weak.reindex(idx).to_numpy(dtype=float)
        base["rests_on_name"] = np.isin(base["basis"].to_numpy(), ["contextual", "name_only"])
        base.loc[base["best_watchlist_record_id"] == "", "cluster_watchlist_ids"] = ""
        base["status"] = np.where(base["candidates"] == 0, "no candidate proposed",
                                  np.where(base["best_watchlist_record_id"] == "", "only vetoed candidates", "scored"))
        base["_e"] = idx
        out.append(base)
    return pd.concat(out, ignore_index=True)


## 14 · Evidence

One row per kept pair and compared field that is not empty: both values as normalized,
the level, m with its source chain and pair count, the u actually used (value-specific or
field-level) with its source and count, and the bits. A pair's rows sum to its total bits
(checked in the self-tests); the prior is on the candidate row.

The evidence is written chunk by chunk, beside the chunk's candidates (the same part name in
the `evidence` folder), so one pair's evidence is always one small file away.

**Where Splink would do better.** Splink's waterfall chart shows the same breakdown
graphically, per pair, in the browser; this is its table form, filterable in Excel.

In [ ]:
DISPLAY = {"name": lambda F: (F["first"] + " " + F["last"]).str.strip(), "middle": lambda F: F["middle"],
           "org": lambda F: F["org_aliases"].str.replace("|", " / ", regex=False), "dob": lambda F: F["dob"],
           "address": lambda F: F["raw_address"], "phone": lambda F: (F["phone_own"] + " " + F["phone_row"]).str.strip(),
           "spec_cat": lambda F: (F["specialty"] + " / " + F["category"]).str.strip(" /"),
           "co_party": lambda F: F["tie"]}


def decision_cards(evidence, pair_ids=None, max_fields=12):
    """One compact line per pair: field level (bits), strongest first, e.g.
    'ssn exact +19.3; name first_close +11.2; dob differs -5.1'."""
    ev = evidence if pair_ids is None else evidence[evidence["pair_id"].isin(set(pair_ids))]
    if not len(ev):
        return pd.Series(dtype=object)
    ev = ev.assign(_a=-ev["bits"].abs()).sort_values(["pair_id", "_a"], kind="stable")
    txt = ev["field"] + " " + ev["level"] + " " + ev["bits"].map(lambda b: f"{b:+.1f}") + \
        np.where(ev["veto"].astype(bool), " VETO", "")
    return txt.groupby(ev["pair_id"].to_numpy()).agg(lambda s: "; ".join(list(s)[:max_fields]))


def sensitivity(entities_best, prior_df, cfg):
    """How many entities cross 0.5 / 0.8 / 0.9 if the prior or the recall behind it moved."""
    rows = []
    for _, pr_ in prior_df.iterrows():
        sub = entities_best[(entities_best["part"] == pr_["part"]) & (entities_best["prior_group"] == pr_["group"])]
        bits = sub["bits"].to_numpy()
        for mult in cfg.sensitivity_prior_mult:
            for rmult in cfg.sensitivity_recall:
                rec = min(1.0, pr_["recall"] * rmult) if pr_["recall"] > 0 else 0.0
                base = pr_["prior"] * mult * (pr_["recall"] / rec if rec > 0 else 1.0)
                base = min(base, 0.5)
                logit = math.log2(base / (1 - base)) + bits
                p = 1 / (1 + np.exp2(-logit))
                row = {"part": pr_["part"], "group": pr_["group"], "prior_multiplier": mult,
                       "recall_multiplier": rmult, "prior": base, "entities": len(sub)}
                for t in cfg.p_bands:
                    row[f"entities_p>={t}"] = int((p >= t).sum())
                rows.append(row)
    return pd.DataFrame(rows)


def display_values(F):
    """The display value of every displayed field for every party of one frame (the evidence's
    value columns), computed once per frame rather than once per chunk."""
    return {f: DISPLAY[f](F).to_numpy(dtype=object) for f in DISPLAY}


def evidence_rows_fast(pair_ids, levels, scored, part, weights, values_l, values_r, return_positions=False):
    """evidence_rows (matching core), vectorized: the weights are looked up by level code per
    field instead of per pair, and the rows are put in (pair_id, field) order by integer keys
    instead of a string sort. The rows, columns and values are the core's (a self-test compares
    the two). With return_positions, also each row's pair position and field."""
    pid = np.asarray(pair_ids, dtype=object)
    frames, pos, fld = [], [], []
    for f in PART_FIELDS[part]:
        if f not in levels:
            continue
        lv = levels[f].to_numpy()
        keep = lv >= 0
        if not keep.any():
            continue
        idx = np.flatnonzero(keep)
        codes = lv[idx].astype(np.int64)
        wf = weights[weights["field"] == f].set_index("level_code")
        take = lambda col: wf[col].reindex(codes).to_numpy()
        u_all, use_v = used_u(levels, f, weights)
        k = levels[f"k_{f}"].to_numpy()[idx] if f"k_{f}" in levels else np.ones(len(idx))
        frames.append(pd.DataFrame({
            "pair_id": pid[idx], "field": f,
            "value_extracted": values_l.get(f, np.full(len(lv), ""))[idx] if f in values_l else "",
            "value_watchlist": values_r.get(f, np.full(len(lv), ""))[idx] if f in values_r else "",
            "level": [FIELD_LEVELS[f][c] for c in codes],
            "m": take("m"), "m_source": take("m_source"), "m_pairs": take("m_pairs"),
            "u": u_all[idx],
            "u_source": np.where(use_v[idx], take("u_value_specific"), take("u_source")),
            "u_pairs": take("u_pairs"),
            "values_offered": k,
            "bits": scored[f"bits_{f}"].to_numpy()[idx],
            "veto": levels[f"veto_{f}"].to_numpy()[idx] if f"veto_{f}" in levels else False,
        }))
        pos.append(idx)
        fld.append(np.full(len(idx), f, dtype=object))
    if not frames:
        ev = pd.DataFrame(columns=EVIDENCE_COLUMNS)
        return (ev, np.array([], np.int64), np.array([], dtype=object)) if return_positions else ev
    ev = pd.concat(frames, ignore_index=True)
    pos, fld = np.concatenate(pos), np.concatenate(fld)
    # (pair_id, field) order: rank the pair ids once, then sort integer keys
    uniq_f = np.array(sorted(set(fld)), dtype=object)
    f_rank = pd.Series(np.arange(len(uniq_f)), index=uniq_f).reindex(fld).to_numpy()
    p_rank = np.empty(len(pid), dtype=np.int64)
    p_rank[np.argsort(pid, kind="stable")] = np.arange(len(pid))
    order = np.lexsort((f_rank, p_rank[pos]))
    ev = ev.iloc[order].reset_index(drop=True)
    return (ev, pos[order], fld[order]) if return_positions else ev


def evidence_for(pr, part, pairs_part, levels_part, S_part, weights, disp=None):
    """Evidence rows of kept entity pairs: the extracted value is the one the member that
    supplied the field's best level holds (its record id in `extracted_record_id`). `disp`:
    display_values of both frames, when already computed ({"X": ..., "W": ...})."""
    X, W = pr.X[part], pr.W[part]
    disp = disp or {"X": display_values(X), "W": display_values(W)}
    r = S_part["r"].to_numpy(dtype=np.int64)
    vl, vr, src_rid = {}, {}, {}
    rid = X["record_id"].to_numpy(dtype=object)
    for f in PART_FIELDS[part]:
        src = levels_part[f"src_{f}"].to_numpy(dtype=np.int64) if f"src_{f}" in levels_part else np.full(len(r), -1)
        ok = src >= 0
        s0 = np.maximum(src, 0)
        if f in DISPLAY:
            xl = disp["X"][f]
            vl[f] = np.where(ok, xl[s0] if len(xl) else "", "")
            vr[f] = disp["W"][f][r]
        elif f"id_{f}" in X:
            xl = X[f"id_{f}"].to_numpy(dtype=object)
            vl[f] = np.where(ok, xl[s0] if len(xl) else "", "")
            vr[f] = W[f"id_{f}"].to_numpy(dtype=object)[r]
        src_rid[f] = np.where(ok, rid[s0] if len(rid) else "", "")
    ev, pos, fld = evidence_rows_fast(pairs_part["pair_id"].to_numpy(), levels_part, S_part, part, weights, vl, vr,
                                      return_positions=True)
    if len(ev):
        xr = np.empty(len(ev), dtype=object)
        for f in set(fld):
            m = fld == f
            xr[m] = src_rid[f][pos[m]]
        ev.insert(2, "extracted_record_id", xr)
    return ev


def evidence_rows_for(evidence, pair_chunks):
    """Evidence rows of the given pairs. `pair_chunks`: {chunk part name: pair ids} (a pair's
    evidence is in the part named like its candidates part)."""
    got = []
    for name, ids in pair_chunks.items():
        ev = evidence.read_part(name)
        if ev is not None and len(ev):
            got.append(ev[ev["pair_id"].isin(set(ids))])
    return pd.concat(got, ignore_index=True) if got else pd.DataFrame(columns=EVIDENCE_COLUMNS + ["extracted_record_id"])


def decision_cards_for(evidence, rows):
    """decision_cards for the pairs in `rows` (columns pair_id and _chunk)."""
    if not len(rows):
        return pd.Series(dtype=object)
    by = rows.groupby("_chunk")["pair_id"].agg(list).to_dict()
    return decision_cards(evidence_rows_for(evidence, by))


## 15 · Tables, workbook and manifest

**Tables.** The three large tables (candidates, evidence, dedup_pairs) are folders of Parquet parts in `out/<dataset>/tables/`, one table each (`pd.read_parquet(folder)` reads it whole; every part has the same columns); every other table is a CSV in the same folder. The full tables are: **entities** (one
row per extracted entity), **rows** (one row per extracted party, pointing back to its entity's
result), **candidates** (every kept entity x watchlist pair with its bits per field and cluster
status), **evidence** (pair x field: both values, level, m and u with their sources, bits;
the rows of a pair sum to its bits), **claims**, **dedup_pairs** (extracted x extracted pairs,
and whether they pooled), **weights** (every m and u, both models, with intervals and flags),
the self-check tables, and the **manifest**.

**Workbook: a summary, never overwhelmed.** `record_linkage_<dataset>.xlsx` holds: *summary*
(counts and the self-check headline), *flagged_entities* (best p >= `flag_p`, strongest first,
at most `workbook_flagged_max` rows; all of them are in the entities table), *best_matches*
(each flagged entity's best match with its decision card), *samples* (candidate pairs sampled
per basis x p band, with decision cards), *self_check* (held-out identifier and noise-test
metrics, B1 and baseline), *baseline_disagreements* (when the baseline is on), *weights* and
*manifest*. Full candidates and evidence are only in the table files.

The manifest holds: the run configuration, input hashes,
input reports, the address standardizer's counts, unmapped category values, blocking counts,
deduplication counts, every m, u and bits value of both models with source, pair count, 95%
bootstrap interval and flags (unstable weights are listed apart), every training pass, the
priors with their intervals, the sensitivity table, cluster counts, data-quality findings,
vetoes, the self-check funnel (how many true pairs the candidate stage lost) and metrics, and the
timing of each step.

**Where Splink would do better.** Splink writes predictions straight to a database table or
Parquet from DuckDB or Spark and its model JSON records the trained parameters; on Databricks
it writes Spark tables without a pandas round trip. It has no workbook: the workbook is this
package's requirement, now a summary so its row limit never binds.

In [ ]:
def manifest_rows(section, df, key_cols, value_col, source_col=None, pairs_col=None, lo=None, hi=None, note_col=None):
    out = []
    for _, r in df.iterrows():
        out.append({"section": section, "key": "|".join(str(r[c]) for c in key_cols),
                    "value": r[value_col], "source": r[source_col] if source_col else "",
                    "pairs": r[pairs_col] if pairs_col else "",
                    "ci_low": r[lo] if lo else "", "ci_high": r[hi] if hi else "",
                    "note": r[note_col] if note_col else ""})
    return out


def _clean_cell(v):
    if v is None:
        return ""
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return ""
        return float(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.bool_, bool)):
        return bool(v)
    return v if isinstance(v, (int, float, str)) else str(v)


def _column_values(s):
    """A column as Python values xlsxwriter writes directly: NaN and '' become blanks."""
    a = s.to_numpy()
    if a.dtype.kind in "fc":
        return [None if v != v else float(v) for v in a.tolist()]
    if a.dtype.kind in "iu":
        return a.tolist()
    if a.dtype.kind == "b":
        return a.tolist()
    return [None if (v is None or v == "" or (isinstance(v, float) and v != v)) else
            (v if isinstance(v, (str, int, float, bool)) else str(v)) for v in a.tolist()]


def write_workbook(path, tables, row_limit, filters=None):
    """tables: ordered {sheet name: DataFrame}. Streams rows (xlsxwriter constant_memory).
    Returns {sheet: [sheet names written]}."""
    import xlsxwriter
    wb = xlsxwriter.Workbook(str(path), {"constant_memory": True, "strings_to_urls": False,
                                         "strings_to_formulas": False, "nan_inf_to_errors": True})
    head = wb.add_format({"bold": True, "bg_color": "#DDE3EA"})
    written = {}
    for name, df in tables.items():
        cols = [str(c) for c in df.columns]
        n = len(df)
        parts = max(1, math.ceil(n / row_limit))
        written[name] = []
        cols_py = [_column_values(df[c]) for c in df.columns]
        for k in range(parts):
            sname = name if k == 0 else f"{name} ({k + 1})"
            ws = wb.add_worksheet(sname[:31])
            written[name].append(sname)
            ws.write_row(0, 0, cols, head)
            lo, hi = k * row_limit, min(n, (k + 1) * row_limit)
            for i in range(lo, hi):
                ws.write_row(i - lo + 1, 0, [col[i] for col in cols_py])
            ws.autofilter(0, 0, max(1, hi - lo), max(0, len(cols) - 1))
            ws.freeze_panes(1, 0)
    wb.close()
    return written


def weights_output(models):
    return pd.concat([models[k].weights[p] for k in ("link", "dedup") for p in ("person", "business")],
                     ignore_index=True)


def workbook_tables(run, selfcheck, manifest, cfg):
    """The summary workbook's sheets (the full tables are the Parquet parts)."""
    ent, pairs, ev = run["entities"], run["pairs"], run["evidence"]
    flagged = ent[ent["p"] >= cfg.flag_p].sort_values(["p", "entity_id"], ascending=[False, True], kind="stable")
    cut = flagged.head(cfg.workbook_flagged_max)
    n_pairs = len(pairs)
    best = pd.DataFrame()
    if n_pairs:
        want = set(cut["entity_id"] + "|" + cut["best_watchlist_record_id"])
        got = []
        for name, P in pairs.iter():
            m = (P["veto"] == "") & (P["entity_id"] + "|" + P["watchlist_record_id"]).isin(want)
            if m.any():
                got.append(P[m].assign(_chunk=name))
        if got:
            sel = pd.concat(got, ignore_index=True)
            cards = decision_cards_for(ev, sel)
            best = sel.drop(columns=["_e", "_r"])
            best = best.assign(decision_card=best["pair_id"].map(cards).fillna(""))
            best = best.sort_values(["p", "pair_id"], ascending=[False, True], kind="stable").drop(columns="_chunk")
    samples = pd.DataFrame()
    if n_pairs:
        rng = np.random.default_rng(cfg.seed + 5)
        sizes = run["groups"]
        choose = {}
        for g in sorted(sizes):
            n = min(sizes[g], cfg.workbook_sample_per_stratum)
            choose[g] = set(np.sort(rng.choice(sizes[g], n, replace=False)).tolist())
        seen, picks = defaultdict(int), defaultdict(list)
        for name, P in pairs.iter():
            band = p_band(P["p"].to_numpy())
            for (b, bd), idx in pd.Series(np.arange(len(P))).groupby([P["basis"].to_numpy(), band]).groups.items():
                g = (b, bd)
                idx = np.asarray(idx)
                pos = seen[g] + np.arange(len(idx))
                keep = np.isin(pos, list(choose.get(g, ())))
                if keep.any():
                    picks[g].append(P.iloc[idx[keep]].assign(band=bd, _chunk=name))
                seen[g] += len(idx)
        chosen = [pd.concat(picks[g]) for g in sorted(sizes) if picks.get(g)]
        if chosen:
            samples = pd.concat(chosen, ignore_index=True)
            cards = decision_cards_for(ev, samples)
            samples = samples.drop(columns=["_e", "_r", "_chunk"])
            samples["decision_card"] = samples["pair_id"].map(cards).fillna("")
    summ = [("dataset", cfg.dataset), ("exhaustive", cfg.exhaustive),
            ("address_standardizer", cfg.address_standardizer), ("baseline", cfg.baseline),
            ("extracted parties", int(len(run["rows"]))), ("extracted entities", int(len(ent))),
            ("pooled entities (more than one record)", int((ent["records"] > 1).sum())),
            ("candidate pairs kept", int(n_pairs)),
            (f"entities with best p >= {cfg.flag_p}", int(len(flagged))),
            (f"  of which rest on the name (contextual or name only)", int(flagged["rests_on_name"].sum()) if len(flagged) else 0),
            ("entities with a consistent watchlist cluster", int((ent["match_status"] == "linked in a consistent cluster").sum())),
            ("entities with conflicting candidates", int((ent["conflicting_candidates"] != "").sum()))]
    if selfcheck and len(selfcheck.get("bands", [])):
        for _, r in selfcheck["bands"].iterrows():
            if r["method"] == "baseline" or r["threshold"] in (0.5, 0.9):
                summ.append((f"self-check {r['test']} {r['method']} at {r['threshold']}",
                             f"precision {r['precision']:.4f}, recall {r['recall']:.4f} ({r['tp']} of {r['true_pairs']} true pairs, {r['fp']} false)"))
        for _, r in selfcheck["funnel"].iterrows():
            if r["method"] == "B1":
                summ.append((f"self-check {r['test']}: true pairs lost by the candidate stage",
                             f"{r['lost_at_candidates']} of {r['true_pairs']}"))
    tables = {"summary": pd.DataFrame(summ, columns=["item", "value"]).astype({"value": str}),
              "flagged_entities": cut.drop(columns=["_e"]), "best_matches": best, "samples": samples}
    if selfcheck:
        sc = []
        for k in ("funnel", "bands", "basis", "bins"):
            t = selfcheck.get(k)
            if isinstance(t, pd.DataFrame) and len(t):
                sc.append(t.assign(table=k))
        tables["self_check"] = pd.concat(sc, ignore_index=True) if sc else pd.DataFrame()
        if cfg.baseline:
            tables["baseline_disagreements"] = selfcheck.get("disagreements", pd.DataFrame())
            tables["baseline_ambiguity"] = selfcheck.get("ambiguity", pd.DataFrame())
    tables["weights"] = weights_output(run["models"])
    tables["manifest"] = manifest
    return tables


## 15a · Table I/O: CSV

The two inputs are CSVs read from the paths given in the Settings cell. The small output tables
are written as `out/<dataset>/tables/<name>.csv`; the large ones are the Parquet folders the run
writes as it goes (section 12), optionally exported as CSV parts (`export_csv_parts`).

**Where Splink would do better.** Splink reads Parquet or a database table straight into
DuckDB without a pandas copy, so a 1M-row input costs little memory. Here pandas holds the
rows as text, and every write is a full-file overwrite (no history, no time travel).


In [ ]:
def plain_table(df):
    """A frame any writer takes: object columns as text (None -> ''), other dtypes kept."""
    d = df.copy()
    for c in d.columns:
        if d[c].dtype == object or str(d[c].dtype).startswith("str"):
            d[c] = d[c].map(lambda v: "" if v is None or (isinstance(v, float) and v != v) else
                            (v if isinstance(v, str) else (json.dumps(v, default=str) if isinstance(v, (list, dict)) else str(v))))
    d.columns = [str(c) for c in d.columns]
    return d


class TableIO:
    """Reads the two input CSVs and writes every output table as a CSV under
    out/<dataset>/tables/. `run_id` is a timestamp stamped into the manifest, so two runs on
    the same dataset are still distinguishable in the output."""

    def __init__(self, cfg, run_id=None):
        self.cfg = cfg
        self.run_id = run_id or time.strftime("%Y%m%dT%H%M%S")
        self.info = {"run_id": self.run_id}
        self.out_dir = cfg.paths["tables"]

    def read_inputs(self, xpath, wpath):
        """(extracted frame, report, watchlist frame, report, input metadata)."""
        x, xr = load_input(xpath, "extracted")
        w, wr = load_input(wpath, "watchlist")
        meta = {"extracted.path": str(xpath), "watchlist.path": str(wpath),
                "extracted.sha256": sha256_file(xpath), "watchlist.sha256": sha256_file(wpath)}
        self.info.update(meta)
        return x, xr, w, wr, meta

    def write(self, name, df):
        """Write one output table as out/<dataset>/tables/<name>.csv (a full overwrite)."""
        d = plain_table(df.drop(columns=[c for c in df.columns if str(c).startswith("_")], errors="ignore"))
        self.out_dir.mkdir(parents=True, exist_ok=True)
        d.to_csv(self.out_dir / f"{name}.csv", index=False, encoding="utf-8")
        return {"table": name, "rows": len(d)}


def export_csv_parts(handle, folder):
    """A large table as CSV parts (one per Parquet part, same names), for tools that cannot read
    Parquet. Returns the rows written."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    n = 0
    for name, df in handle.iter():
        plain_table(df.drop(columns=[c for c in df.columns if str(c).startswith("_")], errors="ignore")) \
            .to_csv(folder / f"{name}.csv", index=False, encoding="utf-8")
        n += len(df)
    return n


## 16 · Diagnostics and the manifest

The diagnostics that need individual pairs (blocking recall on identifier anchors, the truth
pairs of a synthetic run) are answered while the run streams: the pairs they ask about are
known before scoring, so each chunk reports the ones it holds (`Probes`).

**Where Splink would do better.** Splink's diagnostics are charts: match-weight
distributions, parameter-estimate comparisons across training passes, m/u per level, unlinkable
records, cluster studio. These are tables in the manifest and printed summaries.

In [ ]:
class Probes:
    """Questions a run answers while it streams (its pairs are never all in memory at once):
    `keys`: (entity, watchlist party) keys whose proposal by the candidate stage is recorded;
    `evidence_keys`: evidence only for these pairs (None: every pair); `consumers`: functions
    applied to each chunk's final candidates in the second pass; `start`: called with the
    prepared parties, before anything is scored."""
    no_evidence = False
    write_candidates = True
    truth = None

    def start(self, pr):
        pass

    def keys(self, pr, part, ent):
        return None

    def evidence_keys(self, pr, part, ent):
        return None

    def consumers(self, pr, part, ent):
        return ()


class Consumer:
    """A named function f(shared, P) applied to each chunk's final candidates P."""

    def __init__(self, name, fn):
        self.name, self.fn = name, fn

    def __call__(self, S2, P):
        return self.fn(S2, P)


class MainProbes(Probes):
    """The main run's probes: the pairs sharing a single-holder identifier (which rules
    proposed them: blocking recall on identifier anchors) and, with a truth file, the truth
    pairs (proposed, kept, p, rank, basis)."""

    def __init__(self, truth=None):
        self.truth = truth
        self.anchor, self.truth_keys, self.want = {}, {}, {}

    def _anchor_pairs(self, pr, part, ent):
        X, W = pr.X[part], pr.W[part]
        ids = ["ssn", "npi", "dl", "license", "email", "vin", "plate"] if part == "person" else ["tin", "cnpi", "email"]
        single = pr.ctx.stats.single
        found = []
        for t in ids:
            a = pd.DataFrame({"k": X[f"id_{t}"].to_numpy(dtype=object), "l": np.arange(len(X))})
            b = pd.DataFrame({"k": W[f"id_{t}"].to_numpy(dtype=object), "r": np.arange(len(W))})
            a = a[(a["k"] != "") & a["k"].isin(single.get(t, set()))]
            b = b[b["k"] != ""]
            found.append(a.merge(b, on="k")[["l", "r"]])
        A = pd.concat(found).drop_duplicates() if found else pd.DataFrame(columns=["l", "r"])
        n_raw = len(A)
        if len(A):
            A = pd.DataFrame({"e": ent.e_of[A["l"].to_numpy(dtype=np.int64)], "r": A["r"].to_numpy()}).drop_duplicates()
        return A, n_raw

    def _truth_pairs(self, pr, part, ent):
        if self.truth is None:
            return None
        X, W = pr.X[part], pr.W[part]
        t = self.truth[self.truth["part"] == part]
        xi = pd.Series(np.arange(len(X)), index=X["record_id"].to_numpy())
        wi = pd.Series(np.arange(len(W)), index=W["record_id"].to_numpy())
        t = t[t["x_record_id"].isin(xi.index) & t["w_record_id"].isin(wi.index)]
        if not len(t):
            return None
        e = ent.e_of[xi[t["x_record_id"]].to_numpy()]
        r = wi[t["w_record_id"]].to_numpy()
        return t, e, r, _key(e, r)

    def keys(self, pr, part, ent):
        A, n_raw = self._anchor_pairs(pr, part, ent)
        self.anchor[part] = (A, n_raw)
        tk = self._truth_pairs(pr, part, ent)
        self.truth_keys[part] = tk
        ks = [_key(A["e"].to_numpy(dtype=np.int64), A["r"].to_numpy(dtype=np.int64))] if len(A) else []
        if tk is not None:
            ks.append(tk[3])
        self.want[part] = np.unique(np.concatenate(ks)) if ks else np.array([], np.int64)
        return self.want[part]

    def consumers(self, pr, part, ent):
        want = self.want[part] if part in self.want else self.keys(pr, part, ent)
        cols = ["proposed_by", "p", "rank", "basis", "rests_on_name", "cluster_status"]

        def rows(S2, P, want=want):
            if not len(want) or not len(P):
                return None
            k = _key(P["_e"].to_numpy(dtype=np.int64), P["_r"].to_numpy(dtype=np.int64))
            m = np.isin(k, want)
            return P.loc[m, cols].assign(_k=k[m]) if m.any() else None
        return (Consumer("probe_rows", rows),)


def identifier_anchor_recall(pr, scored, probes, part):
    """Pairs sharing a valid single-holder identifier: share of them (as entity pairs) that at
    least one non-identifier rule also proposed (the name rules alone)."""
    A, n_raw = getattr(probes, "anchor", {}).get(part, (pd.DataFrame(columns=["e", "r"]), 0))
    if not n_raw or not scored["link"][part][3]["kept"]:
        return {"part": part, "identifier_anchor_pairs": n_raw, "proposed_by_name_rules": 0, "recall": float("nan")}
    rows = scored["probe_rows"].get(part)
    rules = rows.set_index("_k")["proposed_by"] if rows is not None and len(rows) else pd.Series(dtype=object)
    got = rules.reindex(_key(A["e"].to_numpy(dtype=np.int64), A["r"].to_numpy(dtype=np.int64))).fillna("")
    name_rule = got.map(lambda s: any(x and x not in IDENTIFIER_RULES and x != "exhaustive" for x in s.split(",")))
    return {"part": part, "identifier_anchor_pairs": len(A), "proposed_by_name_rules": int(name_rule.sum()),
            "recall": float(name_rule.mean())}


def truth_recall(pr, scored, probes, part):
    """Truth pairs (x_record_id, part, w_record_id) against the run: proposed (candidate
    stage), kept, ranked first, p, basis. Each extracted row is judged through its entity."""
    tk = probes.truth_keys.get(part) if getattr(probes, "truth_keys", None) else None
    if tk is None:
        return {"part": part, "true_pairs": 0}
    t, e, r, keys = tk
    ent = scored["entities"][part]
    proposed = np.isin(keys, scored["link"][part][3]["proposed"])
    rows = scored["probe_rows"].get(part)
    kp = rows.set_index("_k") if rows is not None and len(rows) else \
        pd.DataFrame(columns=["p", "rank", "basis", "rests_on_name", "cluster_status"])
    tb = kp.reindex(keys)
    ev = t.copy()
    ev["entity_id"] = ent.ids[e]
    ev["proposed"] = proposed
    ev["kept"] = tb["p"].notna().to_numpy()
    ev["p"] = tb["p"].fillna(0.0).to_numpy()
    ev["rank"] = tb["rank"].to_numpy()
    ev["basis"] = tb["basis"].fillna("not kept").to_numpy()
    ev["cluster_status"] = tb["cluster_status"].fillna("").to_numpy()
    k = ev[ev["kept"]]
    return {"part": part, "true_pairs": len(ev), "proposed": int(ev["proposed"].sum()),
            "recall": float(ev["proposed"].mean()), "kept": int(ev["kept"].sum()), "kept_share": float(ev["kept"].mean()),
            "true_pairs_ranked_first": int((k["rank"] == 1).sum()),
            "true_rests_on_name_kept": int(k["basis"].isin(["contextual", "name_only"]).sum()),
            "true_by_basis": k["basis"].value_counts().to_dict(),
            "true_p_ge_0.5": int((k["p"] >= 0.5).sum()), "true_p_ge_0.9": int((k["p"] >= 0.9).sum()),
            "missed_examples": list(ev[~ev["proposed"]]["x_record_id"].head(10))}


def diagnostics(pr, scored, cfg, probes, log=print):
    d = {"anchor_recall": [identifier_anchor_recall(pr, scored, probes, part) for part in ("person", "business")]}
    for a in d["anchor_recall"]:
        log(f"blocking recall on identifier anchors [{a['part']}]: {a.get('proposed_by_name_rules', 0)} of "
            f"{a['identifier_anchor_pairs']} also proposed by a name rule ({a['recall']:.3f})")
    d["truth"] = []
    if getattr(probes, "truth", None) is not None:
        d["truth"] = [truth_recall(pr, scored, probes, part) for part in ("person", "business")]
        for t in d["truth"]:
            if t.get("true_pairs"):
                log(f"truth [{t['part']}]: {t['true_pairs']} true pairs, proposed {t['recall']:.4f}, kept "
                    f"{t['kept_share']:.4f}, ranked first {t['true_pairs_ranked_first']}, p>=0.5 {t['true_p_ge_0.5']}, "
                    f"by basis {t['true_by_basis']}")
    hist = []
    for part in ("person", "business"):
        h = scored["link"][part][3]["hist"]
        for i, b in enumerate(BASIS_ORDER):
            for j in range(len(P_BINS) - 1):
                if h[i, j]:
                    hist.append({"part": part, "basis": b, "p_from": P_BINS[j], "p_to": min(1.0, P_BINS[j + 1]),
                                 "pairs": int(h[i, j])})
    d["hist"] = pd.DataFrame(hist)
    if len(d["hist"]):
        log(d["hist"].pivot_table(index=["part", "basis"], columns="p_from", values="pairs", aggfunc="sum",
                                  fill_value=0).to_string())
    top = scored["top_name_only"]
    d["top_name_only"] = top
    if len(top):
        log("top name-only matches:")
        log(top[["extracted_name", "watchlist_name", "watchlist_record_id", "p", "bits"]].to_string(index=False))
    d["scored_pairs"] = {part: scored["link"][part][3]["scored"] for part in ("person", "business")}
    d["dedup"] = {part: {"extracted_parties": len(pr.X[part]), "entities": len(scored["entities"][part].ids),
                         "pooled_entities": int((scored["entities"][part].size > 1).sum()),
                         "largest": int(scored["entities"][part].size.max()) if len(scored["entities"][part].size) else 0,
                         "refused_links": scored["entities"][part].refused,
                         "scored_pairs": scored["dedup"][part][1]["scored"]}
                  for part in ("person", "business")}
    return d


def _m(rows, section, key, value, source="", pairs="", lo="", hi="", note=""):
    rows.append({"section": section, "key": key, "value": value, "source": source, "pairs": pairs,
                 "ci_low": lo, "ci_high": hi, "note": note})


def build_manifest(cfg, input_meta, ref, input_reports, pr, models, sens, scored, diag, sw, selfcheck=None,
                   io_info=None):
    rows = config_records(cfg)
    _m(rows, "run", "run_id", (io_info or {}).get("run_id", ""), "TableIO", note="stamps every row this run writes")
    rows.append({"section": "version", "key": "matching_core", "value": CORE_VERSION, "source": "notebook"})
    for k, v in input_meta.items():
        _m(rows, "input", k, v if isinstance(v, (int, float, str)) else json.dumps(v, default=str), "read")
    for rep in input_reports:
        for k in ("rows", "all_empty_columns", "added_columns", "passthrough_columns"):
            _m(rows, "input", f"{rep['source']}.{k}", json.dumps(rep[k]), "load")
    for k, v in (io_info or {}).items():
        _m(rows, "io", k, v if isinstance(v, (int, float, str)) else json.dumps(v, default=str), "table I/O")
    for _, r in ref["hash_checks"].iterrows():
        _m(rows, "reference", r["file"], r["actual"], "SOURCES.md", note=r["status"])
    _m(rows, "reference", "rarity_source", json.dumps(pr.reports["rarity_source"]), "Rarity")
    for k in ("parties", "rows", "address_standardizer"):
        _m(rows, "breakdown", k, json.dumps(pr.reports[k], default=str), "breakdown")
    for _, r in pr.reports["empty_rows_extracted"].iterrows():
        _m(rows, "data_quality", "empty_row.extracted", r["record_id"], "breakdown",
           note="no name, business name, TIN or clinic NPI: no party")
    _m(rows, "data_quality", "empty_rows.watchlist", len(pr.reports["empty_rows_watchlist"]), "breakdown")
    for _, r in pr.reports["invalid_values"].iterrows():
        _m(rows, "data_quality", f"invalid.{r['type']}.{r['reason']}", int(r["values"]), "normalize")
    junk = pr.vi[pr.vi["junk"] & pr.vi["valid"]]
    for _, r in junk.iterrows():
        _m(rows, "data_quality", f"junk.{r['type']}", r["value"], "value index", note=r["junk_reason"])
    _m(rows, "data_quality", "category_mismatch", pr.reports["category_mismatch"], "category",
       note="given category disagrees with an inferred one")
    if pr.reports["unmapped"] is not None:
        for _, r in pr.reports["unmapped"].iterrows():
            _m(rows, "unmapped", f"{r['source_field']}:{r['value']}", int(r["parties"]), "category_map.csv", note="-> unknown")
    for part in ("person", "business"):
        D = scored["link"][part][3]
        for rep in D["report"]:
            _m(rows, "blocking", f"{part}.{rep['rule']}", rep["pairs_after_refine"], "key counts before indexing",
               rep["pairs_before_refine"], note=(f"oversized keys {rep['oversized_keys']}, dropped {rep['dropped_keys']} "
                                                  f"({rep['dropped_pairs']} pairs) {rep['dropped_examples']}").strip())
        for k, v in D["rule_counts"].items():
            _m(rows, "blocking.proposed", f"{part}.{k}", v, "indexing (entity x watchlist pairs)")
        _m(rows, "blocking.proposed", f"{part}.scored_pairs", D["scored"], "indexing",
           note=f"{D['member_pairs']} member comparisons; exhaustive={D['exhaustive']}")
        dd = diag["dedup"][part]
        for k, v in dd.items():
            _m(rows, "dedup", f"{part}.{k}", v, "extracted-side deduplication")
    for kind in ("dedup", "link"):
        M = models[kind]
        W = pd.concat([M.weights["person"], M.weights["business"]], ignore_index=True)
        for _, r in W.iterrows():
            k = f"{kind}.{r['part']}.{r['field']}.{r['level']}"
            _m(rows, "m", k, r["m"], r["m_source"], r["m_pairs"], r["m_lo"], r["m_hi"],
               note="; ".join(x for x in (r["flag"], r.get("m_note", ""), f"prior centre {r['m_prior_centre']:.4g} ({r['m_chain']})",
                                         f"passes {r['m_passes']}") if x))
            _m(rows, "u", k, r["u"], r["u_source"], r["u_pairs"],
               note=(f"{r.get('u_method', '')}; value-specific on this level: " + r["u_value_specific"]) if r["u_value_specific"] else r.get("u_method", ""))
            _m(rows, "bits", k, r["bits_field_level"], "log2(m/u), bounded", lo=r.get("bits_lo", ""), hi=r.get("bits_hi", ""),
               note=r["unstable"])
            if r["unstable"]:
                _m(rows, "weights.unstable", k, r["bits_field_level"], "bootstrap / training passes",
                   lo=r.get("bits_lo", ""), hi=r.get("bits_hi", ""), note=r["unstable"])
        for part, ps in M.passes.items():
            for p_ in ps:
                _m(rows, "m.em_passes", f"{kind}.{part}.{p_['pass'].split(':')[-1]}",
                   json.dumps({"lambda": p_.get("lambda"), "iterations": p_.get("iterations"), "converged": p_.get("converged"),
                               "patterns": p_.get("patterns"), "fields": p_.get("fields"), "excluded": p_.get("excluded"),
                               "skipped": p_.get("skipped", "")}, default=str),
                   "EM, u fixed, Dirichlet prior", p_.get("pairs", 0))
        for part, n in M.u_sample_pairs.items():
            _m(rows, "u.sample", f"{kind}.{part}", n, "random pairs behind the fuzzy levels of u")
        for _, r in M.prior_df.iterrows():
            _m(rows, "prior", f"{kind}.{r['part']}.{r['group']}", r["prior"],
               f"strict pairs {r['strict_pairs']} / recall {r['recall']:.3f} / all pairs {r['all_pairs']}",
               r["strict_pairs"], r["prior_lo"], r["prior_hi"], r["flag"])
    for k, v in (models["components"].get("link") or {}).items():
        _m(rows, "u.name_components", f"link.{k}", v, "random pairs")
    for s in models["sources"]:
        _m(rows, "m.prior_centre_sources", f"{s['part']}.{s['source']}", s["pairs"], "pairs")
    for part, e in models["em_loose"].items():
        _m(rows, "m.prior_centre_em", part, json.dumps(e), "loose anchors, EM, u fixed")
    for _, r in sens.iterrows():
        _m(rows, "sensitivity", f"{r['part']}.{r['group']}.prior_x{r['prior_multiplier']}.recall_x{r['recall_multiplier']}",
           json.dumps({k: r[k] for k in r.index if k.startswith("entities")}), f"prior {r['prior']:.3g}")
    for a in diag["anchor_recall"]:
        _m(rows, "diagnostics", f"identifier_anchor_recall.{a['part']}", a["recall"], "name rules only",
           a["identifier_anchor_pairs"])
    for t in diag["truth"]:
        _m(rows, "diagnostics", f"truth.{t['part']}", json.dumps(t, default=str), "truth file (diagnostics only)")
    for part in ("person", "business"):
        vc = scored.get("veto_counts", {}).get(part, {})
        for v, n in sorted(vc.items(), key=lambda kv: (-kv[1], kv[0])):
            _m(rows, "vetoes", f"{part}.{v}", int(n), "score")
        stc = scored.get("status_counts", {}).get(part, {})
        for v, n in sorted(stc.items(), key=lambda kv: (-kv[1], kv[0])):
            _m(rows, "clusters", f"{part}.{v}", int(n), "link clustering (pairs)")
    for r in (selfcheck or {}).get("manifest_rows", []):
        rows.append(r)
    for r in sw.rows:
        _m(rows, "timing", r["step"], r["seconds"], "wall seconds", note=f"peak {r['peak_rss_gb']} GB")
    m = pd.DataFrame(rows)
    for c in ("section", "key", "value", "source", "pairs", "ci_low", "ci_high", "note"):
        if c not in m:
            m[c] = ""
    m = m[["section", "key", "value", "source", "pairs", "ci_low", "ci_high", "note"]].fillna("")
    m["value"] = m["value"].map(lambda v: v if isinstance(v, (int, float, np.integer, np.floating)) else str(v))
    for c in ("pairs", "ci_low", "ci_high"):
        m[c] = m[c].map(lambda v: "" if (isinstance(v, float) and v != v) else v)
    return m


## 16a · Baseline: the current method (switch `baseline`, default off)

A replica of the method in use today, so the self-checks can show it beside B1 on the same
truth:

1. **Name.** Each party's entity name is cleaned: lowercase, punctuation removed, legal
   suffixes (INC, LLC, PC, ...) and titles or credentials (DR, MD, JR, ...) dropped. A person's
   name is first + last as parsed (middle only with `baseline_include_middle`), a business's
   name as written.
2. **Candidates gated by category.** The extracted file's `category` column (the AI-assigned
   category) must equal the watchlist category, which is inferred from specialty and licence
   type (section 5). A pair with a missing category is blocked (`baseline_gate_missing =
   "block"`) or let through (`"pass"`).
3. **Match** when rapidfuzz `token_sort_ratio` of the two cleaned names is at least
   `baseline_threshold` (90). No identifiers, no other field, no probability.

**Assumed defaults.** Threshold 90, the cleaning above, first + last for persons and "block"
for missing categories are this build's assumptions, marked as such in the manifest, until the
user supplies the real settings.

When the switch is on, the held-out identifier check and the noise test report the baseline
beside B1, split the baseline's lost true pairs into *blocked by the category gate* and *name
too different*, and write a disagreement report: false alarms B1 avoids, matches B1 recovers
(and the reverse), and how many watchlist rows each extracted name hits (ambiguity), each row
with B1's decision card (field, level and bits).

**Where Splink would do better.** Nowhere specific: this is the user's method, reproduced
for comparison. Splink's closest counterpart would be a single-comparison model with a
deterministic rule; it would add nothing here.

In [ ]:
BASELINE_LEGAL = {"inc", "llc", "pc", "pllc", "corp", "corporation", "co", "ltd", "llp", "lp", "pa", "company",
                  "incorporated", "plc", "lc"}
BASELINE_TITLES = {"dr", "doctor", "mr", "mrs", "ms", "miss", "prof", "hon", "jr", "sr", "ii", "iii", "iv",
                   "md", "do", "dc", "dpm", "dds", "dmd", "phd", "np", "rn", "esq", "jd", "cpa", "lcsw", "psyd", "dpt"}
BASELINE_ASSUMED = ("threshold 90; cleaning: lowercase, punctuation, legal suffixes, titles; persons first + last; "
                    "missing category blocks: ASSUMED until the user supplies the real settings")


def baseline_clean(name, cfg):
    # _undot first ('M.D.' -> 'MD', so it is recognized as one credential word, not 'm' 'd')
    s = re.sub(r"[^\w\s]", " ", _undot(fold(name)).lower().replace("&", " and "))
    drop = (BASELINE_LEGAL if cfg.baseline_strip_legal else set()) | (BASELINE_TITLES if cfg.baseline_strip_titles else set())
    return " ".join(w for w in s.split() if w not in drop)


def baseline_frame(F, part, cfg, watchlist):
    """Per party: the cleaned entity name and the category the gate reads."""
    if part == "person":
        raw = F["first"] + " " + (F["middle"] + " " if cfg.baseline_include_middle else "") + F["last"]
    else:
        raw = F["business_raw"]
    name = map_unique(raw, lambda s: baseline_clean(s, cfg)).to_numpy(dtype=object)
    cat = (F["category"] if watchlist else F["category_given"]).to_numpy(dtype=object)
    cat = np.where(np.isin(cat, ["", "other"]), "", cat)
    return name, cat


def baseline_scores(xnames, wnames, pairs_l, pairs_r):
    """token_sort_ratio for given pairs."""
    return np.array([_rf_fuzz.token_sort_ratio(xnames[i], wnames[j]) if xnames[i] and wnames[j] else 0.0
                     for i, j in zip(pairs_l, pairs_r)], dtype=float)


def baseline_table(pr, B, rows):
    """The baseline's matches at the row level, with the B1 entity of the extracted row."""
    if not len(B):
        return pd.DataFrame(columns=["part", "x_party_id", "extracted_record_id", "entity_id", "watchlist_record_id",
                                     "score", "p_baseline", "gate_ok", "x_category", "w_category", "baseline_match"])
    ent = rows.set_index("party_id")["entity_id"]
    wr = pd.concat([pr.W[p][["party_id", "record_id"]] for p in ("person", "business")]).set_index("party_id")["record_id"]
    xr = pd.concat([pr.X[p][["party_id", "record_id"]] for p in ("person", "business")]).set_index("party_id")["record_id"]
    return pd.DataFrame({"part": B["part"], "x_party_id": B["x_party_id"],
                         "extracted_record_id": xr.reindex(B["x_party_id"]).to_numpy(),
                         "entity_id": ent.reindex(B["x_party_id"]).to_numpy(),
                         "watchlist_record_id": wr.reindex(B["w_party_id"]).to_numpy(),
                         "score": B["score"], "p_baseline": np.where(B["gate_ok"], B["score"] / 100.0, 0.0),
                         "gate_ok": B["gate_ok"], "x_category": B["x_category"], "w_category": B["w_category"],
                         "baseline_match": B["match"]})


def run_baseline(pr, cfg, log=print):
    """Every extracted x watchlist party pair (same part) whose cleaned names reach the
    threshold, with whether the category gate lets it through. Returns a frame: part, l, r,
    x_party_id, w_party_id, score, gate_ok, match."""
    from rapidfuzz import process
    t0 = time.time()
    out = []
    for part in ("person", "business"):
        X, W = pr.X[part], pr.W[part]
        if not len(X) or not len(W):
            continue
        xn, xc = baseline_frame(X, part, cfg, False)
        wn, wc = baseline_frame(W, part, cfg, True)
        wi = np.flatnonzero(wn != "")
        xi_all = np.flatnonzero(xn != "")
        step = max(1, 20_000_000 // max(1, len(wi)))
        for lo in range(0, len(xi_all), step):
            xi = xi_all[lo:lo + step]
            M = process.cdist(list(xn[xi]), list(wn[wi]), scorer=_rf_fuzz.token_sort_ratio,
                              score_cutoff=cfg.baseline_threshold, dtype=np.uint8, workers=-1)
            a, b = np.nonzero(M)
            if not len(a):
                continue
            l, r = xi[a], wi[b]
            gate = (xc[l] == wc[r]) & (xc[l] != "")
            if cfg.baseline_gate_missing == "pass":
                gate |= (xc[l] == "") | (wc[r] == "")
            out.append(pd.DataFrame({"part": part, "l": l, "r": r, "x_party_id": X["party_id"].to_numpy()[l],
                                     "w_party_id": W["party_id"].to_numpy()[r], "score": M[a, b].astype(float),
                                     "gate_ok": gate, "x_category": xc[l], "w_category": wc[r]}))
    B = pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        columns=["part", "l", "r", "x_party_id", "w_party_id", "score", "gate_ok", "x_category", "w_category"])
    B["match"] = B["gate_ok"].astype(bool)
    log(f"baseline: {len(B):,} pairs with token_sort_ratio >= {cfg.baseline_threshold:g}, "
        f"{int(B['match'].sum()):,} through the category gate ({time.time() - t0:.0f}s)")
    return B


def baseline_columns(pairs, part, ent, pre, cfg):
    """p_baseline on one part's candidate (entity x watchlist) pairs: the best token_sort_ratio
    / 100 any member row of the entity reaches against the watchlist name, 0 when the category
    gate blocks every such row; baseline_match: that score reaches baseline_threshold. `pre`:
    the cleaned names and categories of both sides (baseline_frame), computed once per part."""
    xn, xc, wn, wc = pre
    if not len(pairs):
        return pairs.assign(p_baseline=pd.Series(dtype=float), baseline_match=pd.Series(dtype=bool))
    e = pairs["_e"].to_numpy().astype(np.int64)
    r = pairs["_r"].to_numpy().astype(np.int64)
    n_e = ent.size[e]
    g = np.repeat(np.arange(len(e)), n_e)
    offs = np.arange(len(g)) - np.repeat(np.cumsum(n_e) - n_e, n_e)
    mem = ent.order[ent.start[e][g] + offs]
    rr = r[g]
    s = baseline_scores(xn, wn, mem, rr)
    gate = (xc[mem] == wc[rr]) & (xc[mem] != "")
    if cfg.baseline_gate_missing == "pass":
        gate |= (xc[mem] == "") | (wc[rr] == "")
    best = pd.Series(np.where(gate, s, 0.0)).groupby(g).max().to_numpy()
    out = pairs.copy()
    out["p_baseline"] = best / 100.0
    out["baseline_match"] = out["p_baseline"].to_numpy() * 100.0 >= cfg.baseline_threshold
    return out


## 16b · Self-checks without gold labels

Two measurements of the whole system that need no labels. Both run the complete B1 pipeline
(deduplication, estimation, blocking, scoring, clusters) again on altered inputs, so what they
measure is what a run produces. Their results are read by nothing in the pipeline:
they measure it, they do not tune it.

**Held-out identifier check** (`holdout_ids`: SSN, provider NPI, TIN, driver licence). Where
both an extracted and a watchlist party carry one of these identifiers, the identifier is the
truth: equal values are a match, different values are not (values one digit or one
transposition apart are left out as ambiguous; values held under more than
`holdout_max_holders` holders are left out as unreliable). The identifier's input column is
then blanked in both files and B1 runs on everything else. `holdout_mode = "each"` hides one
identifier per run (each is judged with the others still present); `"all"` hides all of them in
one run.

**Noise-injection test.** `noise_sources` watchlist records are copied 1 to `noise_max_copies`
times with the pessimistic noise table (the simulation engine of the noise section), and
`noise_decoys` fictional parties that are not on the list are added. Truth: each copy matches
its source and the source's own watchlist duplicates (records sharing an SSN, NPI, DL, TIN or
clinic NPI with it); every other pair of a copy or a decoy is a non-match.

**Streamed.** Each self-check run is a full run like the main one, but it keeps no candidates
table: while its second pass streams, it keeps only the pairs whose label is known (true pairs,
and pairs the held-out identifier or the noise copy marks false; for the false ones only p and,
at p >= 0.5, the basis), which is everything the metrics read. It writes evidence only for the
pairs the baseline disagreement table shows. Its files are deleted when it is scored.

**Metrics** (both tests, B1 and the baseline): for each p threshold, true positives, false
positives, false negatives, precision and recall; true and false pairs per p band; precision
per basis at p >= 0.5; the funnel of the true pairs (proposed by the candidate stage, kept,
above 0.5), with the number the **candidate stage lost** also written to the manifest. An
extracted row is judged through the entity it was pooled into, so deduplication errors count.
Denominators are explicit in every table.

**Where Splink would do better.** Splink has no built-in held-out identifier check, but its
labelling tools (`truth_space_table_from_labels_column`, ROC and precision-recall charts) do the
same arithmetic from a column that marks the truth, and it can use a withheld identifier as that
column. Noise injection has no Splink counterpart; Splink's own benchmarks use labelled public
data. The measurement here is ours.

In [ ]:
import dataclasses

HOLDOUT_COLUMNS = {"ssn": ["ssn"], "npi": ["provider_npi"], "tin": ["tin"],
                   "dl": ["driver_license_number", "driver_license_state"]}
HOLDOUT_PART = {"ssn": "person", "npi": "person", "dl": "person", "tin": "business"}
EVAL_BINS = [0.0, 0.1, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0000001]


def evaluate(method, test, pos, pred, label, thresholds, score_col="p", basis_col="basis", proposed=None):
    """Metrics of one method on one test. `pos`: truth positives (x_party_id, w_party_id).
    `pred`: the method's pairs with a score; `label`: 1 true / 0 false / -1 unknown per row of
    pred. A positive the method did not produce scores 0. Returns (bands, bins, basis, funnel)."""
    pred = pred.assign(_label=label)
    known = pred[pred["_label"] >= 0]
    pk = set(zip(pos["x_party_id"], pos["w_party_id"]))
    score_of = dict(zip(zip(known["x_party_id"], known["w_party_id"]), known[score_col]))
    ps = np.array([score_of.get(k, 0.0) for k in zip(pos["x_party_id"], pos["w_party_id"])], dtype=float)
    neg = known[known["_label"] == 0]
    bands = []
    for t in thresholds:
        tp = int((ps >= t).sum())
        fp = int((neg[score_col] >= t).sum())
        fn = len(ps) - tp
        bands.append({"test": test, "method": method, "threshold": t, "true_pairs": len(ps), "tp": tp, "fp": fp, "fn": fn,
                      "precision": tp / (tp + fp) if tp + fp else np.nan, "recall": tp / len(ps) if len(ps) else np.nan})
    bins = []
    for lo, hi in zip(EVAL_BINS[:-1], EVAL_BINS[1:]):
        bins.append({"test": test, "method": method, "p_from": lo, "p_to": min(1.0, hi),
                     "true_pairs": int(((ps >= lo) & (ps < hi)).sum()),
                     "false_pairs": int(((neg[score_col] >= lo) & (neg[score_col] < hi)).sum())})
    basis = []
    if basis_col in known:
        hit = known[known[score_col] >= 0.5]
        for b, g in hit.groupby(basis_col):
            tp = int((g["_label"] == 1).sum()); fp = int((g["_label"] == 0).sum())
            basis.append({"test": test, "method": method, "basis": b, "tp_at_0.5": tp, "fp_at_0.5": fp,
                          "precision": tp / (tp + fp) if tp + fp else np.nan,
                          "share_of_true_found": tp / len(ps) if len(ps) else np.nan})
    funnel = {"test": test, "method": method, "true_pairs": len(ps),
              "proposed": int(proposed.sum()) if proposed is not None else np.nan,
              "lost_at_candidates": int((~proposed).sum()) if proposed is not None else np.nan,
              "kept": int((ps > 0).sum()), "p_ge_0.5": int((ps >= 0.5).sum()), "p_ge_0.9": int((ps >= 0.9).sum()),
              "known_false_pairs_scored": len(neg)}
    return pd.DataFrame(bands), pd.DataFrame(bins), pd.DataFrame(basis), funnel


def baseline_eval(test, pos, B, label_fn, thresholds, part_names):
    """The baseline on the same truth: a match is a pair through the gate at or above the
    threshold (score 1), everything else 0; lost true pairs split by reason."""
    pred = B[B["match"]].rename(columns={"score": "ts"}).assign(p=1.0, basis="name (baseline)")
    lab = label_fn(pred) if len(pred) else np.array([], dtype=int)
    bands, bins, basis, funnel = evaluate("baseline", test, pos, pred, lab, [0.5], "p", "basis")
    bands["threshold"] = f"token_sort >= {part_names['threshold']:g} + category gate"
    # why each lost true pair was lost
    allb = {(x, w): (s, g) for x, w, s, g in zip(B["x_party_id"], B["w_party_id"], B["score"], B["gate_ok"])}
    reasons = []
    for x, w, part in zip(pos["x_party_id"], pos["w_party_id"], pos["part"]):
        s, g = allb.get((x, w), (None, None))
        if s is not None and g:
            continue
        reasons.append("blocked by category gate" if s is not None else "name too different")
    funnel["lost_blocked_by_category_gate"] = int(sum(r == "blocked by category gate" for r in reasons))
    funnel["lost_name_too_different"] = int(sum(r == "name too different" for r in reasons))
    return bands, bins, basis, funnel


def disagreements(test, pos, b1pred, b1label, B, blabel, evidence, run, cfg, cap=200):
    """Rows where B1 and the baseline disagree, with B1's decision card."""
    pk = set(zip(pos["x_party_id"], pos["w_party_id"]))
    b1p = dict(zip(zip(b1pred["x_party_id"], b1pred["w_party_id"]), b1pred["p"]))
    b1pair = dict(zip(zip(b1pred["x_party_id"], b1pred["w_party_id"]), b1pred["pair_id"]))
    b1basis = dict(zip(zip(b1pred["x_party_id"], b1pred["w_party_id"]), b1pred["basis"]))
    bm = B[B["match"]]
    bset = dict(zip(zip(bm["x_party_id"], bm["w_party_id"]), bm["score"]))
    bneg = set(k for k, l_ in zip(zip(bm["x_party_id"], bm["w_party_id"]), blabel) if l_ == 0)
    rows = []
    for k in sorted(bneg):                                   # baseline false alarms
        p = b1p.get(k, 0.0)
        rows.append(("false alarm B1 avoids" if p < 0.5 else "false alarm in both", k, bset[k], p))
    for k in sorted(pk):
        inb = k in bset
        p = b1p.get(k, 0.0)
        if not inb and p >= 0.5:
            rows.append(("match B1 recovers", k, np.nan, p))
        elif inb and p < 0.5:
            rows.append(("match the baseline finds and B1 misses", k, bset[k], p))
    if not rows:
        return pd.DataFrame(columns=["test", "kind", "x_party_id", "w_party_id", "baseline_score", "b1_p", "b1_basis",
                                     "decision_card"])
    df = pd.DataFrame(rows, columns=["kind", "k", "baseline_score", "b1_p"])
    df["x_party_id"] = [k[0] for k in df["k"]]
    df["w_party_id"] = [k[1] for k in df["k"]]
    df = df.drop(columns="k")
    df = df.groupby("kind", group_keys=False).head(cap).reset_index(drop=True)
    pid = [b1pair.get((x, w), "") for x, w in zip(df["x_party_id"], df["w_party_id"])]
    cards = decision_cards(evidence, [p for p in pid if p])
    df["b1_basis"] = [b1basis.get((x, w), "not a B1 candidate") for x, w in zip(df["x_party_id"], df["w_party_id"])]
    df["decision_card"] = [cards.get(p, "not kept by B1 (below the keep rule or not proposed)") if p else
                           "not kept by B1 (below the keep rule or not proposed)" for p in pid]
    names = pd.concat([run["pr"].X[p][["party_id", "display_name"]] for p in ("person", "business")]).set_index("party_id")["display_name"]
    wnames = pd.concat([run["pr"].W[p][["party_id", "display_name"]] for p in ("person", "business")]).set_index("party_id")["display_name"]
    df.insert(0, "test", test)
    df.insert(4, "extracted_name", names.reindex(df["x_party_id"]).fillna("").to_numpy())
    df.insert(5, "watchlist_name", wnames.reindex(df["w_party_id"]).fillna("").to_numpy())
    return df


def ambiguity(test, B):
    """How many watchlist rows each extracted name hits under the baseline."""
    bm = B[B["match"]]
    if not len(bm):
        return pd.DataFrame(columns=["test", "watchlist_rows_per_name", "extracted_parties"])
    per = bm.groupby("x_party_id").size()
    band = pd.cut(per, [0, 1, 2, 5, 10, 50, np.inf], labels=["1", "2", "3-5", "6-10", "11-50", ">50"])
    t = band.value_counts().sort_index().rename("extracted_parties").reset_index()
    t.columns = ["watchlist_rows_per_name", "extracted_parties"]
    t.insert(0, "test", test)
    return t


# ---- the held-out identifier check ---------------------------------------------------------
def holdout_truth(pr, t, cfg):
    """Truth for identifier t from the unaltered run: positives (equal values held under at
    most holdout_max_holders holders) and the value of every party that carries t."""
    part = HOLDOUT_PART[t]
    X, W = pr.X[part], pr.W[part]
    col = f"id_{t}"
    vi = pr.vi[pr.vi["type"] == t]
    ok = set(vi.loc[vi["distinct_names"] <= cfg.holdout_max_holders, "value"])
    xv = pd.Series(X[col].to_numpy(dtype=object), index=X["party_id"].to_numpy())
    wv = pd.Series(W[col].to_numpy(dtype=object), index=W["party_id"].to_numpy())
    xv, wv = xv[xv != ""], wv[wv != ""]
    a = pd.DataFrame({"x_party_id": xv.index, "v": xv.to_numpy()})
    b = pd.DataFrame({"w_party_id": wv.index, "v": wv.to_numpy()})
    pos = a[a["v"].isin(ok)].merge(b, on="v")[["x_party_id", "w_party_id"]]
    pos["part"] = part
    return pos, xv.to_dict(), wv.to_dict(), ok


def holdout_label(xval, wval, ok, t):
    num = (lambda v: v.partition(":")[2]) if t == "dl" else (lambda v: v)

    def lab(df):
        out = np.full(len(df), -1, dtype=int)
        for i, (x, w) in enumerate(zip(df["x_party_id"], df["w_party_id"])):
            a, b = xval.get(x), wval.get(w)
            if not a or not b:
                continue
            if a == b:
                out[i] = 1 if a in ok else -1
            elif not _near(num(a), num(b)):
                out[i] = 0
        return out
    return lab


def hide_identifier(df, ids):
    d = df.copy()
    for t in ids:
        for c in HOLDOUT_COLUMNS[t]:
            d[c] = ""
    return d


def _sub_config(cfg):
    return dataclasses.replace(cfg, self_check=False, core=dataclasses.replace(cfg.core))


# ---- the noise-injection test ------------------------------------------------------------
def watchlist_duplicate_groups(W, stats):
    """Watchlist parties of one part joined by a shared valid SSN, NPI, DL, TIN or clinic
    NPI: group id per party."""
    n = len(W)
    a, b = [], []
    for c in ("id_ssn", "id_npi", "id_dl", "id_tin", "id_cnpi"):
        if c not in W:
            continue
        d = pd.DataFrame({"pos": np.arange(n), "v": W[c].to_numpy(dtype=object)})
        d = d[d["v"] != ""]
        first = d.groupby("v")["pos"].transform("min")
        m = d["pos"] != first
        a.append(first[m].to_numpy()); b.append(d["pos"][m].to_numpy())
    a = np.concatenate(a) if a else np.array([], np.int64)
    b = np.concatenate(b) if b else np.array([], np.int64)
    cid, _, _ = constrained_clusters(n, a, b, np.ones(len(a)), {}, fields=[])
    return cid


def noise_test_inputs(wdf, maps, ref, cfg):
    """Noisy copies of watchlist rows (1..noise_max_copies each) and fictional decoys, with
    the truth of the copies (x_record_id, part, w_record_id)."""
    rng = np.random.default_rng(cfg.seed + 101)
    has_party = (wdf["first_name"] != "") | (wdf["last_name"] != "") | (wdf["business_name"] != "") | \
        (wdf["tin"] != "") | (wdf["clinic_npi"] != "")
    cand = np.flatnonzero(has_party.to_numpy())
    pick = np.sort(rng.choice(cand, size=min(cfg.noise_sources, len(cand)), replace=False)) if len(cand) else np.array([], int)
    src = wdf.iloc[pick][SCHEMA].reset_index(drop=True)
    cm = maps["category_map"]
    spec = dict(zip(cm.loc[cm["source_field"] == "specialty", "value"].map(fold), cm.loc[cm["source_field"] == "specialty", "category"]))
    lic = dict(zip(cm.loc[cm["source_field"] == "license_type", "value"].map(fold), cm.loc[cm["source_field"] == "license_type", "category"]))
    src["category"] = [spec.get(fold(s), "") or lic.get(fold(l), "") for s, l in zip(src["provider_specialty"], src["professional_license_type"])]
    k = rng.integers(1, cfg.noise_max_copies + 1, len(src))
    rep = src.loc[src.index.repeat(k)].reset_index(drop=True)
    j = rep.groupby("record_id").cumcount()
    source_rid = rep["record_id"].to_numpy().copy()
    rep["record_id"] = rep["record_id"] + "#" + j.astype(str)
    fake = Fake(ref, rng)
    copies, _ = apply_noise(rep, maps["simulation_noise"], fake, rng, scale=1.0, id_prefix="nt")
    fict = pd.DataFrame(columns=SCHEMA)
    if cfg.noise_decoys:
        fict, _, _, _, _ = make_dataset(cfg.noise_decoys, max(1, cfg.noise_decoys // 5), 0, ref, maps["simulation_noise"],
                                        cfg.seed + 102, x_share=1.0, w_share=0.0, include_edges=False, prefix="ND")
        fict = fict.iloc[:cfg.noise_decoys]
    X = pd.concat([copies, fict], ignore_index=True)
    order = rng.permutation(len(X))
    X["claim_id"] = [f"NC{i // 3:06d}" for i in np.argsort(order)]
    X["note_id"] = X["claim_id"] + "-N0"
    truth = []
    person = ((rep["first_name"] != "") | (rep["last_name"] != "")).to_numpy()
    business = ((rep["business_name"] != "") | (rep["tin"] != "") | (rep["clinic_npi"] != "")).to_numpy()
    for xr, wr, pp, bb in zip(copies["record_id"], source_rid, person, business):
        if pp:
            truth.append((xr, "person", wr))
        if bb:
            truth.append((xr, "business", wr))
    return X.fillna("").astype(str), pd.DataFrame(truth, columns=["x_record_id", "part", "w_record_id"])


def noise_truth(run, truth):
    """Positives (a copy and its source, plus the source's watchlist duplicates), the label
    function, and the extracted parties in scope (copies and decoys: all of them)."""
    pos, scope, true_sets = [], {}, {}
    for part in ("person", "business"):
        X, W = run["pr"].X[part], run["pr"].W[part]
        grp = watchlist_duplicate_groups(W, run["pr"].ctx.stats)
        wpos = pd.Series(np.arange(len(W)), index=W["record_id"].to_numpy())
        members = pd.Series(W["party_id"].to_numpy()).groupby(grp).agg(list)
        xp = pd.Series(X["party_id"].to_numpy(), index=X["record_id"].to_numpy())
        t = truth[(truth["part"] == part) & truth["x_record_id"].isin(xp.index) & truth["w_record_id"].isin(wpos.index)]
        for xr, wr in zip(t["x_record_id"], t["w_record_id"]):
            xid = xp[xr]
            ws = members[grp[wpos[wr]]]
            true_sets[xid] = set(ws)
            for w in ws:
                pos.append((xid, w, part))
        scope[part] = set(X["party_id"])
    pos = pd.DataFrame(pos, columns=["x_party_id", "w_party_id", "part"]).drop_duplicates()

    def lab(df):
        return np.array([1 if w in true_sets.get(x, ()) else 0 for x, w in zip(df["x_party_id"], df["w_party_id"])],
                        dtype=int)
    return pos, lab, scope


class SelfCheckProbes(Probes):
    """Probes of one self-check run. `tests_fn(pr)` -> {test: (part, positives, scope, label_fn)},
    built once the run's parties exist; `extra[test]`: (x_party, w_party) keys whose rows are
    always kept (the baseline disagreement table). Evidence is written only for the positives
    and the extra keys, and only when the disagreement table needs it."""
    write_candidates = False

    def __init__(self, tests_fn, extra=None, evidence=False):
        self.tests_fn, self.extra, self.need_evidence = tests_fn, extra or {}, evidence
        self.no_evidence = not evidence
        self.truth = None
        self.pos_keys = {}

    def start(self, pr):
        self.tests = self.tests_fn(pr)

    def _party_keys(self, pr, part, ent, xs, ws):
        X, W = pr.X[part], pr.W[part]
        xi = pd.Series(np.arange(len(X)), index=X["party_id"].to_numpy())
        wi = pd.Series(np.arange(len(W)), index=W["party_id"].to_numpy())
        a = xi.reindex(xs).to_numpy(dtype=float)
        b = wi.reindex(ws).to_numpy(dtype=float)
        ok = ~np.isnan(a) & ~np.isnan(b)
        k = np.full(len(a), -1, dtype=np.int64)
        if ok.any():
            k[ok] = _key(ent.e_of[a[ok].astype(np.int64)], b[ok].astype(np.int64))
        return k

    def _tests(self, part):
        return [t for t, v in self.tests.items() if v[0] == part]

    def keys(self, pr, part, ent):
        ks = []
        for t in self._tests(part):
            pos = self.tests[t][1]
            k = self._party_keys(pr, part, ent, pos["x_party_id"].to_numpy(dtype=object),
                                 pos["w_party_id"].to_numpy(dtype=object))
            self.pos_keys[t] = k
            ks.append(k[k >= 0])
        return np.unique(np.concatenate(ks)) if ks else np.array([], np.int64)

    def evidence_keys(self, pr, part, ent):
        if not self.need_evidence:
            return None
        ks = []
        for t in self._tests(part):
            pos = self.tests[t][1]
            ex = self.extra.get(t, [])
            xs = np.r_[pos["x_party_id"].to_numpy(dtype=object), np.array([x for x, _ in ex], dtype=object)]
            ws = np.r_[pos["w_party_id"].to_numpy(dtype=object), np.array([w for _, w in ex], dtype=object)]
            k = self._party_keys(pr, part, ent, xs, ws)
            ks.append(k[k >= 0])
        return np.unique(np.concatenate(ks)) if ks else np.array([], np.int64)

    def consumers(self, pr, part, ent):
        X, W = pr.X[part], pr.W[part]
        x_pid = X["party_id"].to_numpy(dtype=object)
        w_pid = W["party_id"].to_numpy(dtype=object)
        out = []
        for t in self._tests(part):
            _, pos, scope, label = self.tests[t]
            in_scope = X["party_id"].isin(scope).to_numpy()
            extra = set(self.extra.get(t, []))

            def eval_rows(S2, P, in_scope=in_scope, label=label, extra=extra):
                """b1_row_pairs for this chunk, restricted to the parties in scope and labelled;
                keeps the true rows, the extra keys, and the false rows' p (basis at p >= 0.5)."""
                if not len(P):
                    return None
                e = P["_e"].to_numpy(dtype=np.int64)
                n_e = ent.size[e]
                g = np.repeat(np.arange(len(P)), n_e)
                offs = np.arange(len(g)) - np.repeat(np.cumsum(n_e) - n_e, n_e)
                mem = ent.order[ent.start[e][g] + offs]
                m = in_scope[mem]
                g, mem = g[m], mem[m]
                if not len(g):
                    return None
                rows = pd.DataFrame({"x_party_id": x_pid[mem], "entity_id": P["entity_id"].to_numpy()[g],
                                     "w_party_id": w_pid[P["_r"].to_numpy(dtype=np.int64)[g]],
                                     "p": P["p"].to_numpy()[g], "basis": P["basis"].to_numpy()[g],
                                     "rank": P["rank"].to_numpy()[g], "cluster_status": P["cluster_status"].to_numpy()[g],
                                     "pair_id": P["pair_id"].to_numpy()[g], "veto": P["veto"].to_numpy()[g]})
                lab = np.asarray(label(rows))
                keep_extra = np.array([(x, w) in extra for x, w in zip(rows["x_party_id"], rows["w_party_id"])], bool) \
                    if extra else np.zeros(len(rows), bool)
                neg = lab == 0
                pv = rows["p"].to_numpy(dtype=float)
                return {"pos_rows": rows[lab == 1], "extra_rows": rows[keep_extra],
                        "neg_p": pv[neg], "neg_hi_basis": rows["basis"].to_numpy()[neg & (pv >= 0.5)]}
            out.append(Consumer(f"eval:{t}", eval_rows))
        return tuple(out)

    def collect(self, run, test):
        """One test's evaluation input: (true rows, extra rows, false p values, false basis at
        p >= 0.5, proposed mask of the positives)."""
        part = self.tests[test][0]
        res = [f[f"eval:{test}"] for f in run["final"].get(part, []) if f.get(f"eval:{test}") is not None]
        cat = lambda k: pd.concat([r[k] for r in res], ignore_index=True) if res else None
        pos_rows = cat("pos_rows")
        extra_rows = cat("extra_rows")
        neg_p = np.concatenate([r["neg_p"] for r in res]) if res else np.array([], float)
        neg_hi = np.concatenate([r["neg_hi_basis"] for r in res]) if res else np.array([], dtype=object)
        k = self.pos_keys.get(test, np.array([], np.int64))
        proposed = (k >= 0) & np.isin(k, run["scored"]["link"][part][3]["proposed"])
        return pos_rows, extra_rows, neg_p, neg_hi, proposed


def evaluate_stream(method, test, pos, pos_rows, neg_p, neg_hi_basis, thresholds, proposed=None):
    """evaluate() from what the stream kept: the true rows, the false rows' p values and their
    basis at p >= 0.5. Every number is the one evaluate() computes on the full labelled rows."""
    known_true = pos_rows if pos_rows is not None else pd.DataFrame(columns=["x_party_id", "w_party_id", "p", "basis"])
    score_of = dict(zip(zip(known_true["x_party_id"], known_true["w_party_id"]), known_true["p"]))
    ps = np.array([score_of.get(k, 0.0) for k in zip(pos["x_party_id"], pos["w_party_id"])], dtype=float)
    neg_p = np.asarray(neg_p, dtype=float)
    bands = []
    for t in thresholds:
        tp = int((ps >= t).sum())
        fp = int((neg_p >= t).sum())
        fn = len(ps) - tp
        bands.append({"test": test, "method": method, "threshold": t, "true_pairs": len(ps), "tp": tp, "fp": fp, "fn": fn,
                      "precision": tp / (tp + fp) if tp + fp else np.nan, "recall": tp / len(ps) if len(ps) else np.nan})
    bins = []
    for lo, hi in zip(EVAL_BINS[:-1], EVAL_BINS[1:]):
        bins.append({"test": test, "method": method, "p_from": lo, "p_to": min(1.0, hi),
                     "true_pairs": int(((ps >= lo) & (ps < hi)).sum()),
                     "false_pairs": int(((neg_p >= lo) & (neg_p < hi)).sum())})
    basis = []
    hit_t = known_true[known_true["p"].to_numpy(dtype=float) >= 0.5]["basis"].value_counts()
    hit_f = pd.Series(np.asarray(neg_hi_basis, dtype=object)).value_counts()
    for b in sorted(set(hit_t.index) | set(hit_f.index)):
        tp = int(hit_t.get(b, 0)); fp = int(hit_f.get(b, 0))
        basis.append({"test": test, "method": method, "basis": b, "tp_at_0.5": tp, "fp_at_0.5": fp,
                      "precision": tp / (tp + fp) if tp + fp else np.nan,
                      "share_of_true_found": tp / len(ps) if len(ps) else np.nan})
    funnel = {"test": test, "method": method, "true_pairs": len(ps),
              "proposed": int(proposed.sum()) if proposed is not None else np.nan,
              "lost_at_candidates": int((~proposed).sum()) if proposed is not None else np.nan,
              "kept": int((ps > 0).sum()), "p_ge_0.5": int((ps >= 0.5).sum()), "p_ge_0.9": int((ps >= 0.9).sum()),
              "known_false_pairs_scored": len(neg_p)}
    return pd.DataFrame(bands), pd.DataFrame(bins), pd.DataFrame(basis), funnel


def _b1pred(pos_rows, extra_rows):
    """The B1 rows the disagreement table reads (true pairs and the baseline's false alarms)."""
    frames = [f for f in (pos_rows, extra_rows) if f is not None and len(f)]
    if not frames:
        return pd.DataFrame(columns=["x_party_id", "w_party_id", "p", "pair_id", "basis"])
    return pd.concat(frames, ignore_index=True).drop_duplicates(["x_party_id", "w_party_id"])


def run_self_checks(xdf, wdf, maps, ref, cfg, main_run, log=print, sw=None):
    """Held-out identifier check and noise-injection test, B1 and (switch on) the baseline, each
    a full streamed run evaluated as it streams."""
    mark = sw.mark if sw is not None else (lambda s: None)
    res = {"bands": [], "bins": [], "basis": [], "funnel": [], "disagreements": [], "ambiguity": [], "manifest_rows": []}
    sub = _sub_config(cfg)
    thresholds = list(cfg.p_bands)
    pr0 = main_run["pr"]
    base_work = Path(main_run["store"].work_root) / "selfcheck"
    truths = {}
    for t in cfg.holdout_ids:
        pos, xv, wv, ok = holdout_truth(pr0, t, cfg)
        if len(pos) >= cfg.holdout_min_pairs:
            truths[t] = (pos, xv, wv, ok)
        else:
            res["manifest_rows"].append({"section": "selfcheck.holdout", "key": f"{t}.not_run", "value": len(pos),
                                         "source": "held-out identifier check",
                                         "note": f"fewer than {cfg.holdout_min_pairs} true pairs carry this identifier on both sides"})
    B0 = (main_run["baseline_raw"] if "baseline_raw" in main_run else run_baseline(pr0, cfg, log)) if cfg.baseline else None

    def holdout_probes(ts):
        extra = {}
        if B0 is not None:
            for t in ts:
                part = HOLDOUT_PART[t]
                pos, xv, wv, ok = truths[t]
                Bm = B0[(B0["part"] == part) & B0["x_party_id"].isin(set(xv)) & B0["match"]]
                lab = holdout_label(xv, wv, ok, t)(Bm) if len(Bm) else np.array([], int)
                extra[f"holdout:{t}"] = [(x, w) for x, w, l_ in zip(Bm["x_party_id"], Bm["w_party_id"], lab) if l_ == 0]

        def tests_fn(pr):
            return {f"holdout:{t}": (HOLDOUT_PART[t], truths[t][0], set(truths[t][1]),
                                     holdout_label(truths[t][1], truths[t][2], truths[t][3], t)) for t in ts}
        return SelfCheckProbes(tests_fn, extra, evidence=B0 is not None)

    def evaluate_holdout(t, run, probes):
        pos, xv, wv, ok = truths[t]
        part = HOLDOUT_PART[t]
        test = f"holdout:{t}"
        pos_rows, extra_rows, neg_p, neg_hi, prop = probes.collect(run, test)
        bd, bn, bs, fn = evaluate_stream("B1", test, pos, pos_rows, neg_p, neg_hi, thresholds, proposed=prop)
        for k, v in (("bands", bd), ("bins", bn), ("basis", bs)):
            res[k].append(v)
        res["funnel"].append(fn)
        if B0 is not None:
            lab = holdout_label(xv, wv, ok, t)
            Bp = B0[(B0["part"] == part) & B0["x_party_id"].isin(set(xv))]
            bd2, bn2, bs2, fn2 = baseline_eval(test, pos, Bp, lab, thresholds, {"threshold": cfg.baseline_threshold})
            for k, v in (("bands", bd2), ("bins", bn2), ("basis", bs2)):
                res[k].append(v)
            res["funnel"].append(fn2)
            Bm = Bp[Bp["match"]]
            pred = _b1pred(pos_rows, extra_rows)
            res["disagreements"].append(disagreements(test, pos, pred, lab(pred), Bp, lab(Bm),
                                                      run["evidence"].read(), run, cfg))
            res["ambiguity"].append(ambiguity(test, Bp))

    if cfg.holdout_mode == "all" and truths:
        probes = holdout_probes(list(truths))
        store = RunStore(base_work / "holdout_all", base_work / "holdout_all" / "work")
        run = run_pipeline(hide_identifier(xdf, list(truths)), hide_identifier(wdf, list(truths)), maps, ref, sub,
                           store=store, probes=probes)
        for t in truths:
            evaluate_holdout(t, run, probes)
        del run
        shutil.rmtree(base_work / "holdout_all", ignore_errors=True)
    else:
        for t in truths:
            log(f"held-out identifier check: {t} hidden, full B1 run")
            probes = holdout_probes([t])
            store = RunStore(base_work / f"holdout_{t}", base_work / f"holdout_{t}" / "work")
            run = run_pipeline(hide_identifier(xdf, [t]), hide_identifier(wdf, [t]), maps, ref, sub,
                               store=store, probes=probes)
            evaluate_holdout(t, run, probes)
            del run
            shutil.rmtree(base_work / f"holdout_{t}", ignore_errors=True)
    mark("16b held-out identifier check")
    # ---- noise-injection test ----
    Xn, truth_n = noise_test_inputs(wdf, maps, ref, cfg)
    log(f"noise-injection test: {len(Xn):,} extracted rows ({int(truth_n['x_record_id'].nunique()):,} noisy copies), full B1 run")
    Xn, _ = validate_input(Xn, "noise test")
    holder = {}

    def noise_fn(pr):
        pos_n, lab_n, scope = noise_truth({"pr": pr}, truth_n)
        holder["pos"], holder["lab"], holder["scope"] = pos_n, lab_n, scope
        return {f"noise:{part}": (part, pos_n[pos_n["part"] == part], scope[part], lab_n) for part in ("person", "business")
                if len(pos_n[pos_n["part"] == part])}
    probes_n = SelfCheckProbes(noise_fn, None, evidence=cfg.baseline)
    store = RunStore(base_work / "noise", base_work / "noise" / "work")
    run_n = run_pipeline(Xn, wdf, maps, ref, sub, store=store, probes=probes_n)
    for part in ("person", "business"):
        pp = holder["pos"][holder["pos"]["part"] == part]
        if not len(pp):
            continue
        test = f"noise:{part}"
        pos_rows, extra_rows, neg_p, neg_hi, prop = probes_n.collect(run_n, test)
        bd, bn, bs, fn = evaluate_stream("B1", test, pp, pos_rows, neg_p, neg_hi, thresholds, proposed=prop)
        for k, v in (("bands", bd), ("bins", bn), ("basis", bs)):
            res[k].append(v)
        res["funnel"].append(fn)
        if cfg.baseline:
            lab_n = holder["lab"]
            Bn = run_baseline(run_n["pr"], cfg, log)
            Bp = Bn[(Bn["part"] == part) & Bn["x_party_id"].isin(holder["scope"][part])]
            bd2, bn2, bs2, fn2 = baseline_eval(test, pp, Bp, lab_n, thresholds, {"threshold": cfg.baseline_threshold})
            for k, v in (("bands", bd2), ("bins", bn2), ("basis", bs2)):
                res[k].append(v)
            res["funnel"].append(fn2)
            pred = _b1pred(pos_rows, extra_rows)
            res["disagreements"].append(disagreements(test, pp, pred, lab_n(pred), Bp, lab_n(Bp[Bp["match"]]),
                                                      run_n["evidence"].read(), run_n, cfg))
            res["ambiguity"].append(ambiguity(test, Bp))
    del run_n
    shutil.rmtree(base_work / "noise", ignore_errors=True)
    mark("16b noise-injection test")
    out = {k: (pd.concat(v, ignore_index=True) if v and k not in ("funnel", "manifest_rows") else v)
           for k, v in res.items()}
    out["funnel"] = pd.DataFrame(res["funnel"])
    for k in ("bands", "bins", "basis", "disagreements", "ambiguity"):
        if not isinstance(out[k], pd.DataFrame):
            out[k] = pd.DataFrame()
    for f in res["funnel"]:
        for k in ("true_pairs", "lost_at_candidates", "proposed", "kept", "p_ge_0.5", "p_ge_0.9",
                  "lost_blocked_by_category_gate", "lost_name_too_different"):
            if k in f and not (isinstance(f[k], float) and np.isnan(f[k])):
                out["manifest_rows"].append({"section": "selfcheck.funnel", "key": f"{f['test']}.{f['method']}.{k}",
                                             "value": f[k], "source": "self-check (truth: held-out identifier or noise copy)",
                                             "note": "true pairs the candidate stage lost" if k == "lost_at_candidates" else ""})
    if len(out["bands"]):
        for _, r in out["bands"].iterrows():
            out["manifest_rows"].append({"section": "selfcheck.metrics", "key": f"{r['test']}.{r['method']}.{r['threshold']}",
                                         "value": json.dumps({"precision": r["precision"], "recall": r["recall"], "tp": r["tp"],
                                                              "fp": r["fp"], "fn": r["fn"], "true_pairs": r["true_pairs"]}),
                                         "source": "self-check"})
    if cfg.baseline:
        out["manifest_rows"].append({"section": "selfcheck.baseline", "key": "settings", "value": BASELINE_ASSUMED,
                                     "source": "RunConfig.baseline_*"})
    log("self-check metrics:\n" + (out["bands"].to_string(index=False, float_format=lambda v: f"{v:.4f}") if len(out["bands"]) else "none"))
    log("self-check funnel:\n" + out["funnel"].to_string(index=False))
    return out


### Edge-case check

Every hand-written case in `edge_cases()` against the run's candidates and entities: the
expected pair is visible, with the expected basis, veto, rank and levels, and the party carries
the expected category fields. Used by the diagnostics and by the self-tests.

In [ ]:
def check_edge_cases(pr, pairs, rows, scored, edges, evidence=None):
    """One row per expectation: ok / failed with the reason. An extracted record is judged
    through the entity it was pooled into. `pairs`: the candidates table (a DataFrame);
    expected levels are read from the evidence (a field with no evidence row is empty)."""
    out = []
    empty = set(pr.reports["empty_rows_extracted"]["record_id"])
    ent_of = rows.set_index(["record_id", "part"])["entity_id"] if len(rows) else pd.Series(dtype=object)
    ev_levels = None
    for c in edges:
        for ex in c["expect"]:
            problems = []
            if ex.get("empty_row"):
                if ex["x"] not in empty:
                    problems.append("not reported as an empty row")
                out.append({"case": c["case"], "desc": c["desc"], "x": ex["x"], "part": ex["part"], "w": "",
                            "ok": not problems, "problems": "; ".join(problems)})
                continue
            part = ex["part"]
            X = pr.X[part]
            xi = X.index[X["record_id"] == ex["x"]]
            if ex.get("party"):
                if not len(xi):
                    problems.append("party missing")
                else:
                    e = X.loc[xi[0]]
                    for k, v in ex["party"].items():
                        got = e[k]
                        if bool(got == v) is False:
                            problems.append(f"{k}={got!r}, expected {v!r}")
            if ex.get("w"):
                eid = ent_of.get((ex["x"], part), None)
                P = pairs[(pairs["part"] == part) & (pairs["entity_id"] == eid) &
                          (pairs["watchlist_record_id"] == ex["w"])] if eid is not None and len(pairs) else pairs.iloc[:0]
                if not len(P):
                    if not ex.get("if_visible"):      # a strong negative may fall below the keep rule
                        problems.append("pair not visible")
                else:
                    pr_ = P.iloc[0]
                    if "basis" in ex and pr_["basis"] != ex["basis"]:
                        problems.append(f"basis {pr_['basis']}, expected {ex['basis']}")
                    if "basis_not" in ex and pr_["basis"] == ex["basis_not"]:
                        problems.append(f"basis must not be {ex['basis_not']}")
                    if "veto" in ex and pr_["veto"] != ex["veto"]:
                        problems.append(f"veto {pr_['veto']!r}, expected {ex['veto']!r}")
                    if ex.get("veto") and pr_["p"] != 0:
                        problems.append("vetoed pair must have p = 0")
                    if ex.get("rank") and pr_["rank"] != ex["rank"]:
                        problems.append(f"rank {pr_['rank']}, expected {ex['rank']}")
                    if ex.get("levels"):
                        if ev_levels is None:
                            ev = evidence.read() if evidence is not None else pd.DataFrame(columns=["pair_id", "field", "level"])
                            ev_levels = ev.set_index(["pair_id", "field"])["level"].to_dict()
                        for f, want in ex["levels"].items():
                            got = ev_levels.get((pr_["pair_id"], f), "empty")
                            if got != want:
                                problems.append(f"{f} level {got}, expected {want}")
            out.append({"case": c["case"], "desc": c["desc"], "x": ex["x"], "part": part, "w": ex.get("w", ""),
                        "ok": not problems, "problems": "; ".join(problems)})
    return pd.DataFrame(out)


### The whole pipeline as one function

`run_pipeline` chains steps 2-16 on two validated input frames. The large tables stream to the
run's store (Parquet folders); everything else is returned in memory. The
Run section below calls the steps one by one so each prints its own output; the self-tests
call this function on small inputs.

In [ ]:
def assemble_outputs(pr, models, scored, xdf, cfg, store, probes, log=print):
    """Tables from the scored run: the second pass over every chunk (cluster statuses, the
    candidates table, the entity pieces, the probes' consumers), the dedup table, the roll-ups,
    sensitivity and diagnostics. Large tables stay in `store`; the rest is returned."""
    ents = scored["entities"]
    parts_with_pairs = {p for p in ("business", "person") if scored["link"][p][3]["kept"]}
    columns = candidate_columns(cfg, parts_with_pairs)
    final, tops, sens_rows = {}, [], []
    groups = defaultdict(int)
    scored["probe_rows"], scored["veto_counts"], scored["status_counts"] = {}, {}, {}
    for part in ("business", "person"):
        ent = ents[part]
        final[part] = finish_link_part(pr, part, ent, scored["clusters"][part], cfg, store,
                                       scored["link"][part][3]["chunks"], columns,
                                       consumers=probes.consumers(pr, part, ent), write=probes.write_candidates)
        got = [f for f in final[part] if f.get("rows")]
        rows_ = [f["probe_rows"] for f in got if f.get("probe_rows") is not None]
        scored["probe_rows"][part] = pd.concat(rows_) if rows_ else None
        tops += [f["top_name_only"] for f in got if len(f["top_name_only"])]
        sens_rows += [f["sens"] for f in got]
        vc, sc_ = defaultdict(int), defaultdict(int)
        for f in got:
            for k, v in f["veto_counts"].items():
                vc[k] += int(v)
            for k, v in f["status_counts"].items():
                sc_[k] += int(v)
            for k, v in f["groups"].items():
                groups[k] += int(v)
        scored["veto_counts"][part], scored["status_counts"][part] = dict(vc), dict(sc_)
        if sc_:
            by = defaultdict(int)
            for k, v in sc_.items():
                by[k.split(":")[0]] += v
            log(f"  clusters [{part}]: pair status {dict(sorted(by.items(), key=lambda kv: -kv[1]))}")
    scored["top_name_only"] = pd.concat(tops).sort_values("p", ascending=False, kind="stable").head(20) \
        if tops else pd.DataFrame()
    entities = rollup_entities(pr, final, {p: scored["link"][p][2] for p in ("person", "business")}, ents,
                               scored["clusters"], cfg)
    rows = rollup_rows(pr, entities, ents)
    claims = rollup_claims(rows, xdf.groupby("claim_id").size())
    n_dedup = None
    if probes.write_candidates:
        n_dedup = sum(finish_dedup_part(pr, part, ents[part], store, scored["dedup"][part][1]["chunks"])
                      for part in ("person", "business"))
    best = pd.concat(sens_rows, ignore_index=True) if sens_rows else pd.DataFrame()
    if len(best):
        best = best.merge(entities[["entity_id", "prior_group"]], on="entity_id", how="left")
    else:
        best = pd.DataFrame({"part": [], "prior_group": [], "bits": []})
    sens = sensitivity(best, models["link"].prior_df, cfg)
    diag = diagnostics(pr, scored, cfg, probes, log=log)
    n_pairs = sum(scored["link"][p][3]["kept"] for p in ("business", "person"))
    out = {"pairs": TableHandle(store, "candidates", n_pairs if probes.write_candidates else None),
           "entities": entities, "rows": rows, "claims": claims,
           "evidence": TableHandle(store, "evidence"), "dedup": TableHandle(store, "dedup_pairs", n_dedup),
           "sens": sens, "diag": diag, "groups": dict(groups), "final": final}
    if cfg.baseline:
        out["baseline_raw"] = run_baseline(pr, cfg, log)
        out["baseline_matches"] = baseline_table(pr, out["baseline_raw"], rows)
    return out


def run_pipeline(xdf, wdf, maps, ref, cfg, truth_file=None, log=None, sw=None, models=None, store=None, probes=None,
                 keep_scored=False):
    """Steps 2-16 on two validated input frames. The large tables (candidates, evidence, dedup
    pairs) stream to `store` (default: the run's tables folder), everything else is returned.
    `models`: reuse already-trained parameters. `probes`: what the run must answer while it
    streams (default: the main run's diagnostics). `keep_scored`: also keep every scored pair
    and its levels in memory (small runs: the self-tests)."""
    log = log or (lambda *a, **k: None)
    mark = sw.mark if sw is not None else (lambda s: None)
    if store is None:
        store = RunStore(cfg.paths["tables"], cfg.paths["out"] / "work")
    if probes is None:
        truth = pd.read_csv(truth_file, dtype=str, keep_default_na=False) \
            if truth_file is not None and Path(truth_file).exists() else None
        probes = MainProbes(truth)
    for t in ("_dedup_raw", "_link_raw", "candidates", "evidence", "dedup_pairs"):
        store.drop(t)
    pr = prepare(xdf, wdf, maps, ref, cfg, log=log)
    pr.parties = pr.details = pr.ties = None     # read only inside prepare: freed before the workers start
    probes.start(pr)
    mark("2-6 normalize, standardize addresses, breakdown, value index, category, rarity")
    if models is None:
        models = estimate_models(pr, maps, ref, cfg, wdf, log=log)
    else:
        pr.ctx.components = models["components"].get("link", {})
        pr.ctx_dedup.components = models["components"].get("dedup", {})
    mark("8-11 u, m, prior (dedup and link models)")
    scored = score_all(pr, models, cfg, store, log=log, probes=probes, keep=keep_scored)
    mark("7, 12 candidates, deduplication, linking, clusters")
    out = assemble_outputs(pr, models, scored, xdf, cfg, store, probes, log)
    for t in ("_dedup_raw", "_link_raw"):
        store.drop(t)
    if hasattr(PAR, "release"):             # a cluster executor pool frees what it shipped for this run
        PAR.release()
    mark("13-14, 16 roll-ups, evidence, diagnostics")
    link_w = pd.concat([models["link"].weights["person"], models["link"].weights["business"]], ignore_index=True)
    out.update({"pr": pr, "models": models, "scored": scored, "xdf": xdf, "store": store, "probes": probes,
                "params": {"weights": models["link"].weights, "components": models["components"].get("link"),
                           "sources": models["sources"], "em": models["em_loose"], "all_weights": link_w},
                "prior_df": models["link"].prior_df})
    return out


## Run

**Settings.** The next cell holds what to change before running this on your own data:

| Setting | Values |
|---|---|
| `EXTRACTED_TABLE` | your extracted entities: a Delta table (`catalog.schema.table`) with the input schema (section 1) |
| `WATCHLIST_TABLE` | your watchlist: a Delta table with the same schema |
| `OUTPUT_SCHEMA` | the `catalog.schema` the output Delta tables go to (created if it does not exist) |
| `OUTPUT_PREFIX` | output table names: `<OUTPUT_SCHEMA>.<OUTPUT_PREFIX>candidates`, `...entities`, ... (default `rl_`) |
| `WORK_FOLDER` | a Unity Catalog volume folder (`/Volumes/<catalog>/<schema>/<volume>/...`) for the chunk files, the workbook and the manifest |
| `RUN_NAME` | names this run's subfolder of `WORK_FOLDER`; `"synthetic"`: the built-in test set instead of your tables |
| `EXHAUSTIVE`, `ADDRESS_STANDARDIZER`, `BASELINE`, `SELF_CHECK` | as in the pandas notebook |
| `PAIRS_PER_TASK` | comparisons one Spark task holds at a time (default 250,000); lower it if executors run out of memory |
| `OPTIMIZE_OUTPUTS` | `True` (default): Z-order the candidates and evidence tables so a lookup reads only a few files |
| `RUN_SELF_TESTS` | `True` (default): run the self-tests on the cluster (a few minutes) |

**Outputs.** Delta tables in `OUTPUT_SCHEMA`: `candidates`, `evidence`, `dedup_pairs`,
`entities`, `rows`, `claims`, `weights`, `manifest` and the self-check tables, each with the
prefix; the workbook and the manifest JSON in `WORK_FOLDER/<RUN_NAME>/`. Delta does not allow
spaces or `=` in column names, so the claims table's per-basis-and-band counts are named like
`address_p_ge_0.9` there (`address p>=0.9` in the pandas notebook's CSV).


In [ ]:
# ---- Settings: edit these, then run every cell ------------------------------------------------------
EXTRACTED_TABLE = "main.record_linkage.extracted_entities"   # <- your extracted entities (Delta table)
WATCHLIST_TABLE = "main.record_linkage.watchlist"            # <- your watchlist (Delta table)
OUTPUT_SCHEMA = "main.record_linkage"                        # <- catalog.schema for the output Delta tables
OUTPUT_PREFIX = "rl_"                                        # output tables are <schema>.<prefix><name>
WORK_FOLDER = "/Volumes/main/record_linkage/work"            # <- a volume folder every executor can write
RUN_NAME = "production"                                      # "synthetic": the built-in test set
EXHAUSTIVE = False                                           # True: compare every same-type pair (no blocking)
ADDRESS_STANDARDIZER = "usaddress"                           # "usaddress" (offline) | "given" | "smarty"
BASELINE = False                                             # also run the token-sort-name + category baseline
SELF_CHECK = True                                            # held-out identifier check + noise-injection test
PAIRS_PER_TASK = 250_000                                     # comparisons per Spark task
OPTIMIZE_OUTPUTS = True                                      # Z-order candidates / evidence for fast lookups
RUN_SELF_TESTS = True                                        # the self-tests, on the cluster
# ------------------------------------------------------------------------------------------------------
try:
    spark                                                    # noqa: F821 -- the Databricks session
except NameError:
    spark = local_spark()                                    # outside Databricks: a local Spark with Delta
PAR = SparkParallel(spark, PAIRS_PER_TASK)
CFG = make_config(RUN_NAME, exhaustive=EXHAUSTIVE, address_standardizer=ADDRESS_STANDARDIZER, baseline=BASELINE,
                  self_check=SELF_CHECK, workers=PAR.workers, pairs_per_task=PAIRS_PER_TASK, out_root=WORK_FOLDER)
PATHS = CFG.paths
SELFTEST_TMP = str(Path(WORK_FOLDER) / "selftest_tmp")       # the self-tests' runs, where executors can write
Path(SELFTEST_TMP).mkdir(parents=True, exist_ok=True)
if OUTPUT_SCHEMA:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_SCHEMA}")
SW = Stopwatch()
print(f"dataset: {CFG.dataset}   exhaustive: {CFG.exhaustive}   addresses: {CFG.address_standardizer}   "
      f"baseline: {CFG.baseline}   self-check: {CFG.self_check}")
print(f"spark {spark.version}: {PAR.workers} executor cores; {CFG.pairs_per_task:,} comparisons per task; "
      f"work folder {PATHS['out']}")
print(f"python {platform.python_version()}  pandas {pd.__version__}  numpy {np.__version__}  "
      f"recordlinkage {rl.__version__}  matching core v{CORE_VERSION}")
MAPS = load_mappings(PATHS["mappings"])
REF = load_reference(PATHS["reference"], CFG.core)
print(REF["hash_checks"][["file", "status"]].to_string(index=False))
SW.mark("setup: mappings and reference tables")


### Inputs for this run

Both inputs are read from Delta (the table version is recorded in the manifest, so the run can be
reproduced with Delta time travel) into the driver's memory: a few hundred thousand rows are a few
hundred MB. Spark hands a table's rows over in no fixed order, and a few steps sample rows by
position (the seeded random pairs behind the u rates, the self-checks), so `delta_input` puts them
in `record_id` order: the same table version always gives the same numbers. (The pandas notebook
takes a CSV's rows in file order; sort the CSV by `record_id` to compare the two number for number.)
With `RUN_NAME = "synthetic"` the built-in synthetic set is first written to two Delta
tables in `OUTPUT_SCHEMA` and read back from them, so the whole Delta path is exercised; its truth
file stays on the volume and is read only by the diagnostics and self-tests.


In [ ]:
TABLES = lambda name: delta_table_name(OUTPUT_SCHEMA, OUTPUT_PREFIX, name)
TF = None
if CFG.dataset == "synthetic":
    X_, W_, truth_, _, _ = make_dataset(CFG.synth_persons, CFG.synth_businesses, 5000, REF, MAPS["simulation_noise"], CFG.seed)
    EXTRACTED_TABLE, WATCHLIST_TABLE = TABLES("synthetic_extracted"), TABLES("synthetic_watchlist")
    for t_, df_ in ((EXTRACTED_TABLE, X_), (WATCHLIST_TABLE, W_)):
        # _row keeps the generation order (delta_input reads the rows back in that order)
        spark.createDataFrame(df_.astype(str).assign(_row=np.arange(len(df_)))).write.format("delta") \
            .mode("overwrite").option("overwriteSchema", "true").saveAsTable(t_)
    TF = PATHS["out"] / "synthetic_truth.csv"
    truth_.to_csv(TF, index=False)
SW.mark("inputs ready")
XDF, X_VERSION = delta_input(spark, EXTRACTED_TABLE)
WDF, W_VERSION = delta_input(spark, WATCHLIST_TABLE)
XDF, XREP = validate_input(XDF, "extracted")
WDF, WREP = validate_input(WDF, "watchlist")
INPUT_META = {"extracted.table": EXTRACTED_TABLE, "extracted.version": X_VERSION,
              "watchlist.table": WATCHLIST_TABLE, "watchlist.version": W_VERSION}
IO_INFO = {"run_id": time.strftime("%Y%m%dT%H%M%S"), "engine": f"spark {spark.version}, {PAR.workers} cores"}
for rep in (XREP, WREP):
    print(f"{rep['source']}: {rep['rows']:,} rows; all-empty columns: {rep['all_empty_columns'] or 'none'}; "
          f"passthrough: {rep['passthrough_columns'] or 'none'}")
print(f"inputs: {INPUT_META}")
SW.mark("1 load and validate")


### Steps 2-16: normalize, deduplicate, estimate, link, cluster, roll up

One call to `run_pipeline`: every step prints what it did. The candidates, evidence and dedup
pairs are written to their Parquet folders chunk by chunk as they are scored; `PAIRS`,
`EVIDENCE` and `RUN["dedup"]` are handles to those folders (`.read()` loads a whole table,
`.iter()` streams it part by part).


In [ ]:
RUN = run_pipeline(XDF, WDF, MAPS, REF, CFG, truth_file=TF, log=print, sw=SW)
PR, MODELS, SCORED = RUN["pr"], RUN["models"], RUN["scored"]
PAIRS, ENTITIES, ROWS, CLAIMS, EVIDENCE = RUN["pairs"], RUN["entities"], RUN["rows"], RUN["claims"], RUN["evidence"]
DIAG, PARAMS = RUN["diag"], RUN["params"]
WEIGHTS = weights_output(MODELS)
show = WEIGHTS[WEIGHTS["model"] == "link"][["part", "field", "level", "m", "m_lo", "m_hi", "u", "u_method",
                                             "bits_field_level", "flag"]]
print(show.to_string(index=False, max_rows=200, float_format=lambda v: f"{v:.4g}"))
print(ENTITIES.groupby(["part", "basis"]).agg(entities=("p", "size"), p_ge_05=("p", lambda s: int((s >= 0.5).sum())),
                                              p_ge_09=("p", lambda s: int((s >= 0.9).sum()))).to_string())
print(f"candidates: {len(PAIRS):,} rows in {PAIRS.folder}")


### Step 16b: self-checks (held-out identifiers, noise injection; baseline when on)

Each runs the full pipeline again on altered inputs (section 16b), so this is the slowest cell.

In [ ]:
SELFCHECK = run_self_checks(XDF, WDF, MAPS, REF, CFG, RUN, log=print, sw=SW) if CFG.self_check else None
if CFG.dataset == "synthetic":
    EDGE = check_edge_cases(PR, PAIRS.read(), ROWS, SCORED, edge_cases(), EVIDENCE)
    print(f"hand-written edge cases: {int(EDGE['ok'].sum())}/{len(EDGE)} expectations met")
    if not EDGE["ok"].all():
        print(EDGE[~EDGE["ok"]].to_string(index=False))


### Step 15: Delta tables, workbook and manifest

Every table goes to Delta in `OUTPUT_SCHEMA`: the three large ones straight from their Parquet
parts on the volume (Spark reads and writes them in parallel, then optionally Z-orders them by
entity and pair), the small ones from the driver. The workbook (the summary) and the manifest
JSON go to the run's volume folder. The run's intermediate files are deleted at the end.


In [ ]:
def output_tables(run, selfcheck, manifest):
    """Every table that is not a Parquet folder already, by output name."""
    t = {"entities": run["entities"], "rows": run["rows"], "claims": run["claims"],
         "weights": weights_output(run["models"]), "manifest": manifest}
    if "baseline_matches" in run:
        t["baseline_matches"] = run["baseline_matches"]
    if selfcheck:
        for k in ("bands", "funnel", "basis", "bins"):
            t[f"selfcheck_{k}"] = selfcheck[k]
        if CFG.baseline:
            t["baseline_disagreements"] = selfcheck["disagreements"]
            t["baseline_ambiguity"] = selfcheck["ambiguity"]
    return t


SW.mark("15 before writing")
BIG_TABLES = {"candidates": (PAIRS, "entity_id"), "evidence": (EVIDENCE, "pair_id"), "dedup_pairs": (RUN["dedup"], None)}
for name, (h, _) in BIG_TABLES.items():
    IO_INFO[f"output.{name}"] = json.dumps({"format": "delta", "table": TABLES(name), "rows": len(h)})
MANIFEST = build_manifest(CFG, INPUT_META, REF, (XREP, WREP), PR, MODELS, RUN["sens"], SCORED, DIAG, SW,
                          SELFCHECK, IO_INFO)
WB_PATH = PATHS["out"] / f"record_linkage_{CFG.dataset}.xlsx"
WRITTEN = write_workbook(WB_PATH, workbook_tables(RUN, SELFCHECK, MANIFEST, CFG), CFG.excel_row_limit)
DELTA_ROWS = {}
for name, df in output_tables(RUN, SELFCHECK, MANIFEST).items():
    DELTA_ROWS[TABLES(name)] = write_delta_pandas(spark, df, TABLES(name))
for name, (h, zkey) in BIG_TABLES.items():
    if len(h):
        DELTA_ROWS[TABLES(name)] = write_delta_folder(spark, h.folder, TABLES(name),
                                                      zorder=zkey if OPTIMIZE_OUTPUTS else None)
(PATHS["out"] / "manifest.json").write_text(json.dumps(MANIFEST.to_dict("records"), indent=1, default=str), encoding="utf-8")
SW.mark("15 Delta tables and workbook")
print("Delta tables: " + ", ".join(f"{k} {v:,}" for k, v in DELTA_ROWS.items()))
print(f"workbook: {WB_PATH}  sheets: {sum(len(v) for v in WRITTEN.values())}")
print(f"total {SW.rows[-1]['elapsed']:.1f}s, driver peak memory {SW.peak / 1e9:.2f} GB")
(PATHS["out"] / "timing.json").write_text(json.dumps(SW.rows, indent=1), encoding="utf-8")


## 17 · Self-tests

`unittest` cases for every part of the package (normalizers, breakdown, category, candidates,
comparison levels, score invariants, parameters, output), an end-to-end run on a small
synthetic set with its hand-written edge cases, the parallel pieces against the core functions they stand in for (and, on Linux, a whole
run on several workers against the same run on one), and the hash check of the matching core. They
run on their own small inputs, so they are the same for every dataset.

**Where Splink would do better.** Splink has a large test suite across DuckDB, Spark and
SQLite backends; these tests cover this package only.

In [ ]:
import unittest, tempfile, io

_T = {}


def _env():
    """Mappings, reference tables and a small configuration, loaded once."""
    if "cfg" not in _T:
        cfg = make_config("selftest", random_pairs=150_000, dedup_random_pairs=150_000, sim_records=4_000,
                          chunk_size=700, bootstrap_reps=40, exhaustive=False, baseline=False,
                          address_standardizer="usaddress", self_check=False)
        _T["cfg"] = cfg
        _T["maps"] = load_mappings(cfg.paths["mappings"])
        _T["ref"] = load_reference(cfg.paths["reference"], cfg.core)
    return _T["cfg"], _T["maps"], _T["ref"]


def _temp_store(name):
    """A run store of its own in a temporary folder, so test runs never share tables."""
    d = Path(tempfile.mkdtemp(prefix=f"rl_selftest_{name}_", dir=globals().get("SELFTEST_TMP")))
    return RunStore(d / "tables", d / "work")


def _small_run():
    """One end-to-end run on a small synthetic set with every edge case (cached); it keeps its
    scored pairs and their levels in memory so the invariants below can read them."""
    if "run" not in _T:
        cfg, maps, ref = _env()
        X, W, truth, edges, _ = make_dataset(400, 100, 1200, ref, maps["simulation_noise"], cfg.seed)
        X, _ = validate_input(X, "X"); W, _ = validate_input(W, "W")
        store = _temp_store("main")
        tf = store.work_root.parent / "truth.csv"
        truth.to_csv(tf, index=False)
        _T["run"] = run_pipeline(X, W, maps, ref, cfg, truth_file=tf, store=store, keep_scored=True)
        _T["inputs"] = (X, W, truth, edges)
    return _T["run"]


def _tframe(rows):
    df = pd.DataFrame([blank_row(r["record_id"], **{k: v for k, v in r.items() if k != "record_id"})
                       for r in rows], columns=SCHEMA)
    return validate_input(df, "t")[0]


def _parties(rows, source="X", watchlist=False):
    cfg, maps, ref = _env()
    nick = Nicknames(ref["nicknames"])
    return rows_to_parties(_tframe(rows), source, maps, nick, cfg.core, watchlist)


class TestNormalizers(unittest.TestCase):
    def test_ssn(self):
        self.assertEqual(norm_ssn("212-45-6781"), ("212456781", True, ""))
        for bad in ("000-12-3456", "666123456", "912345678", "123456789", "111111111", "12345"):
            self.assertFalse(norm_ssn(bad)[1], bad)

    def test_tin(self):
        self.assertTrue(norm_tin("45-1234567")[1])
        self.assertFalse(norm_tin("07-1234567")[1])
        self.assertFalse(norm_tin("000000000")[1])

    def test_npi_luhn(self):
        self.assertTrue(npi_luhn_ok("1234567893"))
        self.assertFalse(npi_luhn_ok("1234567890"))
        self.assertEqual(norm_npi("0000000000"), ("", False, ""))
        self.assertEqual(norm_npi("1234567890")[2], "check_digit")
        self.assertTrue(npi_luhn_ok(luhn_npi("123456789")))

    def test_phone_email(self):
        self.assertEqual(norm_phone("1 (718) 392-4411")[0], "7183924411")
        self.assertFalse(norm_phone("2125550123")[1])
        self.assertFalse(norm_phone("0123456789")[1])
        self.assertEqual(norm_email(" A.B@Mail.Test ")[0], "a.b@mail.test")
        self.assertFalse(norm_email("none@none.com")[1])
        self.assertFalse(norm_email("not an email")[1])

    def test_vin(self):
        self.assertTrue(norm_vin("1M8GDM9AXKP042788")[1])
        self.assertEqual(norm_vin("1M8GDM9AYKP042788")[2], "check_digit")
        self.assertEqual(norm_vin("1M8GDM9AXKP04278I")[2], "format")

    def test_dob(self):
        self.assertEqual(norm_dob("3/4/1961")[0], "1961-03-04")
        self.assertEqual(norm_dob("19610304")[0], "1961-03-04")
        self.assertEqual(norm_dob("1900-01-01")[2], "placeholder")
        self.assertEqual(norm_dob("02/30/1960")[2], "impossible")
        self.assertEqual(norm_dob("2090-01-01")[2], "range")

    def test_qualified_ids(self):
        self.assertEqual(norm_dl("d123-456", "New York")[0], "NY:D123456")
        self.assertEqual(norm_plate("abc 123", "")[0], ":ABC123")
        self.assertFalse(norm_license("00000", "NY")[1])

    def test_address(self):
        self.assertEqual(parse_street_line("2161 UNIVERSITY AVENUE W, STE 5"), ("2161", "", "UNIVERSITY", "AVE", "5"))
        self.assertEqual(parse_street_line("P O BOX 2161")[:3], ("2161", "", "PO BOX"))
        self.assertEqual(parse_street_line("222 N W 45TH AVE")[1], "NW")
        self.assertEqual(norm_street_parts("12", "", "Main Street", "", "Apt 4B"), ("12", "", "MAIN", "ST", "4B"))
        self.assertEqual(norm_state("new york"), "NY")
        self.assertEqual(norm_zip("2134"), "02134")
        self.assertEqual(norm_city("St. Louis"), "SAINT LOUIS")

    def test_person_names(self):
        self.assertEqual(clean_person("Dr. John", "a.", "Smith Jr."), ("JOHN", "A", "SMITH"))
        self.assertEqual(clean_person("", "", "Moy, Marvin"), ("MARVIN", "", "MOY"))
        self.assertEqual(clean_person("William A Weiner", "", ""), ("WILLIAM", "A", "WEINER"))
        self.assertEqual(clean_person("José", "", "O'Brien"), ("JOSE", "", "OBRIEN"))
        self.assertEqual(parse_person("WILLIAM A. WEINER, D.O.")["creds"], ["DO"])

    def test_org_aliases(self):
        self.assertEqual(org_aliases("Rutland Medical P.C. d/b/a Soul Radiology Medical Imaging"),
                         [["RUTLAND", "MEDICAL"], ["SOUL", "RADIOLOGY", "MEDICAL", "IMAGING"]])
        self.assertEqual(org_aliases("AMERICAN TRANSIT INS. CO."), [["AMERICAN", "TRANSIT", "INSURANCE"]])
        self.assertEqual(org_aliases("Smith & Jones LLP"), [["SMITH", "JONES"]])
        self.assertEqual(org_alias_string("Nexray Medical Imaging, P.C."), "NEXRAY MEDICAL IMAGING")

    def test_nicknames(self):
        _, _, ref = _env()
        n = Nicknames(ref["nicknames"])
        self.assertTrue(n.roots("Bill") & n.roots("William"))
        self.assertFalse(n.roots("Robert") & n.roots("William"))
        self.assertEqual(nysiis("Weiner"), nysiis("Wiener"))

    def test_near(self):
        self.assertTrue(_near("123456789", "123456780"))
        self.assertTrue(_near("123456789", "123457689"))
        self.assertFalse(_near("123456789", "123456798x"))
        self.assertFalse(_near("123456789", "123450780"))


class TestBreakdown(unittest.TestCase):
    def test_person_and_business_tied(self):
        rows, parties, details, ties, empty, _ = _parties([
            {"record_id": "r1", "first_name": "Ann", "last_name": "Lee", "business_name": "Lee Chiropractic PC",
             "tin": "45-1234567", "work_phone": "7183924411", "home_phone": "7183924412", "email": "ann@lee.test",
             "street_number": "5", "street_name": "Main", "street_type": "St", "zip": "11211", "state": "NY"}])
        self.assertEqual(sorted(parties["part"]), ["business", "person"])
        self.assertEqual(len(ties), 1)
        P = parties[parties["part"] == "person"].iloc[0]
        B = parties[parties["part"] == "business"].iloc[0]
        self.assertEqual(P["tie"], B["party_id"])
        self.assertEqual((P["id_email"], B["id_email"]), ("ann@lee.test", ""))
        self.assertEqual((P["phone_own"], P["phone_row"], B["phone_row"]), ("7183924412", "7183924411", "7183924411"))
        self.assertEqual(B["id_tin"], "451234567")
        self.assertEqual(P["addr_street"], B["addr_street"])
        own = details.set_index(["party_id", "type", "value"])["ownership"]
        self.assertEqual(own[(P["party_id"], "phone", "7183924411")], "row")
        self.assertEqual(own[(P["party_id"], "phone", "7183924412")], "own")

    def test_tin_only_and_empty(self):
        _, parties, _, _, empty, _ = _parties([{"record_id": "t", "tin": "451234567"},
                                               {"record_id": "e", "city": "Brooklyn"},
                                               {"record_id": "b", "business_name": "Acme Towing", "email": "x@acme.test"}])
        self.assertEqual(list(parties["record_id"]), ["t", "b"])
        self.assertEqual(list(empty["record_id"]), ["e"])
        self.assertEqual(parties.set_index("record_id").loc["b", "id_email"], "x@acme.test")

    def test_invalid_values_kept_visible_not_compared(self):
        _, parties, details, _, _, _ = _parties([{"record_id": "r", "first_name": "A", "last_name": "B",
                                                   "provider_npi": "1234567890", "ssn": "123456789"}])
        self.assertEqual(parties.iloc[0]["id_npi"], "")
        d = details.set_index("type")
        self.assertFalse(bool(d.loc["npi", "valid"]))
        self.assertEqual(d.loc["ssn", "reason"], "placeholder")

    def test_value_index(self):
        _, p1, d1, _, _, _ = _parties([{"record_id": "a", "first_name": "Ann", "last_name": "Lee", "ssn": "212456781"},
                                       {"record_id": "b", "first_name": "A", "last_name": "Lee", "ssn": "212456781"},
                                       {"record_id": "c", "first_name": "Bo", "last_name": "Kim", "ssn": "313456781"},
                                       {"record_id": "d", "first_name": "Cy", "last_name": "Poe", "ssn": "313456781"}])
        vi = mark_junk(value_index_fast(p1, d1), CoreParams(junk_holders=1)).set_index(["type", "value"])
        self.assertTrue(bool(vi.loc[("ssn", "212456781"), "single_holder"]))      # Ann Lee = A. Lee
        self.assertFalse(bool(vi.loc[("ssn", "313456781"), "single_holder"]))
        self.assertTrue(bool(vi.loc[("ssn", "313456781"), "junk"]))


class TestCategory(unittest.TestCase):
    def test_extracted_precedence(self):
        cfg, maps, _ = _env()
        _, p, _, _, _, _ = _parties([
            {"record_id": "npi", "category": "legal", "first_name": "A", "last_name": "B", "provider_npi": luhn_npi("123456789")},
            {"record_id": "giv", "category": "witness", "first_name": "C", "last_name": "D", "business_name": "D Auto Body"},
            {"record_id": "kw", "business_name": "Pemberton & Vasquez LLP"},
            {"record_id": "oth", "category": "other", "first_name": "E", "last_name": "F"},
            {"record_id": "lic", "first_name": "G", "last_name": "H", "professional_license_type": "DC"}])
        e = p.set_index(["record_id", "part"])
        self.assertEqual(e.loc[("npi", "person"), "category"], "medical")
        self.assertTrue(bool(e.loc[("npi", "person"), "category_mismatch"]))
        self.assertEqual(e.loc[("giv", "person"), "category"], "witness")          # given before keyword
        self.assertEqual(e.loc[("giv", "person"), "category_inferred"], "repair shop")
        self.assertEqual(e.loc[("giv", "person"), "prior_group"], "private")
        self.assertEqual(e.loc[("giv", "business"), "prior_group"], "business")
        self.assertEqual(e.loc[("kw", "business"), "category"], "legal")
        self.assertEqual(e.loc[("oth", "person"), "category"], "")                 # other = no information
        self.assertEqual(e.loc[("oth", "person"), "prior_group"], "unknown")
        self.assertEqual(e.loc[("lic", "person"), "cat_strength"], "strong")
        self.assertNotIn("witness", set(p["category_inferred"]))

    def test_watchlist_mapping_and_unmapped(self):
        _, p, _, _, _, unm = _parties([
            {"record_id": "w1", "first_name": "A", "last_name": "B", "provider_specialty": "CHIROPRACTIC"},
            {"record_id": "w2", "first_name": "C", "last_name": "D", "professional_license_type": "LAW PRACTICE"},
            {"record_id": "w3", "first_name": "E", "last_name": "F", "provider_specialty": "BASKET WEAVING"}],
            source="W", watchlist=True)
        self.assertEqual(list(p["category"]), ["medical", "legal", ""])
        self.assertIn("BASKET WEAVING", set(unm["value"]))


def _mini_ctx(xp, wp):
    cfg, maps, ref = _env()
    nick = Nicknames(ref["nicknames"])
    rar = Rarity(ref["surnames"], ref["first_names"], ref["org_tokens"], ref["meta"], cfg.core)
    stats = build_value_stats(xp, wp)
    dba = declared_dba_pairs(pd.concat([xp["org_aliases"], wp["org_aliases"]]))
    return CompareContext(cfg.core, rar, nick, stats, dba,
                          {"first_nick": 0.005, "first_close": 0.005, "first_agrees_fuzzy": 0.01, "first_initial": 0.05,
                           "first_empty": 0.05, "first_differs": 0.9, "last_close": 0.002, "last_sound": 0.004})


def _pair_levels(xrow, wrow, part="person"):
    _, xp, _, _, _, _ = _parties([xrow])
    _, wp, _, _, _, _ = _parties([wrow], source="W", watchlist=True)
    X, W = split_parts(xp)[part], split_parts(wp)[part]
    ctx = _mini_ctx(xp, wp)
    lv = compare_pairs(X, W, [0], [0], part, ctx)
    return lv.iloc[0], ctx


def _lev(lv, f):
    c = int(lv[f])
    return "empty" if c < 0 else FIELD_LEVELS[f][c]


class TestComparisons(unittest.TestCase):
    def test_name_levels(self):
        cases = [(("John", "Smith"), ("John", "Smith"), "exact"),
                 (("Bill", "Szczepanski"), ("William", "Szczepanski"), "first_nick"),
                 (("Steven", "Kowalczyk"), ("Stephen", "Kowalczyk"), "first_close"),
                 (("Maria", "Smyth"), ("Maria", "Smith"), "last_sound_first_agrees"),
                 (("Maria", "Garcia-Villanueva"), ("Maria", "Garcia"), "last_close_first_agrees"),
                 (("R", "Featherstone"), ("Rupert", "Featherstone"), "initial_agrees"),
                 (("Chidi", "Nwachukwu"), ("Nwachukwu", "Chidi"), "swapped"),
                 (("", "Webb"), ("Henry", "Webb"), "first_empty"),
                 (("Alan", "Webb"), ("Henry", "Webb"), "first_differs"),
                 (("Alan", "Webb"), ("Henry", "Moss"), "else")]
        for (f1, l1), (f2, l2), want in cases:
            row = {"record_id": "x", "first_name": f1, "last_name": l1}
            if not l1:
                row["business_name"] = "placeholder biz"    # keep a party with no surname
                row["first_name"] = ""
                row["middle_name"] = ""
            lv, _ = _pair_levels(row, {"record_id": "w", "first_name": f2, "last_name": l2})
            got = _lev(lv, "name") if l1 else "empty"
            self.assertEqual(got, want, (f1, l1, f2, l2))

    def test_dob_levels(self):
        for a, b, want in (("1975-04-09", "1975-04-09", "exact"), ("1975-04-09", "1975-09-04", "swap_or_typo"),
                           ("1975-04-09", "1975-04-08", "swap_or_typo"), ("1975-04-09", "1975-04-21", "year_month"),
                           ("1975-04-09", "1975-11-21", "year"), ("1975-04-09", "1980-11-21", "differs"),
                           ("1975-04-09", "1900-01-01", "empty")):
            lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "dob": a},
                                 {"record_id": "w", "first_name": "A", "last_name": "B", "dob": b})
            self.assertEqual(_lev(lv, "dob"), want, (a, b))

    def test_address_levels(self):
        base = {"street_number": "5", "street_name": "Main", "street_type": "St", "unit": "2", "city": "Brooklyn",
                "state": "NY", "zip": "11211"}
        for change, want in (({}, "exact"), ({"unit": "3"}, "street"), ({"street_number": "7"}, "zip"),
                             ({"street_number": "7", "zip": "11222"}, "city_state"),
                             ({"street_number": "7", "zip": "11222", "city": "Albany"}, "state"),
                             ({"street_number": "7", "zip": "90026", "city": "Los Angeles", "state": "CA"}, "differs")):
            lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", **base},
                                 {"record_id": "w", "first_name": "A", "last_name": "B", **{**base, **change}})
            self.assertEqual(_lev(lv, "address"), want, change)

    def test_identifier_levels_and_veto(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "212456781"})
        self.assertEqual(_lev(lv, "ssn"), "exact")
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "212456718"})
        self.assertEqual(_lev(lv, "ssn"), "near")
        self.assertFalse(bool(lv["veto_ssn"]))
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "ssn": "212456781"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "ssn": "313999222"})
        self.assertEqual(_lev(lv, "ssn"), "differs")
        self.assertTrue(bool(lv["veto_ssn"]))
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "driver_license_number": "D1234567", "driver_license_state": "NY"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "driver_license_number": "K9876543", "driver_license_state": "NJ"})
        self.assertFalse(bool(lv["veto_dl"]))                                       # two states: no veto
        lv, _ = _pair_levels({"record_id": "x", "business_name": "A Corp", "clinic_npi": luhn_npi("111111112")},
                             {"record_id": "w", "business_name": "A Corp", "clinic_npi": luhn_npi("222222223")}, "business")
        self.assertEqual(_lev(lv, "cnpi"), "differs")
        self.assertFalse(bool(lv["veto_cnpi"]))                                     # clinic NPI never vetoes

    def test_phone_levels(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "home_phone": "7183924411"},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "home_phone": "7183924411"})
        self.assertEqual(_lev(lv, "phone"), "exact_owned_single")
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "work_phone": "7183924411"},
                             {"record_id": "w", "first_name": "C", "last_name": "D", "work_phone": "7183924411"})
        self.assertEqual(_lev(lv, "phone"), "exact_shared")

    def test_org_levels(self):
        cases = [("Lakeshore Med Ctr", "Lakeshore Medical Center Inc", "exact"),
                 ("Kestrel Imaging LLC d/b/a Harborview Radiology", "Harborview Radiology", "exact"),
                 ("Pinecrest Orthopedic", "Pinecrest Orthopedic Rehabilitation", "short_form"),
                 ("Allport Indemnity Company", "Allport Casualty Surety Company", "sibling"),
                 ("Quixotic Holdings", "Unrelated Towing", "none")]
        for a, b, want in cases:
            lv, _ = _pair_levels({"record_id": "x", "business_name": a}, {"record_id": "w", "business_name": b}, "business")
            self.assertEqual(_lev(lv, "org"), want, (a, b))

    def test_spec_cat(self):
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "last_name": "B", "provider_specialty": "Chiropractor", "provider_npi": luhn_npi("123456789")},
                             {"record_id": "w", "first_name": "A", "last_name": "B", "provider_specialty": "CHIROPRACTIC"})
        self.assertEqual(_lev(lv, "spec_cat"), "specialty")


class TestCandidates(unittest.TestCase):
    def setUp(self):
        run = _small_run()
        self.pr = run["pr"]
        self.cfg = _env()[0]

    def test_chunked_equals_unchunked(self):
        for part in ("person", "business"):
            X, W = self.pr.X[part], self.pr.W[part]
            plan = CandidatePlan(X, W, part, self.pr.ctx.rarity, self.cfg, self.pr.ctx.dba_pairs)
            one = plan.chunk(0, len(X)).sort_values(["l", "r"]).reset_index(drop=True)
            many = pd.concat([plan.chunk(lo, min(len(X), lo + 97)) for lo in range(0, len(X), 97)])
            many = many.sort_values(["l", "r"]).reset_index(drop=True)
            self.assertTrue(one.equals(many), part)

    def test_each_rule_proposes_its_target(self):
        X = _tframe([{"record_id": "x1", "first_name": "Bill", "last_name": "Kowalski"},
                    {"record_id": "x6", "first_name": "Rita", "last_name": "Oldname", "dob": "1970-01-02"},
                    {"record_id": "x2", "first_name": "Ann", "last_name": "Zyx", "ssn": "212456781"},
                    {"record_id": "x3", "first_name": "Mo", "last_name": "Qwe", "home_phone": "7183924411"},
                    {"record_id": "x4", "first_name": "Chidi", "last_name": "Nwachukwu"},
                    {"record_id": "x5", "last_name": "Plover", "dob": "1981-01-01"}])
        W = _tframe([{"record_id": "w1", "first_name": "William", "last_name": "Kowalski"},
                    {"record_id": "w6", "first_name": "Rita", "last_name": "Newname", "dob": "1970-01-02"},
                    {"record_id": "w2", "first_name": "Zed", "last_name": "Other", "ssn": "212456781"},
                    {"record_id": "w3", "first_name": "Zed", "last_name": "Else", "home_phone": "7183924411"},
                    {"record_id": "w4", "first_name": "Nwachukwu", "last_name": "Chidi"},
                    {"record_id": "w5", "first_name": "Ruth", "last_name": "Plover"}])
        cfg, maps, ref = _env()
        pr = prepare(X, W, maps, ref, cfg, log=lambda *a: None)
        plan = CandidatePlan(pr.X["person"], pr.W["person"], "person", pr.ctx.rarity, cfg)
        c = plan.chunk(0, len(pr.X["person"]))
        got = {(pr.X["person"]["record_id"][l], pr.W["person"]["record_id"][r]): set(n.split(","))
               for l, r, n in zip(c["l"], c["r"], plan.rule_names(c["rules"]))}
        self.assertIn("nysiis_canon_initial", got[("x1", "w1")])
        self.assertNotIn("nysiis_initial", got[("x1", "w1")])
        self.assertIn("dob_initial", got[("x6", "w6")])
        self.assertIn("ssn", got[("x2", "w2")])
        self.assertIn("phone", got[("x3", "w3")])
        self.assertIn("swapped", got[("x4", "w4")])
        self.assertIn("nysiis_first_missing", got[("x5", "w5")])

    def test_cross_file_same_part_only(self):
        S = _small_run()["scored"]
        for part in ("person", "business"):
            s = S["link"][part][0]
            ents = S["entities"][part]
            self.assertTrue((s["e"] < len(ents.ids)).all() and (s["r"] < len(self.pr.W[part])).all())

    def test_strict_rules_subset_of_blocking(self):
        cfg = self.cfg
        for part in ("person", "business"):
            X, W = self.pr.X[part], self.pr.W[part]
            l, r = strict_pairs(X, W, part, self.pr.ctx)
            sc = _small_run()["scored"]
            keys = sc["link"][part][3]["keys"]
            k = _key(sc["entities"][part].e_of[l], r)
            self.assertTrue(np.isin(k, keys).all(), part)


class TestScoring(unittest.TestCase):
    def setUp(self):
        self.run = _small_run()

    def test_empty_fields_are_zero_bits(self):
        for part in ("person", "business"):
            S, L = self.run["scored"]["link"][part][0], self.run["scored"]["link"][part][1]
            for f in COMPARED_FIELDS[part]:
                self.assertTrue((S.loc[L[f].to_numpy() < 0, f"bits_{f}"] == 0).all(), (part, f))

    def test_evidence_sums_to_total(self):
        ev = self.run["evidence"].read().groupby("pair_id")["bits"].sum()
        pairs = self.run["pairs"].read().set_index("pair_id")["bits"]
        diff = (pairs.reindex(ev.index) - ev).abs()
        self.assertLess(float(diff.max()), 1e-6)
        zero = pairs[~pairs.index.isin(ev.index)]
        self.assertTrue((zero.abs() < 1e-9).all())

    def test_agreement_never_lowers_p(self):
        part = "person"
        S, L = self.run["scored"]["link"][part][0], self.run["scored"]["link"][part][1]
        w = self.run["params"]["weights"][part]
        lv = L.copy()
        base = score_pairs(lv, part, w, S["prior_logit"].to_numpy())
        for f in ("ssn", "npi", "email", "dob", "name"):
            lv2 = lv.copy()
            empty = lv2[f].to_numpy() < 0
            lv2.loc[empty, f] = 0
            lv2.loc[empty, f"uv_{f}"] = 1e-6
            more = score_pairs(lv2, part, w, S["prior_logit"].to_numpy())
            ok = base["veto"].to_numpy() == ""
            self.assertTrue((more["p"].to_numpy()[ok] >= base["p"].to_numpy()[ok] - 1e-12).all(), f)

    def test_veto_visible_with_p_zero(self):
        P = self.run["pairs"].read()
        v = P[P["veto"] != ""]
        self.assertGreater(len(v), 0)
        self.assertTrue((v["p"] == 0).all())

    def test_name_only_never_identifier(self):
        part = "person"
        S, L = self.run["scored"]["link"][part][0], self.run["scored"]["link"][part][1]
        idn = S["basis"].to_numpy() == "identifier"
        any_id = np.zeros(len(L), bool)
        for f in IDENTIFIER_FIELDS:
            if f in L:
                any_id |= (L[f].to_numpy() == 0) & (S[f"bits_{f}"].to_numpy() > 0)
        self.assertTrue((any_id[idn]).all())
        self.assertTrue((S.loc[~any_id, "basis"] != "identifier").all())

    def test_coparty_needs_anchor_and_name(self):
        sc = self.run["scored"]
        S, L = sc["link"]["person"][0], sc["link"]["person"][1]
        cp = S["bits_co_party"].to_numpy() > 0
        self.assertGreater(int(cp.sum()), 0)
        self.assertTrue((S["bits_name"].to_numpy()[cp] > 0).all())
        self.assertTrue((L["co_party"].to_numpy()[cp] == 0).all())
        anchors = link_anchors(sc["link"]["business"][0], sc["entities"]["business"], _env()[0])
        X, W = self.run["pr"].X["person"], self.run["pr"].W["person"]
        ep = sc["entities"]["person"]
        for e, r in zip(S["e"].to_numpy()[cp], S["r"].to_numpy()[cp]):
            tl = X["tie_pos"].to_numpy()[ep.members(e)]
            tl = tl[tl >= 0]
            tr = W["tie_pos"].to_numpy()[r]
            ks = _key(anchors["e_of_business"][tl], np.full(len(tl), tr))
            self.assertTrue(np.isin(ks, anchors["keys"]).any())

    def test_deterministic(self):
        cfg, maps, ref = _env()
        X, W, truth, edges = _T["inputs"]
        again = run_pipeline(X, W, maps, ref, cfg, store=_temp_store("again"))
        a = self.run["pairs"].read().drop(columns=["_e", "_r"]).reset_index(drop=True)
        b = again["pairs"].read().drop(columns=["_e", "_r"]).reset_index(drop=True)
        pd.testing.assert_frame_equal(a, b)

    def test_edge_cases(self):
        X, W, truth, edges = _T["inputs"]
        r = check_edge_cases(self.run["pr"], self.run["pairs"].read(), self.run["rows"], self.run["scored"], edges,
                             self.run["evidence"])
        self.assertTrue(r["ok"].all(), r[~r["ok"]].to_string())

    def test_every_true_name_only_pair_visible(self):
        t = [x for x in self.run["diag"]["truth"] if x.get("true_pairs")]
        for x in t:
            self.assertEqual(x["proposed"], x["kept"], x["part"])

    def test_rows_one_per_party_entities_one_per_entity(self):
        # B1: the grain changed. Rows: one per extracted party (as the v1.0 entities sheet was);
        # entities: one per pooled entity; every row points to its entity's result.
        rows, e = self.run["rows"], self.run["entities"]
        self.assertTrue(rows["party_id"].is_unique)
        n = sum(len(self.run["pr"].X[p]) for p in ("person", "business"))
        self.assertEqual(len(rows), n)
        self.assertTrue(e["entity_id"].is_unique)
        self.assertTrue(rows["entity_id"].isin(set(e["entity_id"])).all())
        self.assertEqual(int(e["records"].sum()), n)
        m = rows.merge(e[["entity_id", "p"]], on="entity_id", suffixes=("", "_e"))
        self.assertTrue(np.allclose(m["p"], m["p_e"]))


class TestParameters(unittest.TestCase):
    def test_u_closed_form(self):
        lv = pd.DataFrame({"email": np.array([0] * 3 + [1] * 97 + [-1] * 50, dtype=np.int8)})
        u = estimate_u(lv, ["email"], CoreParams(u_pseudo=0.0)).set_index("level")["u"]
        self.assertAlmostEqual(u["exact"], 0.03)
        self.assertAlmostEqual(u["differs"], 0.97)

    def test_anchor_field_excluded(self):
        lv = pd.DataFrame({"ssn": np.array([0, 0, 0, 2], dtype=np.int8), "dob": np.array([0, 0, 4, 4], dtype=np.int8)})
        c = level_counts(lv, ["ssn", "dob"], exclude={"ssn": np.array([True, True, True, False])})
        self.assertEqual(c["ssn"][1], 1.0)
        self.assertEqual(c["dob"][1], 4.0)

    def test_em_recovers_m(self):
        rng = np.random.default_rng(3)
        n, lam = 20000, 0.3
        m = {"a": np.array([0.8, 0.15, 0.05]), "b": np.array([0.7, 0.3]), "c": np.array([0.9, 0.1])}
        u = {"a": np.array([0.01, 0.09, 0.9]), "b": np.array([0.05, 0.95]), "c": np.array([0.1, 0.9])}
        match = rng.random(n) < lam
        data = {}
        for f in m:
            k = len(m[f])
            data[f] = np.where(match, rng.choice(k, n, p=m[f]), rng.choice(k, n, p=u[f])).astype(np.int8)
        lv = pd.DataFrame(data)
        saved = {f: FIELD_LEVELS.get(f) for f in m}
        FIELD_LEVELS.update({"a": ["x", "y", "z"], "b": ["x", "y"], "c": ["x", "y"]})
        try:
            est, lam_hat, _ = em_fixed_u(lv, ["a", "b"], ["c"], {"c": m["c"]}, u, CoreParams())
        finally:
            for f, v in saved.items():
                if v is None:
                    FIELD_LEVELS.pop(f, None)
        self.assertAlmostEqual(lam_hat, lam, delta=0.02)
        self.assertLess(np.abs(est["a"][0] - m["a"]).max(), 0.03)
        self.assertLess(np.abs(est["b"][0] - m["b"]).max(), 0.03)

    def test_combine_chain(self):
        p = CoreParams(alpha=5, n_min=50)
        rows = combine_m("email", [("anchors_strict", np.array([8.0, 2.0]), 10.0),
                                   ("simulation", np.array([900.0, 100.0]), 1000.0)], [0.7, 0.3], p)
        m = rows[0]["m"]
        sim = (900 + 5 * 0.7) / (1000 + 5)
        self.assertAlmostEqual(m, (8 + 5 * sim) / 15)
        rows = combine_m("email", [("anchors_strict", np.array([80.0, 20.0]), 100.0),
                                   ("simulation", np.array([0.0, 1000.0]), 1000.0)], [0.7, 0.3], p)
        self.assertAlmostEqual(rows[0]["m"], (80 + 5 * 0.7) / 105)       # simulation skipped: enough pairs
        self.assertEqual(rows[0]["m_chain"], "anchors_strict(100)")

    def test_prior_formula(self):
        p = CoreParams()
        self.assertAlmostEqual(prior_estimate(10, 1_000_000, 0.5, p)[0], 2e-5)
        v, flagged = prior_estimate(0, 1_000_000, 0.5, p)
        self.assertTrue(flagged)
        self.assertAlmostEqual(v, 1e-6)
        self.assertAlmostEqual(strict_recall({"ssn": 0.5}, lambda f, l: 0.9), 0.45)

    def test_manifest_has_every_parameter(self):
        run = _small_run()
        W = pd.concat([run["params"]["weights"]["person"], run["params"]["weights"]["business"]])
        self.assertTrue((W["m_chain"] != "").all() and W["m_pairs"].notna().all())
        self.assertTrue((W["u_source"].fillna("") != "").all())
        n_levels = sum(len(FIELD_LEVELS[f]) for p in ("person", "business") for f in PART_FIELDS[p])
        self.assertEqual(len(W), n_levels)
        self.assertTrue((run["prior_df"]["prior"] > 0).all())


class TestOutput(unittest.TestCase):
    def test_continuation_sheets(self):
        import openpyxl
        with tempfile.TemporaryDirectory() as d:
            path = Path(d) / "t.xlsx"
            df = pd.DataFrame({"a": np.arange(25), "b": ["x"] * 25, "c": [True] * 25, "d": [np.nan] * 25})
            written = write_workbook(path, {"cand": df, "manifest": df.head(2)}, row_limit=10)
            self.assertEqual(written["cand"], ["cand", "cand (2)", "cand (3)"])
            wb = openpyxl.load_workbook(path, read_only=True)
            self.assertEqual(wb.sheetnames, ["cand", "cand (2)", "cand (3)", "manifest"])
            rows = list(wb["cand (3)"].iter_rows(values_only=True))
            self.assertEqual(rows[0], ("a", "b", "c", "d"))
            self.assertEqual(len(rows), 6)
            wb.close()

    def test_manifest_sections(self):
        cfg, maps, ref = _env()
        run = _small_run()
        sw = Stopwatch(); sw.mark("t")
        m = build_manifest(cfg, {"x": "0"}, ref, ({"source": "x", "rows": 1, "all_empty_columns": [], "added_columns": [],
                                                   "passthrough_columns": []},), run["pr"], run["models"],
                           run["sens"], run["scored"], run["diag"], sw, None, {"run_id": "r1"})
        for s in ("config", "config.core", "run", "reference", "m", "u", "bits", "prior", "blocking", "dedup",
                  "m.em_passes", "u.sample", "data_quality", "timing"):
            self.assertIn(s, set(m["section"]), s)


class TestInputs(unittest.TestCase):
    def test_validation(self):
        with self.assertRaises(InputError):
            validate_input(pd.DataFrame({"record_id": ["a", "a"]}), "t")
        with self.assertRaises(InputError):
            validate_input(pd.DataFrame({"first_name": ["a"]}), "t")
        df, rep = validate_input(pd.DataFrame({"record_id": ["a"], "odd": ["1"]}), "t")
        self.assertIn("x_odd", df.columns)
        self.assertIn("ssn", rep["all_empty_columns"])

    def test_mappings_valid(self):
        cfg, maps, ref = _env()
        self.assertEqual(set(ref["hash_checks"]["status"]), {"ok"})
        self.assertGreater(len(maps["category_map"]), 280)


class TestClosedFormU(unittest.TestCase):
    def test_closed_form_matches_brute_force(self):
        # A small watchlist: closed-form email-exact u must equal a brute-force count over all
        # extracted x watchlist pairs.
        _, xp, _, _, _, _ = _parties([
            {"record_id": "x1", "first_name": "Ann", "last_name": "Lee", "email": "a@x.test"},
            {"record_id": "x2", "first_name": "Bo", "last_name": "Kim", "email": "b@x.test"},
            {"record_id": "x3", "first_name": "Cy", "last_name": "Poe", "email": ""}])
        _, wp, _, _, _, _ = _parties([
            {"record_id": "w1", "first_name": "Zed", "last_name": "Zoo", "email": "a@x.test"},
            {"record_id": "w2", "first_name": "Yan", "last_name": "Yoo", "email": "c@x.test"}],
            source="W", watchlist=True)
        closed = closed_form_u(xp, wp, "person")
        u_email, n = closed["email"]["exact"]
        brute_num = sum(1 for a in ("a@x.test", "b@x.test", "") for b in ("a@x.test", "c@x.test") if a and b and a == b)
        brute_den = sum(1 for a in ("a@x.test", "b@x.test", "") for b in ("a@x.test", "c@x.test") if a and b)
        self.assertAlmostEqual(u_email, brute_num / brute_den)
        self.assertEqual(n, brute_den)

    def test_closed_form_dedup_distinct_parties_only(self):
        # same_frame=True: agreement counted among distinct parties (i < j), never a party with
        # itself, and never double-counted.
        _, xp, _, _, _, _ = _parties([
            {"record_id": "x1", "first_name": "Ann", "last_name": "Lee", "email": "a@x.test"},
            {"record_id": "x2", "first_name": "Bo", "last_name": "Kim", "email": "a@x.test"},
            {"record_id": "x3", "first_name": "Cy", "last_name": "Poe", "email": "a@x.test"}])
        closed = closed_form_u(xp, xp, "person", same_frame=True)
        u_email, n = closed["email"]["exact"]
        self.assertAlmostEqual(u_email, 1.0)     # every pair agrees
        self.assertEqual(n, 3)                    # C(3,2) = 3 pairs, not 9

    def test_combine_u_rescales_remaining_mass(self):
        cfg, maps, ref = _env()
        u_s = estimate_u(pd.DataFrame({"ssn": np.array([0, 1, 2, 1, 2], dtype=np.int8)}), ["ssn"], cfg.core)
        closed = {"ssn": {"exact": (0.002, 500.0)}}
        out = combine_u(u_s, closed, "ssn", cfg.core)
        # closed form gets the same pseudo-count smoothing as the sample (a level never seen
        # among all pairs is rare, not impossible): (0.002*500 + 0.5) / (500 + 0.5*3)
        expect = (0.002 * 500.0 + cfg.core.u_pseudo) / (500.0 + cfg.core.u_pseudo * 3)
        self.assertAlmostEqual(float(out.loc[out["level"] == "exact", "u"].iloc[0]), expect, delta=1e-6)
        self.assertAlmostEqual(out["u"].sum(), 1.0, delta=1e-6)
        self.assertEqual(out.loc[out["level"] == "exact", "u_method"].iloc[0], "closed form")


class TestEMBootstrap(unittest.TestCase):
    def test_dirichlet_prior_shrinks_thin_pattern(self):
        # A field with almost no data among the selected pairs should barely move from the
        # prior centre m0 (the point of the Dirichlet/Beta prior in EM).
        fields = ["a"]
        FIELD_LEVELS["a"] = ["x", "y"]
        try:
            pat = np.array([[0], [1]], dtype=np.int64)
            counts = np.array([1.0, 1.0])          # one informative pair each: thin
            u_tab = {"a": np.array([0.9, 0.1])}
            m0 = {"a": np.array([0.5, 0.5])}
            est = em_patterns(pat, counts, fields, u_tab, m0, alpha=50.0, params=CoreParams())
            self.assertLess(abs(est["m"]["a"][0] - 0.5), 0.05)
        finally:
            del FIELD_LEVELS["a"]

    def test_bounds_no_field_below_bits_min(self):
        cfg, maps, ref = _env()
        rows = [{"field": "middle", "level": lev, "level_code": i, "m": 0.0001, "m_lo": np.nan, "m_hi": np.nan,
                 "m_source": "x", "m_chain": "x", "m_pairs": 10, "m_prior_centre": 0.0001, "m_published": 0.0001,
                 "m_passes": "{}", "bits_lo": np.nan, "bits_hi": np.nan, "unstable": ""}
                for i, lev in enumerate(FIELD_LEVELS["middle"])]
        u_df = pd.DataFrame({"field": ["middle"] * 3, "level": FIELD_LEVELS["middle"], "level_code": [0, 1, 2],
                             "u": [0.9999, 1e-9, 1e-9]})
        w = weights_table(rows, u_df, {}, cfg.core)
        self.assertTrue((w["bits_field_level"] >= cfg.core.bits_min).all())
        self.assertTrue((w["bits_field_level"] <= cfg.core.bits_max).all())
        # a synthetic middle-name "differs" case cannot produce an absurd weight (the user's
        # -16-bit example): m is floored, so bits never blow up to -16 from a near-zero count.
        self.assertGreater(w.loc[w["level"] == "differs", "bits_field_level"].iloc[0], cfg.core.bits_min)

    def test_final_weights_flag_unstable(self):
        run = _small_run()
        w = pd.concat([run["models"]["link"].weights["person"], run["models"]["link"].weights["business"]])
        self.assertIn("unstable", w.columns)
        # some levels are thin on the tiny synthetic set: at least the mechanism fires somewhere,
        # or every relevant level has a tight enough interval (either is a valid run).
        self.assertTrue((w["unstable"] == "").all() or (w["unstable"] != "").any())


class TestRicherComparisons(unittest.TestCase):
    def test_name_similarity_measures(self):
        sims = name_similarity("SMITH", "SMYTH")
        for k in ("jaro_winkler", "levenshtein", "token_sort", "token_set", "nysiis", "metaphone"):
            self.assertIn(k, sims)
        self.assertGreater(sims["jaro_winkler"], 0.8)
        self.assertTrue(sims["metaphone"])   # SMITH/SMYTH: same metaphone code (SM0), different NYSIIS

    def test_org_close_level(self):
        # rarity-weighted token-set similarity: two aliases sharing most of their word-rarity
        # mass and a high token-set ratio grade as 'close', stronger than a generic shared word.
        lv, _ = _pair_levels({"record_id": "x", "business_name": "Ridgeview Medical Imaging Center"},
                             {"record_id": "w", "business_name": "Ridgeview Medical Imaging Ctr"}, "business")
        self.assertIn(_lev(lv, "org"), ("exact", "close"))

    def test_middle_conditional_on_surname(self):
        # a middle name agreeing between two DIFFERENT surnames must not count: it is compared
        # only when the surname group already agrees (no double counting of the name group).
        lv, _ = _pair_levels({"record_id": "x", "first_name": "A", "middle_name": "J", "last_name": "Smith"},
                             {"record_id": "w", "first_name": "A", "middle_name": "J", "last_name": "Ortiz"})
        self.assertEqual(_lev(lv, "middle"), "empty")
        lv2, _ = _pair_levels({"record_id": "x", "first_name": "A", "middle_name": "James", "last_name": "Smith"},
                              {"record_id": "w", "first_name": "A", "middle_name": "James", "last_name": "Smith"})
        self.assertEqual(_lev(lv2, "middle"), "exact")


class TestAddressStandardizer(unittest.TestCase):
    def test_given_mode_passes_through(self):
        std = AddressStandardizer("given")
        df = pd.DataFrame({"street_number": ["5"], "street_direction": [""], "street_name": ["Main St"],
                           "street_type": [""], "unit": ["2"], "city": ["Brooklyn"], "state": ["NY"], "zip": ["11211"]})
        out = std.standardize(df)
        self.assertEqual(out["name"].iloc[0], "Main St")     # unparsed: kept exactly as given

    def test_usaddress_mode_splits_a_whole_line(self):
        std = AddressStandardizer("usaddress")
        df = pd.DataFrame({"street_number": [""], "street_direction": [""], "street_name": ["2161 University Ave W Ste 5"],
                           "street_type": [""], "unit": [""], "city": ["Saint Paul"], "state": ["MN"], "zip": ["55114"]})
        out = std.standardize(df)
        self.assertEqual(out["number"].iloc[0], "2161")
        self.assertEqual(out["name"].iloc[0].upper(), "UNIVERSITY")
        self.assertEqual(std.report["usaddress_parsed"], 1)

    def test_smarty_uses_injected_post_and_caches(self):
        calls = []

        def fake_post(url, params, payload):
            calls.append(payload)
            return [{"input_id": r["input_id"], "components": {"primary_number": "9", "street_name": "Oak",
                                                               "street_suffix": "Ave", "city_name": "X",
                                                               "state_abbreviation": "NY", "zipcode": "10001"}}
                    for r in payload]
        with tempfile.TemporaryDirectory() as d:
            cache = Path(d) / "cache.json"
            std = AddressStandardizer("smarty", cache_path=cache, post=fake_post, env={"SMARTY_AUTH_ID": "id",
                                                                                       "SMARTY_AUTH_TOKEN": "tok"})
            df = pd.DataFrame({"street_number": ["9"], "street_direction": [""], "street_name": ["Oak Ave"],
                               "street_type": [""], "unit": [""], "city": ["X"], "state": ["NY"], "zip": ["10001"]})
            out1 = std.standardize(df)
            out2 = std.standardize(df)     # second call: cache hit, no new post
        self.assertEqual(len(calls), 1)
        self.assertEqual(std.report["smarty_cache_hits"], 1)
        self.assertEqual(out1["name"].iloc[0], "Oak")

    def test_smarty_without_credentials_falls_back_safely(self):
        std = AddressStandardizer("smarty", env={})   # no credentials: never calls the network
        df = pd.DataFrame({"street_number": [""], "street_direction": [""], "street_name": ["5 Main St"],
                           "street_type": [""], "unit": [""], "city": ["Brooklyn"], "state": ["NY"], "zip": ["11211"]})
        out = std.standardize(df)
        self.assertGreater(std.report["smarty_no_credentials"], 0)
        self.assertEqual(std.report["smarty_fallback_usaddress"], 1)
        self.assertEqual(out["number"].iloc[0], "5")   # usaddress fallback still parsed it


class TestExtractedDedup(unittest.TestCase):
    def test_pools_two_rows_of_one_party(self):
        run = _small_run()
        pr = run["pr"]
        ent = run["scored"]["entities"]["person"]
        self.assertGreater(int((ent.size > 1).sum()), 0)
        self.assertEqual(int(ent.size.sum()), len(pr.X["person"]))

    def test_veto_prevents_pooling_two_different_ssns(self):
        cfg, maps, ref = _env()
        X = _tframe([{"record_id": "d1", "first_name": "Greta", "last_name": "Okonkwo", "ssn": _ssn(901)},
                    {"record_id": "d2", "first_name": "Greta", "last_name": "Okonkwo", "ssn": _ssn(902)}])
        W = _tframe([{"record_id": "dw", "first_name": "Greta", "last_name": "Okonkwo", "ssn": _ssn(901)}])
        pr = prepare(X, W, maps, ref, cfg, log=lambda *a: None)
        n = len(pr.X["person"])
        a, b = np.array([0], np.int64), np.array([1], np.int64)
        cid, acc, why = constrained_clusters(n, a, b, np.array([0.99]), single_values(pr.X["person"], pr.ctx.stats))
        self.assertFalse(bool(acc[0]))
        self.assertIn("ssn", why[0])

    def test_dedup_entity_records_sum(self):
        run = _small_run()
        for part in ("person", "business"):
            ent = run["scored"]["entities"][part]
            e = run["entities"]
            sub = e[e["part"] == part]
            self.assertEqual(int(sub["records"].sum()), len(run["pr"].X[part]))


class TestClusters(unittest.TestCase):
    def test_transitivity(self):
        # a-b and b-c linked: constrained_clusters must place a, b, c in one cluster.
        n = 3
        a = np.array([0, 1], dtype=np.int64)
        b = np.array([1, 2], dtype=np.int64)
        p = np.array([0.99, 0.99])
        cid, acc, why = constrained_clusters(n, a, b, p, {})
        self.assertTrue(acc.all())
        self.assertEqual(cid[0], cid[1])
        self.assertEqual(cid[1], cid[2])

    def test_conflicting_identifier_blocks_cluster_join(self):
        n = 3
        node_vals = {"ssn": [frozenset(["111"]), frozenset(["111"]), frozenset(["222"])]}
        a = np.array([0, 1], dtype=np.int64)
        b = np.array([1, 2], dtype=np.int64)
        p = np.array([0.99, 0.99])
        cid, acc, why = constrained_clusters(n, a, b, p, node_vals, fields=["ssn"])
        self.assertTrue(acc[0])          # 0-1 share the same SSN: fine
        self.assertFalse(acc[1])         # 1-2 would merge conflicting SSNs: refused
        self.assertNotEqual(cid[0], cid[2])

    def test_name_only_links_never_join_a_cluster(self):
        run = _small_run()
        for part in ("person", "business"):
            pairs = run["pairs"].read()
            p = pairs[pairs["part"] == part]
            name_only = p[p["cluster_status"].astype(str).str.startswith("name only")]
            self.assertTrue((name_only["cluster_status"] == "name only: visible, not clustered").all())


class TestBaseline(unittest.TestCase):
    def test_clean_strips_suffixes_and_titles(self):
        cfg, _, _ = _env()
        self.assertEqual(baseline_clean("Dr. John A. Smith, M.D.", cfg), "john a smith")
        cfg2 = dataclasses.replace(cfg, baseline_strip_titles=False)
        self.assertIn("md", baseline_clean("Dr. John A. Smith, M.D.", cfg2).split())
        self.assertEqual(baseline_clean("Acme Medical Group, LLC", cfg), "acme medical group")

    def test_p_baseline_column_present_and_named(self):
        cfg, maps, ref = _env()
        cfg2 = make_config("selftest", random_pairs=150_000, dedup_random_pairs=150_000, sim_records=4_000,
                           chunk_size=700, bootstrap_reps=10, baseline=True)
        X, W, truth, edges = _T["inputs"]
        run = run_pipeline(X, W, maps, ref, cfg2, store=_temp_store("baseline"))
        P = run["pairs"].read()
        self.assertIn("p_baseline", P.columns)
        self.assertTrue(((P["p_baseline"] >= 0) & (P["p_baseline"] <= 1)).all())

    def test_category_gate_blocks_a_mismatched_pair(self):
        cfg, maps, ref = _env()
        _, xp, _, _, _, _ = _parties([{"record_id": "x", "first_name": "Alva", "last_name": "Quintero",
                                       "category": "legal"}])
        _, wp, _, _, _, _ = _parties([{"record_id": "w", "first_name": "Alva", "last_name": "Quintero",
                                       "provider_specialty": "CHIROPRACTIC"}], source="W", watchlist=True)
        xn, xc = baseline_frame(xp, "person", cfg, False)
        wn, wc = baseline_frame(wp, "person", cfg, True)
        self.assertEqual(xn[0], wn[0])                # identical cleaned names
        self.assertNotEqual(xc[0], wc[0])             # but categories disagree: legal vs medical
        self.assertTrue(xc[0] and wc[0] and xc[0] != wc[0])


class TestSelfChecks(unittest.TestCase):
    def test_holdout_truth_and_label(self):
        run = _small_run()
        cfg, maps, ref = _env()
        pos, xv, wv, ok = holdout_truth(run["pr"], "ssn", cfg)
        if len(pos):
            lab = holdout_label(xv, wv, ok, "ssn")(pos)
            self.assertTrue(set(np.unique(lab)) <= {-1, 0, 1})
            self.assertTrue((lab != -1).any() or len(pos) == 0)

    def test_noise_test_inputs_have_truth(self):
        cfg, maps, ref = _env()
        cfg2 = make_config("selftest", noise_sources=50, noise_decoys=20, noise_max_copies=2,
                           dedup_random_pairs=50_000, random_pairs=50_000, bootstrap_reps=5)
        X, W, truth, edges = _T["inputs"]
        Xn, truth_n = noise_test_inputs(W, maps, ref, cfg2)
        self.assertGreater(len(Xn), 0)
        self.assertGreater(len(truth_n), 0)
        self.assertTrue(set(truth_n["x_record_id"]) <= set(Xn["record_id"]))

    def test_watchlist_duplicate_groups_join_on_shared_identifier(self):
        cfg, maps, ref = _env()
        _, wp, _, _, _, _ = _parties([
            {"record_id": "w1", "first_name": "Ines", "last_name": "Faraday", "provider_npi": _npi(551)},
            {"record_id": "w2", "first_name": "Ines", "last_name": "Faraday", "provider_npi": _npi(551)},
            {"record_id": "w3", "first_name": "Otis", "last_name": "Marlowe", "provider_npi": _npi(552)}],
            source="W", watchlist=True)
        stats = build_value_stats(wp, wp)
        grp = watchlist_duplicate_groups(wp, stats)
        self.assertEqual(grp[0], grp[1])
        self.assertNotEqual(grp[0], grp[2])


class TestTableIO(unittest.TestCase):
    def test_csv_roundtrip(self):
        cfg, maps, ref = _env()
        with tempfile.TemporaryDirectory() as d:
            cfg2 = make_config("selftest", root=d)
            Path(cfg2.paths["tables"]).mkdir(parents=True, exist_ok=True)
            io_ = TableIO(cfg2)
            df = pd.DataFrame({"a": [1, 2], "b": ["x", "y"]})
            io_.write("t", df)
            back = pd.read_csv(cfg2.paths["tables"] / "t.csv")
            self.assertEqual(list(back["a"]), [1, 2])


def _square(x):
    return x * x


def _fails(x):
    raise ValueError(f"task {x} fails on purpose")


def _dies_once(x):
    """Task 3 stops its own worker process the first time it runs, as the system does when
    memory runs out; every task returns x squared."""
    flag = _SHARED["flag"]
    if x == 3 and not os.path.exists(flag):
        open(flag, "w").close()
        os.kill(os.getpid(), 9)
    return x * x


class TestParallel(unittest.TestCase):
    """The parallel, streaming run computes what one process computes: the pieces that replace a
    core function against the core function, and (where the platform can fork) a whole run on
    several workers against the same run on one."""

    def _sample(self, n=6000):
        run = _small_run()
        pr = run["pr"]
        X, W = pr.X["person"], pr.W["person"]
        l, r = random_pairs(len(X), len(W), n, 3, False)
        return run, pr, X, W, l, r

    def test_counts_equal_the_core(self):
        run, pr, X, W, l, r = self._sample()
        lv = with_coparty(compare_pairs(X, W, l, r, "person", pr.ctx), X, W, "person")
        cnt, comp = random_pair_counts(X, W, l, r, "person", pr.ctx, step=1_000)
        pd.testing.assert_frame_equal(estimate_u(lv, PART_FIELDS["person"], pr.ctx.params),
                                      estimate_u_counts(cnt, PART_FIELDS["person"], pr.ctx.params))
        self.assertEqual(name_components(lv), name_components_counts(comp))

    def test_compare_in_slices_equals_one_call(self):
        run, pr, X, W, l, r = self._sample()
        a = compare_pairs(X, W, l, r, "person", pr.ctx)
        b = compare_parallel(X, W, l, r, "person", pr.ctx, step=700)
        pd.testing.assert_frame_equal(a, b)

    def test_bootstrap_equals_the_core(self):
        run, pr, X, W, l, r = self._sample()
        lv = with_coparty(compare_pairs(X, W, l, r, "person", pr.ctx), X, W, "person")
        w = run["models"]["link"].weights["person"]
        fields = [f for f in PART_FIELDS["person"] if f not in ("co_party", "dob")]
        u_tab = {f: w[w["field"] == f].sort_values("level_code")["u"].to_numpy() for f in fields}
        m0 = {f: w[w["field"] == f].sort_values("level_code")["m"].to_numpy() for f in fields}
        p = run["pr"].ctx.params
        global BOOT_PARALLEL_MIN
        saved = BOOT_PARALLEL_MIN
        try:
            BOOT_PARALLEL_MIN = 0                 # spread the replicates even on this small sample
            a = training_pass_em(lv, fields, u_tab, m0, p, "t", np.random.default_rng(5), 6)
            b = training_pass_em_parallel(lv, fields, u_tab, m0, p, "t", np.random.default_rng(5), 6)
        finally:
            BOOT_PARALLEL_MIN = saved
        self.assertEqual(a["lambda"], b["lambda"])
        for f in fields:
            np.testing.assert_array_equal(a["m"][f], b["m"][f])
            for x, y in zip(a["boot"], b["boot"]):
                np.testing.assert_array_equal(x["m"][f], y["m"][f])

    def test_fast_evidence_equals_the_core(self):
        run = _small_run()
        for part in ("person", "business"):
            S, L = run["scored"]["link"][part][0], run["scored"]["link"][part][1]
            if not len(S):
                continue
            ent, W = run["scored"]["entities"][part], run["pr"].W[part]
            pid = ent.ids[S["e"].to_numpy()] + "~" + W["party_id"].to_numpy()[S["r"].to_numpy()]
            w = run["models"]["link"].weights[part]
            vals = {"name": np.full(len(S), "x", dtype=object)}
            a = evidence_rows(pid, L, S, part, w, vals, vals)
            b = evidence_rows_fast(pid, L, S, part, w, vals, vals)
            pd.testing.assert_frame_equal(a, b)

    def test_budget_cuts_cover_everything_in_order(self):
        cuts = budget_cuts([5, 1, 1, 9, 2, 2, 0], 6)
        self.assertEqual(cuts[0][0], 0)
        self.assertEqual(cuts[-1][1], 7)
        self.assertTrue(all(lo < hi for lo, hi in cuts))
        self.assertTrue(all(a[1] == b[0] for a, b in zip(cuts, cuts[1:])))

    def test_candidate_parts_share_one_schema(self):
        run = _small_run()
        parts = run["pairs"].store.parts("candidates")
        self.assertGreater(len(parts), 0)
        self.assertEqual(len({tuple(pd.read_parquet(p).columns) for p in parts}), 1)

    def test_workers_keep_order_and_report_failures(self):
        if not _fork_available():
            self.skipTest("needs fork (Linux): on this platform the tasks run in one process")
        par = Parallel(3)
        self.assertEqual(par.map(_square, range(25)), [i * i for i in range(25)])
        with self.assertRaises(RuntimeError):
            par.map(_fails, [1, 2, 3])

    def test_a_stopped_worker_is_replaced_and_its_task_run_again(self):
        if not _fork_available():
            self.skipTest("needs fork (Linux): on this platform the tasks run in one process")
        global WORKER_RECYCLE_TASKS
        flag = Path(tempfile.mkdtemp(prefix="rl_selftest_kill_", dir=globals().get("SELFTEST_TMP"))) / "stopped"
        saved = WORKER_RECYCLE_TASKS
        try:
            WORKER_RECYCLE_TASKS = 2              # workers are also recycled every other task
            got = Parallel(3).map(_dies_once, range(20), {"flag": str(flag)})
        finally:
            WORKER_RECYCLE_TASKS = saved
        self.assertTrue(flag.exists())
        self.assertEqual(got, [i * i for i in range(20)])

    def test_auto_worker_count_stays_within_cores_and_cap(self):
        n = auto_worker_count(8, 250_000)
        self.assertGreaterEqual(n, 1)
        self.assertLessEqual(n, min(8, os.cpu_count() or 1))
        self.assertTrue(Parallel(0).auto and not Parallel(4).auto)

    def test_spark_run_equals_one_process(self):
        global PAR
        if type(PAR).__name__ != "SparkParallel":
            self.skipTest("runs where the executor pool is a Spark cluster (the Spark notebook)")
        cfg, maps, ref = _env()
        cfg2 = dataclasses.replace(cfg, pairs_per_task=1_500, bootstrap_reps=8)
        _small_run()
        X, W, truth, edges = _T["inputs"]
        spark_par = PAR
        try:
            PAR = Parallel(1)
            one = run_pipeline(X, W, maps, ref, cfg2, store=_temp_store("one"))
        finally:
            PAR = spark_par
        many = run_pipeline(X, W, maps, ref, cfg2, store=_temp_store("spark"))
        pd.testing.assert_frame_equal(one["pairs"].read(), many["pairs"].read())
        pd.testing.assert_frame_equal(one["entities"], many["entities"])
        ev = lambda r: r["evidence"].read().sort_values(["pair_id", "field"]).reset_index(drop=True)
        pd.testing.assert_frame_equal(ev(one), ev(many))
        for part in ("person", "business"):
            pd.testing.assert_frame_equal(one["models"]["link"].weights[part], many["models"]["link"].weights[part])

    def test_parallel_run_equals_one_process(self):
        if not _fork_available():
            self.skipTest("needs fork (Linux): on this platform the tasks run in one process")
        global PAR
        cfg, maps, ref = _env()
        cfg2 = dataclasses.replace(cfg, pairs_per_task=1_500, bootstrap_reps=8)
        _small_run()
        X, W, truth, edges = _T["inputs"]
        saved = PAR
        try:
            PAR = Parallel(1)
            one = run_pipeline(X, W, maps, ref, cfg2, store=_temp_store("one"))
            PAR = Parallel(3)
            many = run_pipeline(X, W, maps, ref, cfg2, store=_temp_store("many"))
        finally:
            PAR = saved
        pd.testing.assert_frame_equal(one["pairs"].read(), many["pairs"].read())
        pd.testing.assert_frame_equal(one["entities"], many["entities"])
        ev = lambda r: r["evidence"].read().sort_values(["pair_id", "field"]).reset_index(drop=True)
        pd.testing.assert_frame_equal(ev(one), ev(many))
        for part in ("person", "business"):
            pd.testing.assert_frame_equal(one["models"]["link"].weights[part], many["models"]["link"].weights[part])


CORE_SHA256 = "43aa8da763beb6be47d67a41250171376c884b91ac95599d298da639f3e9cf73"   # v1.1 (re-recorded: comments only; no matching logic changed)


NL, CR = chr(10), chr(13)


def notebook_core_hash(nb_path):
    """sha256 over the code cells between the 'Matching core v' heading and its end marker,
    exactly as recorded when the notebook was assembled."""
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    inside, parts = False, []
    for c in nb["cells"]:
        src = "".join(c["source"])
        if c["cell_type"] == "markdown" and src.startswith("## Matching core v"):
            inside = True
            continue
        if c["cell_type"] == "markdown" and src.strip().startswith("*End of the matching core.*"):
            break
        if inside and c["cell_type"] == "code":
            parts.append(NL.join(l.rstrip() for l in src.replace(CR + NL, NL).split(NL)))
    return hashlib.sha256((NL + "# ---- cell ----" + NL).join(parts).encode("utf-8")).hexdigest()


class TestCoreHash(unittest.TestCase):
    def test_core_section_matches_recorded_hash(self):
        nbp = _env()[0].paths["notebook"]
        if not nbp.exists() or CORE_SHA256.startswith("<"):
            self.skipTest("runs inside the assembled notebook")
        self.assertEqual(notebook_core_hash(nbp), CORE_SHA256,
                         f"the matching core v{CORE_VERSION} changed: bump the version and re-record the hash")


def run_selftests(verbosity=1):
    suite = unittest.TestSuite()
    loader = unittest.TestLoader()
    for cls in (TestNormalizers, TestBreakdown, TestCategory, TestComparisons, TestCandidates, TestScoring,
                TestParameters, TestOutput, TestInputs, TestClosedFormU, TestEMBootstrap,
                TestRicherComparisons, TestAddressStandardizer, TestExtractedDedup, TestClusters, TestBaseline,
                TestSelfChecks, TestTableIO, TestParallel, TestCoreHash):
        suite.addTests(loader.loadTestsFromTestCase(cls))
    stream = io.StringIO()
    res = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    return res, stream.getvalue()

In [ ]:
if RUN_SELF_TESTS:
    SELFTEST, SELFTEST_TEXT = run_selftests(verbosity=1)
    print(SELFTEST_TEXT[-3000:])
    print(f"self-tests: {SELFTEST.testsRun} run, {len(SELFTEST.failures)} failures, {len(SELFTEST.errors)} errors, "
          f"{len(SELFTEST.skipped)} skipped; matching core v{CORE_VERSION} sha256 {CORE_SHA256}")
    assert SELFTEST.wasSuccessful(), "self-tests failed"
else:
    SELFTEST = None


## Acceptance

Each criterion measured on this run where the dataset allows it: the edge cases and blocking
recall on the built-in `synthetic` set; nothing here changes a result, it only reports.


In [ ]:
def acceptance_table(cfg, diag, sw, pairs, params, selftest, selfcheck=None):
    rows = []
    add = lambda c, target, value, met: rows.append({"criterion": c, "target": target, "value": value,
                                                    "met": "yes" if met else ("n/a" if met is None else "NO")})
    add("self-tests pass", "all", f"{selftest.testsRun} run, {len(selftest.failures) + len(selftest.errors)} failing",
        selftest.wasSuccessful())
    total = sw.rows[-1]["elapsed"] if sw.rows else float("nan")
    gen = next((r["elapsed"] for r in sw.rows if r["step"] == "inputs ready"), 0.0)
    total = total - gen          # making the test inputs is not part of a run
    add("input generation / export (not counted)", "-", f"{gen:.0f} s", None)
    truth = {t["part"]: t for t in diag["truth"] if t.get("true_pairs")}
    tp = sum(t["true_pairs"] for t in truth.values())
    prop = sum(t["proposed"] for t in truth.values())
    kept = sum(t["kept"] for t in truth.values())
    rec = prop / tp if tp else float("nan")
    if cfg.dataset == "synthetic":
        # B1 spends compute on quality (bootstrapped EM, several training passes, and, with
        # SELF_CHECK on, the whole pipeline rerun per held-out identifier and for the noise
        # test): pipeline time is reported, not gated, unless self-checks are off.
        note = "" if cfg.self_check else " (self-checks off)"
        add(f"pipeline time (informational; B1 prioritizes quality over time){note}", "-",
            f"{total:.0f} s pipeline", None)
        add("blocking recall on true synthetic pairs", ">= 0.99", f"{rec:.4f} ({prop}/{tp})", rec >= 0.99)
        from_edges = check_edge_cases(PR, PAIRS.read(), ROWS, SCORED, edge_cases(), EVIDENCE)
        add("every hand-written case: expected basis, veto, top rank", "all",
            f"{int(from_edges['ok'].sum())}/{len(from_edges)}", bool(from_edges["ok"].all()))
    if tp:
        add("every true pair proposed is visible (kept), incl. name-only", "kept = proposed", f"{kept}/{prop}", kept == prop)
    W_ = pd.concat([params["weights"]["person"], params["weights"]["business"]])
    add("every parameter in the manifest with source and count", "all m/u rows",
        f"{len(W_)} level rows, {int((W_['m_source'] == '').sum())} without m source",
        bool((W_["m_source"] != "").all() and (W_["u_source"].fillna("") != "").all()))
    add("scored pairs", "-", f"{sum(diag['scored_pairs'].values()):,}", None)
    for part, dd in diag.get("dedup", {}).items():
        add(f"extracted-side deduplication [{part}]", "-",
            f"{dd['extracted_parties']:,} parties -> {dd['entities']:,} entities "
            f"({dd['pooled_entities']:,} pooled, {dd['refused_links']} refused)", None)
    if selfcheck is not None and len(selfcheck.get("funnel", pd.DataFrame())):
        f = selfcheck["funnel"]
        b1 = f[f["method"] == "B1"]
        if len(b1):
            lost = int(b1["lost_at_candidates"].sum())
            tot = int(b1["true_pairs"].sum())
            add("self-check: true pairs the candidate stage lost", "as low as feasible",
                f"{lost}/{tot} ({lost / tot:.4f})" if tot else "n/a (no truth)", None)
        bd = selfcheck.get("bands", pd.DataFrame())
        b1b = bd[(bd["method"] == "B1") & (bd["threshold"] == 0.5)] if len(bd) else bd
        if len(b1b):
            add("self-check: B1 precision at p >= 0.5 (held-out identifier + noise tests)", "-",
                "; ".join(f"{r['test']} {r['precision']:.4f}" for _, r in b1b.iterrows()), None)
    return pd.DataFrame(rows)


ACCEPTANCE = acceptance_table(CFG, DIAG, SW, PAIRS, PARAMS, SELFTEST, SELFCHECK) if SELFTEST is not None else pd.DataFrame()
print(ACCEPTANCE.to_string(index=False))
(PATHS["out"] / "acceptance.json").write_text(ACCEPTANCE.to_json(orient="records", indent=1), encoding="utf-8")

In [ ]:
# The chunk files are in Delta now: free the volume (set KEEP_CHUNK_FILES = True to keep them).
KEEP_CHUNK_FILES = False
if not KEEP_CHUNK_FILES:
    RUN["store"].drop_work()
    for t_ in RunStore.PUBLISHED:
        RUN["store"].drop(t_)


## Look up an entity or claim

Give it a `record_id` (one extracted party) or a `claim_id`, and it prints what was produced for
it -- the entity it rolled up into, its best watchlist match (if any) with the full
field-by-field evidence behind that match, every other candidate considered, and any veto. It
reads the entity's rows from the candidates and evidence Delta tables (Z-ordered by entity and
pair, so a lookup touches a few files). `SUGGESTED_LOOKUPS` picks a handful of ids actually
present in this run, one per match type.

**Where Splink would do better.** Splink's `waterfall_chart` and `cluster_studio_dashboard`
give an interactive, visual version of this per record, in a browser; this is a plain print.


In [ ]:
from pyspark.sql import functions as F

CANDIDATES_TABLE, EVIDENCE_TABLE = TABLES("candidates"), TABLES("evidence")


def entity_pairs(entity_id):
    """The candidate pairs of one entity, from the candidates Delta table."""
    P = spark.read.table(CANDIDATES_TABLE).filter(F.col("entity_id") == entity_id).toPandas()
    return P, None


def _pair_evidence(pair_id, name=None):
    """Full evidence rows for one pair, strongest field first (same order as its decision card)."""
    ev = spark.read.table(EVIDENCE_TABLE).filter(F.col("pair_id") == pair_id).toPandas()
    if not len(ev):
        return ev
    ev = ev.sort_values(["pair_id", "field"], kind="stable")
    return ev.assign(_a=-ev["bits"].abs()).sort_values("_a", kind="stable").drop(columns="_a")


def _print_pair(P, name, watchlist_record_id, label):
    """One candidate's decision card plus its full per-field evidence table."""
    pr_ = P[P["watchlist_record_id"] == watchlist_record_id]
    if not len(pr_):
        return
    row = pr_.iloc[0]
    veto_note = f"  VETO: {row['veto']}" if row["veto"] else ""
    print(f"  {label}: {watchlist_record_id} \"{row['watchlist_name']}\"  p={row['p']:.4g}  basis={row['basis']}{veto_note}")
    ev = _pair_evidence(row["pair_id"], name)
    if len(ev):
        card = decision_cards(ev, [row["pair_id"]])
        if len(card):
            print(f"    decision card: {card.iloc[0]}")
        cols = ["field", "value_extracted", "value_watchlist", "level", "m", "u", "bits", "veto"]
        print("    " + ev[cols].to_string(index=False).replace("\n", "\n    "))


def lookup_record(record_id):
    """Print everything produced for one extracted party: its entity, its best match (if
    any) with full evidence, every other candidate considered, and vetoes. Returns False if
    record_id is not a known record_id, so `lookup` can fall back to a claim_id."""
    row = ROWS[ROWS["record_id"] == record_id]
    if not len(row):
        return False
    r = row.iloc[0]
    print(f"=== extracted party {record_id} ({r['part']}) ===")
    print(f"name: {r['name']}   claim: {r['claim_id']}   note: {r['note_id']}   "
          f"category used: {r['category_used']}")
    ent_rows = ENTITIES[ENTITIES["entity_id"] == r["entity_id"]]
    ent = ent_rows.iloc[0] if len(ent_rows) else None
    print(f"entity: {r['entity_id']}  ({r['entity_records']} extracted record(s) pooled into it)")
    if ent is not None and r["entity_records"] > 1:
        print(f"  other records in this entity: {ent['record_ids']}")
    print(f"status: {r['status']}")
    P, name = entity_pairs(r["entity_id"])
    if r["status"] == "scored" and r["best_watchlist_record_id"]:
        _print_pair(P, name, r["best_watchlist_record_id"], label="best match")
        if ent is not None:
            if ent["conflicting_candidates"]:
                print(f"  conflicting candidates (a different, inconsistent cluster): {ent['conflicting_candidates']}")
            if ent["other_candidates"]:
                print(f"  other candidates considered: {ent['other_candidates']}")
    elif r["status"] == "only vetoed candidates":
        for w in P.loc[P["veto"] != "", "watchlist_record_id"]:
            _print_pair(P, name, w, label="vetoed candidate")
    else:
        print("  no watchlist candidate was proposed for this party.")
    return True


def lookup_claim(claim_id):
    """Print a claim's rollup and every party in it. Returns False if claim_id is unknown."""
    row = CLAIMS[CLAIMS["claim_id"] == claim_id]
    if not len(row):
        return False
    r = row.iloc[0]
    print(f"=== claim {claim_id} ===")
    print(f"rows: {int(r['rows'])}   parties: {int(r['parties'])}   entities: {int(r['entities'])}   "
          f"max p: {r['max_p']:.4g} (party {r['max_p_party']}, \"{r['max_p_name']}\", basis={r['max_p_basis']})")
    members = ROWS[ROWS["claim_id"] == claim_id]
    print("parties in this claim:")
    print(members[["record_id", "part", "name", "status", "basis", "p"]].to_string(index=False))
    print("(look up one of the record_ids above for its full decision card)")
    return True


def lookup(query):
    """Look up an exact record_id or claim_id and print what the pipeline produced for it."""
    q = str(query).strip()
    if not q:
        print("enter a record_id or claim_id -- see SUGGESTED_LOOKUPS above.")
        return
    if lookup_record(q):
        return
    if lookup_claim(q):
        return
    print(f'no record_id or claim_id matches "{q}". Try one of SUGGESTED_LOOKUPS above.')


In [ ]:
def suggest_lookups():
    """A handful of record_ids and claim_ids worth looking up, one per match type actually
    present in this run (not hand-picked, not hardcoded) -- so there is always something
    concrete to try. Any category absent from this run (e.g. no vetoes occurred) is skipped."""
    picks = []

    def pick(label, df, sort_col=None):
        if not len(df):
            return
        d = df.sort_values(sort_col, ascending=False, kind="stable") if sort_col else df
        key = "record_id" if "record_id" in d.columns else "claim_id"
        picks.append({"what it shows": label, "try this id": d.iloc[0][key]})

    scored = ROWS[ROWS["status"] == "scored"]
    pick("high-confidence identifier match", scored[scored["basis"] == "identifier"], "p")
    pick("name-only match (visible, never auto-confirmed)",
         scored[scored["basis"].isin(["name_only", "contextual"]) & (scored["p"] >= 0.5)], "p")
    pick("low-confidence match", scored[(scored["p"] > 0.05) & (scored["p"] < 0.5)], "p")
    pick("conflicting identifier (vetoed candidate)", ROWS[ROWS["status"] == "only vetoed candidates"])
    pick("no candidate found on the watchlist", ROWS[ROWS["status"] == "no candidate proposed"])
    pick("several extracted records pooled into one entity", ROWS[ROWS["entity_records"] > 1], "entity_records")
    if len(CLAIMS):
        pick("a claim with a strong match", CLAIMS[CLAIMS["max_p"] >= 0.9], "max_p")
        pick("a claim with no strong match", CLAIMS[CLAIMS["max_p"].fillna(0) < 0.5])
    return pd.DataFrame(picks, columns=["what it shows", "try this id"])


SUGGESTED_LOOKUPS = suggest_lookups()
print("ids worth looking up, present in this run:")
print(SUGGESTED_LOOKUPS.to_string(index=False) if len(SUGGESTED_LOOKUPS) else "(none of the usual categories occurred in this run)")


In [ ]:
# ---- Look something up: edit this and rerun the cell ------------------------------------------
LOOKUP_QUERY = SUGGESTED_LOOKUPS["try this id"].iloc[0] if len(SUGGESTED_LOOKUPS) else ""   # <- a record_id or claim_id
lookup(LOOKUP_QUERY)
